# Definitions

Model, losses, training, calibration, and baselines. No experiments run in this part.

### Imports and Device

Imports, seeding, and device selection (CUDA/MPS/CPU).

In [ ]:
from __future__ import annotations

# 1. Standard Library Imports
import importlib
import math
import os
import random
import sys
import time
import warnings
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Callable, Optional, Tuple

warnings.filterwarnings("ignore")

# 2. Data Manipulation & Scientific Computing
import joblib
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from scipy.stats import pearsonr
from scipy.stats import mannwhitneyu, pearsonr, ttest_rel, norm as scipy_norm
from scipy.signal import find_peaks

import matplotlib
matplotlib.use("Agg")
import matplotlib as mpl
import matplotlib.patheffects as path_effects
import matplotlib.patches as mpatches
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import gridspec
from matplotlib.font_manager import FontProperties
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import seaborn as sns

# 4. Machine Learning: Scikit-Learn
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression, Ridge, LinearRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# 5. Deep Learning: PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

In [ ]:
STAGE_LABELS: dict = {1: "Normal", 2: "Elevated", 3: "HTN-1", 4: "HTN-2"}
STAGE_ORDER:  list = ["Normal", "Elevated", "HTN-1", "HTN-2"]

STAGE_NAMES = {1: "Normal", 2: "Elevated", 3: "HTN-1", 4: "HTN-2"}
STAGE_COLORS = {1: "#2196F3", 2: "#4CAF50", 3: "#FF9800", 4: "#F44336"}
STAGE_SBP_RANGE_LABEL = {1: "< 120", 2: "120 - 129", 3: "130 - 139", 4: ">= 140"}
 
 
def assign_stages(sbp_arr, dbp_arr=None):
    """
    ACC/AHA 2017 BP staging.
 
    If dbp_arr is None, falls back to SBP-only boundaries (same numeric
    cutoffs, just without the DBP OR-condition) — used only by the
    histogram background bands in build_figure_0, which are SBP-axis
    only by construction and have no DBP to plot against. Any print/
    summary path MUST pass dbp_arr so the OR-condition is applied.
    """
    sbp = np.asarray(sbp_arr, float)
    if dbp_arr is None:
        return np.where(sbp >= 140, 4,
               np.where(sbp >= 130, 3,
               np.where(sbp >= 120, 2, 1))).astype(int)
    dbp = np.asarray(dbp_arr, float)
    return np.where((sbp >= 140) | (dbp >= 90), 4,
           np.where((sbp >= 130) | (dbp >= 80), 3,
           np.where( sbp >= 120,                2,
                                                 1))).astype(int)

def make_stages(*args, **kwargs):
    raise RuntimeError(
        "make_stages() is DEPRECATED (JNC-7). Use assign_stages() from mcnhm_metrics.")


# ── 2. COHEN\'S D  (Fix #1, #8) ───────────────────────────────────────────────

def cohens_d(a, b) -> float:
    """Signed pooled Cohen\'s d.  Positive => mean(a) > mean(b)."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    n1, n2 = len(a), len(b)
    if n1 < 2 or n2 < 2: return float("nan")
    pooled = np.sqrt(((n1-1)*np.var(a,ddof=1)+(n2-1)*np.var(b,ddof=1))/(n1+n2-2))
    return float((np.mean(a)-np.mean(b))/(pooled+1e-12))


# ── 3. ICC  ───────────────────────────────────────────────────────────────────

def _between_within_var(groups_list):
    """(Vb, Vw) = (between-subject variance of per-subject means, mean of within-
    subject variances), sample variance (ddof=1) throughout.  SINGLE SOURCE OF
    TRUTH for BOTH the ICC(1) ratio (compute_icc) and the Vb/Vw dispersion metric
    (_vbvw / _vbvw_from_groups).  Returns (nan, nan) if <2 subjects have >=2 beats."""
    means, wvars = [], []
    for vs in groups_list:
        if len(vs) >= 2:
            means.append(vs.mean()); wvars.append(vs.var(ddof=1))
    if len(means) < 2:
        return float("nan"), float("nan")
    return float(np.var(means, ddof=1)), float(np.mean(wvars))

def compute_icc(values, subj_arr) -> Tuple[float, float]:
    """Approx ICC(1): vb/(vb+vw).  Returns (icc, vb_vw).  Uses _between_within_var
    (SSoT, ddof=1/1e-9) -- unified with _vbvw*; was ddof=0/1e-6 before consolidation."""
    vb, vw = _between_within_var([values[subj_arr == s] for s in np.unique(subj_arr)])
    if vb != vb:   # nan -> <2 usable subjects
        return float("nan"), float("nan")
    return vb / (vb + vw + 1e-9), vb / (vw + 1e-9)


# ── 4. BOOTSTRAP CIs  (Fix #11) ──────────────────────────────────────────────

def _block_resample(subj_arr, n_boot, rng):
    """Subject-block bootstrap: yields (idx, s_new) for each replicate."""
    subjs = np.unique(subj_arr)
    for _ in range(n_boot):
        chosen = rng.choice(subjs, len(subjs), replace=True)
 # Fix: iterate with pseudo-ID so duplicate subjects get UNIQUE IDs
        idx, s_new = [], []
        for pseudo_id, s in enumerate(chosen):
            mask = np.where(subj_arr == s)[0]
            idx.extend(mask.tolist())
            s_new.extend([pseudo_id] * len(mask))
        idx, s_new = np.array(idx), np.array(s_new, dtype=int)
        yield idx, s_new

def bootstrap_icc_ci(values, subj_arr, n_boot=1000, seed=42):
    """ICC + 95% CI via true subject-block bootstrap.  Fix #11."""
    rng   = np.random.default_rng(seed)
    point, _ = compute_icc(values, subj_arr)
    boot  = [v for idx,s in _block_resample(subj_arr,n_boot,rng)
             if not np.isnan(v:=compute_icc(values[idx],s)[0])]
    if not boot: return float(point),float("nan"),float("nan")
    return float(point),float(np.percentile(boot,2.5)),float(np.percentile(boot,97.5))

def _vbvw_from_groups(groups_list):
    """Vb/Vw dispersion ratio (between-subject var / within-subject var) from
    pre-grouped per-subject arrays.  Delegates to _between_within_var (SSoT): every
    Vb/Vw point estimate (cells 3 / 29 / 51) AND the ICC ratio (compute_icc) now
    share that one estimator (ddof=1 / 1e-9)."""
    vb, vw = _between_within_var(groups_list)
    return vb / (vw + 1e-9) if vb == vb else float("nan")

def _vbvw(values, subj_arr):
    """Vb/Vw from flat (values, subj_arr); groups then delegates to _vbvw_from_groups."""
    return _vbvw_from_groups([values[subj_arr == s] for s in np.unique(subj_arr)])

def _bhs_grade(p5, p10, p15):
    """British Hypertension Society grade -- SINGLE SOURCE OF TRUTH.
    Standard thresholds on (%<=5, %<=10, %<=15) mmHg:
    A>=(60/85/95), B>=(50/75/90), C>=(40/65/85); else D."""
    if   p5 >= 60 and p10 >= 85 and p15 >= 95: return "A"
    elif p5 >= 50 and p10 >= 75 and p15 >= 90: return "B"
    elif p5 >= 40 and p10 >= 65 and p15 >= 85: return "C"
    return "D"

def bootstrap_vbvw_ci(values, subj_arr, n_boot=1000, seed=42):
    """Vb/Vw + 95% CI via true subject-block bootstrap.  Fix #11."""
    rng  = np.random.default_rng(seed)
    pt   = _vbvw(values,subj_arr)
    boot = [v for idx,s in _block_resample(subj_arr,n_boot,rng)
            if not np.isnan(v:=_vbvw(values[idx],s))]
    if not boot: return float(pt),float("nan"),float("nan")
    return float(pt),float(np.percentile(boot,2.5)),float(np.percentile(boot,97.5))

def bootstrap_cohens_d_ci(g1, g2, n_boot=1000, seed=42):
    """Cohen\'s d + 95% CI.  Fix #11."""
    rng  = np.random.default_rng(seed)
    pt   = cohens_d(g1,g2)
    boot = [cohens_d(rng.choice(g1,len(g1),True),rng.choice(g2,len(g2),True))
            for _ in range(n_boot)]
    return float(pt),float(np.percentile(boot,2.5)),float(np.percentile(boot,97.5))


# ── 5. LATENT CORRELATION  (Fix #2) ──────────────────────────────────────────

# ── 6. SUBJECT-LEVEL MAE  (Fix #8) ───────────────────────────────────────────

def subj_mae(pred, true, subj_arr, eval_mask=None) -> Tuple[float,float]:
    """MAE of subject means: mean_s|mean_pred_s - mean_true_s|.  Fix #8."""
    if eval_mask is None: eval_mask = np.ones(len(pred),bool)
    rs,rd=[],[]
    for s in np.unique(subj_arr[eval_mask]):
        ms=(subj_arr==s)&eval_mask
        if not ms.any(): continue
        rs.append(abs(pred[ms,0].mean()-true[ms,0].mean()))
        rd.append(abs(pred[ms,1].mean()-true[ms,1].mean()))
    if not rs: return float("nan"),float("nan")
    return float(np.mean(rs)),float(np.mean(rd))


# ── 7. BLAND-ALTMAN LoA  (Fix #14) ───────────────────────────────────────────

# ── 8. SUBJECT MEAN C_SLOW  (Fix #1) ─────────────────────────────────────────

def subj_mean_cs(cs, sbp, subj_arr, dbp=None) -> Tuple[np.ndarray,np.ndarray]:
    """Per-subject mean c_slow + ACC/AHA 2017 stage.  Fix #1."""
    if dbp is None: dbp=np.zeros_like(sbp)
    stages=assign_stages(sbp,dbp)
    sc,st={},{}
    for i,s in enumerate(subj_arr):
        sc.setdefault(s,[]).append(cs[i]); st.setdefault(s,[]).append(stages[i])
    sl=sorted(sc)
    return (np.array([np.mean(sc[s]) for s in sl]),
            np.array([int(np.round(np.mean(st[s]))) for s in sl]))


# ── 9. MAP FROM CUFF  (Fix #10) ──────────────────────────────────────────────

# ══════════════════════════════════════════════════════════════════════════════
# IMPORTS + DEVICE
# ══════════════════════════════════════════════════════════════════════════════
def seed_everything(seed: int = 42) -> None:
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
        
seed_everything(42)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps"  if torch.backends.mps.is_available() else "cpu"
)
    

### Configuration

Hyperparameter dataclasses for the architecture, Windkessel model, calibration, training, loss weights, and physiological limits.

In [ ]:
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

DATA_DIR     = "outputs/phase_1/data/pre_split"
OUT_DIR      = "outputs/phase_2_v192"
CKPT_DIR     = os.path.join(OUT_DIR, "checkpoints")
CKPT_DIR_GRL = os.path.join(OUT_DIR, "checkpoints_grl")
EXPORT_DIR   = os.path.join(OUT_DIR, "exports")
FIG_DIR      = os.path.join(OUT_DIR, "figures")
HIST_DIR     = os.path.join(OUT_DIR, "hist")

for _d in [OUT_DIR, CKPT_DIR, CKPT_DIR_GRL, EXPORT_DIR, FIG_DIR, HIST_DIR]:
    os.makedirs(_d, exist_ok=True)

@dataclass
class _ArchCfg:
    max_subjects:    int   = 0
    embed_size:      int   = 15000
    window_stride:   int   = 3
    n_beats:         int   = 10
    seq_len:         int   = 128
    latent_dim:      int   = 64
    batch:           int   = 256
    head_feat_dim:   int   = 64
    bp_stat_dim:     int   = 16
    c_slow_head_dim: int   = 32

@dataclass
class _WKCfg:
    steps:           int   = 128
    dt:              float = 1.0 / 125 
    fs:              int   = 125            
    c_max:           float = 8.0
    r_max:           float = 5.0
    c_min:           float = 0.5
    c_sigmoid_temp:  float = 2.0
    c_target:        float = 1.5
    r_target:        float = 1.0
    tau_min:         float = 0.30
    tau_max:         float = 4.00
    target_r_std:    float = 0.28
    target_c_std:    float = 0.80
    @property
    def tau_target(self) -> float: return self.r_target * self.c_target

@dataclass
class _CalibCfg:
    ref_beats:   int   = 3
    eval_min:    int   = 1
    min_beats:   int   = 3
    ols_gap:     int   = 1
    ridge_alpha: float = 10.0
    sbp_mae_target:  float = 7.6
    icc_cslow_thresh: float = 0.80
    cohens_d_thresh:  float = 1.0

@dataclass
class _TrainCfg:
    epochs_p1:    int   = 80
    epochs_p2:    int   = 80
    force_retrain: bool = False
    enc_lr:        float = 5e-5
    head_lr:       float = 5e-4
    wd:            float = 3e-4
    wk_lr_fixed:   float = 7.5e-4 
    awakening_ep:       int   = 20
    awakening_warmup:   int   = 8
    enc_lr_target:      float = 1e-4
    head_lr_target:     float = 3e-4
    detach_ratio:       float = 0.5
    hybrid_alpha_start: float = 0.80
    hybrid_alpha_end:   float = 0.10
    ste_alpha_start:    float = 0.10
    ste_alpha_end:      float = 0.50
    ste_alpha_ramp_ep:  int   = 20
    soft_sbp_temp:      float = 25.0
    cont_temperature:   float = 0.20
    focal_gamma:        float = 2.0
    focal_alpha:        float = 0.75
    wk_iqr_low:         float = 5.0
    wk_iqr_high:        float = 95.0
    wk_freeze_ep:       int   = 5
    r_var_detach_ratio: float = 0.40
    freeze_direct_ep_start:  int   = 35
    freeze_direct_every:     int   = 15
    freeze_direct_duration:  int   = 4
    gate_barrier_thresh:     float = 0.50
    wk_min_contrib_frac:     float = 0.35
    gate_floor_start:        float = 0.30
    gate_floor_end_ep:       int   = 30
    gate_smooth_start_ep:    int   = 50
    gate_ema_decay:          float = 0.90
    wk_util_start_ep:        int   = 10
    wk_gate_floor_start:     float = 0.35
    wk_gate_floor_end:       float = 0.28
    wk_gate_floor_end_ep:    int   = 100
    qr_indep_start_ep:       int   = 15
    wk_shape_cons_start_ep:  int   = 20
    r_iqr_floor:             float = 0.20
    r_anc_target_mean:       float = 0.80
    wk_bp_delay:       int   = 30
    wk_bp_peak:        float = 0.18
    wk_bp_ramp_ep:     int   = 10
    wk_bp_decay_ep:    int   = 40
    wk_bp_bell_decay:  float = 12.0
    wk_bp_floor:       float = 0.06
    grad_bal_clamp_c:  tuple = (0.70, 1.50)
    grad_bal_clamp_d:  tuple = (0.50, 2.00)
    grad_bal_late_max: float = 1.50
    grad_bal_late_ep:  int   = 40
    c_sat_thresh:   float = 7.5
    c_sat_target:   float = 0.20
    c_prior_mean:   float = 6.00
    c_prior_scale:  float = 1.50
    c_sat_start_ep: int   = 5
    hsic_start_ep:       int   = 10
    hsic_ramp_end_ep:    int   = 40
    lambda_hsic_start:   float = 0.20
    lambda_hsic_max:     float = 0.55
    anticollapse_thresh: float = 0.40
    c_proto_ema_decay:   float = 0.90
    c_proto_start_ep:    int   = 20
    target_c_slow_std:   float = 0.70
    c_slow_prior_mean:   float = 6.0
    c_slow_prior_scale:  float = 1.5
    subj_id_acc_n_cap: int   = 50
    subj_id_acc_cv:    int   = 3
    wkode_warmup_end_ep:  int   = 15
    wkode_plateau_end_ep: int   = 30
    wkode_peak_factor:    float = 3.0
    c_proto_warmup_ep:    int   = 10
    cosine_T0:       int   = 30
    cosine_eta_min:  float = 1e-5
    grad_clip_norm:  float = 1.0
    grad_clip_norm_grl: float = 1.0
    phase_a_end_ep:  int   = 20
    phase_b_end_ep:  int   = 30
    phase_c_end_ep:  int   = 50
    val_frac:    float = 0.15
    test_frac:   float = 0.15
    split_seed:  int   = 42
    ckpt_corr_weight:  float = 1.0
    ckpt_corr_start_ep: int  = 0

@dataclass
class _LossWeights:
    sbp:         float = 2.8
    wave:        float = 1.00
    deriv:       float = 0.20
    peak:        float = 0.50
    bpaux:       float = 0.80
    wavebp:      float = 6.00
    ens_mmhg:    float = 1.50
    wkode:       float = 0.01
    wkprior:     float = 0.002
    wksprd_r:    float = 3.00
    wkcorr:      float = 0.10
    qanchor:     float = 0.70
    qtemp:       float = 0.15
    htn:         float = 0.50
    aux:         float = 0.15
    cont:        float = 0.60
    dir_mmhg:    float = 1.00
    dir_rank:    float = 1.00
    dir_ext:     float = 5.00
    gate_ent:    float = 0.15
    gate_barrier: float = 1.50
    wk_util:     float = 0.60
    qr_indep:    float = 0.80
    wk_shape_cons: float = 0.08
    r_physio:    float = 0.50
    r_anc:       float = 0.15
    c_sat:       float = 2.00
    c_prior:     float = 0.30
    c_slow_spread: float = 0.15
    c_slow_prior:  float = 0.05
    c_proto:       float = 0.02
    anticollapse:  float = 2.00
    wkode_shp:   float = 0.20
    wkode_scl:   float = 0.10
    wkode_smo:   float = 0.10
    wkprior_tau: float = 0.10
    wkprior_r:   float = 0.02
    wkprior_c:   float = 0.01
    ens_bp_weight: float = 0.70
    ens_wk_weight: float = 0.30
    htn_main_weight: float = 0.60
    htn_aux_weight:  float = 0.40
    adv: float = 0.30

@dataclass
class _PhysioLimits:
    sbp_norm_max:       float = 220.0
    dbp_norm_max:       float = 120.0
    bp_scale_mmhg:      float = 50.0
    dbp_sbp_ratio:      float = 0.65
    baseline_scale:     float = 100.0
    loa_z_score:        float = 1.96
    cont_bp_dist_scale: float = 50.0
    cont_pos_temp:      float = 0.05
    cont_neg_temp:      float = 0.10
    rank_sbp_thresh:    float = 5.0
    rank_dbp_thresh:    float = 3.0
    spread_target_ratio: float = 0.75
    spread_floor_ratio:  float = 0.60

ARCH   = _ArchCfg()
WK     = _WKCfg()
CALIB  = _CalibCfg()
TRAIN  = _TrainCfg()
LW     = _LossWeights()
PHYSIO = _PhysioLimits()

HTN_SBP, HTN_DBP = 130, 80

### Data Loader

Loads the pre-split data and builds the DataLoaders.

In [ ]:
# DATA (Pre-split Loader)
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class Split:
    X:    np.ndarray
    Q:    np.ndarray
    Yabp: np.ndarray
    Ybp:  np.ndarray
    dur:  np.ndarray
    htn:  np.ndarray
    subj: np.ndarray
    dom:  np.ndarray
    base: np.ndarray

@dataclass
class DataSplits:
    train: Split
    val:   Split
    test:  Split
    pop_mu:  float = 0.0
    pop_sig: float = 1.0
    sbp_mu:  float = 0.0
    sbp_sig: float = 1.0
    dbp_mu:  float = 0.0
    dbp_sig: float = 1.0
    n_subjects: int = 0

SPLITS: Optional[DataSplits] = None

def _load_data(load_dir=DATA_DIR) -> DataSplits:
    global SPLITS
    print(f"  Loading Pre-split Data from {load_dir} ...", flush=True)

    def _load_single_split(filename):
        path = os.path.join(load_dir, filename)
        d = np.load(path)
        return Split(X=d['X'], Q=d['Q'], Yabp=d['Yabp'], Ybp=d['Ybp'], 
                     dur=d['dur'], htn=d['htn'], subj=d['subj'], 
                     dom=d['dom'], base=d['base'])

    tr_split = _load_single_split("train_split.npz")
    va_split = _load_single_split("val_split.npz")
    te_split = _load_single_split("test_split.npz")
    meta = np.load(os.path.join(load_dir, "split_metadata.npz"))

    print(f"  Windows — train:{len(tr_split.X):,}  val:{len(va_split.X):,}  test:{len(te_split.X):,}", flush=True)

    SPLITS = DataSplits(
        train=tr_split, val=va_split, test=te_split,
        pop_mu=float(meta['pop_mu']), pop_sig=float(meta['pop_sig']),
        sbp_mu=float(meta['sbp_mu']), sbp_sig=float(meta['sbp_sig']),
        dbp_mu=float(meta['dbp_mu']), dbp_sig=float(meta['dbp_sig']),
        n_subjects=int(meta['n_subjects'])
    )
    print(f"  POP: mu={float(meta['pop_mu']):.1f}  sig={float(meta['pop_sig']):.1f}  n_subjects={int(meta['n_subjects'])}", flush=True)
    return SPLITS

def _make_tensor_dataset(split: Split) -> TensorDataset:
    return TensorDataset(
        torch.tensor(split.X), torch.tensor(split.Q), torch.tensor(split.Yabp),
        torch.tensor(split.Ybp), torch.tensor(split.htn),
        torch.tensor(split.subj, dtype=torch.long),
        torch.tensor(split.dur), torch.tensor(split.dom, dtype=torch.long),
        torch.tensor(split.base))

def _to_device(batch): return [t.to(DEVICE) for t in batch]

def make_train_dataloader() -> DataLoader:
    tr = SPLITS.train
    wt = 1.0 / np.bincount(tr.htn.astype(int))[tr.htn.astype(int)]
    return DataLoader(_make_tensor_dataset(tr), batch_size=ARCH.batch,
                      sampler=WeightedRandomSampler(torch.tensor(wt).float(), len(tr.htn), replacement=True),
                      num_workers=0)

def make_val_dataloader()  -> DataLoader:
    return DataLoader(_make_tensor_dataset(SPLITS.val),  batch_size=ARCH.batch, shuffle=False, num_workers=0)

### Model Architecture

DiBP-Net and its components: beat encoder, hemodynamic extractor, Windkessel ODE solver, ABP decoder, and subject prototype.

In [ ]:
# MODEL ARCHITECTURE
# ══════════════════════════════════════════════════════════════════════════════

def _total_epochs() -> int:         return TRAIN.epochs_p1 + TRAIN.epochs_p2
def _ste_alpha(ep: int) -> float:
    t = min(1.0, ep / max(TRAIN.ste_alpha_ramp_ep, 1))
    return TRAIN.ste_alpha_start + (TRAIN.ste_alpha_end - TRAIN.ste_alpha_start) * t
def _curriculum_phase(ep: int) -> str:
    if ep < TRAIN.phase_a_end_ep: return "A"
    if ep < TRAIN.phase_b_end_ep: return "B"
    if ep < TRAIN.phase_c_end_ep: return "C"
    return "D"
def _hybrid_alpha(ep: int) -> float:
    t = min(1.0, ep / max(_total_epochs() - 1, 1))
    return TRAIN.hybrid_alpha_start + (TRAIN.hybrid_alpha_end - TRAIN.hybrid_alpha_start) * t
def _gate_floor(ep: int) -> float:
    if ep >= TRAIN.gate_floor_end_ep: return 0.0
    return TRAIN.gate_floor_start * (1.0 - ep / TRAIN.gate_floor_end_ep)
def _wk_gate_floor(ep: int) -> float:
    t = min(1.0, ep / max(TRAIN.wk_gate_floor_end_ep, 1))
    return TRAIN.wk_gate_floor_start + (TRAIN.wk_gate_floor_end - TRAIN.wk_gate_floor_start) * t
def _lam_wkode(ep: int) -> float:
    if ep < TRAIN.wk_freeze_ep: return 0.0
    warmup_span = max(TRAIN.wkode_warmup_end_ep - TRAIN.wk_freeze_ep, 1)
    if ep < TRAIN.wkode_warmup_end_ep:
        return LW.wkode * TRAIN.wkode_peak_factor * (ep - TRAIN.wk_freeze_ep) / warmup_span
    if ep < TRAIN.wkode_plateau_end_ep:
        return LW.wkode * TRAIN.wkode_peak_factor
    t = min(1.0, (ep - TRAIN.wkode_plateau_end_ep) /
            max(_total_epochs() - TRAIN.wkode_plateau_end_ep, 1))
    return LW.wkode + (LW.wkode * TRAIN.wkode_peak_factor - LW.wkode) * 0.5 * (1.0 + np.cos(np.pi * t))
def _lam_c_proto(ep: int) -> float:
    if ep < TRAIN.c_proto_start_ep: return 0.0
    return LW.c_proto * min(1.0, (ep - TRAIN.c_proto_start_ep) / max(TRAIN.c_proto_warmup_ep, 1))
def _lam_hsic(ep: int) -> float:
    if ep < TRAIN.hsic_start_ep: return 0.0
    t = min(1.0, (ep - TRAIN.hsic_start_ep) / max(TRAIN.hsic_ramp_end_ep - TRAIN.hsic_start_ep, 1))
    return TRAIN.lambda_hsic_start + (TRAIN.lambda_hsic_max - TRAIN.lambda_hsic_start) * t
def _lam_wk_bp(ep: int) -> float:
    if ep < TRAIN.wk_bp_delay: return 0.0
    if ep < TRAIN.wk_bp_delay + TRAIN.wk_bp_ramp_ep:
        return TRAIN.wk_bp_peak * (ep - TRAIN.wk_bp_delay) / max(TRAIN.wk_bp_ramp_ep, 1)
    return max(TRAIN.wk_bp_floor,
               TRAIN.wk_bp_peak * np.exp(-(ep - TRAIN.wk_bp_decay_ep) / TRAIN.wk_bp_bell_decay))
def _grad_bal_clamp(ep: int):
    ph = _curriculum_phase(ep)
    if ph in ("A", "B"): return None
    if ph == "C":        return TRAIN.grad_bal_clamp_c
    return TRAIN.grad_bal_clamp_d
def _should_freeze_direct(ep: int) -> bool:
    if ep < TRAIN.freeze_direct_ep_start: return False
    return (ep - TRAIN.freeze_direct_ep_start) % TRAIN.freeze_direct_every < TRAIN.freeze_direct_duration
def warmup_lr(ep, start, warmup, lr0, lr1):
    if ep < start: return lr0
    if ep < start + warmup: return lr0 + (lr1 - lr0) * (ep - start) / warmup
    return lr1


def soft_max_val(x: torch.Tensor, temp: float = None) -> torch.Tensor:
    temp = temp or TRAIN.soft_sbp_temp
    return (x * torch.softmax(x * temp, dim=-1)).sum(dim=-1)
def soft_min_val(x: torch.Tensor, temp: float = None) -> torch.Tensor:
    temp = temp or TRAIN.soft_sbp_temp
    return (x * torch.softmax(-x * temp, dim=-1)).sum(dim=-1)
def normalize_q(x: torch.Tensor) -> torch.Tensor:
    return x / (x.mean(dim=-1, keepdim=True) + 1e-6)
def _batch_corr(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    x = x - x.mean(); y = y - y.mean()
    return (x * y).mean() / (x.std() * y.std()).clamp(min=1e-6)
def _soft_max_np(x: np.ndarray, temp: float = None) -> np.ndarray:
    temp = temp or TRAIN.soft_sbp_temp
    xs = x - x.max(axis=-1, keepdims=True)
    w = np.exp(xs * temp); w /= w.sum(axis=-1, keepdims=True) + 1e-6
    return (x * w).sum(axis=-1)
def _soft_min_np(x: np.ndarray, temp: float = None) -> np.ndarray:
    temp = temp or TRAIN.soft_sbp_temp
    xs = x - x.min(axis=-1, keepdims=True)
    w = np.exp(-xs * temp); w /= w.sum(axis=-1, keepdims=True) + 1e-6
    return (x * w).sum(axis=-1)

# cohens_d / compute_icc / subj_mean_cs: single source of truth lives in
# Part 0, cell above ("DiBP-Net Single Source of Truth for all metrics") — imported into
# this notebook's global namespace before Part 1 runs, so no local redefinition needed here.


def corr_zm_zh_fn(zm: np.ndarray, zh: np.ndarray) -> float:
    """Canonical per-dim correlation."""
    zm_norm = (zm - zm.mean(axis=0, keepdims=True)) / (zm.std(axis=0, keepdims=True) + 1e-8)
    zh_norm = (zh - zh.mean(axis=0, keepdims=True)) / (zh.std(axis=0, keepdims=True) + 1e-8)
    return float(np.mean(np.mean(zm_norm * zh_norm, axis=0)))

class SelfAttn1D(nn.Module):
    def __init__(self, d, heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        xt = x.permute(0, 2, 1); a, _ = self.attn(xt, xt, xt)
        return self.norm(xt + self.drop(a)).permute(0, 2, 1)

class BeatEncoder(nn.Module):
    def __init__(self, in_ch=2, dropout=0.15):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, 32, 5, padding=2), nn.GELU(), nn.BatchNorm1d(32), nn.MaxPool1d(2),
            nn.Conv1d(32,    64, 5, padding=2), nn.GELU(), nn.BatchNorm1d(64), nn.MaxPool1d(2),
            nn.Conv1d(64,   128, 3, padding=1), nn.GELU(), nn.BatchNorm1d(128))
        self.attn       = SelfAttn1D(128, heads=4)
        self.skip1_proj = nn.Sequential(nn.Conv1d(32, 32, 1), nn.GELU())
        self.skip2_proj = nn.Sequential(nn.Conv1d(64, 64, 1), nn.GELU())
        self.bottleneck = nn.Sequential(
            nn.Linear(128 * 32, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.GELU())
        self.phys_proj = nn.Sequential(nn.Linear(7, 32), nn.GELU(), nn.Linear(32, 32), nn.GELU())
        self.fuse = nn.Sequential(nn.Linear(128 + 32, 128), nn.GELU())

    @staticmethod
    def _morphology_features(x, baseline):
        s = x.squeeze(1); B, T = s.shape; mid = max(1, min(T - 1, T // 2))
        amp      = s.max(dim=-1).values - s.min(dim=-1).values
        early    = s[:, :mid].mean(dim=-1); late = s[:, mid:].mean(dim=-1)
        ratio    = early / (late.abs() + 1e-6)
        upslope  = (torch.diff(s, dim=-1).clamp(min=0).max(dim=-1).values
                    if T > 1 else torch.zeros(B, device=x.device))
        peak_time = s.argmax(dim=-1).float() / float(T)
        return torch.stack([amp, early, late, ratio, upslope, peak_time,
                            baseline / PHYSIO.baseline_scale], dim=-1)

    def forward_with_skips(self, x_ppg, baseline):
        base_ch = baseline.view(-1, 1, 1).expand(-1, 1, x_ppg.shape[-1])
        x  = torch.cat([x_ppg, base_ch], dim=1)
        s1 = self.conv[0:4](x); s2 = self.conv[4:8](s1)
        h  = self.attn(self.conv[8:](s2))
        feat_cnn = self.bottleneck(h.contiguous().view(x.size(0), -1))
        phys     = self.phys_proj(self._morphology_features(x_ppg, baseline))
        feat     = self.fuse(torch.cat([feat_cnn, phys], dim=-1))
        return feat, self.skip1_proj(s1), self.skip2_proj(s2)

class HemodynamicExtractor(nn.Module):
    OUT_DIM = 16
    def __init__(self):
        super().__init__()
        self.lp_conv = nn.Sequential(
            nn.Conv1d(ARCH.n_beats + 1, 24, kernel_size=19, padding=9), nn.GELU(),
            nn.Conv1d(24, 16, kernel_size=11, padding=5),                nn.GELU(),
            nn.AdaptiveAvgPool1d(4))
        self.dias_proj = nn.Sequential(nn.Linear(10, 32), nn.GELU(), nn.Linear(32, 8))
        self.lp_proj   = nn.Sequential(nn.Flatten(), nn.Linear(16 * 4, 32), nn.GELU(), nn.Linear(32, 8))

    def _diastolic_features(self, s):
        B, T = s.shape; half = max(1, T // 2); dias = s[:, half:]
        level = s.mean(dim=-1); d_mean = dias.mean(dim=-1)
        d_std = dias.std(dim=-1).clamp(min=1e-4)
        d_decay = (dias[:, 0] - dias[:, -1]).clamp(-20.0, 20.0)
        sys_area = s[:, :half].mean(dim=-1).abs(); dias_area = dias.mean(dim=-1).abs()
        runoff = dias_area / (sys_area + 1e-6)
        d1 = torch.diff(s, dim=-1, prepend=s[:, :1])
        downslope = (-d1[:, half:]).clamp(min=0).mean(dim=-1)
        d2 = torch.diff(d1, dim=-1, prepend=d1[:, :1])
        dias_curv = d2[:, half:].abs().mean(dim=-1).clamp(max=1.0)
        d_min = s.min(dim=-1).values
        d_end = (dias[:, -1] - d_min) / (d_mean - d_min + 1e-6)
        signal_std = s.std(dim=-1).clamp(min=1e-4)
        pp_proxy = (s.max(dim=-1).values - s.min(dim=-1).values) / (signal_std + 1e-6)
        return torch.stack([level, d_mean, d_std, d_decay, runoff, downslope,
                            dias_curv, d_end, (d_std / (level.abs() + 1e-6)), pp_proxy], dim=-1)

    def forward(self, x_win):
        if x_win.dim() == 4: x_win = x_win.squeeze(2)
        B, K, T = x_win.shape
        dias_emb = self.dias_proj(self._diastolic_features(x_win[:, K // 2]))
        lp_emb   = self.lp_proj(self.lp_conv(x_win))
        return torch.cat([dias_emb, lp_emb], dim=-1)

class WindkesselODESolver(nn.Module):
    def __init__(self):
        super().__init__()
        self.dt = WK.dt; self.steps = WK.steps
    def forward(self, q, R, C, L, P0=None):
        tau = R * C; tau_c = tau.clamp(min=WK.tau_min, max=WK.tau_max)
        scale = (tau_c / (tau + 1e-6)).sqrt()
        R = (R * scale).clamp(min=0.05, max=WK.r_max).view(-1, 1)
        C = (C * scale).clamp(min=0.01, max=WK.c_max).view(-1, 1)
        L = L.view(-1, 1).clamp(min=1e-4)
        P = torch.zeros(q.shape[0], 1, device=q.device) if P0 is None else P0.view(-1, 1)
        alpha = torch.clamp(self.dt / (R * C), max=1.0)
        q_use = q[:, :self.steps]; wave = []
        for t in range(q_use.shape[1]):
            P = (P + self.dt * q_use[:, t:t+1] / C) / (1.0 + alpha + 1e-6)
            dq = (q_use[:, t:t+1] - (q_use[:, t-1:t] if t > 0 else torch.zeros_like(q_use[:, 0:1]))) / self.dt
            wave.append(P + L * dq)
        P_wave = torch.cat(wave, dim=1)
        if P_wave.shape[1] < q.shape[1]:
            P_wave = torch.cat([P_wave, P_wave[:, -1:].expand(-1, q.shape[1] - P_wave.shape[1])], dim=1)
        return ((P_wave - P_wave.mean(dim=1, keepdim=True))
                / P_wave.std(dim=1, keepdim=True).clamp(min=1e-4))

class ABPDecoder(nn.Module):
    def __init__(self, latent=None):
        super().__init__()
        latent = latent or ARCH.latent_dim
        self.expand   = nn.Sequential(nn.Linear(latent, 128 * 32), nn.GELU())
        self.up1      = nn.Sequential(nn.ConvTranspose1d(128, 64, 4, stride=2, padding=1), nn.GELU(), nn.BatchNorm1d(64))
        self.up1_fuse = nn.Sequential(nn.Conv1d(64 + 32, 64, 3, padding=1), nn.GELU())
        self.up2      = nn.Sequential(nn.ConvTranspose1d(64, 32, 4, stride=2, padding=1), nn.GELU(), nn.BatchNorm1d(32))
        self.up2_fuse = nn.Sequential(nn.Conv1d(32 + 64, 32, 3, padding=1), nn.GELU())
        self.wk_proj  = nn.Conv1d(1, 32, 1)
        self.out_conv = nn.Sequential(nn.Conv1d(64, 32, 3, padding=1), nn.GELU(), nn.Conv1d(32, 1, 1))
    def forward(self, z, skip1, skip2, wk_wave):
        f = self.expand(z).view(z.size(0), 128, 32)
        f = self.up1_fuse(torch.cat([self.up1(f), skip1], dim=1))
        f = self.up2_fuse(torch.cat([self.up2(f),
            F.interpolate(skip2, size=128, mode='linear', align_corners=False)], dim=1))
        return self.out_conv(torch.cat([f, self.wk_proj(wk_wave.unsqueeze(1))], dim=1)).squeeze(1)


class DiBPNet(nn.Module):
    def __init__(self, latent=None, n_beats=None, dropout=0.15):
        super().__init__()
        latent  = latent  or ARCH.latent_dim
        n_beats = n_beats or ARCH.n_beats
        self.latent_dim = latent; self.n_beats = n_beats; self.dropout = dropout

        self.beat_enc       = BeatEncoder(in_ch=2, dropout=dropout)
        self.gru            = nn.GRU(128, 128, num_layers=2, batch_first=True, dropout=dropout)
        self.gru_attn_pool  = nn.Sequential(nn.Linear(128, 32), nn.Tanh(), nn.Linear(32, 1))
        self.gru_pool_proj  = nn.Sequential(nn.Linear(128 * 2, 128), nn.GELU(), nn.Dropout(dropout))

        self.morph_proj = nn.Sequential(nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.05), nn.Linear(64, latent), nn.Tanh())
        self.hemo_proj  = nn.Sequential(nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.15), nn.Linear(64, latent), nn.Tanh())
        self.reg_proj   = nn.Sequential(nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.05), nn.Linear(64, latent), nn.Tanh())

        self.bp_stat_enc = nn.Sequential(nn.Linear(2, 32), nn.GELU(), nn.Linear(32, ARCH.bp_stat_dim), nn.Tanh())
        self.wk          = nn.Sequential(nn.Linear(latent + ARCH.bp_stat_dim, 64), nn.GELU(), nn.Linear(64, 3))
        self.wk_solver   = WindkesselODESolver()
        self.q_head      = nn.Sequential(nn.Linear(latent, 64), nn.GELU(), nn.Dropout(0.1), nn.Linear(64, ARCH.seq_len), nn.Softplus())
        self.abp_dec     = ABPDecoder(latent)
        self.p0_head     = nn.Sequential(nn.Linear(latent, 32), nn.GELU(), nn.Linear(32, 1))

        self.htn_head = nn.Sequential(nn.Linear(latent, 64), nn.GELU(), nn.Dropout(dropout),
                                       nn.BatchNorm1d(64), nn.Linear(64, 32), nn.GELU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.aux_head = nn.Sequential(nn.Linear(latent, 64), nn.GELU(), nn.Dropout(0.30), nn.Linear(64, 32), nn.GELU(), nn.Linear(32, 1))

        self.subj_emb  = nn.Embedding(ARCH.embed_size, 8)
        self.subj_proj = nn.Linear(latent, 8, bias=False)
        _bp_in = latent + latent + 8
        self.bp_fc1 = nn.Linear(_bp_in, 256); self.bp_fc2 = nn.Linear(256, 128)
        self.bp_fc3 = nn.Linear(128, ARCH.head_feat_dim); self.bp_fc_out = nn.Linear(ARCH.head_feat_dim, 2)
        self.bp_skip = nn.Linear(_bp_in, ARCH.head_feat_dim)

        self.hemo_ext = HemodynamicExtractor()
        _wave_in = latent + HemodynamicExtractor.OUT_DIM
        self.wave_fc  = nn.Sequential(nn.Linear(_wave_in, 128), nn.GELU(), nn.Dropout(0.15),
                                       nn.Linear(128, ARCH.head_feat_dim), nn.GELU(), nn.Dropout(0.10))
        self.wave_out = nn.Linear(ARCH.head_feat_dim, 2)
        nn.init.normal_(self.wave_out.weight, std=0.01); nn.init.zeros_(self.wave_out.bias)

        self.ens_gate_direct = nn.Sequential(nn.Linear(latent, 16), nn.GELU(), nn.Linear(16, 1))
        self.ens_gate_wk     = nn.Sequential(nn.Linear(latent, 16), nn.GELU(), nn.Linear(16, 1))
        self.slow_C_head     = nn.Sequential(nn.Linear(latent, ARCH.c_slow_head_dim), nn.GELU(), nn.Linear(ARCH.c_slow_head_dim, 1))

        for buf, val in [('pop_mu', 0.), ('pop_sig', 1.), ('sbp_mu', 0.), ('sbp_sig', 1.), ('dbp_mu', 0.), ('dbp_sig', 1.)]:
            self.register_buffer(buf, torch.tensor(val))
        self._subj_z_cache: dict = {}

    def _set_stats_from_splits(self, splits: DataSplits):
        self.pop_mu.fill_(splits.pop_mu);  self.pop_sig.fill_(splits.pop_sig)
        self.sbp_mu.fill_(splits.sbp_mu);  self.sbp_sig.fill_(max(splits.sbp_sig, 1.0))
        self.dbp_mu.fill_(splits.dbp_mu);  self.dbp_sig.fill_(max(splits.dbp_sig, 1.0))

    def _subject_embedding(self, subj_id):
        if not self.training:
            _n_subj = (SPLITS.n_subjects
            if (SPLITS is not None and SPLITS.n_subjects > 0)
                        else self.subj_emb.weight.shape[0])
            pop_mean = self.subj_emb.weight[:_n_subj].mean(0)
            return torch.stack([
                torch.tensor(self._subj_z_cache[sid], dtype=torch.float32, device=subj_id.device)
                if sid in self._subj_z_cache else pop_mean
                for sid in subj_id.cpu().numpy()], dim=0)
        emb = self.subj_emb(subj_id)
        return emb * (torch.rand_like(emb[:, :1]) > 0.30).float()

    def build_subj_cache(self, ref_X, ref_Q, ref_subj, ref_base, batch=128):
        self.eval(); self._subj_z_cache = {}
        with torch.no_grad():
            for i in range(0, len(ref_X), batch):
                sl  = slice(i, min(i + batch, len(ref_X)))
                out = self.forward(torch.tensor(ref_X[sl]).to(DEVICE),
                                   torch.tensor(ref_subj[sl], dtype=torch.long).to(DEVICE),
                                   torch.tensor(ref_base[sl]).to(DEVICE),
                                   torch.tensor(ref_Q[sl, ARCH.n_beats // 2]).to(DEVICE),
                                   hybrid_alpha=0.0, _cache_build=True)
                for b_idx, sid in enumerate(ref_subj[sl]):
                    self._subj_z_cache.setdefault(int(sid), []).append(out['z_reg'].cpu().numpy()[b_idx])
        with torch.no_grad():
            for sid, zs in self._subj_z_cache.items():
                mean_z = torch.tensor(np.mean(zs, axis=0), dtype=torch.float32).to(DEVICE)
                mean_z = (mean_z - mean_z.mean()) / mean_z.std().clamp(min=1e-6)
                self._subj_z_cache[sid] = self.subj_proj(mean_z).cpu().numpy().astype(np.float32)
        print(f"  Subject cache built: {len(self._subj_z_cache)} subjects", flush=True)

    def _augment_and_encode(self, x_win, baseline):
        if x_win.dim() == 4: x_win = x_win.squeeze(2)
        B, K, T = x_win.shape; x_win = x_win.unsqueeze(2)
        if self.training:
            amp = torch.rand(B, K, 1, 1, device=x_win.device) * 0.4 + 0.8
            x_win = x_win * amp + torch.randn_like(x_win) * 0.02
            shift = torch.randint(-5, 5, (B,), device=x_win.device)
            for b in range(B): x_win[b] = torch.roll(x_win[b], shifts=int(shift[b]), dims=-1)
        centre = K // 2; x_flat = x_win.reshape(B * K, 1, T)
        base_flat = baseline.unsqueeze(1).expand(-1, K).reshape(B * K)
        feat, s1_all, s2_all = self.beat_enc.forward_with_skips(x_flat, base_flat)
        return (feat.view(B, K, 128),
                s1_all.view(B, K, 32, s1_all.shape[-1])[:, centre],
                s2_all.view(B, K, 64, s2_all.shape[-1])[:, centre],
                x_win.squeeze(2))

    def _gru_encode_dual(self, beat_feats):
        gru_out = self.gru(beat_feats)[0]
        attn_w  = torch.softmax(self.gru_attn_pool(gru_out), dim=1)
        h_attn  = (gru_out * attn_w).sum(dim=1)
        h_max   = gru_out.max(dim=1).values
        h_pool  = self.gru_pool_proj(torch.cat([h_attn, h_max], dim=1))
        return h_pool, h_attn, h_max

    def _predict_bp_direct(self, z_morph, z_reg_det, s_emb):
        z    = torch.cat([z_morph, z_reg_det, s_emb], dim=-1)
        h    = F.dropout(F.gelu(self.bp_fc1(z)), p=self.dropout, training=self.training)
        h    = F.dropout(F.gelu(self.bp_fc2(h)), p=self.dropout, training=self.training)
        feat = F.gelu(self.bp_fc3(h) + self.bp_skip(z))
        bp_n = self.bp_fc_out(feat)
        bp   = torch.stack([bp_n[:, 0] * self.sbp_sig + self.sbp_mu,
                             bp_n[:, 1] * self.dbp_sig + self.dbp_mu], dim=1)
        return bp, feat

    def _predict_bp_wave(self, z_hemo, x_win_raw):
        hemo_feat = self.hemo_ext(x_win_raw)
        feat      = self.wave_fc(torch.cat([z_hemo, hemo_feat], dim=-1))
        bp_n      = self.wave_out(feat)
        bp        = torch.stack([bp_n[:, 0] * self.sbp_sig + self.sbp_mu,
                                  bp_n[:, 1] * self.dbp_sig + self.dbp_mu], dim=1)
        return bp, feat

    def _softmax_gate(self, z_morph, z_hemo, gate_floor=0.0, wk_gate_floor=0.0):
        weights  = torch.softmax(torch.cat([self.ens_gate_direct(z_morph), self.ens_gate_wk(z_hemo)], dim=-1), dim=-1)
        w_direct = weights[:, 0:1]; w_wk = weights[:, 1:2]
        if gate_floor    > 0.0: w_direct = w_direct.clamp(min=gate_floor); w_wk = 1.0 - w_direct
        if wk_gate_floor > 0.0: w_wk = w_wk.clamp(min=wk_gate_floor); w_direct = 1.0 - w_wk
        return w_direct, w_wk

    def _compute_C(self, tau, k):
        c_raw = tau / k.clamp(min=0.05)
        return WK.c_min + (WK.c_max - WK.c_min) * torch.sigmoid(c_raw / WK.c_sigmoid_temp)

    def forward(self, x_win, subj_id, baseline, q_proxy=None,
                hybrid_alpha=0.0, _cache_build=False,
                gate_floor=0.0, detach_ratio=None, ste_alpha=None, wk_gate_floor=0.0,
                grl_alpha: float = 1.0):
        detach_ratio = detach_ratio if detach_ratio is not None else TRAIN.detach_ratio
        ste_alpha    = ste_alpha    if ste_alpha    is not None else TRAIN.ste_alpha_end

        beat_feats, skip1, skip2, x_win_raw = self._augment_and_encode(x_win, baseline)
        h_pool, h_attn, h_max = self._gru_encode_dual(beat_feats)

        z_morph = self.morph_proj(h_max); z_hemo = self.hemo_proj(h_attn)
        z_reg   = self.reg_proj(0.05 * h_pool.detach() + 0.95 * h_pool)

        if _cache_build: return {'z_reg': z_reg}

        s_emb = self._subject_embedding(subj_id)
        bp_direct, feat_direct = self._predict_bp_direct(z_morph, z_reg.detach(), s_emb)

        baseline_hint = torch.stack([baseline, baseline * PHYSIO.dbp_sbp_ratio], dim=-1)
        bp_relative   = (bp_direct.detach() - baseline_hint.detach()) / PHYSIO.bp_scale_mmhg
        bp_stat       = self.bp_stat_enc(bp_relative)
        wk_raw = self.wk(torch.cat([z_hemo, bp_stat], dim=-1))
        tau = F.softplus(wk_raw[:, 0]) + 0.5; k = torch.sigmoid(wk_raw[:, 1]) * 2.0 + 0.1
        R   = (tau * k).clamp(max=WK.r_max); C = self._compute_C(tau, k)

        if self.training:
            R_center = R.mean().detach(); R_var = R - R.mean()
            R = (R_center + (1.0 - TRAIN.r_var_detach_ratio) * R_var + TRAIN.r_var_detach_ratio * R_var.detach())

        L      = torch.ones_like(tau) * 0.01
        q_pred = normalize_q(self.q_head(z_hemo))

        if q_proxy is not None and hybrid_alpha > 0.0:
            q_obs = normalize_q(q_proxy)
            if self.training:
                mask    = (torch.rand(q_pred.shape[0], 1, device=q_pred.device) < hybrid_alpha).float()
                q_input = mask * q_obs + (1.0 - mask) * q_pred
            else:
                q_input = hybrid_alpha * q_obs + (1.0 - hybrid_alpha) * q_pred
        else:
            q_input = q_pred

        wk_wave     = self.wk_solver(q_input, R, C, L, P0=self.p0_head(z_hemo))
        wk_wave_ste = wk_wave.detach() + ste_alpha * (wk_wave - wk_wave.detach())
        abp_pred    = self.abp_dec(z_hemo, skip1, skip2, wk_wave_ste)

        htn_logit = self.htn_head(z_reg).squeeze(-1)
        aux_logit = self.aux_head(z_hemo).squeeze(-1)
        bp_wave, feat_wave = self._predict_bp_wave(z_hemo.detach(), x_win_raw)

        w_direct, w_wk = self._softmax_gate(z_morph, z_hemo, gate_floor, wk_gate_floor)
        bp_direct_semi = (bp_direct.detach() * detach_ratio + bp_direct * (1.0 - detach_ratio)
                          if self.training and detach_ratio > 0.0 else bp_direct)
        bp_ens = w_direct * bp_direct_semi + w_wk * bp_wave
        gate   = w_direct; RC = R * C

        c_slow = WK.c_min + (WK.c_max - WK.c_min) * torch.sigmoid(
            self.slow_C_head(z_hemo.detach()).squeeze(-1) / WK.c_sigmoid_temp)

        return (abp_pred, wk_wave, q_pred, q_input,
                z_hemo, z_reg, htn_logit, R, C, L,
                bp_direct, bp_wave, bp_ens,
                aux_logit, gate, h_pool,
                feat_direct, feat_wave, z_morph, RC, c_slow)

    def wk_params(self):
        return [p for m in [self.wk, self.wk_solver] for p in m.parameters()]
    def enc_params(self):
        return [p for m in [self.beat_enc, self.gru, self.gru_attn_pool,
                             self.gru_pool_proj, self.morph_proj, self.hemo_proj,
                             self.reg_proj, self.q_head] for p in m.parameters()]
    def head_params_no_wk(self):
        return [p for m in [self.abp_dec, self.htn_head, self.aux_head,
                             self.p0_head, self.subj_emb, self.subj_proj,
                             self.bp_fc1, self.bp_fc2, self.bp_fc3, self.bp_fc_out, self.bp_skip,
                             self.wave_fc, self.wave_out, self.hemo_ext,
                             self.ens_gate_direct, self.ens_gate_wk, self.slow_C_head,
                             self.bp_stat_enc] for p in m.parameters()]
    def direct_bp_params(self):
        return [p for m in [self.bp_fc1, self.bp_fc2, self.bp_fc3, self.bp_fc_out, self.bp_skip]
                for p in m.parameters()]
    def unfreeze(self): [p.requires_grad_(True) for p in self.parameters()]
    def n_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ── EMA subject prototype buffer  ───────────────────────────────────────

class SubjectCPrototype:
    def __init__(self, ema_decay: float = None):
        self.decay = ema_decay or TRAIN.c_proto_ema_decay
        self.proto: dict = {}; self._is_train = True
    def train_mode(self): self._is_train = True
    def eval_mode(self):  self._is_train = False
    def update(self, subj_ids_np, c_slow_batch):
        if not self._is_train: return
        c_det = c_slow_batch.detach().cpu()
        for i, sid in enumerate(subj_ids_np):
            c_val = float(c_det[i].item())
            if sid in self.proto:
                self.proto[sid] = self.decay * self.proto[sid] + (1.0 - self.decay) * c_val
            else:
                self.proto[sid] = c_val
    def get_targets(self, subj_ids_np, device):
        targets, valid = [], []
        for sid in subj_ids_np:
            if sid in self.proto: targets.append(self.proto[sid]); valid.append(True)
            else: targets.append(0.0); valid.append(False)
        return (torch.tensor(targets, dtype=torch.float32, device=device),
                torch.tensor(valid,   dtype=torch.bool,    device=device))
    def reset(self): self.proto = {}


# ── GRL Variant  ──────────────────────────────────────────────────────────

class _GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha): ctx.alpha = alpha; return x.clone()
    @staticmethod
    def backward(ctx, grad): return -ctx.alpha * grad, None

def grad_reverse(x, alpha=1.0): return _GradReverse.apply(x, alpha)

class DiBPNet_v193(DiBPNet):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        n_cls = min(SPLITS.n_subjects, TRAIN.subj_id_acc_n_cap)
        self._adv_classes   = n_cls
        self.subj_adversary = nn.Sequential(
            nn.Linear(ARCH.latent_dim, 64), nn.ReLU(), nn.Linear(64, n_cls))

    def forward(self, *args, grl_alpha: float = 1.0, **kwargs):
        out = super().forward(*args, **kwargs)
        if kwargs.get("_cache_build"): return out
        return out + (self.subj_adversary(grad_reverse(out[4], grl_alpha)),)

### Loss Functions

The differentiable loss terms: waveform, blood pressure, Windkessel-ODE physics, spread, contrastive, rank, focal, and HSIC.

In [ ]:
# LOSS FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def _mmhg_loss(pred, true, sbp_sig, dbp_sig):
    err = pred - true; h = 0.5 * torch.abs(err) + 0.5 * err ** 2
    return LW.sbp * (h[:, 0] / sbp_sig).mean() + (h[:, 1] / dbp_sig).mean()

def _wave_bp_loss(bp_wave, bp_true):
    return LW.sbp * F.l1_loss(bp_wave[:, 0], bp_true[:, 0]) + F.l1_loss(bp_wave[:, 1], bp_true[:, 1])

def waveform_loss(pred, target):
    l_mse   = F.mse_loss(pred, target)
    l_deriv = F.mse_loss(pred[:, 1:] - pred[:, :-1], target[:, 1:] - target[:, :-1])
    l_pk    = (F.huber_loss(soft_max_val(pred), soft_max_val(target), delta=1.0)
               + F.huber_loss(soft_min_val(pred), soft_min_val(target), delta=1.0))
    return l_mse, l_deriv, l_pk

def bp_wave_aux_loss(abp_norm, pop_sig, pop_mu, bp_true, sbp_sig, dbp_sig, sbp_mu, dbp_mu):
    abp_mm = abp_norm * pop_sig + pop_mu
    bp_w   = torch.stack([soft_max_val(abp_mm), soft_min_val(abp_mm)], dim=-1)
    pn = torch.stack([(bp_w[:, 0] - sbp_mu) / sbp_sig, (bp_w[:, 1] - dbp_mu) / dbp_sig], dim=-1)
    tn = torch.stack([(bp_true[:, 0] - sbp_mu) / sbp_sig, (bp_true[:, 1] - dbp_mu) / dbp_sig], dim=-1)
    s = 0.5 * F.l1_loss(pn[:, 0], tn[:, 0]) + 0.5 * F.huber_loss(pn[:, 0], tn[:, 0], delta=0.5)
    d = 0.5 * F.l1_loss(pn[:, 1], tn[:, 1]) + 0.5 * F.huber_loss(pn[:, 1], tn[:, 1], delta=0.5)
    return LW.sbp * s + d

def wk_physics_loss(wk_wave, q_in, R, C, L):
    dQ  = torch.diff(q_in,    prepend=q_in[:, :1],    dim=1)
    dP  = torch.diff(wk_wave, prepend=wk_wave[:, :1], dim=1)
    res = (C.unsqueeze(-1).clamp(min=0.01) * dP - WK.dt * q_in
           + WK.dt * wk_wave / R.unsqueeze(-1).clamp(min=0.05)
           - L.unsqueeze(-1).clamp(min=1e-4) * dQ)
    return res.pow(2).mean().clamp(max=1.0)

def wk_prior_loss(R, C, L):
    tau = R * C
    return (LW.wkprior_tau * F.mse_loss(tau, torch.ones_like(tau) * WK.tau_target)
          + LW.wkprior_r   * ((torch.log(R.clamp(min=0.05)) - np.log(WK.r_target)) ** 2).mean()
          + LW.wkprior_c   * ((C - WK.c_target) / WK.c_target).pow(2).mean())

def wk_spread_loss(R, C):
    return ((R.std(unbiased=False) - WK.target_r_std).pow(2),
            (C.std(unbiased=False) - WK.target_c_std).pow(2))

def c_antisaturation_loss(C, global_ep):
    if global_ep < TRAIN.c_sat_start_ep:
        return C.new_zeros(1).squeeze(), C.new_zeros(1).squeeze()
    return (F.relu(C - TRAIN.c_sat_thresh).pow(2).mean(),
            ((C.mean() - TRAIN.c_prior_mean) / TRAIN.c_prior_scale).pow(2))

def wk_corr_loss(R, sbp):          return -_batch_corr(R, sbp.detach())
def r_physiology_loss(R):
    q75 = torch.quantile(R, 0.75); q25 = torch.quantile(R, 0.25)
    return F.relu(TRAIN.r_iqr_floor - (q75 - q25).clamp(min=0.0)).pow(2)
def r_magnitude_anchor_loss(R):    return (R.abs().mean() - TRAIN.r_anc_target_mean).pow(2)
def qr_independence_loss(q_pred, R):
    q_mean = q_pred.mean(dim=-1); q_c = q_mean - q_mean.mean(); R_c = R - R.mean()
    corr = (q_c * R_c).mean() / (q_c.std().clamp(min=1e-6) * R_c.std().clamp(min=1e-6))
    return corr.pow(2)

def hsic_loss(z_morph, z_hemo, global_ep):
    if global_ep < TRAIN.hsic_start_ep: return z_morph.new_zeros(1).squeeze()
    B = z_morph.shape[0]
    if B < 8: return z_morph.new_zeros(1).squeeze()
    with torch.no_grad():
        def _median_bw(z):
            d2 = torch.cdist(z.detach(), z.detach(), p=2).pow(2)
            return max((d2.median().clamp(min=1e-4) / (2.0 * np.log(B + 1))).sqrt().item(), 0.1)
        sigma_x = _median_bw(z_morph)
        sigma_y = _median_bw(z_hemo)
    Kx = torch.exp(-torch.cdist(z_morph, z_morph, p=2).pow(2) / (2.0 * sigma_x ** 2))
    Ky = torch.exp(-torch.cdist(z_hemo,  z_hemo,  p=2).pow(2) / (2.0 * sigma_y ** 2))
    H  = torch.eye(B, device=z_morph.device) - 1.0 / B
    return ((H @ Kx @ H) * (H @ Ky @ H)).sum() / max((B - 1) ** 2, 1)

def anticollapse_loss(z_morph, z_hemo):
    zm = z_morph.mean(dim=-1); zh = z_hemo.mean(dim=-1)
    zm = zm - zm.mean(); zh = zh - zh.mean()
    corr = (zm * zh).mean() / (zm.std().clamp(min=1e-6) * zh.std().clamp(min=1e-6))
    return F.relu(corr.abs() - TRAIN.anticollapse_thresh).pow(2)

def c_proto_consistency_loss(c_slow, subj_ids_np, proto_buffer, device, global_ep):
    lam = _lam_c_proto(global_ep)
    if lam == 0.0: return c_slow.new_zeros(1).squeeze()
    proto_targets, valid_mask = proto_buffer.get_targets(subj_ids_np, device)
    if valid_mask.sum() < 4: return c_slow.new_zeros(1).squeeze()
    c_valid = c_slow[valid_mask]; proto_valid = proto_targets[valid_mask]
    global_std = c_slow.std(unbiased=False).detach().clamp(min=1e-6)
    return ((c_valid - proto_valid).abs().mean() / (global_std + 1e-6)
            + F.relu(TRAIN.target_c_slow_std - global_std).pow(2))

def c_slow_spread_loss(c_slow):
    return (c_slow.std(unbiased=False) - TRAIN.target_c_slow_std).pow(2)
def c_slow_prior_loss(c_slow):
    return ((c_slow.mean() - TRAIN.c_slow_prior_mean) / TRAIN.c_slow_prior_scale).pow(2)

def wk_shape_consistency_loss(wk_wave, q_input, R):
    qr_product = q_input.mean(dim=-1) * R.detach(); wk_peak = soft_max_val(wk_wave)
    B = wk_wave.shape[0]
    if B < 4: return wk_wave.new_zeros(1).squeeze()
    qr_diff = qr_product.unsqueeze(1) - qr_product.unsqueeze(0)
    wp_diff = wk_peak.unsqueeze(1)    - wk_peak.unsqueeze(0)
    mask    = qr_diff.abs() > 0.05
    return (F.relu(-(qr_diff * wp_diff)[mask]).mean()
            if mask.sum() > 0 else wk_wave.new_zeros(1).squeeze())

def q_anchor_loss(q_pred, q_proxy):
    q_proxy_n = normalize_q(q_proxy).detach()
    d1 = torch.diff(q_pred, dim=-1); d2 = torch.diff(d1, dim=-1)
    l_temp = d1.pow(2).mean() * 0.1 + d2.pow(2).mean() * 0.05
    return 0.6 * F.l1_loss(q_pred, q_proxy_n) + LW.qtemp * l_temp

def wk_direct_bp_loss(wk_wave, bp_true, pop_sig, pop_mu):
    bp_range = (bp_true[:, 0] - bp_true[:, 1]).clamp(min=10.0).mean().detach()
    wk_mm    = wk_wave * pop_sig + pop_mu
    return (LW.sbp * F.l1_loss(soft_max_val(wk_mm) / bp_range, bp_true[:, 0] / bp_range)
            + F.l1_loss(soft_min_val(wk_mm) / bp_range, bp_true[:, 1] / bp_range))

def focal_loss(logits, targets, gamma=None, alpha=None):
    gamma = gamma or TRAIN.focal_gamma; alpha = alpha or TRAIN.focal_alpha
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    pt  = targets * torch.sigmoid(logits) + (1 - targets) * (1 - torch.sigmoid(logits))
    at  = targets * alpha + (1 - targets) * (1 - alpha)
    return (at * (1 - pt).pow(gamma) * bce).mean()

def contrastive_loss(z, bp_vec, temperature=None):
    temperature = temperature or TRAIN.cont_temperature
    z_in = F.normalize(z, dim=-1); sim = torch.mm(z_in, z_in.T) / temperature
    d    = torch.cdist(bp_vec, bp_vec, p=1) / PHYSIO.cont_bp_dist_scale
    pw   = torch.exp(-d / PHYSIO.cont_pos_temp); pw.fill_diagonal_(0.0)
    nw   = torch.exp(d  / PHYSIO.cont_neg_temp); nw.fill_diagonal_(0.0)
    pw   = pw / (pw.sum(dim=1, keepdim=True) + 1e-8)
    nw   = nw / (nw.sum(dim=1, keepdim=True) + 1e-8)
    den  = (torch.exp(sim - sim.max().detach()) * nw).sum(1, keepdim=True) + 1e-8
    return -(pw * (sim - torch.log(den))).sum() / pw.sum().clamp(min=1e-8)

def rank_loss(pred, true):
    N = pred.shape[0]
    if N < 4: return pred.new_zeros(1).squeeze()
    def _rl(p_col, t_col, thresh, weight=1.0):
        pi = p_col.unsqueeze(1) - p_col.unsqueeze(0)
        pj = t_col.unsqueeze(1) - t_col.unsqueeze(0)
        mask = pj.abs() > thresh
        return weight * (F.relu(-(pi * pj.sign())[mask]).mean()
                         if mask.sum() > 0 else pred.new_zeros(1).squeeze())
    return _rl(pred[:, 0], true[:, 0], PHYSIO.rank_sbp_thresh, LW.sbp) + _rl(pred[:, 1], true[:, 1], PHYSIO.rank_dbp_thresh)

def extremes_loss(pred, true, sbp_sig, dbp_sig):
    true_mean = true[:, 0].mean().detach(); true_std = true[:, 0].std().detach().clamp(min=10.0)
    w = (1.0 + 2.0 * ((true[:, 0] - true_mean).abs() / true_std).clamp(max=3.0))
    w = w * (((true[:, 0] >= 160) | (true[:, 0] <= 100)).float() * 2.0 + 1.0)
    return (LW.sbp * (w * torch.abs(pred[:, 0] - true[:, 0])).mean() / sbp_sig
            + torch.abs(pred[:, 1] - true[:, 1]).mean() / dbp_sig)

def gate_entropy_loss(gate):
    g = gate.clamp(1e-6, 1 - 1e-6)
    return (g * g.log() + (1 - g) * (1 - g).log()).mean()
def gate_barrier_loss(gate): return F.relu(gate - TRAIN.gate_barrier_thresh).pow(2).mean()
def wk_utilization_loss(gate):
    excess = (gate - TRAIN.gate_barrier_thresh).clamp(min=0.0)
    return (-torch.log(1.0 - excess.clamp(max=0.999) + 1e-6)).mean()

def compute_grad_balance(l_wk, l_direct, shared_param, clamp):
    try:
        g_wk  = torch.autograd.grad(l_wk,  shared_param, retain_graph=True, allow_unused=True, create_graph=False)[0]
        g_dir = torch.autograd.grad(l_direct, shared_param, retain_graph=True, allow_unused=True, create_graph=False)[0]
        nw = g_wk.norm()  if g_wk  is not None else l_wk.new_tensor(1e-6)
        nd = g_dir.norm() if g_dir is not None else l_direct.new_tensor(1e-6)
        return (nd / (nw + 1e-6)).clamp(clamp[0], clamp[1]).detach()
    except Exception: return l_wk.new_tensor(1.0)

def _wk_iqr_mask(R, C):
    tau = R * C
    def _inlier(x):
        return ((x >= torch.quantile(x, TRAIN.wk_iqr_low / 100.0)) &
                (x <= torch.quantile(x, TRAIN.wk_iqr_high / 100.0)))
    return _inlier(R) & _inlier(tau)

### Batch Runner

Single-batch forward pass and total-loss assembly.

In [ ]:
# BATCH RUNNER + TOTAL LOSS
# ══════════════════════════════════════════════════════════════════════════════

def _run_batch(mdl, bx, bq, babp, bbp, bht, bsubj, bbase,
               hybrid_alpha=0.0, lam_wk_ode=0.01, lam_qanchor=0.70,
               gate_floor=0.0, lam_wk_bp=0.0, global_ep=0,
               ste_alpha=None, gate_ema=None, wk_gate_floor=0.0,
               proto_buffer=None, grl_alpha: float = 1.0):
    ste_alpha  = ste_alpha or TRAIN.ste_alpha_end
    centre_idx = ARCH.n_beats // 2
    abp_true   = babp[:, centre_idx]; abp_true = abp_true.squeeze(1) if abp_true.dim() == 3 else abp_true
    q_centre   = bq[:, centre_idx];  q_centre  = q_centre.squeeze(1)  if q_centre.dim()  == 3 else q_centre

    out = mdl(bx, bsubj, bbase, q_centre, hybrid_alpha,
              gate_floor=gate_floor, ste_alpha=ste_alpha,
              wk_gate_floor=wk_gate_floor, grl_alpha=grl_alpha)

    (abp_pred, wk_wave, q_pred, q_input,
     z_hemo, z_reg, htn_logit, R, C, L,
     bp_direct, bp_wave, bp_ens,
     aux_logit, gate, h_pool,
     feat_direct, feat_wave, z_morph, RC, c_slow, *_extra) = out

    sbp_sig = mdl.sbp_sig.item(); dbp_sig = mdl.dbp_sig.item()
    sbp_mu  = mdl.sbp_mu.item();  dbp_mu  = mdl.dbp_mu.item()
    pop_sig = mdl.pop_sig.item(); pop_mu  = mdl.pop_mu.item()

    wk_weight = _wk_iqr_mask(R.detach(), C.detach()).float()

    l_mse, l_deriv, l_pk = waveform_loss(abp_pred, abp_true)
    l_bpaux  = bp_wave_aux_loss(abp_pred, pop_sig, pop_mu, bbp, sbp_sig, dbp_sig, sbp_mu, dbp_mu)

    wk_wave_m = wk_wave * wk_weight.unsqueeze(1); abp_pred_m = abp_pred.detach() * wk_weight.unsqueeze(1)
    l_wk_shp  = F.mse_loss(wk_wave_m, abp_pred_m)
    l_wk_scl  = F.l1_loss(wk_wave.std(dim=1) * wk_weight, abp_true.std(dim=1) * wk_weight)
    l_wk_smo  = ((wk_wave[:, 2:] - 2 * wk_wave[:, 1:-1] + wk_wave[:, :-2]) ** 2).mean()
    l_wkode   = wk_physics_loss(wk_wave, q_input, R, C, L) + LW.wkode_shp * l_wk_shp + LW.wkode_scl * l_wk_scl + LW.wkode_smo * l_wk_smo
    l_wkp     = wk_prior_loss(R, C, L)
    l_wksprd_R, _ = wk_spread_loss(R, C)
    l_wkcorr  = wk_corr_loss(R, bbp[:, 0])
    l_wk_bp   = wk_direct_bp_loss(wk_wave, bbp, pop_sig, pop_mu)
    l_qr_indep = (qr_independence_loss(q_pred, R)
                  if global_ep >= TRAIN.qr_indep_start_ep else R.new_zeros(1).squeeze())
    l_wk_shcons = (wk_shape_consistency_loss(wk_wave, q_input, R)
                   if global_ep >= TRAIN.wk_shape_cons_start_ep else wk_wave.new_zeros(1).squeeze())
    l_r_physio  = r_physiology_loss(R)
    l_r_anc     = r_magnitude_anchor_loss(R)
    l_c_sat, l_c_prior = c_antisaturation_loss(C, global_ep)

    l_dir_mmhg = _mmhg_loss(bp_direct, bbp, sbp_sig, dbp_sig)
    l_dir_rank  = rank_loss(bp_direct, bbp)
    l_dir_ext   = extremes_loss(bp_direct, bbp, sbp_sig, dbp_sig)

    l_cls  = focal_loss(htn_logit, bht)
    l_aux  = focal_loss(aux_logit, bht)
    l_cont = contrastive_loss(z_hemo,
               torch.stack([bbp[:, 0] / PHYSIO.sbp_norm_max, bbp[:, 1] / PHYSIO.dbp_norm_max], dim=-1))

    l_wav_bp   = _wave_bp_loss(bp_wave, bbp)
    l_ens_mmhg = _mmhg_loss(bp_ens, bbp, sbp_sig, dbp_sig)

    l_qanchor  = q_anchor_loss(q_pred, q_centre)

    l_gate_ent  = gate_entropy_loss(gate)
    l_gate_bar  = gate_barrier_loss(gate)
    l_wk_util   = (wk_utilization_loss(gate)
                   if global_ep >= TRAIN.wk_util_start_ep else gate.new_zeros(1).squeeze())

    l_hsic        = hsic_loss(z_morph, z_hemo, global_ep)
    l_anticollapse = anticollapse_loss(z_morph, z_hemo)

    l_c_proto       = c_slow.new_zeros(1).squeeze()
    l_c_slow_spread = c_slow_spread_loss(c_slow)
    l_c_slow_prior  = c_slow_prior_loss(c_slow)
    if proto_buffer is not None:
        l_c_proto = c_proto_consistency_loss(c_slow, bsubj.cpu().numpy(), proto_buffer, bx.device, global_ep)
        proto_buffer.update(bsubj.cpu().numpy(), c_slow)

    wk_bal = (l_dir_mmhg.detach().abs() / (l_wkode.detach().abs() + 1e-6)).clamp(0.5, 2.0)

    return dict(
        mse=l_mse, deriv=l_deriv, pk=l_pk, bpaux=l_bpaux,
        wkode=l_wkode, wkp=l_wkp, wksprd_R=l_wksprd_R, wkcorr=l_wkcorr, wk_bp=l_wk_bp,
        qr_indep=l_qr_indep, wk_shape_cons=l_wk_shcons,
        qanchor=l_qanchor,
        cls=l_cls, aux=l_aux, cont=l_cont,
        dir_mmhg=l_dir_mmhg, dir_rank=l_dir_rank, dir_ext=l_dir_ext,
        wav_bp=l_wav_bp, ens_mmhg=l_ens_mmhg,
        gate_ent=l_gate_ent, gate_barrier=l_gate_bar, wk_util=l_wk_util,
        hsic=l_hsic, c_proto=l_c_proto, c_slow_spread=l_c_slow_spread, c_slow_prior=l_c_slow_prior,
        c_sat=l_c_sat, c_prior=l_c_prior,
        r_physio=l_r_physio, r_anc=l_r_anc, anticollapse=l_anticollapse,
        wk_balance_factor=float(wk_bal.item()),
        bp_direct=bp_direct, bp_wave=bp_wave, bp_ens=bp_ens,
        R=R, C=C, RC=RC, c_slow=c_slow,
        n_outlier=int((wk_weight < 1).sum().item()),
        gate_mean=float(gate.mean().item()), gate_val=gate.detach(),
        c_mean=float(C.mean().item()),  c_std=float(C.std().item()),
        c_sat_frac=float((C > TRAIN.c_sat_thresh).float().mean().item()),
        c_slow_mean=float(c_slow.mean().item()), c_slow_std=float(c_slow.std().item()),
        adv_logit=(_extra[0] if _extra else None),
    )


def _total_loss(l, mdl, lam_wk_ode, lam_qanchor, lam_wk_bp_val, global_ep, lam_c_proto_val):
    lam_hsic_val = _lam_hsic(global_ep)

    l_wk = (lam_wk_ode * l['wk_balance_factor'] * l['wkode']
          + LW.wksprd_r    * l['wksprd_R']
          + LW.wkcorr      * l['wkcorr']
          + lam_wk_bp_val  * l['wk_bp']
          + LW.r_anc       * l['r_anc']
          + LW.c_sat       * l['c_sat']
          + LW.c_prior     * l['c_prior'])

    l_direct = (LW.dir_mmhg * l['dir_mmhg']
              + LW.dir_rank  * l['dir_rank']
              + LW.dir_ext   * l['dir_ext'])

    l_slow = (lam_c_proto_val  * l['c_proto']
            + LW.c_slow_spread * l['c_slow_spread']
            + LW.c_slow_prior  * l['c_slow_prior'])

    l_shared = (LW.wave         * l['mse']
              + LW.deriv        * l['deriv']
              + LW.peak         * l['pk']
              + LW.bpaux        * l['bpaux']
              + lam_qanchor     * l['qanchor']
              + LW.htn          * l['cls']
              + LW.aux          * l['aux']
              + LW.cont         * l['cont']
              + LW.wavebp       * l['wav_bp']
              + LW.ens_mmhg     * l['ens_mmhg']
              + LW.gate_ent     * l['gate_ent']
              + LW.gate_barrier * l['gate_barrier']
              + LW.wk_util      * l['wk_util']
              + lam_hsic_val    * l['hsic']
              + LW.r_physio     * l['r_physio']
              + LW.anticollapse * l['anticollapse'])

    clamp = _grad_bal_clamp(global_ep)
    if mdl.training and clamp is not None:
        ratio = compute_grad_balance(l_wk, l_direct, mdl.gru_pool_proj[0].weight, clamp)
        if global_ep >= TRAIN.grad_bal_late_ep:
            ratio = ratio.clamp(max=TRAIN.grad_bal_late_max)
    else:
        ratio = 1.0

    return l_direct + ratio * l_wk + l_shared + l_slow, \
           float(ratio if isinstance(ratio, float) else ratio.item())

### Training

Per-epoch training loop, optimiser, and learning-rate schedule.

In [ ]:
# TRAINING CORE
# ══════════════════════════════════════════════════════════════════════════════

def _make_optimizer(mdl: DiBPNet):
    return torch.optim.AdamW([
        {"params": mdl.enc_params(),        "lr": TRAIN.enc_lr,      "weight_decay": 1e-4},
        {"params": mdl.head_params_no_wk(), "lr": TRAIN.head_lr,     "weight_decay": TRAIN.wd},
        {"params": mdl.wk_params(),         "lr": TRAIN.wk_lr_fixed, "weight_decay": TRAIN.wd},
    ])


def _run_epoch(mdl: DiBPNet, dl: DataLoader, opt,
               global_ep: int, gate_ema_tensor=None,
               proto_buffer: SubjectCPrototype = None,
               adv_params=None, grl_alpha: float = 1.0) -> dict:
    mdl.train()
    if proto_buffer is not None: proto_buffer.train_mode()

    for p in mdl.wk.parameters(): p.requires_grad_(global_ep >= TRAIN.wk_freeze_ep)
    freeze_direct = _should_freeze_direct(global_ep)
    for p in mdl.direct_bp_params(): p.requires_grad_(not freeze_direct)

    lam_wk      = _lam_wkode(global_ep)
    lam_wk_bp   = _lam_wk_bp(global_ep)
    lam_c_proto = _lam_c_proto(global_ep)
    lam_qa      = LW.qanchor * (1.0 - 0.5 * global_ep / max(_total_epochs() - 1, 1))
    ste_a       = _ste_alpha(global_ep)
    g_floor     = _gate_floor(global_ep)
    wk_floor    = _wk_gate_floor(global_ep)
    alpha       = _hybrid_alpha(global_ep)

    totals = {k: 0.0 for k in ['loss', 'gate', 'c_mean', 'c_std', 'c_satfrac',
                                 'c_proto', 'c_slow_spread', 'c_slow_prior',
                                 'c_slow_mean', 'c_slow_std', 'hsic', 'anticollapse',
                                 'bal', 'l_ens', 'l_dir', 'l_wk', 'l_wave', 'l_lat',
                                 'adv']}
    ns = 0

    try:
        for batch in dl:
            bx, bq, babp, bbp, bht, bsubj, _, _, bbase = _to_device(batch)
            opt.zero_grad()
            l = _run_batch(mdl, bx, bq, babp, bbp, bht, bsubj, bbase,
                           hybrid_alpha=alpha, lam_wk_ode=lam_wk, lam_qanchor=lam_qa,
                           gate_floor=g_floor, lam_wk_bp=lam_wk_bp,
                           global_ep=global_ep, ste_alpha=ste_a,
                           gate_ema=gate_ema_tensor, wk_gate_floor=wk_floor,
                           proto_buffer=proto_buffer, grl_alpha=grl_alpha)
            total, bal = _total_loss(l, mdl, lam_wk, lam_qa, lam_wk_bp, global_ep, lam_c_proto)

            if l.get('adv_logit') is not None and adv_params is not None:
                l_adv = LW.adv * F.cross_entropy(l['adv_logit'], bsubj % mdl._adv_classes)
                total = total + l_adv
                totals['adv'] += l_adv.item()

            total.backward()
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), TRAIN.grad_clip_norm)
            opt.step()

            totals['loss']         += l['ens_mmhg'].item(); totals['bal'] += bal
            totals['gate']         += l['gate_mean']
            totals['c_mean']       += l['c_mean']; totals['c_std'] += l['c_std']
            totals['c_satfrac']    += l['c_sat_frac']
            totals['c_proto']      += float(l['c_proto'].item())
            totals['c_slow_spread']+= float(l['c_slow_spread'].item())
            totals['c_slow_prior'] += float(l['c_slow_prior'].item())
            totals['c_slow_mean']  += l['c_slow_mean']; totals['c_slow_std'] += l['c_slow_std']
            totals['hsic']         += float(l['hsic'].item())
            totals['anticollapse'] += float(l['anticollapse'].item())
            totals['l_ens']        += float(l['ens_mmhg'].item())
            totals['l_dir']        += float(l['dir_mmhg'].item()) + float(l['dir_rank'].item()) + float(l['dir_ext'].item())
            totals['l_wk']         += float(l['wkode'].item()) + float(l['wksprd_R'].item()) + float(l['c_sat'].item())
            totals['l_wave']       += float(l['mse'].item()) + float(l['wav_bp'].item()) + float(l['bpaux'].item())
            totals['l_lat']        += float(l['hsic'].item()) + float(l['anticollapse'].item()) + float(l['cont'].item())
            ns += 1

            if gate_ema_tensor is not None:
                with torch.no_grad():
                    gate_ema_tensor.mul_(TRAIN.gate_ema_decay).add_(
                        l['gate_val'].mean(0, keepdim=True) * (1.0 - TRAIN.gate_ema_decay))
    finally:
        if freeze_direct:
            for p in mdl.direct_bp_params(): p.requires_grad_(True)
        if proto_buffer is not None: proto_buffer.eval_mode()

    return {k: v / max(ns, 1) for k, v in totals.items()} | {
        'phase': _curriculum_phase(global_ep),
        'lam_hsic': _lam_hsic(global_ep),
        'freeze_direct': freeze_direct,
    }


def _log_epoch(phase_tag: str, ep: int, em: dict, diag: dict):
    corr_pen = max(0.0, diag.get('czz', 0.0)) * TRAIN.ckpt_corr_weight if TRAIN.ckpt_corr_weight > 0 else 0.0
    score_str = f" score={diag.get('sbp_mae', 0) + corr_pen:.3f}" if corr_pen > 0 else ""
    freeze_tag = " [D-FROZEN]" if em['freeze_direct'] else ""
    print(f"  {phase_tag} ep{ep+1:3d}{freeze_tag} "
          f"| SBP={diag.get('sbp_mae',0):.2f} DBP={diag.get('dbp_mae',0):.2f} AUC={diag.get('auc',0):.3f} "
          f"| L_ens={em.get('l_ens',float('nan')):.4f} L_dir={em.get('l_dir',float('nan')):.4f} "
          f"L_wk={em.get('l_wk',float('nan')):.4f} L_wave={em.get('l_wave',float('nan')):.4f} "
          f"L_lat={em.get('l_lat',float('nan')):.4f} "
          f"| HSIC={em.get('lam_hsic',0):.3f} bal={em.get('bal',1):.2f} "
          f"| corr={diag.get('czz',0):+.3f} ICC={diag.get('icc_cs',0):.3f} "
          f"cs={diag.get('c_slow_mean',0):.2f}±{diag.get('c_slow_std',0):.3f}{score_str}", flush=True)


def _record_ep1(hist, global_ep, em, adv=float("nan")):
    hist["ep_global"].append(global_ep)
    hist["train_loss"].append(float(em.get("loss", float("nan"))))
    hist["gate_mean"].append(float(em.get("gate", float("nan"))))
    hist["freeze_direct"].append(int(em.get("freeze_direct", 0)))
    hist["lam_hsic_ep1"].append(float(em.get("lam_hsic", float("nan"))))
    hist["phase_ep1"].append(em.get("phase", "?"))
    hist["adv_loss"].append(adv)


def _record_ep10(hist, global_ep, ms, md, auc, diag, em):
    hist["ep10_global"].append(global_ep)
    for k, v in [("sbp_mae", ms), ("dbp_mae", md), ("auc", auc),
                 ("corr_zm_zh", diag['czz']), ("icc_cs", diag['icc_cs']),
                 ("vb_vw_cs", diag['vb_vw_cs']), ("c_d_cs_h2n", diag['c_d_cs_h2n']),
                 ("c_slow_mean", diag['c_slow_mean']), ("c_slow_std", diag['c_slow_std']),
                 ("c_proto_loss", em['c_proto']), ("c_slow_spread_loss", em['c_slow_spread']),
                 ("c_slow_prior_loss", em['c_slow_prior']), ("lam_hsic", em['lam_hsic']),
                 ("phase", em['phase'])]:
        hist[k].append(v)


def _empty_hist() -> dict:
    ep1_keys  = ["ep_global", "train_loss", "gate_mean", "freeze_direct", "lam_hsic_ep1", "phase_ep1", "adv_loss"]
    ep10_keys = ["ep10_global", "sbp_mae", "dbp_mae", "auc", "corr_zm_zh", "icc_cs",
                 "vb_vw_cs", "c_d_cs_h2n", "c_slow_mean", "c_slow_std",
                 "c_proto_loss", "c_slow_spread_loss", "c_slow_prior_loss", "lam_hsic", "phase"]
    return {k: [] for k in ep1_keys + ep10_keys}


def _two_phase_train(
    mdl: DiBPNet,
    ckpt_tag: str,
    hist: dict,
    *,
    adv_params=None,
    alpha_fn: Callable = None,
    print_prefix: str = "",
) -> tuple:
    proto           = SubjectCPrototype()
    gate_ema_tensor = torch.full((1, 1), 0.80, device=DEVICE)
    best_score      = float("inf")
    global_ep       = 0
    dl              = make_train_dataloader()
    tag = lambda phase: f"{ckpt_tag}_p{phase}_best"

    # ── Phase 1 ───────────────────────────────────────────────────────────────
    opt = _make_optimizer(mdl)
    sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=TRAIN.cosine_T0, eta_min=TRAIN.cosine_eta_min)
    print(f"  {print_prefix}── Phase 1 ({TRAIN.epochs_p1} epochs) ──────────────────────────────────", flush=True)
    _t_p1 = time.time()

    for ep in range(TRAIN.epochs_p1):
        grl_alpha = alpha_fn(global_ep) if alpha_fn else 1.0
        em = _run_epoch(mdl, dl, opt, global_ep, gate_ema_tensor, proto, adv_params, grl_alpha=grl_alpha)
        sch.step()
        global_ep += 1

        _record_ep1(hist, global_ep, em, adv=grl_alpha if alpha_fn else float("nan"))
        if (ep + 1) % 10 == 0:
            proto.eval_mode()
            ms, md, auc = quick_metrics(mdl, SPLITS.val)
            diag = quick_head_diag(mdl)
            proto.train_mode()
            _record_ep10(hist, global_ep, ms, md, auc, diag, em)
            diag.update(sbp_mae=ms, dbp_mae=md, auc=auc)
            _log_epoch(f"{print_prefix}P1", ep, em, diag)
            score = ms + max(0.0, diag.get('czz', 0.0)) * TRAIN.ckpt_corr_weight
            if score < best_score - 0.05:
                best_score = score; save_checkpoint(mdl, hist, tag(1))

    save_checkpoint(mdl, hist, f"{ckpt_tag}_p1")
    print(f"  ⏱  Phase 1 done in {(time.time() - _t_p1)/60:.1f} min", flush=True)

    # ── Phase 2 ───────────────────────────────────────────────────────────────
    mdl.unfreeze()
    opt = _make_optimizer(mdl)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TRAIN.epochs_p2)
    best_score = float("inf"); no_impr = 0
    print(f"\n  {print_prefix}── Phase 2 ({TRAIN.epochs_p2} epochs) ──────────────────────────────────", flush=True)
    _t_p2 = time.time()

    for ep in range(TRAIN.epochs_p2):
        in_awk = TRAIN.awakening_ep <= ep < TRAIN.awakening_ep + TRAIN.awakening_warmup
        if in_awk:
            opt.param_groups[0]["lr"] = warmup_lr(ep, TRAIN.awakening_ep, TRAIN.awakening_warmup, TRAIN.enc_lr, TRAIN.enc_lr_target)
            opt.param_groups[1]["lr"] = warmup_lr(ep, TRAIN.awakening_ep, TRAIN.awakening_warmup, TRAIN.head_lr, TRAIN.head_lr_target)

        grl_alpha = alpha_fn(global_ep) if alpha_fn else 1.0
        em = _run_epoch(mdl, dl, opt, global_ep, gate_ema_tensor, proto, adv_params, grl_alpha=grl_alpha)
        if not in_awk: sch.step()
        global_ep += 1

        _record_ep1(hist, global_ep, em, adv=grl_alpha if alpha_fn else float("nan"))
        if (ep + 1) % 10 == 0:
            proto.eval_mode()
            ms, md, auc = quick_metrics(mdl, SPLITS.val)
            diag = quick_head_diag(mdl)
            proto.train_mode()
            _record_ep10(hist, global_ep, ms, md, auc, diag, em)
            diag.update(sbp_mae=ms, dbp_mae=md, auc=auc)
            _log_epoch(f"{print_prefix}P2", ep, em, diag)
            score = ms + max(0.0, diag.get('czz', 0.0)) * TRAIN.ckpt_corr_weight
            if score < best_score - 0.05:
                best_score = score; no_impr = 0; save_checkpoint(mdl, hist, f"{ckpt_tag}_best")
            else:
                no_impr += 1
            if no_impr >= 5 and ep >= 30:
                print(f"  Early stop (best score={best_score:.3f})", flush=True); break

    print(f"  ⏱  Phase 2 done in {(time.time() - _t_p2)/60:.1f} min", flush=True)
    if _ckpt_exists(f"{ckpt_tag}_best"):
        mdl, _ = load_checkpoint(f"{ckpt_tag}_best")
        save_checkpoint(mdl, hist, f"{ckpt_tag}_best")
    return mdl, hist

### Diagnostics

Per-epoch evaluation callbacks and metric logging.

In [ ]:
# DIAGNOSTICS
# ══════════════════════════════════════════════════════════════════════════════

def _blend_htn_scores(htn_logit, aux_logit):
    """Blend main + auxiliary HTN logits into one score for AUC (matches training loss weighting)."""
    return LW.htn_main_weight * htn_logit + LW.htn_aux_weight * aux_logit


def quick_metrics(mdl, split=None, eval_mask=None):
    if split is None: split = SPLITS.val
    mdl.eval()
    dl = DataLoader(_make_tensor_dataset(split), batch_size=ARCH.batch, shuffle=False, num_workers=0)
    preds, trues, sc_all, yt_cls = [], [], [], []
    
    with torch.no_grad():
        for batch in dl:
            bx, bq, _, bbp, bht, bsubj, _, _, bbase = _to_device(batch)
            q_c = bq[:, ARCH.n_beats // 2]
            if q_c.dim() == 3: q_c = q_c.squeeze(1)
            out = mdl(bx, bsubj, bbase, q_c, 0.0)
            
            preds.append(out[12].cpu().numpy())  # bp_ens
            trues.append(bbp.cpu().numpy())
            sc_all.append(_blend_htn_scores(out[6], out[13]).cpu().numpy())
            yt_cls.append(bht.cpu().numpy())

    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    sbp_mae = float(np.abs(preds[:, 0] - trues[:, 0]).mean())
    dbp_mae = float(np.abs(preds[:, 1] - trues[:, 1]).mean())

    yt_cls = np.concatenate(yt_cls)
    sc_all = np.concatenate(sc_all)
    if eval_mask is not None:
        yt_cls = yt_cls[eval_mask]
        sc_all = sc_all[eval_mask]
    auc = float(roc_auc_score(yt_cls, sc_all)) if len(np.unique(yt_cls)) > 1 else float("nan")

    return sbp_mae, dbp_mae, auc


def quick_head_diag(mdl: DiBPNet, include_subj_id_acc: bool = False) -> dict:
    dl = make_val_dataloader(); mdl.eval()
    pop_sig = mdl.pop_sig.item(); pop_mu = mdl.pop_mu.item()
    acc = {k: [] for k in ['w','d','t','R','C','RC','c_slow','main','aux','gate','zm','zh','subj']}
    with torch.no_grad():
        for batch in dl:
            bx, bq, _, bbp, _, bsubj, _, _, bbase = _to_device(batch)
            q_c = bq[:, ARCH.n_beats // 2]
            if q_c.dim() == 3: q_c = q_c.squeeze(1)
            out = mdl(bx, bsubj, bbase, q_c, 0.0)
            for k, v in [('w', out[11].cpu().numpy()), ('d', out[10].cpu().numpy()),
                         ('t', bbp.cpu().numpy()), ('R', out[7].cpu().numpy()),
                         ('C', out[8].cpu().numpy()), ('RC', out[19].cpu().numpy()),
                         ('c_slow', out[20].cpu().numpy()), ('main', out[6].cpu().numpy()),
                         ('aux', out[13].cpu().numpy()), ('gate', out[14].cpu().numpy()),
                         ('zm', out[18].cpu().numpy()), ('zh', out[4].cpu().numpy()),
                         ('subj', bsubj.cpu().numpy())]:
                acc[k].append(v)
    w = np.concatenate(acc['w']); d = np.concatenate(acc['d']); t = np.concatenate(acc['t'])
    R = np.concatenate(acc['R']); C = np.concatenate(acc['C'])
    c_slow   = np.concatenate(acc['c_slow']); gate_arr = np.concatenate(acc['gate']).squeeze()
    zm = np.concatenate(acc['zm']); zh = np.concatenate(acc['zh']); subj_arr = np.concatenate(acc['subj'])
    icc_cs, vb_vw_cs = compute_icc(c_slow, subj_arr)
    s_cs, s_st = subj_mean_cs(c_slow, t[:, 0], subj_arr, t[:, 1])
    result = dict(
        wm=float(np.abs(w[:, 0] - t[:, 0]).mean()), dm=float(np.abs(d[:, 0] - t[:, 0]).mean()),
        ds=float(np.std(d[:, 0])), ws=float(np.std(w[:, 0])),
        rs=float(np.std(R)), cs_wk=float(np.std(C)), gm=float(gate_arr.mean()),
        rcc=float(np.corrcoef(R, C)[0, 1]),
        czz=corr_zm_zh_fn(zm, zh),
        icc_cs=icc_cs, vb_vw_cs=vb_vw_cs,
        c_d_cs_h2n=cohens_d(s_cs[s_st == 4], s_cs[s_st == 1]),
        c_slow_mean=float(c_slow.mean()), c_slow_std=float(c_slow.std()),
        c_sat_frac=float((C > TRAIN.c_sat_thresh).mean()),
        subj_id_acc_zh=float("nan"), subj_id_acc_zm=float("nan"), subj_id_acc_cs=float("nan"))
    return result

### Checkpoints and History

Saving and loading checkpoints and training history.

In [ ]:
# CHECKPOINTS + HISTORY
# ══════════════════════════════════════════════════════════════════════════════

def _ckpt_path(tag: str) -> str:  return os.path.join(CKPT_DIR, f"{tag}.pt")
def _ckpt_exists(tag: str) -> bool: return os.path.exists(_ckpt_path(tag))

def save_checkpoint(mdl: DiBPNet, hist: dict, tag: str, global_ep=None):
    payload = {
        "state": mdl.state_dict(), 
        "history": hist,
        "torch_rng": torch.get_rng_state(),
        "numpy_rng": np.random.get_state(),
        "python_rng": random.getstate()
    }
    
    if torch.cuda.is_available():
        payload["cuda_rng"] = torch.cuda.get_rng_state()
        
    if global_ep is not None: 
        payload["global_epoch"] = global_ep
        
    torch.save(payload, _ckpt_path(tag))
    print(f"   Checkpoint saved → {tag}.pt", flush=True)


def load_checkpoint(tag: str) -> tuple:
    path = _ckpt_path(tag)
    if not os.path.exists(path): return None, {}
    
    d = torch.load(path, map_location=DEVICE, weights_only=False)
    
    mdl = DiBPNet().to(DEVICE)
    if SPLITS is not None: mdl._set_stats_from_splits(SPLITS)
    
    ms  = mdl.state_dict(); filtered = {}
    for k, v in d["state"].items():
        if k not in ms: continue
        if v.shape == ms[k].shape: 
            filtered[k] = v
        elif v.dim() == 2 and ms[k].dim() == 2 and v.shape[1] == ms[k].shape[1] and "emb" in k:
            w = ms[k].clone(); n = min(v.shape[0], ms[k].shape[0]); w[:n] = v[:n]; filtered[k] = w
    mdl.load_state_dict(filtered, strict=False)
    
    try:
        if "torch_rng" in d:
            state = d["torch_rng"].cpu().to(torch.uint8)
            torch.set_rng_state(state)
            
        if "numpy_rng" in d: 
            np.random.set_state(d["numpy_rng"])
            
        if "python_rng" in d: 
            random.setstate(d["python_rng"])
            
        if torch.cuda.is_available() and "cuda_rng" in d:
            cuda_state = d["cuda_rng"].cpu().to(torch.uint8)
            torch.cuda.set_rng_state(cuda_state)
            
    except Exception as e:
        print(f"  ⚠️  Warning: Could not restore RNG state: {e}")
        
    return mdl, d.get("history", {})


def save_hist_csv(hist: dict, tag: str = "v192") -> str:
    ep1_keys  = ["ep_global", "train_loss", "gate_mean", "freeze_direct", "lam_hsic_ep1", "phase_ep1", "adv_loss"]
    ep10_keys = ["ep10_global", "sbp_mae", "dbp_mae", "auc", "corr_zm_zh", "icc_cs",
                 "vb_vw_cs", "c_d_cs_h2n", "c_slow_mean", "c_slow_std",
                 "c_proto_loss", "c_slow_spread_loss", "c_slow_prior_loss", "lam_hsic", "phase"]
    def _build(keys):
        data = {k: hist[k] for k in keys if k in hist and hist[k]}
        if not data: return pd.DataFrame(columns=keys)
        n = min(len(v) for v in data.values())
        for k in keys:
            if k not in data: data[k] = [("" if k in ("phase_ep1","phase") else float("nan"))] * n
        return pd.DataFrame({k: data[k][:n] for k in keys if k in data})
    p1  = os.path.join(HIST_DIR, f"{tag}_ep1.csv")
    p10 = os.path.join(HIST_DIR, f"{tag}_ep10.csv")
    df1  = _build(ep1_keys);  df1.to_csv(p1,   index=False)
    df10 = _build(ep10_keys); df10.to_csv(p10, index=False)
    print(f"  [HIST] Saved → {p1} ({len(df1)} rows)")
    print(f"  [HIST] Saved → {p10} ({len(df10)} rows)")
    return p1

### Baseline Models

GRU, ResNet, and Transformer baselines for the control experiment.

In [ ]:
# BASELINE MODELS
# ══════════════════════════════════════════════════════════════════════════════

def _apply_augmentation(x_win: torch.Tensor) -> torch.Tensor:
    B, K, C, L = x_win.shape
    amp = torch.rand(B, K, 1, 1, device=x_win.device) * 0.4 + 0.8
    x_win = x_win * amp + torch.randn_like(x_win) * 0.02
    shift = torch.randint(-5, 6, (B,), device=x_win.device)
    for b in range(B): x_win[b] = torch.roll(x_win[b], shifts=int(shift[b].item()), dims=-1)
    return x_win

class _ConvBlock(nn.Module):
    def __init__(self, in_ch=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, 32, 5, padding=2), nn.GELU(), nn.BatchNorm1d(32), nn.MaxPool1d(2),
            nn.Conv1d(32,    64, 5, padding=2), nn.GELU(), nn.BatchNorm1d(64), nn.MaxPool1d(2),
            nn.Conv1d(64,   128, 3, padding=1), nn.GELU(), nn.BatchNorm1d(128))
    def forward(self, x): return self.conv(x)

class SingleStreamGRU(nn.Module):
    def __init__(self, latent=ARCH.latent_dim, n_beats=ARCH.n_beats, dropout=0.15):
        super().__init__()
        self.conv       = _ConvBlock(in_ch=1)
        self.bottleneck = nn.Sequential(nn.Linear(128 * 32, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 128), nn.GELU())
        self.gru  = nn.GRU(128, 128, num_layers=2, batch_first=True, dropout=dropout)
        self.pool = nn.Sequential(nn.Linear(128, 32), nn.Tanh(), nn.Linear(32, 1))
        self.bp_head = nn.Sequential(nn.Linear(128 + 1, 128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.10), nn.Linear(64, 2))
        for buf, val in [("sbp_mu",0.),("sbp_sig",1.),("dbp_mu",0.),("dbp_sig",1.)]:
            self.register_buffer(buf, torch.tensor(val))
    def _set_stats(self):
        self.sbp_mu.fill_(float(SPLITS.sbp_mu)); self.sbp_sig.fill_(float(SPLITS.sbp_sig))
        self.dbp_mu.fill_(float(SPLITS.dbp_mu)); self.dbp_sig.fill_(float(SPLITS.dbp_sig))
    def n_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)
    def forward(self, x_win, bsubj=None, bbase=None, **kwargs):
        if x_win.dim() == 3: x_win = x_win.unsqueeze(2)
        B, T, C, L = x_win.shape
        if self.training: x_win = _apply_augmentation(x_win)
        h    = self.conv(x_win.reshape(B * T, C, L))
        h    = self.bottleneck(h.reshape(B * T, -1)).reshape(B, T, 128)
        h, _ = self.gru(h); z = (h * torch.softmax(self.pool(h), dim=1)).sum(1)
        bf = (bbase / 100.0).unsqueeze(-1) if bbase is not None else torch.zeros(B, 1, device=z.device)
        bp_n = self.bp_head(torch.cat([z, bf], dim=-1))
        return torch.stack([bp_n[:, 0] * self.sbp_sig + self.sbp_mu, bp_n[:, 1] * self.dbp_sig + self.dbp_mu], dim=1)

class _ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, kernel=7):
        super().__init__()
        pad = kernel // 2
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel, stride=stride, padding=pad, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel, padding=pad, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.down  = (nn.Sequential(nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False), nn.BatchNorm1d(out_ch))
                      if stride != 1 or in_ch != out_ch else None)
    def forward(self, x):
        out = F.gelu(self.bn1(self.conv1(x))); out = self.bn2(self.conv2(out))
        return F.gelu(out + (self.down(x) if self.down is not None else x))

class ResNet1D(nn.Module):
    def __init__(self, in_ch=1, dropout=0.15):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(in_ch, 64, 7, stride=2, padding=3, bias=False), nn.BatchNorm1d(64), nn.GELU(), nn.MaxPool1d(3, stride=2, padding=1))
        def _layer(ic, oc, n, s=2): return nn.Sequential(_ResBlock1D(ic, oc, stride=s), *[_ResBlock1D(oc, oc) for _ in range(1, n)])
        self.layer1 = _layer(64, 128, 2); self.layer2 = _layer(128, 256, 2)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.beat_agg = nn.Sequential(nn.Linear(256, 128), nn.GELU(), nn.Dropout(dropout))
        self.bp_head  = nn.Sequential(nn.Linear(128 + 1, 64), nn.GELU(), nn.Dropout(dropout), nn.Linear(64, 2))
        for buf, val in [("sbp_mu",0.),("sbp_sig",1.),("dbp_mu",0.),("dbp_sig",1.)]:
            self.register_buffer(buf, torch.tensor(val))
    def _set_stats(self):
        self.sbp_mu.fill_(float(SPLITS.sbp_mu)); self.sbp_sig.fill_(float(SPLITS.sbp_sig))
        self.dbp_mu.fill_(float(SPLITS.dbp_mu)); self.dbp_sig.fill_(float(SPLITS.dbp_sig))
    def n_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)
    def forward(self, x_win, bsubj=None, bbase=None, **kwargs):
        if x_win.dim() == 3: x_win = x_win.unsqueeze(2)
        B, T, C, L = x_win.shape
        if self.training: x_win = _apply_augmentation(x_win)
        x  = self.stem(x_win.reshape(B * T, C, L))
        x  = self.layer2(self.layer1(x))
        x  = self.gap(x).squeeze(-1).reshape(B, T, 256).mean(dim=1)
        z  = self.beat_agg(x)
        bf = (bbase / 100.0).unsqueeze(-1) if bbase is not None else torch.zeros(B, 1, device=z.device)
        bp_n = self.bp_head(torch.cat([z, bf], dim=-1))
        return torch.stack([bp_n[:, 0] * self.sbp_sig + self.sbp_mu, bp_n[:, 1] * self.dbp_sig + self.dbp_mu], dim=1)

class _PatchEmbedding(nn.Module):
    def __init__(self, patch_len, d_model, in_ch=1):
        super().__init__()
        self.patch_len = patch_len; self.proj = nn.Linear(patch_len * in_ch, d_model); self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        B, C, T = x.shape; n = T // self.patch_len
        x = x[:, :, :n * self.patch_len].reshape(B, C, n, self.patch_len)
        return self.norm(self.proj(x.permute(0, 2, 1, 3).reshape(B, n, -1)))

class TransformerBaseline(nn.Module):
    def __init__(self, n_beats=ARCH.n_beats, seq_len=ARCH.seq_len, patch_len=16, d_model=128,
                 n_heads=4, n_layers=3, d_ff=256, dropout=0.15):
        super().__init__()
        self.d_model = d_model; self.patch_len = patch_len
        self.patch_emb = _PatchEmbedding(patch_len, d_model, in_ch=1)
        max_tokens = n_beats * (seq_len // patch_len) + 1
        self.pos_emb   = nn.Embedding(max_tokens, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
                                               dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.subj_emb    = nn.Embedding(ARCH.embed_size, 8)
        self.subj_proj   = nn.Linear(d_model, 8, bias=False)
        self._subj_cache: dict = {}
        self.head = nn.Sequential(nn.Linear(d_model + 8 + 1, 128), nn.GELU(), nn.Dropout(dropout),
                                   nn.Linear(128, 64), nn.GELU(), nn.Dropout(dropout * 0.5), nn.Linear(64, 2))
        for buf, val in [("sbp_mu",0.),("sbp_sig",1.),("dbp_mu",0.),("dbp_sig",1.)]:
            self.register_buffer(buf, torch.tensor(val))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.head[-1].weight, std=0.01); nn.init.zeros_(self.head[-1].bias)
    def _set_stats(self):
        self.sbp_mu.fill_(float(SPLITS.sbp_mu)); self.sbp_sig.fill_(float(SPLITS.sbp_sig))
        self.dbp_mu.fill_(float(SPLITS.dbp_mu)); self.dbp_sig.fill_(float(SPLITS.dbp_sig))
    def n_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)
    def build_subj_cache(self, ref_X, ref_subj, ref_base, batch=128):
        self.eval(); self._subj_cache = {}; cls_acc = {}; dev = self.pos_emb.weight.device
        with torch.no_grad():
            for i in range(0, len(ref_X), batch):
                sl = slice(i, min(i + batch, len(ref_X)))
                bx = torch.tensor(ref_X[sl]).to(dev); B, K, T = bx.shape
                patches = self.patch_emb(bx.reshape(B * K, 1, T)).reshape(B, K * (T // self.patch_len), self.d_model)
                tokens  = torch.cat([self.cls_token.expand(B, -1, -1), patches], dim=1)
                tokens  = tokens + self.pos_emb(torch.arange(tokens.shape[1], device=dev))
                cls_out = self.transformer(tokens)[:, 0]
                for b_idx, sid in enumerate(ref_subj[sl]):
                    cls_acc.setdefault(int(sid), []).append(cls_out[b_idx].cpu().numpy())
        with torch.no_grad():
            for sid, vecs in cls_acc.items():
                mean_cls = torch.tensor(np.mean(vecs, axis=0), dtype=torch.float32).to(dev)
                self._subj_cache[sid] = self.subj_proj(mean_cls).cpu().numpy()
    def forward(self, x_win, bsubj=None, bbase=None, **kwargs):
        if x_win.dim() == 4: x_win = x_win.squeeze(2)
        B, K, T = x_win.shape
        if self.training:
            x_win = _apply_augmentation(x_win.unsqueeze(2)).squeeze(2)
        patches = self.patch_emb(x_win.reshape(B * K, 1, T)).reshape(B, K * (T // self.patch_len), self.d_model)
        tokens  = torch.cat([self.cls_token.expand(B, -1, -1), patches], dim=1)
        tokens  = tokens + self.pos_emb(torch.arange(tokens.shape[1], device=x_win.device))
        cls_out = self.transformer(tokens)[:, 0]
        dev = x_win.device
        if self.training:
            s_emb = self.subj_emb(bsubj) * (torch.rand_like(self.subj_emb(bsubj)[:, :1]) > 0.30).float()
        else:
            pop_mean = self.subj_emb.weight.mean(0).detach()
            s_emb = torch.stack([torch.tensor(self._subj_cache[int(sid)], dtype=torch.float32, device=dev)
                                  if int(sid) in self._subj_cache else pop_mean
                                  for sid in bsubj.cpu().numpy()], dim=0)
        bf = (bbase / 100.0).unsqueeze(-1) if bbase is not None else torch.zeros(B, 1, device=dev)
        bp_n = self.head(torch.cat([cls_out, s_emb, bf], dim=-1))
        return torch.stack([bp_n[:, 0] * self.sbp_sig + self.sbp_mu, bp_n[:, 1] * self.dbp_sig + self.dbp_mu], dim=1)


# ── SupCon (Supervised Contrastive) baseline ─────────────────────────────────
class SupConLoss(nn.Module):
    """Supervised Contrastive Loss (Khosla et al., NeurIPS 2020), applied to a
    projection of the pooled latent to pull same-subject embeddings together.
    Reference: https://arxiv.org/abs/2004.11362"""
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        B = features.shape[0]
        if B < 2:
            return features.new_zeros(1).squeeze()
        features = F.normalize(features, dim=1)
        sim      = torch.matmul(features, features.T) / self.temperature
        labels   = labels.view(-1, 1)
        mask_eq  = (labels == labels.T).float(); mask_eq.fill_diagonal_(0.0)
        if mask_eq.sum() == 0:
            return features.new_zeros(1).squeeze()
        sim       = sim - sim.max(dim=1, keepdim=True).values.detach()
        eye_mask  = 1 - torch.eye(B, device=features.device)
        log_denom = torch.log((torch.exp(sim) * eye_mask).sum(dim=1, keepdim=True) + 1e-8)
        log_prob  = sim - log_denom
        mean_log_prob = (mask_eq * log_prob).sum(dim=1) / (mask_eq.sum(dim=1) + 1e-8)
        return -mean_log_prob.mean()


class SupConBPModel(nn.Module):
    """Pure-PPG SupCon baseline: CNN+GRU encoder → BP head + SupCon projection
    head. Conforms to the baseline forward interface (x_win, bsubj, bbase) and
    predicts SBP/DBP in mmHg directly. `return_supcon=True` also returns the
    projection features + pooled latent (used only during contrastive training)."""
    def __init__(self, latent=ARCH.latent_dim, seq_len=ARCH.seq_len, dropout=0.15):
        super().__init__()
        self.beat_cnn = nn.Sequential(
            nn.Conv1d(1, 32, 5, padding=2),  nn.GELU(), nn.BatchNorm1d(32),  nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2), nn.GELU(), nn.BatchNorm1d(64),  nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1), nn.GELU(), nn.BatchNorm1d(128),
        )
        self.beat_flat = nn.Sequential(
            nn.Linear(128 * (seq_len // 4), 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, latent), nn.GELU(),
        )
        self.gru       = nn.GRU(latent, latent, num_layers=2, batch_first=True, dropout=dropout)
        self.attn_w    = nn.Sequential(nn.Linear(latent, 32), nn.Tanh(), nn.Linear(32, 1))
        self.pool_proj = nn.Sequential(nn.Linear(latent * 2, latent), nn.GELU(), nn.Dropout(dropout))
        self.bp_head = nn.Sequential(
            nn.Linear(latent + 1, 128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 64),         nn.GELU(), nn.Dropout(dropout * 0.5),
            nn.Linear(64, 2),
        )
        self.supcon_proj = nn.Sequential(nn.Linear(latent, 128), nn.GELU(), nn.Linear(128, 64))

    def n_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def _encode(self, x_beats: torch.Tensor) -> torch.Tensor:
        B, K, C, T = x_beats.shape
        h = self.beat_flat(self.beat_cnn(x_beats.reshape(B * K, C, T)).reshape(B * K, -1))
        return h.reshape(B, K, -1)

    def forward(self, x_win, bsubj=None, bbase=None, return_supcon=False, **kwargs):
        if x_win.dim() == 3: x_win = x_win.unsqueeze(2)          # (B,K,T) -> (B,K,1,T)
        gru_out, _ = self.gru(self._encode(x_win))
        h_attn = (torch.softmax(self.attn_w(gru_out), dim=1) * gru_out).sum(dim=1)
        h_pool = self.pool_proj(torch.cat([h_attn, gru_out.max(dim=1).values], dim=1))
        B  = x_win.shape[0]
        bf = bbase.unsqueeze(-1) if bbase is not None else torch.zeros(B, 1, device=h_pool.device)
        bp_pred = self.bp_head(torch.cat([h_pool, bf], dim=-1))
        if return_supcon:
            return bp_pred, self.supcon_proj(h_pool), h_pool
        return bp_pred


def _baseline_loss(pred, true, sbp_sig, dbp_sig):
    err = pred - true; h = 0.5 * err.abs() + 0.5 * err ** 2
    return LW.sbp * (h[:, 0] / sbp_sig).mean() + (h[:, 1] / dbp_sig).mean()

def _build_subj_cache_baseline(mdl, X, subj_arr, base_arr):
    if not hasattr(mdl, 'build_subj_cache'): return
    ref_X, ref_subj, ref_base = [], [], []
    for s in np.unique(subj_arr):
        m = np.where(subj_arr == s)[0]
        if len(m) < CALIB.min_beats: continue
        idx = m[:min(CALIB.ref_beats, len(m))]
        ref_X.append(X[idx]); ref_subj.append(subj_arr[idx]); ref_base.append(base_arr[idx])
    if ref_X:
        mdl.build_subj_cache(np.concatenate(ref_X), np.concatenate(ref_subj), np.concatenate(ref_base))

_BL_CKPT = {"gru": "checkpoints_gru", "resnet": "checkpoints_resnet", "transformer": "checkpoints_transformer", "supcon": "checkpoints_supcon"}

def run_baseline_model(model_type: str = "gru", force_retrain: bool = False):
    m_type = model_type.lower()
    ckpt_dir = os.path.join(OUT_DIR, _BL_CKPT[m_type]); os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, f"baseline_{m_type}_best.pt")

    factory = {"gru": SingleStreamGRU, "resnet": ResNet1D, "transformer": TransformerBaseline}
    mdl = factory[m_type]().to(DEVICE); mdl._set_stats()
    EPOCHS = TRAIN.epochs_p1 + TRAIN.epochs_p2
    LR     = 3e-4 if m_type == "transformer" else TRAIN.head_lr

    print(f"\n{'='*85}\n  BASELINE : {m_type.upper():12s} | Params : {mdl.n_params():,}")
    print(f"  Epochs : {EPOCHS}  LR : {LR}  Device : {DEVICE}\n{'='*85}")

    opt = torch.optim.AdamW(mdl.parameters(), lr=LR, weight_decay=TRAIN.wd)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

    if not force_retrain and os.path.exists(ckpt_path):
        mdl.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=False)["state"])
        print(f"  Model loaded ← {ckpt_path}")
    else:
        best_mae = float("inf")
        bl_hist = {"ep": [], "train_loss": [], "sbp_mae_ep10": [], "dbp_mae_ep10": [], "ep10": []}
        for ep in range(1, EPOCHS + 1):
            mdl.train(); total_loss = 0.0; ns = 0
            for batch in make_train_dataloader():
                bx, _, _, bbp, _, bsubj, _, _, bbase = _to_device(batch)
                opt.zero_grad()
                loss = _baseline_loss(mdl(bx, bsubj, bbase), bbp, mdl.sbp_sig.item(), mdl.dbp_sig.item())
                loss.backward(); nn.utils.clip_grad_norm_(mdl.parameters(), 5.0); opt.step()
                total_loss += loss.item(); ns += 1
            sch.step()
            bl_hist["ep"].append(ep); bl_hist["train_loss"].append(total_loss / max(ns, 1))
            if ep % 10 == 0:
                _build_subj_cache_baseline(mdl, SPLITS.val.X, SPLITS.val.subj, SPLITS.val.base)
                
                mdl.eval(); raw_va = []
                with torch.no_grad():
                    for i in range(0, len(SPLITS.val.X), ARCH.batch):
                        sl = slice(i, min(i + ARCH.batch, len(SPLITS.val.X)))
                        raw_va.append(mdl(torch.tensor(SPLITS.val.X[sl]).to(DEVICE), 
                                          torch.tensor(SPLITS.val.subj[sl], dtype=torch.long).to(DEVICE),
                                          torch.tensor(SPLITS.val.base[sl]).to(DEVICE)).cpu().numpy())
                raw_va = np.concatenate(raw_va)
                sbp_mae = float(np.mean(np.abs(raw_va[:, 0] - SPLITS.val.Ybp[:, 0])))
                dbp_mae = float(np.mean(np.abs(raw_va[:, 1] - SPLITS.val.Ybp[:, 1])))
                bl_hist["ep10"].append(ep); bl_hist["sbp_mae_ep10"].append(sbp_mae); bl_hist["dbp_mae_ep10"].append(dbp_mae)
                tag_str = ""
                if sbp_mae < best_mae: best_mae = sbp_mae; torch.save({"state": mdl.state_dict()}, ckpt_path); tag_str = " ← best"
                print(f"  ep {ep:3d} | loss={total_loss/ns:.4f} | SBP={sbp_mae:.2f}  DBP={dbp_mae:.2f}{tag_str}")
        mdl.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=False)["state"])
        def _trim(d): n = min(len(v) for v in d.values()) if d else 0; return {k: v[:n] for k, v in d.items()}
        pd.DataFrame(_trim({"ep": bl_hist["ep"], "train_loss": bl_hist["train_loss"]})).to_csv(os.path.join(HIST_DIR, f"baseline_{m_type}_ep1.csv"), index=False)
        pd.DataFrame(_trim({"ep10": bl_hist["ep10"], "sbp_mae_ep10": bl_hist["sbp_mae_ep10"], "dbp_mae_ep10": bl_hist["dbp_mae_ep10"]})).to_csv(os.path.join(HIST_DIR, f"baseline_{m_type}_ep10.csv"), index=False)

    return mdl


def _supcon_bp_loader(split, batch_size, shuffle):
    """Single-channel (PPG-only) loader for SupCon baseline training."""
    from torch.utils.data import TensorDataset, DataLoader
    ds = TensorDataset(
        torch.tensor(split.X,         dtype=torch.float32),
        torch.tensor(split.subj,      dtype=torch.long),
        torch.tensor(split.base,      dtype=torch.float32),
        torch.tensor(split.Ybp[:, 0], dtype=torch.float32),
        torch.tensor(split.Ybp[:, 1], dtype=torch.float32),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)


def run_supcon_baseline(epochs=160, lr=1e-3, weight_decay=1e-4,
                        temp_supcon=0.07, lam_supcon=0.10, force_retrain=False):
    """Train (or load) the pure-PPG SupCon baseline. Same checkpoint/history
    contract as run_baseline_model → checkpoints_supcon/baseline_supcon_best.pt;
    L1 BP loss + SupCon contrastive loss on the projection head. best selected
    on raw VAL SBP MAE (like the other baselines)."""
    ckpt_dir  = os.path.join(OUT_DIR, _BL_CKPT["supcon"]); os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, "baseline_supcon_best.pt")

    torch.manual_seed(42); np.random.seed(42)
    mdl = SupConBPModel().to(DEVICE)
    print(f"\n{'='*85}\n  BASELINE : {'SUPCON':12s} | Params : {mdl.n_params():,}")
    print(f"  Epochs : {epochs}  LR : {lr}  Device : {DEVICE}\n{'='*85}")

    if not force_retrain and os.path.exists(ckpt_path):
        mdl.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=False)["state"])
        print(f"  Model loaded ← {ckpt_path}")
        return mdl

    supcon_fn    = SupConLoss(temperature=temp_supcon)
    opt          = torch.optim.Adam(mdl.parameters(), lr=lr, weight_decay=weight_decay)
    sch          = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    train_loader = _supcon_bp_loader(SPLITS.train, ARCH.batch, shuffle=True)

    best_mae = float("inf")
    bl_hist  = {"ep": [], "train_loss": [], "sbp_mae_ep10": [], "dbp_mae_ep10": [], "ep10": []}
    for ep in range(1, epochs + 1):
        mdl.train(); total = 0.0; ns = 0
        for x, subj_ids, base, sbp, dbp in train_loader:
            x = x.to(DEVICE); subj_ids = subj_ids.to(DEVICE); base = base.to(DEVICE)
            bp_true = torch.stack([sbp, dbp], dim=1).to(DEVICE)
            bp_pred, sc_feat, _ = mdl(x, subj_ids, base, return_supcon=True)
            loss = F.l1_loss(bp_pred, bp_true) + lam_supcon * supcon_fn(sc_feat, subj_ids)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(mdl.parameters(), 1.0); opt.step()
            total += loss.item(); ns += 1
        sch.step()
        bl_hist["ep"].append(ep); bl_hist["train_loss"].append(total / max(ns, 1))
        if ep % 10 == 0:
            mdl.eval(); raw_va = []
            with torch.no_grad():
                for i in range(0, len(SPLITS.val.X), ARCH.batch):
                    sl = slice(i, min(i + ARCH.batch, len(SPLITS.val.X)))
                    raw_va.append(mdl(torch.tensor(SPLITS.val.X[sl]).to(DEVICE),
                                      torch.tensor(SPLITS.val.subj[sl], dtype=torch.long).to(DEVICE),
                                      torch.tensor(SPLITS.val.base[sl]).to(DEVICE)).cpu().numpy())
            raw_va  = np.concatenate(raw_va)
            sbp_mae = float(np.mean(np.abs(raw_va[:, 0] - SPLITS.val.Ybp[:, 0])))
            dbp_mae = float(np.mean(np.abs(raw_va[:, 1] - SPLITS.val.Ybp[:, 1])))
            bl_hist["ep10"].append(ep); bl_hist["sbp_mae_ep10"].append(sbp_mae); bl_hist["dbp_mae_ep10"].append(dbp_mae)
            tag = ""
            if sbp_mae < best_mae:
                best_mae = sbp_mae; torch.save({"state": mdl.state_dict()}, ckpt_path); tag = " ← best"
            print(f"  ep {ep:3d} | loss={total/ns:.4f} | SBP={sbp_mae:.2f}  DBP={dbp_mae:.2f}{tag}")
    mdl.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=False)["state"])
    def _trim(d): n = min(len(v) for v in d.values()) if d else 0; return {k: v[:n] for k, v in d.items()}
    pd.DataFrame(_trim({"ep": bl_hist["ep"], "train_loss": bl_hist["train_loss"]})).to_csv(
        os.path.join(HIST_DIR, "baseline_supcon_ep1.csv"), index=False)
    pd.DataFrame(_trim({"ep10": bl_hist["ep10"], "sbp_mae_ep10": bl_hist["sbp_mae_ep10"],
                        "dbp_mae_ep10": bl_hist["dbp_mae_ep10"]})).to_csv(
        os.path.join(HIST_DIR, "baseline_supcon_ep10.csv"), index=False)
    return mdl


### Experiment Runners

Top-level entry point that trains DiBP-Net variants and baselines.

In [ ]:
# EXPERIMENT RUNNERS
# ══════════════════════════════════════════════════════════════════════════════

def run_training() -> tuple:
    global SPLITS
    if SPLITS is None: _load_data()
    if not TRAIN.force_retrain and _ckpt_exists("v192_best"):
        mdl, hist = load_checkpoint("v192_best")
        if mdl is not None:
            ms, md, auc = quick_metrics(mdl, SPLITS.val)
            print(f"  Loaded v192_best — SBP MAE: {ms:.2f}  DBP MAE: {md:.2f}  AUC: {auc:.4f}", flush=True)
            save_hist_csv(hist, tag="v192")
            return mdl, hist
    mdl = DiBPNet().to(DEVICE); mdl._set_stats_from_splits(SPLITS)
    print(f"\n  Model parameters: {mdl.n_params():,}", flush=True)
    hist = _empty_hist()
    mdl, hist = _two_phase_train(mdl, "v192", hist, print_prefix="")
    save_hist_csv(hist, tag="v192")
    return mdl, hist


def run_ablation(label, ckpt_tag, override_lw, override_train, description_lines) -> tuple:
    global SPLITS
    if SPLITS is None: _load_data()

    orig_lw    = {k: getattr(LW, k) for k in override_lw}
    orig_train = {k: getattr(TRAIN, k) for k in override_train}
    for k, v in override_lw.items():    setattr(LW, k, v)
    for k, v in override_train.items(): setattr(TRAIN, k, v)

    print("=" * 70, flush=True)
    print(f"  {label}", flush=True)
    for line in description_lines: print(f"  {line}", flush=True)
    print("=" * 70, flush=True)

    if _ckpt_exists(f"{ckpt_tag}_best"):
        mdl, hist = load_checkpoint(f"{ckpt_tag}_best")
        if mdl is not None:
            mdl._set_stats_from_splits(SPLITS)
            ms, md, auc = quick_metrics(mdl, SPLITS.val)
            print(f"  Loaded {ckpt_tag}_best — SBP MAE: {ms:.2f}  DBP MAE: {md:.2f}  AUC: {auc:.4f}", flush=True)
            save_hist_csv(hist, tag=ckpt_tag)
            for k, v in orig_lw.items():    setattr(LW, k, v)
            for k, v in orig_train.items(): setattr(TRAIN, k, v)
            return mdl, hist or {}

    mdl = DiBPNet().to(DEVICE); mdl._set_stats_from_splits(SPLITS)
    print(f"  Model parameters: {mdl.n_params():,}", flush=True)
    hist = _empty_hist()
    try:  
        mdl, hist = _two_phase_train(mdl, ckpt_tag, hist, print_prefix=f"[{label[:10]}] ")
        save_hist_csv(hist, tag=ckpt_tag)
    finally:
        for k, v in orig_lw.items():    setattr(LW, k, v)
        for k, v in orig_train.items(): setattr(TRAIN, k, v)
    return mdl, hist


def run_grl(mdl_v192_ckpt: str = "v192_best", force_retrain: bool = False) -> tuple:
    global SPLITS
    if SPLITS is None: _load_data()
    os.makedirs(CKPT_DIR_GRL, exist_ok=True)
    ckpt_path = os.path.join(CKPT_DIR_GRL, "v193_best.pt")

    mdl = DiBPNet_v193().to(DEVICE); mdl._set_stats_from_splits(SPLITS)
    v192_path = _ckpt_path(mdl_v192_ckpt)
    if os.path.exists(v192_path):
        sd_src = torch.load(v192_path, map_location=DEVICE, weights_only=False)["state"]
        sd_dst = mdl.state_dict()
        mdl.load_state_dict({k: v for k, v in sd_src.items() if k in sd_dst and v.shape == sd_dst[k].shape}, strict=False)
        print(f"  [v193] Warm-started from {v192_path}")

    if not force_retrain and os.path.exists(ckpt_path):
        ckpt_data = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        mdl.load_state_dict(ckpt_data["state"], strict=False)
        print(f"  [v193] Loaded existing checkpoint ← {ckpt_path}")
        return mdl, ckpt_data.get("history", {})

    print(f"\n{'='*85}\n  DiBPNet_v193 (GRL)  Params: {mdl.n_params():,}  adv_classes: {mdl._adv_classes}\n{'='*85}")

    def alpha_fn(global_ep):
        total = TRAIN.epochs_p1 + TRAIN.epochs_p2
        return float(2.0 / (1.0 + np.exp(-10.0 * global_ep / max(total, 1))) - 1.0)

    adv_params = list(mdl.subj_adversary.parameters())

    hist = _empty_hist()
    mdl, hist = _two_phase_train(mdl, "v193", hist, adv_params=adv_params, alpha_fn=alpha_fn, print_prefix="[GRL] ")
    torch.save({"state": mdl.state_dict(), "history": hist}, ckpt_path)
    save_hist_csv(hist, tag="v193")
    return mdl, hist

### Calibration

Global Ridge + Isotonic calibration pipeline, fit on the validation set.

In [ ]:
# CALIBRATION UTILITIES
# ══════════════════════════════════════════════════════════════════════════════
@dataclass
class CalibState:
    """Fitted Ridge + Isotonic correctors; built on VAL, applied to VAL + TEST."""
    ridge_sbp: Optional[Ridge]               = None
    ridge_dbp: Optional[Ridge]               = None
    iso_s:     Optional[IsotonicRegression]  = None
    iso_d:     Optional[IsotonicRegression]  = None

CALIB_STATE = CalibState()


def _build_subj_cache(mdl, X, Q, subj_arr, base_arr, batch=ARCH.batch):
    """Causal subject-embedding cache -- SINGLE SOURCE OF TRUTH for DiBP-Net.
    Uses only the first CALIB.ref_beats beats per subject (min_beats gate), so the
    personalization vector never sees the beat it later predicts [FIX-LEAK].
    Called by both _build_subj_cache_for_split() (main eval) and every later
    experiment. (Baselines use _build_subj_cache_baseline(); their forward has a
    different signature.)"""
    _is_transformer = "TransformerBaseline" in type(mdl).__name__
    beat_idx = ARCH.n_beats // 2
    mdl.eval()
    mdl._subj_z_cache = {}

    ref_idx_parts = []
    for s in np.unique(subj_arr):
        m = np.where(subj_arr == s)[0]
        if len(m) < CALIB.min_beats:
            continue
        ref_idx_parts.append(m[:min(CALIB.ref_beats, len(m))])
    if not ref_idx_parts:
        return
    ref_idx = np.concatenate(ref_idx_parts)

    with torch.no_grad():
        for i in range(0, len(ref_idx), batch):
            sl_idx = ref_idx[i:i + batch]
            x_in = (torch.tensor(X[sl_idx, beat_idx, :], dtype=torch.float32).to(DEVICE)
                    if _is_transformer else torch.tensor(X[sl_idx]).to(DEVICE))
            q_c = torch.tensor(Q[sl_idx, beat_idx]).to(DEVICE)
            if q_c.dim() == 3:
                q_c = q_c.squeeze(1)
            out = mdl(x_in,
                      torch.tensor(subj_arr[sl_idx], dtype=torch.long).to(DEVICE),
                      torch.tensor(base_arr[sl_idx]).to(DEVICE),
                      q_c, 0.0, _cache_build=True)
            for b_idx, sid in enumerate(subj_arr[sl_idx]):
                mdl._subj_z_cache.setdefault(int(sid), []).append(
                    out["z_reg"].cpu().numpy()[b_idx])
    with torch.no_grad():
        for sid, zs in mdl._subj_z_cache.items():
            mz = torch.tensor(np.mean(zs, axis=0), dtype=torch.float32).to(DEVICE)
            mz = (mz - mz.mean()) / mz.std().clamp(min=1e-6)
            mdl._subj_z_cache[sid] = mdl.subj_proj(mz).cpu().numpy().astype(np.float32)


def _build_subj_cache_for_split(mdl: DiBPNet, split: Split):
    """Thin Split-object wrapper over _build_subj_cache() (SSoT)."""
    _build_subj_cache(mdl, split.X, split.Q, split.subj, split.base)


def _stack_ridge_feats_10(bp_direct, bp_wave, sbp_wk, dbp_wk, R, C, base):
    """10-column DiBP-Net Ridge feature matrix -- SINGLE SOURCE OF TRUTH
    (get_raw_predictions, _collect_ridge_features, _build_ridge_feats_10 all route
    here). Cols: [direct_sbp, direct_dbp, wave_sbp, wave_dbp, WK_sbp, WK_dbp,
    R, C, R*C, base].  Model defines RC = R*C, so the product is used directly."""
    return np.stack([bp_direct[:, 0], bp_direct[:, 1],
                     bp_wave[:, 0],   bp_wave[:, 1],
                     sbp_wk, dbp_wk, R, C, R * C,
                     base.astype(np.float32)], axis=1)


def _collect_ridge_features(mdl: DiBPNet, split: Split) -> np.ndarray:
    """Collect 10-column feature matrix for Ridge corrector."""
    mdl.eval(); feats = []
    pop_sig = mdl.pop_sig.item(); pop_mu = mdl.pop_mu.item()
    with torch.no_grad():
        for i in range(0, len(split.X), ARCH.batch):
            sl  = slice(i, min(i + ARCH.batch, len(split.X)))
            q_c = torch.tensor(split.Q[sl, ARCH.n_beats // 2]).to(DEVICE)
            if q_c.dim() == 3: q_c = q_c.squeeze(1)
            out    = mdl(torch.tensor(split.X[sl]).to(DEVICE),
                         torch.tensor(split.subj[sl], dtype=torch.long).to(DEVICE),
                         torch.tensor(split.base[sl]).to(DEVICE), q_c, 0.0)
            abp_mm = out[0].cpu().numpy() * pop_sig + pop_mu
            R_np   = out[7].cpu().numpy(); C_np = out[8].cpu().numpy()
            feats.append(_stack_ridge_feats_10(
                out[10].cpu().numpy(), out[11].cpu().numpy(),
                _soft_max_np(abp_mm), _soft_min_np(abp_mm),
                R_np, C_np, split.base[sl]))
    return np.concatenate(feats)


def fit_ridge_on_val(mdl: DiBPNet):
    """Fit Ridge correctors on VAL split; stores results into CALIB_STATE."""
    feats = _collect_ridge_features(mdl, SPLITS.val)
    CALIB_STATE.ridge_sbp = Ridge(alpha=CALIB.ridge_alpha).fit(feats, SPLITS.val.Ybp[:, 0])
    CALIB_STATE.ridge_dbp = Ridge(alpha=CALIB.ridge_alpha).fit(feats, SPLITS.val.Ybp[:, 1])
    mae_s = float(np.mean(np.abs(CALIB_STATE.ridge_sbp.predict(feats) - SPLITS.val.Ybp[:, 0])))
    mae_d = float(np.mean(np.abs(CALIB_STATE.ridge_dbp.predict(feats) - SPLITS.val.Ybp[:, 1])))
    print(f"  Ridge fit on VAL — SBP MAE: {mae_s:.2f}  DBP MAE: {mae_d:.2f}", flush=True)


def apply_ridge(feats: np.ndarray) -> Optional[np.ndarray]:
    if CALIB_STATE.ridge_sbp is None: return None
    return np.stack([CALIB_STATE.ridge_sbp.predict(feats),
                     CALIB_STATE.ridge_dbp.predict(feats)], axis=1).astype(np.float32)


def fit_isotonic_on_val(raw_preds: np.ndarray):
    """Fit Isotonic correctors on VAL raw preds; stores results into CALIB_STATE."""
    CALIB_STATE.iso_s = IsotonicRegression(out_of_bounds="clip")
    CALIB_STATE.iso_d = IsotonicRegression(out_of_bounds="clip")
    idx_s = np.argsort(raw_preds[:, 0]); idx_d = np.argsort(raw_preds[:, 1])
    CALIB_STATE.iso_s.fit(raw_preds[idx_s, 0], SPLITS.val.Ybp[idx_s, 0])
    CALIB_STATE.iso_d.fit(raw_preds[idx_d, 1], SPLITS.val.Ybp[idx_d, 1])


def apply_isotonic(preds: np.ndarray) -> np.ndarray:
    if CALIB_STATE.iso_s is None: return preds
    out = preds.copy()
    out[:, 0] = CALIB_STATE.iso_s.predict(preds[:, 0])
    out[:, 1] = CALIB_STATE.iso_d.predict(preds[:, 1])
    return out.astype(np.float32)


def get_raw_predictions(mdl: DiBPNet, split: Split) -> tuple:
    """
    Run inference on split → (corrected_preds, raw_ensemble_preds).
    Ridge + Isotonic in CALIB_STATE must be fitted on VAL before calling.
    Returns: primary (Ridge+Iso corrected), ens_all (raw bp_ens+WK blend).
    """
    mdl.eval(); N = len(split.X)
    ens_all     = np.zeros((N, 2), np.float32)
    ridge_feats = np.zeros((N, 10), np.float32)
    pop_sig = mdl.pop_sig.item(); pop_mu = mdl.pop_mu.item()
    with torch.no_grad():
        for i in range(0, N, ARCH.batch):
            sl  = slice(i, min(i + ARCH.batch, N))
            q_c = torch.tensor(split.Q[sl, ARCH.n_beats // 2]).to(DEVICE)
            if q_c.dim() == 3: q_c = q_c.squeeze(1)
            out    = mdl(torch.tensor(split.X[sl]).to(DEVICE),
                         torch.tensor(split.subj[sl], dtype=torch.long).to(DEVICE),
                         torch.tensor(split.base[sl]).to(DEVICE), q_c, 0.0)
            abp_mm = out[0].cpu().numpy() * pop_sig + pop_mu
            sbp_wk = _soft_max_np(abp_mm); dbp_wk = _soft_min_np(abp_mm)
            R_np   = out[7].cpu().numpy(); C_np = out[8].cpu().numpy()
            ens_all[sl]     = out[12].cpu().numpy()
            ridge_feats[sl] = _stack_ridge_feats_10(
                out[10].cpu().numpy(), out[11].cpu().numpy(),
                sbp_wk, dbp_wk, R_np, C_np, split.base[sl])
    ridge_pred = apply_ridge(ridge_feats)
    primary    = apply_isotonic(ridge_pred if ridge_pred is not None else ens_all)
    return primary, ens_all


# ══════════════════════════════════════════════════════════════════════════════
# BASELINE INFERENCE and CALIBRATION UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def _build_ridge_feats_5(raw_preds: np.ndarray, base_arr: np.ndarray) -> np.ndarray:
    """5-column Ridge feature matrix for baseline models [sbp, dbp, sbp², dbp², base]."""
    return np.stack([raw_preds[:, 0], raw_preds[:, 1],
                     raw_preds[:, 0] ** 2, raw_preds[:, 1] ** 2,
                     base_arr.astype(np.float32)], axis=1)


def _get_raw_preds_baseline(mdl, X: np.ndarray,
                             subj_arr: np.ndarray, base_arr: np.ndarray) -> np.ndarray:
    """Run inference on a baseline model (GRU / ResNet / Transformer).
    Returns (N, 2) float32 array in mmHg."""
    mdl.eval()
    N = len(X)
    out_all = np.zeros((N, 2), np.float32)
    with torch.no_grad():
        for i in range(0, N, ARCH.batch):
            sl  = slice(i, min(i + ARCH.batch, N))
            raw = mdl(
                torch.tensor(X[sl]).to(DEVICE),
                torch.tensor(subj_arr[sl], dtype=torch.long).to(DEVICE),
                torch.tensor(base_arr[sl]).to(DEVICE),
            )
            out_all[sl] = raw.cpu().numpy()
    return out_all


def _get_primary_baseline(uncal: np.ndarray, base_arr: np.ndarray,
                           ridge_s, ridge_d, iso_s, iso_d) -> tuple:
    """Apply 5-feature Ridge + Isotonic to baseline raw predictions.
    Returns: (gcal (N,2) float32, feats (N,5) float32)."""
    feats = _build_ridge_feats_5(uncal, base_arr)
    rp    = np.stack([ridge_s.predict(feats), ridge_d.predict(feats)], axis=1)
    gcal  = rp.copy()
    gcal[:, 0] = iso_s.predict(rp[:, 0])
    gcal[:, 1] = iso_d.predict(rp[:, 1])
    return gcal.astype(np.float32), feats


def _cls_score_baseline(cls_head, gcal: np.ndarray, feats: np.ndarray) -> np.ndarray:
    """Optional: score AUC probability from a classification head for baselines.
    Returns NaN array when cls_head cannot produce valid output."""
    try:
        inp = torch.tensor(np.concatenate([gcal, feats], axis=1),
                           dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            prob = torch.softmax(cls_head.to(DEVICE)(inp), dim=1)
        return prob[:, 1].cpu().numpy()
    except Exception:
        return np.full(len(gcal), np.nan)

### Load All Models

In [ ]:
# ── 0. Bootstrap ──────────────────────────────────────────────────────────

def ensure_data_loaded():
    if SPLITS is None:
        _load_data()


# ── 1. Auto-train missing checkpoints ────────────────────────────────────

_ABL_CFG = {
    "v192nodis": dict(override_lw={"anticollapse": 0.0},
                      override_train={"lambda_hsic_max": 0.0},
                      description_lines=["TRAIN.lambda_hsic_max=0  LW.anticollapse=0"]),
    "ablA":      dict(override_lw={"c_sat": 0.0, "c_prior": 0.0, "c_slow_spread": 0.0},
                      override_train={},
                      description_lines=["LW.c_sat=0  LW.c_prior=0  LW.c_slow_spread=0"]),
    "ablB":      dict(override_lw={"anticollapse": 0.0},
                      override_train={},
                      description_lines=["LW.anticollapse=0"]),
}


def train_missing_checkpoints():
    """Auto-train any missing DiBP-Net / GRL / baseline checkpoints (no-op if all exist)."""
    missing_mcnhm = [t for t in ("v192", "v192nodis", "ablA", "ablB")
                     if not os.path.exists(os.path.join(CKPT_DIR, f"{t}_best.pt"))]
    if missing_mcnhm:
        print(f"  Missing DiBP-Net: {missing_mcnhm} — training …")
        if "v192" in missing_mcnhm:
            global mdl_v192_for_grl
            mdl_v192_for_grl, _ = run_training()
        for tag in missing_mcnhm:
            if tag != "v192":
                run_ablation(f"Auto-train {tag}", tag, **_ABL_CFG[tag])

    if not os.path.exists(os.path.join(CKPT_DIR_GRL, "v193_best.pt")):
        print("  Missing v193 — training GRL …")
        run_grl(mdl_v192_ckpt="v192_best", force_retrain=False)

    for tag in ("gru", "resnet", "transformer"):
        if not os.path.exists(os.path.join(OUT_DIR, f"checkpoints_{tag}", f"baseline_{tag}_best.pt")):
            print(f"  Missing baseline {tag} — training …")
            run_baseline_model(tag)

    if not os.path.exists(os.path.join(OUT_DIR, "checkpoints_supcon", "baseline_supcon_best.pt")):
        print("  Missing baseline supcon — training …")
        run_supcon_baseline()


# ── 2. ModelCalib & shared helpers ───────────────────────────────────────

@dataclass
class ModelCalib:
    """Per-model Ridge + Isotonic calibration bundle; isolated from global state."""
    ridge_sbp: object = None
    ridge_dbp: object = None
    iso_s:     object = None
    iso_d:     object = None
    cls_head:  object = None  # nn.Linear HTN classifier (baseline only)

    @property
    def is_fitted(self) -> bool:
        return self.ridge_sbp is not None and self.iso_s is not None


@contextmanager
def _calib_ctx(calib: Optional[ModelCalib]):
    """Temporarily swap CALIB_STATE to match `calib`, restore on exit."""
    prev = CalibState(ridge_sbp=CALIB_STATE.ridge_sbp, ridge_dbp=CALIB_STATE.ridge_dbp,
                      iso_s=CALIB_STATE.iso_s,          iso_d=CALIB_STATE.iso_d)
    CALIB_STATE.ridge_sbp = calib.ridge_sbp if calib else None
    CALIB_STATE.ridge_dbp = calib.ridge_dbp if calib else None
    CALIB_STATE.iso_s     = calib.iso_s     if calib else None
    CALIB_STATE.iso_d     = calib.iso_d     if calib else None
    try:    yield
    finally:
        CALIB_STATE.ridge_sbp = prev.ridge_sbp
        CALIB_STATE.ridge_dbp = prev.ridge_dbp
        CALIB_STATE.iso_s     = prev.iso_s
        CALIB_STATE.iso_d     = prev.iso_d


def get_raw_predictions_safe(mdl, split, calib: ModelCalib) -> tuple:
    """get_raw_predictions() with per-model calib injected; no global state leak."""
    with _calib_ctx(calib):
        return get_raw_predictions(mdl, split)


def _fit_calib_on_val(mdl) -> ModelCalib:
    """Fit DiBP-Net Ridge + Isotonic on VAL. Returns ModelCalib; CALIB_STATE unchanged."""
    prev = CalibState(ridge_sbp=CALIB_STATE.ridge_sbp, ridge_dbp=CALIB_STATE.ridge_dbp,
                      iso_s=CALIB_STATE.iso_s,          iso_d=CALIB_STATE.iso_d)
    try:
        _build_subj_cache_for_split(mdl, SPLITS.val)
        fit_ridge_on_val(mdl)
        preds_va, _ = get_raw_predictions(mdl, SPLITS.val)
        fit_isotonic_on_val(preds_va)
        return ModelCalib(ridge_sbp=CALIB_STATE.ridge_sbp, ridge_dbp=CALIB_STATE.ridge_dbp,
                          iso_s=CALIB_STATE.iso_s,          iso_d=CALIB_STATE.iso_d)
    finally:
        CALIB_STATE.ridge_sbp = prev.ridge_sbp
        CALIB_STATE.ridge_dbp = prev.ridge_dbp
        CALIB_STATE.iso_s     = prev.iso_s
        CALIB_STATE.iso_d     = prev.iso_d


def _fit_calib_baseline(mdl) -> ModelCalib:
    """Fit Ridge + Isotonic + HTN cls_head on VAL for a baseline model."""
    _build_subj_cache_baseline(mdl, SPLITS.val.X, SPLITS.val.subj, SPLITS.val.base)
    raw_va   = _get_raw_preds_baseline(mdl, SPLITS.val.X, SPLITS.val.subj, SPLITS.val.base)
    feats_va = _build_ridge_feats_5(raw_va, SPLITS.val.base)
    Ybp_va   = SPLITS.val.Ybp
    r_s  = Ridge(alpha=CALIB.ridge_alpha).fit(feats_va, Ybp_va[:, 0])
    r_d  = Ridge(alpha=CALIB.ridge_alpha).fit(feats_va, Ybp_va[:, 1])
    rp   = np.stack([r_s.predict(feats_va), r_d.predict(feats_va)], axis=1)
    idx_s = np.argsort(rp[:, 0]); idx_d = np.argsort(rp[:, 1])
    i_s  = IsotonicRegression(out_of_bounds="clip").fit(rp[idx_s, 0], Ybp_va[idx_s, 0])
    i_d  = IsotonicRegression(out_of_bounds="clip").fit(rp[idx_d, 1], Ybp_va[idx_d, 1])
    mae_s = float(np.mean(np.abs(i_s.predict(rp[:, 0]) - Ybp_va[:, 0])))
    mae_d = float(np.mean(np.abs(i_d.predict(rp[:, 1]) - Ybp_va[:, 1])))
    print(f"       VAL SBP MAE {mae_s:.2f}  DBP MAE {mae_d:.2f}", flush=True)

 # Train a tiny logistic HTN head (on VAL calibrated features → TEST AUC still honest)
    gcal_va = np.column_stack([i_s.predict(rp[:, 0]),
                                i_d.predict(rp[:, 1])]).astype(np.float32)
    htn_va  = ((Ybp_va[:, 0] >= HTN_SBP) | (Ybp_va[:, 1] >= HTN_DBP)).astype(np.int64)
    inp_va  = np.concatenate([gcal_va, feats_va], axis=1).astype(np.float32)
    cls = nn.Linear(inp_va.shape[1], 2)
    _opt = torch.optim.Adam(cls.parameters(), lr=1e-2, weight_decay=1e-3)
    _Xt  = torch.tensor(inp_va); _yt = torch.tensor(htn_va)
    for _ in range(300):
        _opt.zero_grad(); F.cross_entropy(cls(_Xt), _yt).backward(); _opt.step()
    cls.eval()

    return ModelCalib(ridge_sbp=r_s, ridge_dbp=r_d, iso_s=i_s, iso_d=i_d, cls_head=cls)


def _quick_metrics_baseline(mdl, calib: ModelCalib, split) -> tuple:
    """Beat-level MAE + AUC for a baseline model.
    Returns (raw_sbp, raw_dbp, pre_sbp, pre_dbp, post_sbp, post_dbp, auc)."""
    _build_subj_cache_baseline(mdl, split.X, split.subj, split.base)
    uncal = _get_raw_preds_baseline(mdl, split.X, split.subj, split.base)
    raw_s = float(np.abs(uncal[:, 0] - split.Ybp[:, 0]).mean())
    raw_d = float(np.abs(uncal[:, 1] - split.Ybp[:, 1]).mean())
    gcal, feats = _get_primary_baseline(uncal, split.base,
                                         calib.ridge_sbp, calib.ridge_dbp,
                                         calib.iso_s, calib.iso_d)
    cal_s = float(np.abs(gcal[:, 0] - split.Ybp[:, 0]).mean())
    cal_d = float(np.abs(gcal[:, 1] - split.Ybp[:, 1]).mean())
 # Post: subject-bias correction (SSoT apply_subject_calibration), eval-window MAE
    scal, m = apply_subject_calibration(gcal, split.Ybp, split.subj)
    post_s = float(np.abs(scal[m, 0] - split.Ybp[m, 0]).mean())
    post_d = float(np.abs(scal[m, 1] - split.Ybp[m, 1]).mean())
    auc = float("nan")
    if calib.cls_head is not None:
        htn_true = ((split.Ybp[:, 0] >= HTN_SBP) | (split.Ybp[:, 1] >= HTN_DBP)).astype(int)
        prob = _cls_score_baseline(calib.cls_head, gcal, feats)
        if len(np.unique(htn_true)) > 1 and not np.all(np.isnan(prob)):
            auc = float(roc_auc_score(htn_true, prob))
    return raw_s, raw_d, cal_s, cal_d, post_s, post_d, auc


def _load_calib_from_pkl(pkl_path: str) -> Optional[ModelCalib]:
    if not os.path.exists(pkl_path):
        return None
    d = joblib.load(pkl_path)
    return ModelCalib(
        ridge_sbp=d.get("ridge_sbp") or d.get("ridge_s"),
        ridge_dbp=d.get("ridge_dbp") or d.get("ridge_d"),
        iso_s=d.get("iso_s"), iso_d=d.get("iso_d"),
        cls_head=d.get("cls_head"),
    )


def _save_calib_to_pkl(calib: ModelCalib, mdl, pkl_path: str):
    sc = getattr(mdl, "_subj_z_cache", None) or getattr(mdl, "_subj_cache", {})
    joblib.dump({"ridge_sbp": calib.ridge_sbp, "ridge_dbp": calib.ridge_dbp,
                 "iso_s":     calib.iso_s,     "iso_d":     calib.iso_d,
                 "subj_cache": sc,
                 "cls_head":  calib.cls_head}, pkl_path)


def _restore_subj_cache(mdl, pkl_path: str):
    try:
        sc = joblib.load(pkl_path).get("subj_cache")
        if sc:
            if   hasattr(mdl, "_subj_z_cache"): mdl._subj_z_cache = sc
            elif hasattr(mdl, "_subj_cache"):    mdl._subj_cache   = sc
    except Exception:
        pass


# ── 3. DiBP-Net v19.2 family ─────────────────────────────────────────────

_VAL_METRICS: dict = {}  # tag → (raw_sbp, raw_dbp, pre_sbp, pre_dbp, post_sbp, post_dbp, auc) on VAL


def _load_mcnhm(tag: str, calib_name: str = None) -> tuple:
    """Load DiBP-Net checkpoint + calibration. Returns (mdl, calib, hist) or (None, None, {})."""
    mdl, hist = load_checkpoint(tag)
    if mdl is None:
        print(f"  ⚠  {tag:<20} checkpoint not found")
        return None, None, {}

    mdl._set_stats_from_splits(SPLITS); mdl.eval()
    key      = calib_name or tag.replace("_best", "").replace("_p1", "")
    pkl_path = os.path.join(CKPT_DIR, f"calibration_{key}.pkl")
    calib    = _load_calib_from_pkl(pkl_path)

 # Raw: model with empty subject cache (pop_mean for all subjects)
    mdl._subj_z_cache = {}
    rs, rd, auc = quick_metrics(mdl, SPLITS.val)

 # Cal: 3-beat subject cache + Ridge + Isotonic
    if calib is not None and calib.is_fitted:
        _restore_subj_cache(mdl, pkl_path)
        calib_tag = "disk"
    else:
        calib = _fit_calib_on_val(mdl)
        _save_calib_to_pkl(calib, mdl, pkl_path)
        calib_tag = "fit★"

    with _calib_ctx(calib):
        gcal_va, _ = get_raw_predictions(mdl, SPLITS.val)
    cs = float(np.abs(gcal_va[:, 0] - SPLITS.val.Ybp[:, 0]).mean())
    cd = float(np.abs(gcal_va[:, 1] - SPLITS.val.Ybp[:, 1]).mean())
 # Post: subject-bias correction (SSoT apply_subject_calibration), eval-window MAE
    scal_va, m_va = apply_subject_calibration(gcal_va, SPLITS.val.Ybp, SPLITS.val.subj)
    ps = float(np.abs(scal_va[m_va, 0] - SPLITS.val.Ybp[m_va, 0]).mean())
    pd = float(np.abs(scal_va[m_va, 1] - SPLITS.val.Ybp[m_va, 1]).mean())
    _VAL_METRICS[key] = (rs, rd, cs, cd, ps, pd, auc)
    print(f"  {key:<13}  {calib_tag:<5}  {rs:5.2f}/{rd:5.2f} → {cs:5.2f}/{cd:5.2f}  AUC {auc:.4f}")
    save_hist_csv(hist or {}, tag=key)
    return mdl, calib, hist or {}


def load_dibpnet_v192_family():
    """Load v19.2 base model + its 3 ablations (nodis, ablA, ablB). Prints summary table."""
    print("\nDiBP-Net v19.2 " + "─" * 46)
    print(f"  {'Tag':<13}  {'Calib':<5}  {'SBP/DBP (raw)':^13}     {'SBP/DBP (cal)':^13}  AUC↑")
    model,     calib_v192,  training_hist = _load_mcnhm("v192_best",      calib_name="v192")
    mdl_nodis, calib_nodis, hist_nodis    = _load_mcnhm("v192nodis_best", calib_name="v192nodis")
    mdl_ablA,  calib_ablA,  hist_ablA     = _load_mcnhm("ablA_best",      calib_name="ablA")
    mdl_ablB,  calib_ablB,  hist_ablB     = _load_mcnhm("ablB_best",      calib_name="ablB")
    return (model, calib_v192, training_hist,
            mdl_nodis, calib_nodis, hist_nodis,
            mdl_ablA, calib_ablA, hist_ablA,
            mdl_ablB, calib_ablB, hist_ablB)


# ── 4. DiBP-Net v19.3 GRL ────────────────────────────────────────────────

def load_dibpnet_v193():
    """Load the v19.3 GRL model + calibration, if its checkpoint exists.
    Returns (mdl_v193, calib_v193, hist_v193); mdl_v193 is None if unavailable."""
    mdl_v193 = None; calib_v193 = None; hist_v193 = {}
    _ckpt_193 = os.path.join(CKPT_DIR_GRL, "v193_best.pt")
    _pkl_193  = os.path.join(CKPT_DIR_GRL, "calibration_v193.pkl")

    print("\nDiBP-Net v19.3 GRL " + "─" * 43)
    print(f"  {'Tag':<13}  {'Calib':<5}  {'SBP/DBP (raw)':^13}     {'SBP/DBP (cal)':^13}  AUC↑")

    if not os.path.exists(_ckpt_193):
        print(f"  ⚠  v193 not found: {_ckpt_193}")
        return mdl_v193, calib_v193, hist_v193

    try:
 # DiBPNet_v193 + _GradReverse are the single source of truth in the architecture section (Model
 # Architecture, cell 9). The old inline fallback here was dead code -- the
 # guard `"DiBPNet_v193" not in globals()` is always False once the architecture section has run --
 # and it had drifted out of sync with the architecture section (missing self._adv_classes). Removed.
        ckpt      = torch.load(_ckpt_193, map_location=DEVICE, weights_only=False)
        hist_v193 = ckpt.get("history", {})
        mdl_v193  = DiBPNet_v193().to(DEVICE)
        mdl_v193._set_stats_from_splits(SPLITS)
        dst = mdl_v193.state_dict()
        mdl_v193.load_state_dict(
            {k: v for k, v in ckpt["state"].items() if k in dst and v.shape == dst[k].shape},
            strict=False)
        mdl_v193.eval()

        calib_v193 = _load_calib_from_pkl(_pkl_193)
        if calib_v193 is not None and calib_v193.is_fitted:
            _restore_subj_cache(mdl_v193, _pkl_193)
            _ctag = "disk"
        else:
            calib_v193 = _fit_calib_on_val(mdl_v193)
            _save_calib_to_pkl(calib_v193, mdl_v193, _pkl_193)
            _ctag = "fit★"

 # Raw: pop_mean (empty subject cache), matching the master evaluation + the other DiBP-Net models
        _saved_cache = mdl_v193._subj_z_cache
        mdl_v193._subj_z_cache = {}
        rs, rd, auc = quick_metrics(mdl_v193, SPLITS.val)
        mdl_v193._subj_z_cache = _saved_cache   # restore personalized cache for Pre/Post
        with _calib_ctx(calib_v193):
            gcal_v193, _ = get_raw_predictions(mdl_v193, SPLITS.val)
        cs = float(np.abs(gcal_v193[:, 0] - SPLITS.val.Ybp[:, 0]).mean())
        cd = float(np.abs(gcal_v193[:, 1] - SPLITS.val.Ybp[:, 1]).mean())
 # Post: subject-bias correction (SSoT apply_subject_calibration), eval-window MAE
        scal_v193, m_v193 = apply_subject_calibration(gcal_v193, SPLITS.val.Ybp, SPLITS.val.subj)
        ps = float(np.abs(scal_v193[m_v193, 0] - SPLITS.val.Ybp[m_v193, 0]).mean())
        pd = float(np.abs(scal_v193[m_v193, 1] - SPLITS.val.Ybp[m_v193, 1]).mean())
        _VAL_METRICS["v193"] = (rs, rd, cs, cd, ps, pd, auc)
        print(f"  {'v193':<13}  {_ctag:<5}  {rs:5.2f}/{rd:5.2f} → {cs:5.2f}/{cd:5.2f}  AUC {auc:.4f}")

    except Exception as _e:
        import traceback; traceback.print_exc()
        print(f"  ⚠  v193 load failed: {_e}")
        mdl_v193 = None; calib_v193 = None

    return mdl_v193, calib_v193, hist_v193


# ── 5. Baseline models ────────────────────────────────────────────────────

_BL_CKPT_DIRS = {k: os.path.join(OUT_DIR, v) for k, v in _BL_CKPT.items()}


def _load_baseline(cls_name: str, ckpt_path: str, calib_pkl: str) -> tuple:
    """Load baseline model + calibration. Returns (mdl, calib) or (None, None)."""
    _SHORT = {"SingleStreamGRU": "GRU", "ResNet1D": "ResNet-1D",
              "TransformerBaseline": "Transformer", "SupConBPModel": "SupCon"}
    label = _SHORT.get(cls_name, cls_name)
    CLS = globals().get(cls_name)
    if CLS is None:
        print(f"  ⚠  {label:<14} class not defined"); return None, None
    if not os.path.exists(ckpt_path):
        print(f"  ⚠  {label:<14} checkpoint not found"); return None, None

    mdl = CLS().to(DEVICE)
    if   hasattr(mdl, "_set_stats"):             mdl._set_stats()
    elif hasattr(mdl, "_set_stats_from_splits"): mdl._set_stats_from_splits(SPLITS)
    mdl.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=False)["state"])
    mdl.eval()

    calib = _load_calib_from_pkl(calib_pkl)
    if calib is not None and calib.is_fitted:
        _restore_subj_cache(mdl, calib_pkl)
        calib_tag = "disk"
    else:
        calib = _fit_calib_baseline(mdl)
        _save_calib_to_pkl(calib, mdl, calib_pkl)
        calib_tag = "fit★"

    _KEY_MAP = {"SingleStreamGRU": "gru", "ResNet1D": "resnet",
                "TransformerBaseline": "transformer", "SupConBPModel": "supcon"}
    bl_key = _KEY_MAP.get(cls_name, cls_name)
    rs, rd, cs, cd, ps, pd, auc_bl = _quick_metrics_baseline(mdl, calib, SPLITS.val)
    _VAL_METRICS[bl_key] = (rs, rd, cs, cd, ps, pd, auc_bl)
    _auc_str = f"AUC {auc_bl:.4f}" if not (auc_bl != auc_bl) else ""
    print(f"  {label:<13}  {calib_tag:<5}  {rs:5.2f}/{rd:5.2f} → {cs:5.2f}/{cd:5.2f}  {_auc_str}")
    return mdl, calib


def load_baselines():
    """Load GRU, ResNet-1D, Transformer baselines + calibration. Prints summary table."""
    print("\nBaseline models " + "─" * 45)
    print(f"  {'Label':<13}  {'Calib':<5}  {'SBP/DBP (raw)':^13}     {'SBP/DBP (cal)':^13}")
    mdl_gru,         calib_gru         = _load_baseline(
        "SingleStreamGRU",     os.path.join(_BL_CKPT_DIRS["gru"],         "baseline_gru_best.pt"),
                               os.path.join(_BL_CKPT_DIRS["gru"],         "calibration_gru.pkl"))
    mdl_resnet,      calib_resnet      = _load_baseline(
        "ResNet1D",            os.path.join(_BL_CKPT_DIRS["resnet"],      "baseline_resnet_best.pt"),
                               os.path.join(_BL_CKPT_DIRS["resnet"],      "calibration_resnet.pkl"))
    mdl_transformer, calib_transformer = _load_baseline(
        "TransformerBaseline", os.path.join(_BL_CKPT_DIRS["transformer"], "baseline_transformer_best.pt"),
                               os.path.join(_BL_CKPT_DIRS["transformer"], "calibration_transformer.pkl"))
    mdl_supcon,      calib_supcon      = _load_baseline(
        "SupConBPModel",       os.path.join(_BL_CKPT_DIRS["supcon"],      "baseline_supcon_best.pt"),
                               os.path.join(_BL_CKPT_DIRS["supcon"],      "calibration_supcon.pkl"))
    return (mdl_gru, calib_gru, mdl_resnet, calib_resnet,
            mdl_transformer, calib_transformer, mdl_supcon, calib_supcon)


# ── 6. Registry ───────────────────────────────────────────────────────────

def build_model_registry(model, calib_v192, mdl_nodis, calib_nodis,
                          mdl_ablA, calib_ablA, mdl_ablB, calib_ablB,
                          mdl_v193, calib_v193,
                          mdl_gru, calib_gru, mdl_resnet, calib_resnet,
                          mdl_transformer, calib_transformer,
                          mdl_supcon, calib_supcon) -> dict:
    return {
        "v192":        (model,           calib_v192),
        "v192nodis":   (mdl_nodis,       calib_nodis),
        "ablA":        (mdl_ablA,        calib_ablA),
        "ablB":        (mdl_ablB,        calib_ablB),
        "v193":        (mdl_v193,        calib_v193),
        "gru":         (mdl_gru,         calib_gru),
        "resnet":      (mdl_resnet,      calib_resnet),
        "transformer": (mdl_transformer, calib_transformer),
        "supcon":      (mdl_supcon,      calib_supcon),
    }


def _load_bl_hist(m_type: str) -> dict:
    """Load baseline training history from HIST_DIR CSVs."""
    h = {}
    for suffix in ("ep1", "ep10"):
        p = os.path.join(HIST_DIR, f"baseline_{m_type}_{suffix}.csv")
        if os.path.exists(p):
            h.update(pd.read_csv(p).to_dict(orient="list"))
    return h


def build_hist_registry(training_hist, hist_nodis, hist_ablA, hist_ablB, hist_v193) -> dict:
    """Combine DiBP-Net training histories with freshly-loaded baseline histories."""
    return {
        "v192":        training_hist, "v192nodis": hist_nodis,
        "ablA":        hist_ablA,     "ablB":      hist_ablB,
        "v193":        hist_v193,
        "gru":         _load_bl_hist("gru"),
        "resnet":      _load_bl_hist("resnet"),
        "transformer": _load_bl_hist("transformer"),
        "supcon":      _load_bl_hist("supcon"),
    }


DISPLAY_NAMES = {
    "v192":        "DiBP-Net v19.2 (proposed)",
    "v192nodis":   "DiBP-Net NoDis",
    "ablA":        "DiBP-Net Ablation A",
    "ablB":        "DiBP-Net Ablation B",
    "v193":        "DiBP-Net v19.3 (GRL)",
    "gru":         "GRU Baseline",
    "resnet":      "ResNet-1D Baseline",
    "transformer": "Transformer Baseline",
    "supcon":      "SupCon Baseline",
}

# Registry key → CSV "Model" name in master_ablation_summary.csv
_CSV_NAME = {
    "v192":        "DiBP-Net-v192",
    "v192nodis":   "AblNoDis",
    "ablA":        "AblA-HSIConly",
    "ablB":        "AblB-cSlow",
    "v193":        "DiBP-Net-v193 (GRL)",
    "gru":         "GRU",
    "resnet":      "ResNet-1D",
    "transformer": "Transformer",
    "supcon":      "SupCon",
}

_DIBPNET_KEYS = ["v192", "v192nodis", "ablA", "ablB", "v193"]


def _load_master_summary_csv() -> Optional[pd.DataFrame]:
    """Load master_ablation_summary.csv (Val split), or None if unavailable/empty."""
    master_csv = os.path.join(OUT_DIR, "master_ablation_summary.csv")
    try:
        if os.path.exists(master_csv) and os.path.getsize(master_csv) > 0:
            mdf = pd.read_csv(master_csv)
            return mdf[mdf["Split"] == "Val"].set_index("Model")
        print(f"  [SKIP] {master_csv} not found yet — run Part 3 to generate it.")
    except pd.errors.EmptyDataError:
        print(f"  [SKIP] {master_csv} exists but has no data yet — run Part 3 to generate it.")
    return None


def print_summary_table(registry: dict, val_metrics: dict, width: int = 92):
    """3-stage (raw/pre/post) summary table, matching the paper's eval_window=True view."""
    baseline_keys = [k for k in registry if k not in _DIBPNET_KEYS]
    mdf_val = _load_master_summary_csv()

    print("\n" + "═" * width)
    print(f"  {'Model':<30} {'Cal':^5}  {'─ Raw SBP/DBP ─':^17}  {'─ Pre SBP/DBP ─':^17}  "
          f"{'─ Post SBP/DBP ─':^17}  {'AUC↑':>7}")
    print("  " + "─" * (width - 2))

    for k in _DIBPNET_KEYS + ["---"] + baseline_keys:
        if k == "---":
            print("  " + "·" * (width - 2)); continue
        m, c = registry[k]
        cal_flag = "✅" if (c is not None and c.is_fitted) else "⚠ "
        csv_key = _CSV_NAME.get(k, "")

        if mdf_val is not None and csv_key in mdf_val.index:
            row  = mdf_val.loc[csv_key]
            raw  = f"{row['Beat SBP (Raw)']:5.2f}/{row['Beat DBP (Raw)']:5.2f}"
            pre  = f"{row['Beat SBP (Pre)']:5.2f}/{row['Beat DBP (Pre)']:5.2f}"
            post = f"{row['Beat SBP (Post)']:5.2f}/{row['Beat DBP (Post)']:5.2f}"
            auc  = f"{row['AUC (HTN)']:.4f}" if not (row['AUC (HTN)'] != row['AUC (HTN)']) else "   —  "
        elif k in val_metrics:
            rs, rd, cs, cd, ps, pd, a = val_metrics[k]
            raw  = f"{rs:5.2f}/{rd:5.2f}"
            pre  = f"{cs:5.2f}/{cd:5.2f}"
            post = f"{ps:5.2f}/{pd:5.2f}"
            auc  = f"{a:.4f}" if not (a != a) else "   —  "
        else:
            raw = pre = post = "  — / —  "; auc = "    —  "

        print(f"  {DISPLAY_NAMES[k]:<30} {cal_flag:^5}  {raw:^17}  {pre:^17}  {post:^17}  {auc:>7}")

    print("═" * width)
    n_fit = sum(1 for v in val_metrics.values() if v is not None)
    if mdf_val is not None:
        print("  Source: master_ablation_summary.csv  |  eval_window=True, VAL split  |  ★ = freshly fitted on VAL")
    else:
        print("  [WARN] master_ablation_summary.csv not found — showing VAL quick_metrics "
              "(Raw/Pre all-beats; Post = subject-cal on eval window)")
    print(f"  {n_fit} DiBP-Net + {len(baseline_keys)} baseline models ready.")
    print("  Ready → MODEL_REGISTRY + get_raw_predictions_safe()")


# ── 7. Orchestration ──────────────────────────────────────────────────────

def initialize_all_models() -> dict:
    """Run the full pipeline: bootstrap data, train missing ckpts, load every
    model + calibration, build MODEL_REGISTRY + HIST_REGISTRY, print the
    summary table. Returns a dict of everything downstream cells need."""
    ensure_data_loaded()
    train_missing_checkpoints()

    (model, calib_v192, training_hist,
     mdl_nodis, calib_nodis, hist_nodis,
     mdl_ablA, calib_ablA, hist_ablA,
     mdl_ablB, calib_ablB, hist_ablB) = load_dibpnet_v192_family()

    mdl_v193, calib_v193, hist_v193 = load_dibpnet_v193()

    (mdl_gru, calib_gru, mdl_resnet, calib_resnet,
     mdl_transformer, calib_transformer, mdl_supcon, calib_supcon) = load_baselines()

    registry = build_model_registry(
        model, calib_v192, mdl_nodis, calib_nodis,
        mdl_ablA, calib_ablA, mdl_ablB, calib_ablB,
        mdl_v193, calib_v193,
        mdl_gru, calib_gru, mdl_resnet, calib_resnet,
        mdl_transformer, calib_transformer,
        mdl_supcon, calib_supcon,
    )

    hist_registry = build_hist_registry(training_hist, hist_nodis, hist_ablA, hist_ablB, hist_v193)

    print_summary_table(registry, _VAL_METRICS)

    return {
        "model": model, "calib_v192": calib_v192, "training_hist": training_hist,
        "mdl_nodis": mdl_nodis, "calib_nodis": calib_nodis, "hist_nodis": hist_nodis,
        "mdl_ablA": mdl_ablA, "calib_ablA": calib_ablA, "hist_ablA": hist_ablA,
        "mdl_ablB": mdl_ablB, "calib_ablB": calib_ablB, "hist_ablB": hist_ablB,
        "mdl_v193": mdl_v193, "calib_v193": calib_v193, "hist_v193": hist_v193,
        "mdl_gru": mdl_gru, "calib_gru": calib_gru,
        "mdl_resnet": mdl_resnet, "calib_resnet": calib_resnet,
        "mdl_transformer": mdl_transformer, "calib_transformer": calib_transformer,
        "mdl_supcon": mdl_supcon, "calib_supcon": calib_supcon,
        "MODEL_REGISTRY": registry,
        "HIST_REGISTRY": hist_registry,
        "_VAL_METRICS": _VAL_METRICS,
    }

### Evaluation, Validation, and Export

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PREREQUISITES and CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════

STAGE_LABELS = {1: "Normal", 2: "Elevated", 3: "HTN-1", 4: "HTN-2"}

def check_prerequisites():
    """Bootstrap data if needed and assert everything Cell 1 / Cell 2 must have
    already produced is present in globals()."""
    if SPLITS is None:
        _load_data()

    assert "MODEL_REGISTRY" in globals(), "MODEL_REGISTRY not found — run Cell 2 first"
    assert "HIST_REGISTRY" in globals(), "HIST_REGISTRY not found — run Cell 2 first"

    required_from_cell1 = (
        "subj_mae", "cohens_d", "compute_icc", "assign_stages",
        "bootstrap_cohens_d_ci", "bootstrap_icc_ci", "bootstrap_vbvw_ci",
        "corr_zm_zh_fn", "subj_mean_cs",
    )
    missing_cell1 = [f for f in required_from_cell1 if f not in globals()]
    assert not missing_cell1, (
        f"Missing from globals(): {missing_cell1} — run Cell 1 (mcnhm_metrics "
        f"module, Part 0) before running this cell."
    )


# ══════════════════════════════════════════════════════════════════════════════
# UNIVERSAL CALIBRATION and LATENT EXTRACTION
# ══════════════════════════════════════════════════════════════════════════════

def apply_subject_calibration(primary: np.ndarray, Ybp: np.ndarray, subj: np.ndarray) -> tuple:
    """Universal subject median bias correction (Unified for both DiBP-Net & Baseline)."""
    cal_pred = primary.copy()
    eval_mask = np.zeros(len(primary), dtype=bool)

    for s in np.unique(subj):
        m = np.where(subj == s)[0]
        n = len(m)
        if n < CALIB.min_beats: continue

        n_ref = (CALIB.ref_beats if n >= CALIB.ref_beats + CALIB.ols_gap + CALIB.eval_min
                 else max(1, n - CALIB.ols_gap - CALIB.eval_min))

        ref_idx = m[:n_ref]
        eval_idx = m[n_ref + CALIB.ols_gap:]
        if len(eval_idx) < CALIB.eval_min: continue

        cal_pred[eval_idx, 0] = primary[eval_idx, 0] + float(np.median(Ybp[ref_idx, 0] - primary[ref_idx, 0]))
        cal_pred[eval_idx, 1] = primary[eval_idx, 1] + float(np.median(Ybp[ref_idx, 1] - primary[ref_idx, 1]))
        eval_mask[eval_idx] = True

    return cal_pred, eval_mask


def extract_all_outputs(mdl, split) -> dict:
    """Extract full forward pass latent outputs for DiBP-Net models."""
    mdl.eval()
    N = len(split.X)
    accum = {k: [] for k in ("R", "C_wk", "c_slow", "gate", "zm", "zh")}

    with torch.no_grad():
        for i in range(0, N, ARCH.batch):
            sl = slice(i, min(i + ARCH.batch, N))
            q_c = torch.tensor(split.Q[sl, ARCH.n_beats // 2]).to(DEVICE)
            if q_c.dim() == 3: q_c = q_c.squeeze(1)

            out = mdl(
                torch.tensor(split.X[sl]).to(DEVICE),
                torch.tensor(split.subj[sl], dtype=torch.long).to(DEVICE),
                torch.tensor(split.base[sl]).to(DEVICE),
                q_c, 0.0,
            )

            accum["R"].append(out[7].cpu().numpy())
            accum["C_wk"].append(out[8].cpu().numpy())
            accum["gate"].append(out[14].squeeze(-1).cpu().numpy())
            accum["zm"].append(out[18].cpu().numpy())
            accum["zh"].append(out[4].cpu().numpy())
            accum["c_slow"].append(out[20].cpu().numpy())

    return {k: np.concatenate(v) for k, v in accum.items()}


def _baseline_calib_as_dict(calib_obj, raw_pkl_path: str) -> dict:
    result = {}
    if calib_obj is not None and calib_obj.is_fitted:
        result["ridge_s"] = calib_obj.ridge_sbp; result["ridge_d"] = calib_obj.ridge_dbp
        result["iso_s"] = calib_obj.iso_s; result["iso_d"] = calib_obj.iso_d
        result["cls_head"] = calib_obj.cls_head  # prefer in-memory (freshly fitted)
    elif os.path.exists(raw_pkl_path):
        raw = joblib.load(raw_pkl_path)
        result["ridge_s"] = raw.get("ridge_s") or raw.get("ridge_sbp")
        result["ridge_d"] = raw.get("ridge_d") or raw.get("ridge_dbp")
        result["iso_s"] = raw.get("iso_s"); result["iso_d"] = raw.get("iso_d")
        result["cls_head"] = raw.get("cls_head")
    return result


# ══════════════════════════════════════════════════════════════════════════════
# METRICS and STATISTICAL HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _bootstrap_stats(data, func, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(data)
    if n == 0: return np.nan, np.nan, np.nan
    point = func(data)
    boot = np.array([func(rng.choice(data, size=n, replace=True)) for _ in range(n_boot)])
    return point, float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))

def _bootstrap_vbvw(c_slow, subj, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    unique_subjs = np.unique(subj)

    point = _vbvw(c_slow, subj)   # SSoT: canonical _vbvw() from Part 1 (cell 3)
 # PERF FIX: see _bootstrap_icc — group once, compute directly from groups
 # each draw instead of concatenating + rescanning with `sb == s`.
    groups = {s: c_slow[subj == s] for s in unique_subjs}
    boot = []
    for _ in range(n_boot):
        bs = rng.choice(unique_subjs, size=len(unique_subjs), replace=True)
        v = _vbvw_from_groups([groups[s] for s in bs])
        if not np.isnan(v): boot.append(v)
    boot = np.array(boot)
    return point, float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))

# SSOT: identical implementation lives in bootstrap_cohens_d_ci (PART 5.2);
# kept as a thin alias so existing call sites are unchanged.
def _aami_bhs_metrics(errors):
    if len(errors) == 0:
        return dict(ME=np.nan, SD=np.nan, pct5=np.nan, pct10=np.nan, pct15=np.nan, BHS="—")
    ae = np.abs(errors)
    p5, p10, p15 = 100.0 * np.mean(ae <= 5), 100.0 * np.mean(ae <= 10), 100.0 * np.mean(ae <= 15)

    grade = _bhs_grade(p5, p10, p15)   # SSoT (cell 3); was hardcoded w/ p10>=60 (BHS-C bug -> now 65)

    return dict(ME=float(np.mean(errors)), SD=float(np.std(errors, ddof=1)), pct5=p5, pct10=p10, pct15=p15, BHS=grade)


# ══════════════════════════════════════════════════════════════════════════════
# UNIFIED EVALUATION LOGIC
# ══════════════════════════════════════════════════════════════════════════════

def _eval_split_metrics(sp, eval_mask, preds_dict, out_d=None, auc_val=np.nan, n_boot=1000):
    """Calculates all metrics (3 Stages MAE, Latents, AUC, AAMI) in a single loop."""
    res = {"AUC (HTN)": auc_val}
    m = eval_mask
    mae_fn = lambda err: np.mean(np.abs(err))

 # 1. Loop 3-Stage Prediction Metrics
    prefix_map = {"Raw": "U", "Pre": "G", "Post": "S"}
    for stage, pred in preds_dict.items():
        prefix = prefix_map.get(stage, stage[0])

        if not m.any() or pred is None:
            res.update({
                f"Beat SBP ({stage})": np.nan, f"SBP_{prefix}_CI": (np.nan, np.nan),
                f"Beat DBP ({stage})": np.nan, f"DBP_{prefix}_CI": (np.nan, np.nan),
                f"Subj SBP ({stage})": np.nan, f"Subj DBP ({stage})": np.nan,
            })
            continue

        sbp_err = pred[m, 0] - sp.Ybp[m, 0]
        dbp_err = pred[m, 1] - sp.Ybp[m, 1]

        sbp_mae, sbp_lo, sbp_hi = _bootstrap_stats(sbp_err, mae_fn, n_boot)
        dbp_mae, dbp_lo, dbp_hi = _bootstrap_stats(dbp_err, mae_fn, n_boot)
        subj_sbp_mae, subj_dbp_mae = subj_mae(pred, sp.Ybp, sp.subj, m)

        res.update({
            f"Beat SBP ({stage})": sbp_mae, f"SBP_{prefix}_CI": (sbp_lo, sbp_hi),
            f"Beat DBP ({stage})": dbp_mae, f"DBP_{prefix}_CI": (dbp_lo, dbp_hi),
            f"Subj SBP ({stage})": subj_sbp_mae, f"Subj DBP ({stage})": subj_dbp_mae,
        })

 # Calculate AAMI/BHS specifically for Subject Calibrated stage
        if stage == "Post":
            for k, v in _aami_bhs_metrics(sbp_err).items(): res[f"SBP {k}"] = v
            for k, v in _aami_bhs_metrics(dbp_err).items(): res[f"DBP {k}"] = v

 # Fallback if Post fails
    if "SBP ME" not in res:
        for k, v in _aami_bhs_metrics([]).items(): res[f"SBP {k}"] = v; res[f"DBP {k}"] = v

 # 2. Latent Metrics (For DiBP-Net)
    res.update({
        "corr(z_morph, z_hemo)": np.nan, "ICC_C(c_slow)": np.nan, "ICC CI lo": np.nan, "ICC CI hi": np.nan,
        "Vb/Vw(c_slow)": np.nan, "Vb/Vw CI lo": np.nan, "Vb/Vw CI hi": np.nan,
        "Cohen's d (HTN-2 vs Normal)": np.nan, "Cd CI lo": np.nan, "Cd CI hi": np.nan,
        "Cohen's d (R)": np.nan, "Cd_R CI lo": np.nan, "Cd_R CI hi": np.nan,
        "c_slow_mean": np.nan, "Gate mean": np.nan, "acc(z_hemo)": np.nan, "acc(z_morph)": np.nan, "Adversary acc": np.nan
    })

    if out_d is not None and m.any():
        try:
            subj_eval = sp.subj[m]
            c_slow, zm, zh = out_d["c_slow"][m], out_d["zm"][m], out_d["zh"][m]

            res["corr(z_morph, z_hemo)"] = corr_zm_zh_fn(zm, zh)
            res["ICC_C(c_slow)"], res["ICC CI lo"], res["ICC CI hi"] = bootstrap_icc_ci(c_slow, subj_eval, n_boot)
            res["Vb/Vw(c_slow)"], res["Vb/Vw CI lo"], res["Vb/Vw CI hi"] = _bootstrap_vbvw(c_slow, subj_eval, n_boot)
            res["c_slow_mean"] = float(c_slow.mean())
            res["Gate mean"] = float(out_d["gate"][m].mean())

 # Subj ID Classification
            def _acc(z):
                try:
                    unique = np.unique(subj_eval)
                    if len(unique) > TRAIN.subj_id_acc_n_cap:
                        chosen = np.random.default_rng(123).choice(unique, TRAIN.subj_id_acc_n_cap, replace=False)
                        mask = np.isin(subj_eval, chosen)
                        z_sub, s_sub = z[mask], subj_eval[mask]
                    else:
                        z_sub, s_sub = z, subj_eval
                    _, s_int = np.unique(s_sub, return_inverse=True)
                    sc = StandardScaler().fit_transform(z_sub if z_sub.ndim == 2 else z_sub.reshape(len(z_sub), -1))
                    return cross_val_score(KNeighborsClassifier(n_neighbors=5), sc, s_int, cv=min(TRAIN.subj_id_acc_cv, len(np.unique(s_int))), scoring='accuracy').mean()
                except Exception: return np.nan

            res["acc(z_hemo)"] = float(_acc(zh))
            res["acc(z_morph)"] = float(_acc(zm))

 # HTN-2 vs Normal Cohen's d (R and c_slow)
            _sl = np.unique(subj_eval)
            s_st = assign_stages(np.array([sp.Ybp[m, 0][subj_eval == s].mean() for s in _sl]),
                                 np.array([sp.Ybp[m, 1][subj_eval == s].mean() for s in _sl]))

            cs_norm = np.array([c_slow[subj_eval == s].mean() for s in _sl])[s_st == 1]
            cs_htn2 = np.array([c_slow[subj_eval == s].mean() for s in _sl])[s_st == 4]
            cs_norm, cs_htn2 = cs_norm[~np.isnan(cs_norm)], cs_htn2[~np.isnan(cs_htn2)]

            if len(cs_norm) >= 2 and len(cs_htn2) >= 2:
                res["Cohen's d (HTN-2 vs Normal)"], res["Cd CI lo"], res["Cd CI hi"] = bootstrap_cohens_d_ci(cs_htn2, cs_norm, n_boot)

            R_eval = out_d.get("R", np.full(m.sum(), np.nan))
            if len(R_eval) == len(sp.subj): R_eval = R_eval[m]

            r_norm = np.array([R_eval[subj_eval == s].mean() for s in _sl])[s_st == 1]
            r_htn2 = np.array([R_eval[subj_eval == s].mean() for s in _sl])[s_st == 4]
            r_norm, r_htn2 = r_norm[~np.isnan(r_norm)], r_htn2[~np.isnan(r_htn2)]

            if len(r_norm) >= 2 and len(r_htn2) >= 2:
                res["Cohen's d (R)"], res["Cd_R CI lo"], res["Cd_R CI hi"] = bootstrap_cohens_d_ci(r_htn2, r_norm, n_boot)
        except Exception:
            pass

    return res


def run_full_evaluation(mdl, calib_obj, is_baseline=False, raw_pkl_path=None, n_boot=1000) -> dict:
    """Unified pipeline runner for both DiBP-Net and Baseline models."""
    results = {}

    if is_baseline:
        calib_dict = _baseline_calib_as_dict(calib_obj, raw_pkl_path)
        if calib_dict.get("ridge_s") is None: return {}

    for split_name, sp in [("Train", SPLITS.train), ("Val", SPLITS.val), ("Test", SPLITS.test)]:
        out_d, auc_val = None, np.nan

        if is_baseline:
            _build_subj_cache_baseline(mdl, sp.X, sp.subj, sp.base)
            uncal = _get_raw_preds_baseline(mdl, sp.X, sp.subj, sp.base)
            gcal, feats = _get_primary_baseline(uncal, sp.base, calib_dict["ridge_s"], calib_dict["ridge_d"], calib_dict["iso_s"], calib_dict["iso_d"])
            scal, m = apply_subject_calibration(gcal, sp.Ybp, sp.subj)

            htn_true = ((sp.Ybp[:, 0] >= HTN_SBP) | (sp.Ybp[:, 1] >= HTN_DBP)).astype(int)
            cls_head = calib_dict.get("cls_head")
            eidx = np.where(m)[0]
            if len(np.unique(htn_true[eidx])) > 1 and cls_head is not None:
                prob = _cls_score_baseline(cls_head, gcal[eidx], feats[eidx])
                if not np.all(np.isnan(prob)): auc_val = float(roc_auc_score(htn_true[eidx], prob))
        else:
 # Raw: empty subject cache (pop_mean) — no personalization
            mdl._subj_z_cache = {}
            with _calib_ctx(calib_obj):
                _, uncal = get_raw_predictions(mdl, sp)
 # Cal: 3-beat subject cache + Ridge + Isotonic
            _build_subj_cache_for_split(mdl, sp)
            with _calib_ctx(calib_obj):
                gcal, _ = get_raw_predictions(mdl, sp)
            scal, m = apply_subject_calibration(gcal, sp.Ybp, sp.subj)

            try:
                out_d = extract_all_outputs(mdl, sp)
                with _calib_ctx(calib_obj):
                    _, _, auc_val = quick_metrics(mdl, sp, eval_mask=m)
            except Exception: pass

        preds_dict = {"Raw": uncal, "Pre": gcal, "Post": scal}
        res = _eval_split_metrics(sp, m, preds_dict, out_d, auc_val, n_boot)

        res["_preds"] = preds_dict
        res["_mask"] = m
        res["_out_d"] = out_d
        results[split_name] = res

    return results


# ══════════════════════════════════════════════════════════════════════════════
# UNIFIED CSV EXPORT and HELPER
# ══════════════════════════════════════════════════════════════════════════════

def export_beat_csv(eval_results, export_tag):
    """Unified CSV Dataframe export handling both Baselines and DiBP-Net."""
    export_dir = os.path.join(EXPORT_DIR, export_tag)
    os.makedirs(export_dir, exist_ok=True)

    for split_name, r in eval_results.items():
        if '_preds' not in r: continue

        sp = getattr(SPLITS, split_name.lower())
        m = r['_mask']
        preds = r['_preds']
        out_d = r.get('_out_d') or {}

        stages = assign_stages(sp.Ybp[:, 0], sp.Ybp[:, 1])
        nan_col = np.full(len(sp.X), np.nan, np.float32)

        u, g, s = preds.get("Raw"), preds.get("Pre"), preds.get("Post")
        if g is None: continue
        u = u if u is not None else np.column_stack((nan_col, nan_col))

        df = pd.DataFrame({
            "split": split_name.lower(),
            "subj_id": sp.subj,
            "stage": stages,
            "stage_label": [STAGE_LABELS.get(st, "?") for st in stages],
            "true_sbp": sp.Ybp[:, 0].astype(np.float32),
            "true_dbp": sp.Ybp[:, 1].astype(np.float32),
            "pred_sbp_raw": u[:, 0], "pred_dbp_raw": u[:, 1],
            "pred_sbp_pre": g[:, 0], "pred_dbp_pre": g[:, 1],
            "pred_sbp_post": s[:, 0], "pred_dbp_post": s[:, 1],
            "abs_err_sbp_raw": np.abs(u[:, 0] - sp.Ybp[:, 0]),
            "abs_err_dbp_raw": np.abs(u[:, 1] - sp.Ybp[:, 1]),
            "abs_err_sbp_pre": np.abs(g[:, 0] - sp.Ybp[:, 0]),
            "abs_err_dbp_pre": np.abs(g[:, 1] - sp.Ybp[:, 1]),
            "abs_err_sbp_post": np.abs(s[:, 0] - sp.Ybp[:, 0]),
            "abs_err_dbp_post": np.abs(s[:, 1] - sp.Ybp[:, 1]),
            "err_sbp_post": s[:, 0] - sp.Ybp[:, 0],
            "err_dbp_post": s[:, 1] - sp.Ybp[:, 1],
            "eval_window": m,
        })

        if out_d:
            df["R"] = out_d.get("R", nan_col)
            df["C_wk"] = out_d.get("C_wk", nan_col)
            df["c_slow"] = out_d.get("c_slow", nan_col)
            df["gate"] = out_d.get("gate", nan_col)

        df.to_csv(os.path.join(export_dir, f"beats_{split_name.lower()}.csv"), index=False)


def _build_master_rows(model_label, eval_results):
    rows = []
    _INTERNAL = {"_preds", "_out_d", "_mask"}
    for split_name in ["Train", "Val", "Test"]:
        if split_name not in eval_results: continue
        r = {k: v for k, v in eval_results[split_name].items() if k not in _INTERNAL}
        rows.append({"Model": model_label, "Split": split_name, **r})
    return rows


# ══════════════════════════════════════════════════════════════════════════════
# WINDKESSEL STAGE ANALYSIS and TABLES
# ══════════════════════════════════════════════════════════════════════════════

def _cohens_d_magnitude(d):
    if d is None or (isinstance(d, float) and np.isnan(d)): return "—"
    ad = abs(d)
    if ad < 0.2: return "negligible"
    if ad < 0.5: return "small"
    if ad < 0.8: return "medium"
    return "large"

def _stage_analysis(out_d: dict, sp, eval_mask: np.ndarray, n_boot: int = 1000):
    if out_d is None: return pd.DataFrame()
    m = eval_mask
    stages = assign_stages(sp.Ybp[:, 0], sp.Ybp[:, 1])
    rows = []

    for var_name, vals in {"R": out_d.get("R"), "C": out_d.get("C_wk"), "c_slow": out_d.get("c_slow")}.items():
        if vals is None: continue
        vals_m, stages_m, subj_m = vals[m], stages[m], sp.subj[m]

        subj_stage_vals = {}
        for s in np.unique(subj_m):
            sm = subj_m == s
            subj_stage_vals.setdefault(int(np.round(stages_m[sm].mean())), []).append(float(vals_m[sm].mean()))

        prev_subj_arr = None
        for stg in [1, 2, 3, 4]:
            beat_mask = stages_m == stg
            n_beats = int(beat_mask.sum())

            if n_beats == 0:
                rows.append({"variable": var_name, "stage": stg, "stage_label": STAGE_LABELS.get(stg, str(stg)), "n_beats": 0, "n_subj": 0, "beat_mean": np.nan, "beat_sd": np.nan, "subj_mean": np.nan, "subj_sd": np.nan, "p_vs_prev": np.nan, "cd_vs_prev": np.nan, "cd_ci_lo": np.nan, "cd_ci_hi": np.nan, "cd_magnitude": "—"})
                prev_subj_arr = None; continue

            beat_vals = vals_m[beat_mask]
            s_arr = np.array(subj_stage_vals.get(stg, []), dtype=float)
            s_arr = s_arr[~np.isnan(s_arr)]
            n_subj = len(s_arr)

            p_val, cd, cd_lo, cd_hi, cd_mag = np.nan, np.nan, np.nan, np.nan, "—"
            if prev_subj_arr is not None and len(prev_subj_arr) >= 2 and n_subj >= 2:
                cd, cd_lo, cd_hi = bootstrap_cohens_d_ci(s_arr, prev_subj_arr, n_boot)
                cd_mag = _cohens_d_magnitude(cd)
                try:
                    _, p_val = scipy_stats.ttest_ind(s_arr, prev_subj_arr, equal_var=False)
                    p_val = float(p_val)
                except Exception: p_val = np.nan

            rows.append({"variable": var_name, "stage": stg, "stage_label": STAGE_LABELS.get(stg, str(stg)), "n_beats": n_beats, "n_subj": n_subj, "beat_mean": float(beat_vals.mean()), "beat_sd": float(beat_vals.std(ddof=1)) if n_beats > 1 else np.nan, "subj_mean": float(s_arr.mean()) if n_subj > 0 else np.nan, "subj_sd": float(s_arr.std(ddof=1)) if n_subj > 1 else np.nan, "p_vs_prev": p_val, "cd_vs_prev": cd, "cd_ci_lo": cd_lo, "cd_ci_hi": cd_hi, "cd_magnitude": cd_mag})
            prev_subj_arr = s_arr

    return pd.DataFrame(rows)


def print_stage_analysis(stage_df: pd.DataFrame, model_label: str):
    if stage_df.empty: return
    W = 108
    print(f"\n  {'█'*W}\n  WINDKESSEL STAGE ANALYSIS — {model_label}  (Test Set)\n  {'█'*W}")
    for var in ["R", "C", "c_slow"]:
        sub = stage_df[stage_df["variable"] == var]
        if sub.empty: continue
        print(f"\n  {'─'*W}\n  {var}\n  {'─'*W}")
        print((f"  {'Stage':<10} {'Beat mean±SD':>15} {'Subj mean±SD':>15} {'N_b':>6} {'N_s':>5} "
               f"{'p (vs prev)':>12} {'Cd vs prev [CI]':>22} {'Magnitude':>11}\n  {'─'*W}"))
        for _, row in sub.iterrows():
            bm = f"{row.beat_mean:+.3f}±{row.beat_sd:.3f}" if not np.isnan(row.beat_mean) else "—"
            sm = f"{row.subj_mean:+.3f}±{row.subj_sd:.3f}" if not np.isnan(row.subj_mean) else "—"
            p_str = "<0.001" if row.p_vs_prev < 0.001 else f"{row.p_vs_prev:.3f}" if not np.isnan(row.p_vs_prev) else "—"
            cd_str = f"{row.cd_vs_prev:+.3f} [{row.cd_ci_lo:+.3f},{row.cd_ci_hi:+.3f}]" if not np.isnan(row.cd_ci_lo) else f"{row.cd_vs_prev:+.3f}" if not np.isnan(row.cd_vs_prev) else "—"
            print(f"  {str(row.stage_label):<10} {bm:>15} {sm:>15} {int(row.n_beats):>6} {int(row.n_subj):>5} {p_str:>12} {cd_str:>22} {row.cd_magnitude:>11}")
    print("  " + "█" * W)


# ══════════════════════════════════════════════════════════════════════════════
# MASTER SUMMARY TABLE PRINTER
# ══════════════════════════════════════════════════════════════════════════════

_COL_DEFS = [
    ("BeatSBP-U", "Beat SBP (Raw)", 9, "f2"), ("SBP-U-CI", "SBP_U_CI", 13, "ci_tuple"),
    ("BeatDBP-U", "Beat DBP (Raw)", 9, "f2"), ("DBP-U-CI", "DBP_U_CI", 13, "ci_tuple"),
    ("BeatSBP-G", "Beat SBP (Pre)", 9, "f2"),   ("SBP-G-CI", "SBP_G_CI", 13, "ci_tuple"),
    ("BeatDBP-G", "Beat DBP (Pre)", 9, "f2"),   ("DBP-G-CI", "DBP_G_CI", 13, "ci_tuple"),
    ("BeatSBP-S", "Beat SBP (Post)", 9, "f2"),     ("SBP-S-CI", "SBP_S_CI", 13, "ci_tuple"),
    ("BeatDBP-S", "Beat DBP (Post)", 9, "f2"),     ("DBP-S-CI", "DBP_S_CI", 13, "ci_tuple"),

    ("SubjSBP-U", "Subj SBP (Raw)", 9, "f2"), ("SubjDBP-U", "Subj DBP (Raw)", 9, "f2"),
    ("SubjSBP-G", "Subj SBP (Pre)", 9, "f2"),   ("SubjDBP-G", "Subj DBP (Pre)", 9, "f2"),
    ("SubjSBP-S", "Subj SBP (Post)", 9, "f2"),     ("SubjDBP-S", "Subj DBP (Post)", 9, "f2"),

    ("AUC", "AUC (HTN)", 7, "f4"),
    ("SBP-ME", "SBP ME", 7, "sgn"), ("SBP-SD", "SBP SD", 7, "f2"), ("SBP BHS", "SBP BHS", 7, "str"),
    ("DBP-ME", "DBP ME", 7, "sgn"), ("DBP-SD", "DBP SD", 7, "f2"), ("DBP BHS", "DBP BHS", 7, "str"),
    ("corr(zm,zh)", "corr(z_morph, z_hemo)", 11, "sgn"),
    ("ICC", "ICC_C(c_slow)", 8, "f4"), ("ICC-CI", None, 15, "ci_icc"),
    ("Vb/Vw", "Vb/Vw(c_slow)", 7, "f1"), ("Vb/Vw-CI", None, 13, "ci_vbvw"),
    ("Cd", "Cohen's d (HTN-2 vs Normal)", 7, "f3"), ("Cd-CI", None, 13, "ci_cd"),
    ("Cd(R)", "Cohen's d (R)", 7, "f3"), ("Cd(R)-CI", None, 13, "ci_cdR"),
    ("c_slow", "c_slow_mean", 7, "f3"), ("Gate", "Gate mean", 6, "f3"),
    ("acc(zh)", "acc(z_hemo)", 8, "f4"), ("acc(zm)", "acc(z_morph)", 8, "f4"), ("Adv acc", "Adversary acc", 8, "f4"),
]

def print_master_summary(combined: pd.DataFrame, n_boot_used: int = 1000):
    df = combined.copy()
    df["_so"] = df["Split"].map({"Train": 1, "Val": 2, "Test": 3}).fillna(9).astype(int)
    df["_mo"] = df["Model"].map(lambda x: {"GRU": 1, "ResNet-1D": 2, "Transformer": 3, "AblNoDis": 4, "AblA-HSIConly": 5, "AblB-cSlow": 6, "DiBP-Net-v193 (GRL)": 7, "DiBP-Net-v192": 8}.get(x, 99))
    df = df.sort_values(["_so", "_mo"]).drop(columns=["_so", "_mo"])

    aw = [max(len(h), dw) for h, _, dw, _ in _COL_DEFS]; MW = 22; SW = 6
    header = "".join([f"  {'Model':<{MW}} {'Split':<{SW}}"] + [f" {h:>{aw[i]}}" for i, (h, _, _, _) in enumerate(_COL_DEFS)])
    sep = "  " + "─" * (len(header) - 2)

    print(f"\n  {'█'*(len(header)-2)}\n  MASTER SUMMARY TABLE (U=Raw | G=Pre | S=Post | CI n={n_boot_used})\n  {'█'*(len(header)-2)}\n{header}\n{sep}")
    prev = None
    for _, row in df.iterrows():
        if row["Split"] != prev and prev: print(sep)
        prev, r = row["Split"], row.to_dict()
        line = [f"  {str(r['Model'])[:MW-1]:<{MW}} {str(r['Split']):<{SW}}"]
        for i, (_, key, _, fmt) in enumerate(_COL_DEFS):
            val = r.get(key, np.nan) if key else np.nan

 # Formatting Logic
            if fmt == "ci_icc":
                v_str = f"[{r.get('ICC CI lo', np.nan):.4f},{r.get('ICC CI hi', np.nan):.4f}]" if not np.isnan(r.get('ICC CI lo', np.nan)) else "—"
            elif fmt == "ci_vbvw":
                v_str = f"[{r.get('Vb/Vw CI lo', np.nan):.0f},{r.get('Vb/Vw CI hi', np.nan):.0f}]" if not np.isnan(r.get('Vb/Vw CI lo', np.nan)) else "—"
            elif fmt in ("ci_cd", "ci_cdR"):
                if fmt == "ci_cd":
                    lo_key, hi_key = "Cd CI lo", "Cd CI hi"
                else:
                    lo_key, hi_key = "Cd_R CI lo", "Cd_R CI hi"
                v_str = f"[{r.get(lo_key, np.nan):.3f},{r.get(hi_key, np.nan):.3f}]" if not np.isnan(r.get(lo_key, np.nan)) else "—"
            elif fmt == "ci_tuple":
                v_str = f"[{val[0]:.2f},{val[1]:.2f}]" if isinstance(val, tuple) and len(val) == 2 and not np.isnan(val[0]) else "—"
            elif fmt == "str" or (isinstance(val, float) and np.isnan(val)):
                v_str = str(val) if val == val and val != "—" else "—"
            else:
                v_str = f"{float(val):>{aw[i]}.{fmt[-1]}f}" if fmt.startswith("f") else f"{float(val):>+{aw[i]}.3f}"

            line.append(f" {v_str:>{aw[i]}}")
        print("".join(line))
    print(f"  {'█'*(len(header)-2)}\n")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN RUNNER
# ══════════════════════════════════════════════════════════════════════════════

def _compute_adversary_acc(mdl_v193, split) -> float:
    try:
        mdl_v193.eval()
        ap, at = [], []
        with torch.no_grad():
            for batch in DataLoader(_make_tensor_dataset(split), batch_size=ARCH.batch, shuffle=False, num_workers=0):
                bx, bq, _, _, _, bsubj, _, _, bbase = _to_device(batch)
                out = mdl_v193(bx, bsubj, bbase, bq[:, ARCH.n_beats // 2].squeeze(1) if bq[:, ARCH.n_beats // 2].dim() == 3 else bq[:, ARCH.n_beats // 2], 0.0)
                ap.append(out[21].argmax(-1).cpu().numpy()); at.append((bsubj % mdl_v193._adv_classes).cpu().numpy())
        return float((np.concatenate(ap) == np.concatenate(at)).mean())
    except Exception: return np.nan


_DIBPNET_EVAL_ORDER = [
    ("v192", "DiBP-Net-v192"), ("v192nodis", "AblNoDis"), ("ablA", "AblA-HSIConly"),
    ("ablB", "AblB-cSlow"), ("v193", "DiBP-Net-v193 (GRL)"),
]
_BASELINE_EVAL_ORDER = [("gru", "GRU"), ("resnet", "ResNet-1D"), ("transformer", "Transformer"), ("supcon", "SupCon")]


def evaluate_dibpnet_models(n_boot: int = 1000) -> tuple:
    """Evaluate v192 family + v193, export per-model CSVs, run stage analysis
    on v192's Test split. Returns (master_rows, stage_results)."""
    rows = []
    stage_results = {}

    for key, label in _DIBPNET_EVAL_ORDER:
        mdl, calib = MODEL_REGISTRY.get(key, (None, None))
        if mdl is None: continue
        print(f"  → {label} ...", flush=True, end="")
        try:
            res = run_full_evaluation(mdl, calib, is_baseline=False, n_boot=n_boot)
            if key == "v193":
                for sn, sp in [("Val", SPLITS.val), ("Test", SPLITS.test)]:
                    if sn in res: res[sn]["Adversary acc"] = _compute_adversary_acc(mdl, sp)
            if key == "v192" and "Test" in res and res["Test"]["_out_d"]:
                stage_results[label] = _stage_analysis(res["Test"]["_out_d"], SPLITS.test, res["Test"]["_mask"], n_boot=n_boot)

            rows.extend(_build_master_rows(label, res))
            export_beat_csv(res, key)
            pd.DataFrame(_build_master_rows(label, res)).to_csv(
                os.path.join(EXPORT_DIR, key, "overall_summary.csv"), index=False)
            if label in stage_results and not stage_results[label].empty:
                stage_results[label].to_csv(os.path.join(EXPORT_DIR, key, "stage_analysis_test.csv"), index=False)

            hist = {k: v for k, v in HIST_REGISTRY.get(key, {}).items() if isinstance(v, list) and v}
            if hist:
                pd.DataFrame({k: list(v) + [None] * (max(len(x) for x in hist.values()) - len(v))
                              for k, v in hist.items()}).to_csv(
                    os.path.join(EXPORT_DIR, key, "training_history.csv"),
                    index=True, index_label="checkpoint_ep")
            print(" ✅")
        except Exception as _e:
            print(f" ⚠️  {_e}")

    return rows, stage_results


def evaluate_baseline_models(n_boot: int = 1000) -> list:
    """Evaluate GRU / ResNet-1D / Transformer, export per-model CSVs.
    Returns master_rows."""
    rows = []

    for key, label in _BASELINE_EVAL_ORDER:
        mdl, calib_obj = MODEL_REGISTRY.get(key, (None, None))
        if mdl is None: continue
        print(f"  → {label} ...", flush=True, end="")
        try:
            res = run_full_evaluation(
                mdl, calib_obj, is_baseline=True,
                raw_pkl_path=os.path.join(OUT_DIR, f"checkpoints_{key}", f"calibration_{key}.pkl"),
                n_boot=n_boot)
            if not res: continue
            rows.extend(_build_master_rows(label, res))
            export_beat_csv(res, key)
            pd.DataFrame(_build_master_rows(label, res)).to_csv(
                os.path.join(EXPORT_DIR, key, "overall_summary.csv"), index=False)
            _bl_hist = {k: v for k, v in HIST_REGISTRY.get(key, {}).items() if isinstance(v, list) and v}
            if _bl_hist:
                _max_len = max(len(v) for v in _bl_hist.values())
                pd.DataFrame({k: list(v) + [None] * (_max_len - len(v)) for k, v in _bl_hist.items()}).to_csv(
                    os.path.join(EXPORT_DIR, key, "training_history.csv"),
                    index=True, index_label="checkpoint_ep")
            print(" ✅")
        except Exception as _e:
            print(f" ⚠️  {_e}")

    return rows


def finalize_outputs(all_master_rows: list, stage_results: dict, n_boot: int = 1000) -> pd.DataFrame:
    """Write master CSV + Excel workbook, print master summary table +
    v192 stage analysis. Returns the combined master DataFrame."""
    master_df = pd.DataFrame(all_master_rows)
    master_df.to_csv(os.path.join(OUT_DIR, "master_ablation_summary.csv"), index=False)
    print(f"\n  ✅ Master CSV   → {os.path.join(OUT_DIR, 'master_ablation_summary.csv')}")

    try:
        with pd.ExcelWriter(os.path.join(OUT_DIR, "master_ablation_summary.xlsx"), engine="openpyxl") as _w:
            master_df.to_excel(_w, sheet_name="Combined_Summary", index=False)
            for _lbl in master_df["Model"].unique():
                master_df[master_df["Model"] == _lbl].drop(columns=["Model"]).to_excel(_w, sheet_name=str(_lbl)[:31], index=False)
            for _lbl, _sdf in stage_results.items():
                if not _sdf.empty:
                    _sdf.to_excel(_w, sheet_name=f"Stage_{str(_lbl)[:25]}", index=False)
        print(f"  ✅ Master Excel → {os.path.join(OUT_DIR, 'master_ablation_summary.xlsx')}")
    except Exception as _e:
        print(f"  ⚠️  Excel export failed: {_e}")

    print_master_summary(master_df, n_boot_used=n_boot)
    if "DiBP-Net-v192" in stage_results and not stage_results["DiBP-Net-v192"].empty:
        print_stage_analysis(stage_results["DiBP-Net-v192"], "DiBP-Net-v192")

    return master_df


def run_cell3_evaluation(n_boot: int = 1000) -> dict:
    """Main entrypoint for Cell 3: check prerequisites → evaluate every
    DiBP-Net + baseline model → export CSV/Excel → print summary tables.
    Returns {'master_df', 'all_master_rows', 'stage_results'}."""
    check_prerequisites()
    print(f"\n  Evaluating Models (bootstrap n={n_boot}) ...", flush=True)

    dibpnet_rows, stage_results = evaluate_dibpnet_models(n_boot)
    baseline_rows = evaluate_baseline_models(n_boot)
    all_master_rows = dibpnet_rows + baseline_rows

    master_df = finalize_outputs(all_master_rows, stage_results, n_boot)

    return {
        "master_df": master_df,
        "all_master_rows": all_master_rows,
        "stage_results": stage_results,
    }

### Visualization

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4-A — IMPORTS & PATH SETUP
# ══════════════════════════════════════════════════════════════════════════════

import __main__ as _main_fig

BASE_DIR   = getattr(_main_fig, "OUT_DIR",     "outputs/phase_2_v192")
EXPORT_DIR = getattr(_main_fig, "EXPORT_DIR",  os.path.join(BASE_DIR, "exports"))
SAVE_DIR   = getattr(_main_fig, "FIG_DIR",     os.path.join(BASE_DIR, "figures"))
DATA_PATH  = getattr(_main_fig, "DATA_PATH",   "outputs/phase_1/data/universal_tensors.npz")
CKPT_DIR   = getattr(_main_fig, "CKPT_DIR",    os.path.join(BASE_DIR, "checkpoints"))
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"[4-A] Paths ready. SAVE_DIR = {SAVE_DIR}")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 4-B — FONT SETUP & MATPLOTLIB RC PARAMS
# ══════════════════════════════════════════════════════════════════════════════

_COL1_IN = 3.5          # IEEE single-column width (inches)
_TNR     = ["Times New Roman", "Georgia", "DejaVu Serif"]
_TNR_NAME = "Times New Roman"
_FS      = 8          # global base font size (pt)

def _ensure_times_new_roman():
    import urllib.request
    import matplotlib.font_manager as fm
    if "Times New Roman" in {f.name for f in fm.fontManager.ttflist}:
        return
    dst_dir = "/usr/share/fonts/truetype/msttcorefonts"
    os.makedirs(dst_dir, exist_ok=True)
    variants = [
        ("Times_New_Roman.ttf",
         "https://github.com/justrajdeep/fonts/raw/master/Times%20New%20Roman.ttf"),
        ("Times_New_Roman_Bold.ttf",
         "https://github.com/justrajdeep/fonts/raw/master/Times%20New%20Roman%20Bold.ttf"),
        ("Times_New_Roman_Italic.ttf",
         "https://github.com/justrajdeep/fonts/raw/master/Times%20New%20Roman%20Italic.ttf"),
        ("Times_New_Roman_Bold_Italic.ttf",
         "https://github.com/justrajdeep/fonts/raw/master/Times%20New%20Roman%20Bold%20Italic.ttf"),
    ]
    downloaded = 0
    for fname, url in variants:
        dst = os.path.join(dst_dir, fname)
        if not os.path.exists(dst):
            try:
                urllib.request.urlretrieve(url, dst); downloaded += 1
            except Exception:
                pass
    if downloaded:
        import subprocess
        subprocess.run(["fc-cache", "-f"], capture_output=True)
        cache = mpl.get_cachedir()
        for cf in os.listdir(cache):
            if cf.startswith("fontlist") and cf.endswith(".json"):
                try: os.remove(os.path.join(cache, cf))
                except OSError: pass
        fm.fontManager.__init__()

_ensure_times_new_roman()

plt.rcParams.update({
    "font.family":           "serif",
    "font.serif":            _TNR,
    "font.size":             _FS,
    "text.usetex":           False,
    "axes.titlesize":        _FS,
    "axes.titleweight":      "bold",
    "axes.titlepad":         6,
    "axes.labelsize":        _FS,
    "axes.labelweight":      "normal",
    "axes.labelpad":         4,
    "xtick.labelsize":       _FS,
    "ytick.labelsize":       _FS,
    "xtick.direction":       "in",
    "ytick.direction":       "in",
    "xtick.major.size":      3.5,
    "ytick.major.size":      3.5,
    "xtick.major.pad":       3,
    "ytick.major.pad":       3,
    "legend.fontsize":       _FS - 1,
    "legend.title_fontsize": _FS - 1,
    "legend.framealpha":     0.95,
    "legend.edgecolor":      "#B0BEC5",
    "figure.titlesize":      _FS,
    "figure.titleweight":    "bold",
    "mathtext.fontset":      "stix",
    "mathtext.rm":           "serif",
    "mathtext.it":           "serif:italic",
    "mathtext.bf":           "serif:bold",
    "axes.spines.top":       False,
    "axes.spines.right":     False,
    "axes.linewidth":        0.8,
    "axes.edgecolor":        "#333333",
    "axes.grid":             True,
    "grid.alpha":            0.25,
    "grid.linestyle":        "--",
    "grid.linewidth":        0.6,
    "grid.color":            "#AAAAAA",
    "lines.linewidth":       1.5,
    "lines.markersize":      5,
    "figure.dpi":            150,
    "savefig.dpi":           400,
    "savefig.bbox":          "tight",
    "savefig.pad_inches":    0.08,
})

# IEEE paper figure style — use with plt.rc_context(_IEEE_RC):
_IEEE_RC = {
    "font.family":           "serif",
    "font.serif":            _TNR,
    "font.size":             8,
    "axes.titlesize":        8,
    "axes.titleweight":      "bold",
    "axes.labelsize":        8,
    "xtick.labelsize":       8,
    "ytick.labelsize":       8,
    "legend.fontsize":       8,
    "legend.title_fontsize": 8,
    "mathtext.fontset":      "stix",
    "mathtext.rm":           "serif",
    "mathtext.it":           "serif:italic",
    "mathtext.bf":           "serif:bold",
    "savefig.dpi":           400,
    "figure.dpi":            150,
}

print(f"[4-B] Font & RC params ready. Base font size = {_FS} pt")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 4-C — GLOBAL STYLE, PALETTE & CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════
STAGE_ORDER   = ["Normal", "Elevated", "HTN-1", "HTN-2"]
STAGE_PALETTE = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]
C_VAL, C_TEST, C_MED = "#1D4E89", "#C53030", "#C53030"

C_ARCH = {
    "input":   "#546E7A", "encoder": "#1565C0",
    "morph":   "#2E7D32", "hemo":    "#C62828",
    "slow":    "#6A1B9A", "bp":      "#E65100",
    "concat":  "#37474F", "hsic":    "#F57F17",
    "arrow":   "#263238", "stop":    "#FFCDD2",
    "stop_bg": "#FFCDD2",          # ← background fill for stop-grad box
    "stop_tc": "#B71C1C",          # ← text/border colour for stop-grad box
}

STYLE = {
    "nodis":    ("#6C757D", (0, (4, 2)), 1.6),
    "ablA":     ("#D62728", "--",        1.6),
    "ablB":     ("#1F77B4", "-.",        1.6),
    "proposed": ("#2CA02C", "-",         2.2),
    "grl":      ("#9467BD", (0, (5, 1, 1, 1)), 1.6),
}
LABEL = {
    "nodis":    "NoDis",
    "ablA":     r"Abl.A — w/o $c_\mathrm{slow}$",
    "ablB":     "Abl.B — w/o anchor",
    "proposed": "Proposed (Full)",
    "grl":      "Proposed + GRL",
}
MODEL_TAGS = {"nodis": "v192nodis", "ablA": "ablA",
              "ablB": "ablB", "proposed": "v192", "grl": "v193"}

JITTER = {
    ("nodis","best"): -2, ("ablB","best"): +2, ("ablA","best"): -3,
    ("proposed","best"): 0, ("grl","best"): +3,
    ("nodis","stop"): -2, ("ablB","stop"): +2, ("ablA","stop"): -3,
    ("proposed","stop"): 0, ("grl","stop"): +3,
}

TARGET_CORR = getattr(_main_fig, "TARGET_CORR", 0.30)
ICC_TARGET  = getattr(_main_fig, "ICC_TARGET",  0.80)
CD_TARGET   = getattr(_main_fig, "CD_TARGET",   1.0)

print("[4-C] Style, palette & constants ready.")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 4-D — DATA LOADERS
# ══════════════════════════════════════════════════════════════════════════════

def _load_training_history(tag):
    primary = os.path.join(EXPORT_DIR, tag, "training_history.csv")
    if not os.path.exists(primary): return pd.DataFrame()
    try:
        df = pd.read_csv(primary, index_col=0, dtype=str).reset_index(drop=True)
        ep_col = "ep10_global" if "ep10_global" in df.columns else df.columns[0]
        if ep_col != "ep10_global": df = df.rename(columns={ep_col: "ep10_global"})
        df["ep10_global"] = pd.to_numeric(df["ep10_global"], errors="coerce")
        df = df[df["ep10_global"].notna()].copy().reset_index(drop=True)
        df["ep10_global"] = df["ep10_global"].astype(float).round(0).astype(int)
        for col in df.columns:
            if col not in ("ep10_global", "phase", "phase_ep1"):
                df[col] = pd.to_numeric(df[col], errors="coerce")
        return df
    except Exception: return pd.DataFrame()


def _load_overall_summary(tag, split="test"):
    path = os.path.join(EXPORT_DIR, tag, "overall_summary.csv")
    if not os.path.exists(path): return {}
    try:
        df  = pd.read_csv(path)
        row = df[df["Split"].str.lower() == split.lower()]
        return row.iloc[0].to_dict() if len(row) else {}
    except Exception: return {}


def load_beats(split, tag="v192", eval_only=True):
    path = os.path.join(EXPORT_DIR, tag, f"beats_{split}.csv")
    if not os.path.exists(path):
        legacy = os.path.join(EXPORT_DIR, tag, f"subjects_{split}.csv")
        if os.path.exists(legacy): path = legacy
        else: return pd.DataFrame()
    try:
        df = pd.read_csv(path)
        if eval_only and "eval_window" in df.columns:
            df = df[df["eval_window"] == True].copy().reset_index(drop=True)
        return df
    except Exception: return pd.DataFrame()


def _resolve_col(df, possible_names):
    for col in possible_names:
        if col in df.columns: return col
    return None


def _best_stop_from_history(tag):
    df = _load_training_history(tag)
    if df.empty: return None, None
    ep_col    = "ep10_global" if "ep10_global" in df.columns else df.columns[0]
    valid_ep  = df.dropna(subset=[ep_col]).reset_index(drop=True)
    if valid_ep.empty: return None, None
    early_stop = int(round(float(valid_ep[ep_col].iat[-1])))
    search_df  = (valid_ep[valid_ep[ep_col] > 80].reset_index(drop=True)
                  if early_stop > 80 else valid_ep.copy())
    best_ckpt, _THRESH = None, 0.05
    for cname in ["sbp_mae", "val_sbp_mae", "val_mae", "mae"]:
        if cname in search_df.columns:
            _s = pd.to_numeric(search_df[cname], errors="coerce").values
            best_score = float("inf")
            for sc, ep in zip(_s, search_df[ep_col].values):
                if pd.notna(sc) and sc < best_score - _THRESH:
                    best_score, best_ckpt = sc, int(round(float(ep)))
            if best_ckpt is not None: break
    return best_ckpt if best_ckpt is not None else early_stop, early_stop


def _build_model_data(key):
    tag = MODEL_TAGS[key]
    df  = _load_training_history(tag)
    EP  = list(range(10, 170, 10))

    def _col(name):
        if df.empty or name not in df.columns: return [None] * len(EP)
        ep_col = "ep10_global" if "ep10_global" in df.columns else df.columns[0]
        ep_map = {}
        for i in range(len(df)):
            try:   ek = int(round(float(df.iat[i, df.columns.get_loc(ep_col)])))
            except: continue
            try:   ep_map[ek] = (None if pd.isna(df.iat[i, df.columns.get_loc(name)])
                                 else float(df.iat[i, df.columns.get_loc(name)]))
            except: ep_map[ek] = None
        return [ep_map.get(ep) for ep in EP]

    b, s = _best_stop_from_history(tag)
    return dict(sbp=_col("sbp_mae"), corr=_col("corr_zm_zh"),
                icc=_col("icc_cs"),  cd=_col("c_d_cs_h2n"),
                best_ckpt=b, early_stop=s)


_MODELS_CACHE = []

def _build_all_models():
    global _MODELS_CACHE
    if not _MODELS_CACHE:
        _MODELS_CACHE = [(k, _build_model_data(k))
                         for k in ["nodis", "ablA", "ablB", "proposed", "grl"]]
    return _MODELS_CACHE


def _load_icc_vbvw_trajectory(tag="v192"):
    df = _load_training_history(tag)
    if df.empty: return [], [], []
    ep_col = "ep10_global" if "ep10_global" in df.columns else df.columns[0]
    epochs = [int(v) for v in df[ep_col]]
    n      = len(epochs)
    icc    = [float(v) if pd.notna(v) else np.nan for v in df.get("icc_cs",   [np.nan]*n)]
    vbvw   = [float(v) if pd.notna(v) else np.nan for v in df.get("vb_vw_cs", [np.nan]*n)]
    return epochs, icc, vbvw


print("[4-D] Data loaders ready.")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 4-E — UTILITY HELPERS & SHARED PLOT FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def _nan(arr):
    return np.ma.masked_invalid(
        np.array([np.nan if v is None else float(v) for v in arr]))


def _scalar_at(arr, ep, EP_list):
    if ep not in EP_list: return np.nan
    v = arr[EP_list.index(ep)]
    return np.nan if v is None else float(v)


def _phase_line(ax, x=80):
    ax.axvline(x, color="#666", linestyle=":", linewidth=1.0, alpha=0.65, zorder=2)
    ax.text(x + 1.5, 0.97, "P2", transform=ax.get_xaxis_transform(),
            fontsize=_FS, color="#666", va="top", ha="left",
            fontname=_TNR_NAME)


def _add_markers_metric(ax, key, data, metric, EP):
    col = STYLE[key][0]
    for kind, marker, sz in (("best", "*", 120), ("stop", "X", 80)):
        ep = data.get("best_ckpt") if kind == "best" else data.get("early_stop")
        if ep is None or ep not in EP: continue
        v = _scalar_at(data.get(metric, []), ep, EP)
        if not np.isnan(v):
            ax.scatter(ep + JITTER.get((key, kind), 0), v,
                       color=col, marker=marker, s=sz, zorder=6,
                       edgecolors="k", linewidths=0.5)


def staple_bracket(ax, x1, x2, y, d_val, h=0.12, fs=_FS):
    ax.plot([x1, x1, x2, x2], [y - h, y, y, y - h], color="black", lw=1.2)
    ax.text((x1 + x2) / 2, y + 0.04, f"$d$={d_val:.2f}",
            ha="center", va="bottom", fontsize=fs,
            fontweight="bold", fontname=_TNR_NAME)


def _legend_handles():
    mh = [Line2D([0], [0], color=STYLE[k][0], linestyle=STYLE[k][1],
                 linewidth=STYLE[k][2], label=LABEL[k])
          for k in ["nodis", "ablA", "ablB", "proposed", "grl"]]
    eh = [
        Line2D([0], [0], linestyle="none", marker="*", markersize=9,
               color="k", markeredgewidth=0.4, label="Best checkpoint"),
        Line2D([0], [0], linestyle="none", marker="X", markersize=7,
               color="k", markeredgewidth=0.4, label="Early stop"),
        mpatches.Patch(color="#2CA02C", alpha=0.15, label="Target zone"),
    ]
    return mh, eh


def _scatter_panel(ax, true_vals, pred_vals, color, title, lim):
    ax.scatter(true_vals, pred_vals, s=3, alpha=0.25, color=color, rasterized=True)
    lo, hi = lim
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1.0, alpha=0.6)
    try:
        r, _  = pearsonr(true_vals, pred_vals)
        mae   = float(np.mean(np.abs(pred_vals - true_vals)))
        ax.text(0.05, 0.93, f"$r$={r:.3f}\nMAE={mae:.2f}",
                transform=ax.transAxes, va="top",
                bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.85))
    except Exception: pass
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel("True (mmHg)")
    ax.set_ylabel("Predicted (mmHg)")
    ax.set_title(title, fontweight="bold")


def _bland_altman_panel(ax, true_vals, pred_vals, color, title, lim):
    diff  = pred_vals - true_vals
    mean_ = (pred_vals + true_vals) / 2.0
    bias  = float(np.mean(diff))
    sd    = float(np.std(diff, ddof=1))
    lo_a, hi_a = bias - 1.96 * sd, bias + 1.96 * sd

    ax.scatter(mean_, diff, s=2, alpha=0.20, color=color, rasterized=True)
    ax.axhline(bias, color="k",       lw=1.0, ls="-",  zorder=3)
    ax.axhline(hi_a, color="#E65100", lw=0.8, ls="--", zorder=3)
    ax.axhline(lo_a, color="#E65100", lw=0.8, ls="--", zorder=3)
    ax.axhline(0,    color="#999",    lw=0.6, ls=":",  zorder=2)

    ax.set_xlim(lim)

    txt = (f"Bias={bias:+.2f}\n"
           f"+1.96SD={hi_a:.2f}\n"
           f"\u22121.96SD={lo_a:.2f}")
    ax.text(0.03, 0.97, txt,
            transform=ax.transAxes,
            va="top", ha="left", fontsize=8, linespacing=1.4,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.85, lw=0.5))

    ax.set_xlabel("Mean (mmHg)")
    ax.set_ylabel("Pred \u2212 True (mmHg)")
    ax.set_title(title, fontweight="bold")


def _save(fig, stem: str):
    for artist in fig.findobj(plt.Text):
        try:
            if artist.get_text():
                artist.set_fontname(_TNR_NAME)
                if artist.get_fontsize() > _FS:
                    artist.set_fontsize(_FS)
        except Exception:
            pass
    p = os.path.join(SAVE_DIR, f"{stem}.png")
    fig.savefig(p, bbox_inches="tight", pad_inches=0.08,
                facecolor=fig.get_facecolor())
    print(f"  ✅  saved → {p}")
    plt.close(fig)

def _load_test_subject_frame():
    """Shared loader: per-subject mean SBP/DBP on the test split."""
    df_test = load_beats("test", tag="v192", eval_only=False)
    sub_col = _resolve_col(df_test, ("subject_id", "subj_id", "subject", "Subject_ID"))
    sbp_col = _resolve_col(df_test, ("true_sbp", "sbp_true", "true_sbp_eval_mean"))
    dbp_col = _resolve_col(df_test, ("true_dbp", "dbp_true", "true_dbp_eval_mean"))

    if not sub_col or not sbp_col or not dbp_col:
        return None

    df_subj = df_test.groupby(sub_col).agg(
        mean_sbp=(sbp_col, "mean"),
        mean_dbp=(dbp_col, "mean"),
    ).reset_index()
    return df_subj

### Figure Definitions

In [ ]:
def build_figure_0():
    try:
        if not os.path.exists(DATA_PATH):
            print(f"  ⚠️  DATA_PATH not found: {DATA_PATH}")
            return

        data = np.load(DATA_PATH)
        sbp_arr = data["Y_bp"][:, 0]

        FS = 8
        TNR = _TNR[0]

        fig, ax_h = plt.subplots(figsize=(3.5, 2.5))

        sns.histplot(sbp_arr,bins=35,kde=True,ax=ax_h,color="#E58E8E",
            edgecolor="#333333",alpha=0.90,line_kws={"linewidth": 1.8},)

        if ax_h.lines:
            ax_h.lines[0].set_color("#CC1111")

        stage_bounds = {1: (65, 120), 2: (120, 130), 3: (130, 140), 4: (140, 205)}
        legend_patches = []
        for sid, name in STAGE_NAMES.items():
            xmin, xmax = stage_bounds[sid]
            color = STAGE_COLORS[sid]
            ax_h.axvspan(xmin, xmax, color=color, alpha=0.08, zorder=0)
            legend_patches.append(mpatches.Patch(color=color, alpha=0.30, label=name))

        for xb in (120, 130, 140):
            ax_h.axvline(xb, color="#888888", lw=0.8, ls=":", zorder=2)

        ylim_top = ax_h.get_ylim()[1]

        mean_sbp = float(np.mean(sbp_arr))
        ax_h.axvline(mean_sbp, color="#444444", linestyle="--", linewidth=1.5, zorder=3)
        ax_h.annotate(
            f"$\\bar{{x}}={mean_sbp:.2f}$",
            xy=(mean_sbp, ylim_top * 0.5),
            xytext=(mean_sbp + 25, ylim_top * 0.5),
            arrowprops=dict(arrowstyle="-|>", color="#444444", lw=1.2, mutation_scale=12),
            fontsize=FS, va="center", color="#222222",
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#BBBBBB", alpha=0.95),
        )

        ax_h.set_title("SBP Distribution", fontsize=FS, fontweight="bold", pad=6)
        ax_h.set_xlabel("SBP (mmHg)", fontsize=FS)
        ax_h.set_ylabel("Count", fontsize=FS)
        ax_h.set_xlim(65, 205)
        ax_h.tick_params(axis="both", labelsize=FS)

        for lbl in ax_h.get_xticklabels():
            lbl.set_fontname(TNR)
        for lbl in ax_h.get_yticklabels():
            lbl.set_fontname(TNR)

        lgd = ax_h.legend(handles=legend_patches,loc="upper center",bbox_to_anchor=(0.42, -0.2),
            ncol=4,frameon=True,fancybox=False,edgecolor="#CCCCCC",framealpha=0.95,prop={"family": TNR, "size": FS},
            handlelength=1.2, handleheight=1.0,borderpad=0.35,labelspacing=0.30,columnspacing=1.0,
        )
        
        fig.tight_layout()
        p = os.path.join(SAVE_DIR, "fig0_dataset_distribution.png")
        fig.savefig(p, dpi=400, bbox_extra_artists=(lgd,), bbox_inches="tight",
                    pad_inches=0.04, facecolor="white")
        print(f"  ✅  saved → {p}")
        plt.close(fig)

    except Exception as e:
        print(f"  ⚠️  Skipping Fig 0: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# STAGE TABLE — window-level, SBP-axis (matches Fig 0's bands exactly)
# ══════════════════════════════════════════════════════════════════════════════
def print_stage_table():
    if not os.path.exists(DATA_PATH):
        print(f"  ⚠️  DATA_PATH not found: {DATA_PATH}"); return

    data    = np.load(DATA_PATH)
    sbp_arr = data["Y_bp"][:, 0]
    total   = len(sbp_arr)

    stages = assign_stages(sbp_arr)  # SBP-only, same cutoffs as Fig 0 bands

    print("\n" + "=" * 70)
    print(" UCI Cuff-Less Blood Pressure Dataset — Stage Composition (SBP-axis, window-level)")
    print("=" * 70)
    print(f"{'Stage':<12} {'SBP (mmHg)':<15} {'n':>10} {'%':>8}   {'Mean ± SD (mmHg)':<18}")
    print("-" * 70)

    rows = []
    for sid, name in STAGE_NAMES.items():
        mask = stages == sid
        seg  = sbp_arr[mask]
        n    = len(seg)
        pct  = 100.0 * n / total if total > 0 else 0
        mu   = float(np.mean(seg)) if n > 0 else float("nan")
        sd   = float(np.std(seg))  if n > 0 else float("nan")
        print(f"{name:<12} {STAGE_SBP_RANGE_LABEL[sid]:<15} {n:>10,} {pct:>7.1f}%   {mu:.1f} ± {sd:.1f}")
        rows.append({"stage": name, "sbp_range": STAGE_SBP_RANGE_LABEL[sid],
                     "n": n, "pct": round(pct, 1), "sbp_mean": mu, "sbp_sd": sd})

    print("-" * 70)
    mu_all = float(np.mean(sbp_arr)) if total > 0 else float("nan")
    sd_all = float(np.std(sbp_arr))  if total > 0 else float("nan")
    print(f"{'Total':<12} {'---':<15} {total:>10,} {'100.0':>7}%   {mu_all:.1f} ± {sd_all:.1f}")
    print("=" * 70 + "\n")

    rows.append({"stage": "Total", "sbp_range": "---", "n": total,
                 "pct": 100.0, "sbp_mean": mu_all, "sbp_sd": sd_all})
    _p = os.path.join(OUT_DIR, "exp_additional", "table1_uci_stage_composition.csv")
    os.makedirs(os.path.dirname(_p), exist_ok=True)
    pd.DataFrame(rows).to_csv(_p, index=False)
    print(f"  → {_p}")


# ══════════════════════════════════════════════════════════════════════════════
# DATASET STATS — subject-level, full ACC/AHA 2017 (SBP OR DBP)
# ══════════════════════════════════════════════════════════════════════════════
def print_dataset_stats():
    """
    Subject-level ACC/AHA 2017 staging on the test cohort — calls
    assign_stages(sbp, dbp) with BOTH arrays, so the OR-condition applies.
    This is the clinically correct staging; print_stage_table's SBP-only
    numbers will differ from this one by design (different axis, different
    unit of analysis: window vs subject) — that divergence is now explicit
    in each docstring instead of being two silently-different bugs.
    """
    data    = np.load(DATA_PATH)
    sbp_all = data["Y_bp"][:, 0]
    dbp_all = data["Y_bp"][:, 1]
    print("=== Global Stats (full dataset, window-level) ===")
    print(f"SBP: {np.mean(sbp_all):.2f} ± {np.std(sbp_all):.2f} mmHg")
    print(f"DBP: {np.mean(dbp_all):.2f} ± {np.std(dbp_all):.2f} mmHg")

    print("\n=== Split Summary ===")
    split_rows = []
    for split in ("train", "val", "test"):
        df = load_beats(split, tag="v192", eval_only=False)
        if df.empty:
            print(f"  {split}: no data"); continue
        sub_col = _resolve_col(df, ("subject_id", "subj_id", "subject", "Subject_ID"))
        n_subj  = df[sub_col].nunique() if sub_col else "?"
        n_win   = len(df)
        print(f"  {split:6s}: {n_subj:,} subjects, {n_win:,} windows")
        split_rows.append({"split": split, "records": n_subj, "windows": n_win})

    print("\n=== Test Cohort Stage Counts — subject-level, ACC/AHA 2017 (SBP OR DBP) ===")
    print("    (mean SBP/DBP per subject → assign_stages(sbp, dbp))\n")

    df_subj = _load_test_subject_frame()
    if df_subj is None:
        print("  subject/sbp/dbp columns not found"); return

    stages = assign_stages(df_subj["mean_sbp"].values, df_subj["mean_dbp"].values)

    total_subj = len(df_subj)
    stage_rows = []
    for sid, name in STAGE_NAMES.items():
        n = int((stages == sid).sum())
        print(f"  {name:10s} (stage {sid}): {n:4d} subjects  ({100*n/total_subj:.1f}%)")
        stage_rows.append({"stage": name, "n_subjects": n,
                           "pct": round(100 * n / total_subj, 1)})
    print(f"  {'Total':10s}         : {total_subj:4d} subjects")
    stage_rows.append({"stage": "Total", "n_subjects": total_subj, "pct": 100.0})

    _d = os.path.join(OUT_DIR, "exp_additional")
    os.makedirs(_d, exist_ok=True)
    pd.DataFrame([{"metric": "SBP", "mean": float(np.mean(sbp_all)), "sd": float(np.std(sbp_all))},
                  {"metric": "DBP", "mean": float(np.mean(dbp_all)), "sd": float(np.std(dbp_all))}]
                 ).to_csv(os.path.join(_d, "table2_global_bp_stats.csv"), index=False)
    pd.DataFrame(split_rows).to_csv(os.path.join(_d, "table2_split_summary.csv"), index=False)
    pd.DataFrame(stage_rows).to_csv(os.path.join(_d, "test_cohort_subject_stages.csv"), index=False)
    print(f"  → {_d}/table2_split_summary.csv, table2_global_bp_stats.csv, test_cohort_subject_stages.csv")

build_figure_0()


In [ ]:
def build_figure_1(mdl=None):
    try:
        has_c_slow = True  if mdl is None else hasattr(mdl, "slow_C_head")
        has_grl    = False if mdl is None else hasattr(mdl, "subj_adversary")

        FW, FH = 3.5, 5.2
        fig, ax = plt.subplots(figsize=(FW, FH))
        fig.patch.set_facecolor("white")
        ax.set_facecolor("white")
        ax.set_xlim(0,1); ax.set_ylim(-0.02,1.02); ax.axis("off")

        FS_BOX = 8.0
        FS_TTL = 8.0
        LW     = 1.2
        LINE_COL = "#2C3E50"

        ax.set_title("DiBP-Net Architecture", fontsize=FS_TTL, fontweight="bold",
                     fontname=_TNR[0], pad=2)

        xC=0.50; xM=0.28; xH=0.72; xS=0.88
        yI=0.96; yE=0.84; yP=0.72; yZ=0.61
        ySG=0.50
        yML = 0.4075        # gap SG→ML = 0.030
        yCO = 0.3325        # gap ML→CO = 0.030
        yCA = 0.2575        # gap CO→CA = 0.030
        yBP = 0.145
        yO  = 0.040

        boxes = {
            "I" :(0.62, 0.040),
            "E" :(0.66, 0.080),
            "P" :(0.35, 0.075),
            "Z" :(0.26, 0.045),
            "SG":(0.24, 0.080),
            "ML":(0.23, 0.045),
            "CO":(0.23, 0.045),
            "CA":(0.94, 0.045),
            "BP":(0.29, 0.080),
            "O" :(0.62, 0.038),
        }
        xBPs = [0.17, 0.50, 0.83]

        def draw_box(cx,cy,label,w,h,fc,tc="white",bold=False,ec="#333333",ls="-"):
            p = FancyBboxPatch((cx-w/2, cy-h/2), w, h,
                               boxstyle="round,pad=0.00,rounding_size=0.010",
                               facecolor=fc, edgecolor=ec, linewidth=0.9,
                               linestyle=ls, zorder=4, clip_on=False)
            ax.add_patch(p)
            ax.text(cx, cy, label, ha="center", va="center",
                    fontsize=FS_BOX, color=tc,
                    fontweight="bold" if bold else "normal",
                    fontname=_TNR[0], zorder=6, linespacing=1.0)

        def darr(x1,y1,x2,y2,col=LINE_COL,lw=LW,ls="-"):
            dx,dy=x2-x1,y2-y1; dist=np.hypot(dx,dy)
            if dist==0: return
            pull=0.010
            if dist>pull:
                ax.plot([x1, x2-(dx/dist)*pull],
                        [y1, y2-(dy/dist)*pull],
                        color=col, lw=lw, ls=ls, zorder=3, solid_capstyle='butt')
            dir_x=x2-(dx/dist)*0.001; dir_y=y2-(dy/dist)*0.001
            ax.annotate("", xy=(x2,y2), xytext=(dir_x,dir_y),
                arrowprops=dict(arrowstyle="-|>", color=col, facecolor=col,
                                lw=lw, linestyle="-", mutation_scale=10,
                                shrinkA=0, shrinkB=0, joinstyle='miter'),
                zorder=5, clip_on=False)

        def vline(x,y1,y2,col=LINE_COL,lw=LW,ls="-"):
            ax.plot([x,x],[y1,y2],color=col,lw=lw,ls=ls,zorder=3,solid_capstyle='round',clip_on=False)
        def hline(x1,x2,y,col=LINE_COL,lw=LW,ls="-"):
            ax.plot([x1,x2],[y,y],color=col,lw=lw,ls=ls,zorder=3,solid_capstyle='round',clip_on=False)

        # ── boxes ────────────────────────────────────────────────────────────
        draw_box(xC,yI,"PPG Waveform Input",boxes["I"][0],boxes["I"][1],C_ARCH["input"])
        draw_box(xC,yE,"Shared Encoder\n(1-D CNN+GRU)",boxes["E"][0],boxes["E"][1],C_ARCH["encoder"])
        draw_box(xM,yP,"$h_{\\max}$ Max Pool",boxes["P"][0],boxes["P"][1],C_ARCH["morph"])
        draw_box(xH,yP,"$h_{\\mathrm{attn}}$ Attn Pool",boxes["P"][0],boxes["P"][1],C_ARCH["hemo"])
        draw_box(xM,yZ,"$z_{\\mathrm{morph}}$",boxes["Z"][0],boxes["Z"][1],C_ARCH["morph"],bold=True)
        draw_box(xH,yZ,"$z_{\\mathrm{hemo}}$",boxes["Z"][0],boxes["Z"][1],C_ARCH["hemo"],bold=True)
        if has_c_slow:
            draw_box(xS,ySG,"stop-grad\n(detach)",boxes["SG"][0],boxes["SG"][1],
                     C_ARCH["stop_bg"],tc=C_ARCH["stop_tc"],ls="--",ec=C_ARCH["stop_tc"])
            draw_box(xS,yML,"$c_{\\mathrm{slow}}$ MLP",boxes["ML"][0],boxes["ML"][1],C_ARCH["slow"])
            draw_box(xS,yCO,"$c_{\\mathrm{slow}}$",boxes["CO"][0],boxes["CO"][1],C_ARCH["slow"],bold=True)
        draw_box(xC,yCA,
                 "$\\mathbf{z}=[z_{\\mathrm{morph}},z_{\\mathrm{hemo}}" +
                 (",c_{\\mathrm{slow}}]$" if has_c_slow else "]$"),
                 boxes["CA"][0],boxes["CA"][1],C_ARCH["concat"])
        for bx,lbl in zip(xBPs,["Direct\nBP Head","Windkessel\nODE Head","Ensemble\nHead"]):
            draw_box(bx,yBP,lbl,boxes["BP"][0],boxes["BP"][1],C_ARCH["bp"])
        draw_box(xC,yO,"SBP / DBP Predictions",boxes["O"][0],boxes["O"][1],"#1C2833")

        # ── arrows ───────────────────────────────────────────────────────────
        darr(xC,yI-boxes["I"][1]/2, xC,yE+boxes["E"][1]/2)
        bry1=yE-boxes["E"][1]/2-0.007
        vline(xC,yE-boxes["E"][1]/2,bry1); hline(xM,xH,bry1)
        darr(xM,bry1,xM,yP+boxes["P"][1]/2); darr(xH,bry1,xH,yP+boxes["P"][1]/2)
        darr(xM,yP-boxes["P"][1]/2,xM,yZ+boxes["Z"][1]/2,col=C_ARCH["morph"])
        darr(xH,yP-boxes["P"][1]/2,xH,yZ+boxes["Z"][1]/2,col=C_ARCH["hemo"])
        darr(xM,yZ-boxes["Z"][1]/2,xM,yCA+boxes["CA"][1]/2,col=C_ARCH["morph"])
        darr(xH,yZ-boxes["Z"][1]/2,xH,yCA+boxes["CA"][1]/2,col=C_ARCH["hemo"])
        if has_c_slow:
            pull=0.010
            ax.plot([xH+boxes["Z"][0]/2, xS, xS],
                    [yZ, yZ, ySG+boxes["SG"][1]/2+pull],
                    color=C_ARCH["stop_tc"],lw=LW,ls="--",zorder=3,solid_capstyle='butt')
            ax.annotate("",xy=(xS,ySG+boxes["SG"][1]/2),
                        xytext=(xS,ySG+boxes["SG"][1]/2+0.001),
                        arrowprops=dict(arrowstyle="-|>",color=C_ARCH["stop_tc"],
                                        facecolor=C_ARCH["stop_tc"],lw=LW,linestyle="-",
                                        mutation_scale=10,shrinkA=0,shrinkB=0,joinstyle='miter'),
                        zorder=5,clip_on=False)
            darr(xS,ySG-boxes["SG"][1]/2,xS,yML+boxes["ML"][1]/2,col=C_ARCH["slow"])
            darr(xS,yML-boxes["ML"][1]/2,xS,yCO+boxes["CO"][1]/2,col=C_ARCH["slow"])
            darr(xS,yCO-boxes["CO"][1]/2,xS,yCA+boxes["CA"][1]/2,col=C_ARCH["slow"])
        bpbry=yCA-boxes["CA"][1]/2-0.007
        vline(xC,yCA-boxes["CA"][1]/2,bpbry); hline(xBPs[0],xBPs[2],bpbry)
        for bx in xBPs: darr(bx,bpbry,bx,yBP+boxes["BP"][1]/2)
        outbry=yBP-boxes["BP"][1]/2-0.007
        for bx in xBPs: vline(bx,yBP-boxes["BP"][1]/2,outbry)
        hline(xBPs[0],xBPs[2],outbry); darr(xC,outbry,xC,yO+boxes["O"][1]/2)

 # HSIC arc
        ax.annotate("",
            xy    =(xH-boxes["Z"][0]/2-0.008, yZ),
            xytext=(xM+boxes["Z"][0]/2+0.008, yZ),
            arrowprops=dict(arrowstyle="<|-|>",color=C_ARCH["hsic"],facecolor=C_ARCH["hsic"],
                            lw=1.2,mutation_scale=10,shrinkA=2,shrinkB=2,
                            connectionstyle="arc3,rad=-0.1"),
            zorder=5,clip_on=False)
        ax.text(xC,yZ+0.032,"HSIC",ha="center",va="center",fontsize=FS_BOX,
                color=C_ARCH["hsic"],fontweight="bold",fontname=_TNR[0],
                bbox=dict(boxstyle="square,pad=0.18",fc="white",ec=C_ARCH["hsic"],
                          lw=1.0,alpha=1.0),zorder=7)

        # ── legend ───────────────────────────────────────────────────────────
        legend_items = [
            mpatches.Patch(fc=C_ARCH["morph"],label="$z_{\\mathrm{morph}}$ branch",ec="#333",lw=0.5),
            mpatches.Patch(fc=C_ARCH["hemo"], label="$z_{\\mathrm{hemo}}$ branch", ec="#333",lw=0.5),
        ]
        if has_c_slow:
            legend_items.append(
                mpatches.Patch(fc=C_ARCH["slow"],label="$c_{\\mathrm{slow}}$ branch",ec="#333",lw=0.5))
        legend_items += [
            mpatches.Patch(fc=C_ARCH["bp"],label="BP Heads",ec="#333",lw=0.5),
            Line2D([0],[0],color=C_ARCH["stop_tc"],ls="--",lw=1.5,label="Stop-grad"),
            Line2D([0],[0],color=C_ARCH["hsic"],lw=1.5,label="HSIC loss"),
        ]
        if has_grl:
            legend_items.append(
                mpatches.Patch(fc="#FFEBEE",ec=C_ARCH["hemo"],lw=0.5,label="GRL Adversary"))

        fig.subplots_adjust(top=0.80,bottom=0.1,left=0.01,right=0.99)
        fig.legend(handles=legend_items,loc="lower center",bbox_to_anchor=(0.5,-0.01),
                   ncol=2,framealpha=1.0,edgecolor="#333333",fancybox=False,
                   prop={"family":_TNR[0],"size":FS_BOX})

        p = os.path.join(SAVE_DIR, "fig1_architecture.png")
        fig.savefig(p,dpi=400,bbox_inches="tight",pad_inches=0.02,facecolor="white")
        print(f"  ✅  saved → {p}")
        plt.close(fig)

    except Exception as e:
        import traceback
        print(f"  ⚠️  Skipping Fig 1: {e}")
        traceback.print_exc()

In [ ]:
def build_figure_2():
    try:
        import sys, os
        import numpy as np
        import matplotlib.pyplot as plt
        import matplotlib.ticker as mticker
        from matplotlib.gridspec import GridSpec

        _m2   = sys.modules.get("__main__")
        TRAIN = getattr(_m2, "TRAIN", None)

        pa_end     = getattr(TRAIN, "phase_a_end_ep",    20)   if TRAIN else 20
        pb_end     = getattr(TRAIN, "phase_b_end_ep",    30)   if TRAIN else 30
        pc_end     = getattr(TRAIN, "phase_c_end_ep",    50)   if TRAIN else 50
        _total     = ((getattr(TRAIN, "epochs_p1", 80) +
                       getattr(TRAIN, "epochs_p2", 80)) if TRAIN else 160)
        hsic_start = getattr(TRAIN, "hsic_start_ep",     10)   if TRAIN else 10
        hsic_end   = getattr(TRAIN, "hsic_ramp_end_ep",  40)   if TRAIN else 40
        lam_start  = getattr(TRAIN, "lambda_hsic_start", 0.20) if TRAIN else 0.20
        lam_max    = getattr(TRAIN, "lambda_hsic_max",   0.55) if TRAIN else 0.55
        wk_freeze  = getattr(TRAIN, "wk_freeze_ep",       5)   if TRAIN else 5

        ep_x  = np.arange(0, _total + 1, dtype=float)
        t_arr = np.clip((ep_x - hsic_start) / max(hsic_end - hsic_start, 1), 0., 1.)
        lam_y = np.where(ep_x < hsic_start, 0.,
                         lam_start + (lam_max - lam_start) * t_arr)

        PHASES = [
            ("A",  0,      pa_end, "#90CAF9"),
            ("B",  pa_end, pb_end, "#1976D2"),
            ("C",  pb_end, pc_end, "#1565C0"),
            ("D",  pc_end, _total, "#0D47A1"),
        ]
        LOSSES = [
            ("L_BP",     0,          "#E65100", r"$\mathcal{L}_{\mathrm{BP}}$"),
            ("L_WK",     wk_freeze,  "#BF360C", r"$\mathcal{L}_{\mathrm{WK}}$"),
            ("L_anchor", 0,          "#546E7A", r"$\mathcal{L}_{\mathrm{anchor}}$"),
            ("L_anti",   hsic_start, "#C62828", r"$\mathcal{L}_{\mathrm{anti}}$"),
            ("L_HSIC",   hsic_start, "#F57F17", r"$\mathcal{L}_{\mathrm{HSIC}}$"),
            ("L_cslow",  hsic_start, "#6A1B9A", r"$\mathcal{L}_{c_{\mathrm{slow}}}$"),
        ]

        FS   = 8             # 8 pt
        FW, FH   = _COL1_IN, 3.5
        LEFT, RIGHT = 0.21, 0.92
        X_MID    = (LEFT + RIGHT) / 2

        _PAD_R = max(6, int(_total * 0.035))
        X_MIN, X_MAX = -3, _total + _PAD_R

        XTICKS   = list(range(0, _total + 1, 40))
        n = len(LOSSES)

        fig = plt.figure(figsize=(FW, FH), facecolor="white")
        sup = fig.suptitle("DiBP-Net Training Schedule",
                     fontsize=FS, fontweight="bold",
                     x=X_MID, ha="center", y=0.99)

        gs = GridSpec(3, 1, figure=fig,
                      height_ratios=[0.24, 1.70, 1.00],
                      hspace=0.0)

        # ── Phase bar ────────────────────────────────────────────────────────
        ax_a = fig.add_subplot(gs[0])
        ax_a.set_facecolor("white")
        for name, s, e, col in PHASES:
            ax_a.barh(0, e-s, left=s, height=0.7,
                      color=col, edgecolor="white", lw=0.8)
            ax_a.text((s+e)/2, -0.03, name, ha="center", va="center",
                      fontsize=FS, color="white",
                      fontweight="bold")
        ax_a.set_xlim(X_MIN, X_MAX); ax_a.set_ylim(-0.52, 0.52)
        ax_a.axis("off")

        # ── Loss timeline ────────────────────────────────────────────────────
        ax_b = fig.add_subplot(gs[1])
        ax_b.set_facecolor("white")
        bar_h = 0.54
        for i, (key, s, col, lbl) in enumerate(LOSSES):
            if s > 0:
                ax_b.barh(i, s, left=0, height=bar_h,
                          color="#ECEFF1", edgecolor="white", lw=0.3)
            ax_b.barh(i, _total-s, left=s, height=bar_h,
                      color=col, alpha=0.88, edgecolor="white", lw=0.3)
            if s > 0:
                ax_b.text(s+1.0, i, f"  ep {s}",
                          va="center", ha="left",
                          fontsize=FS,
                          color="white", fontweight="bold",
                          bbox=dict(boxstyle="round,pad=0.08",
                                    fc=col, ec="none", alpha=0.90))
        for _, s, _, _ in PHASES[1:]:
            ax_b.axvline(s, color="#B0BEC5", lw=0.7, ls=":", zorder=5)
        ax_b.set_yticks(range(n))
        ax_b.set_yticklabels([l[3] for l in LOSSES], fontsize=FS)
        for lbl in ax_b.get_yticklabels():
            lbl.set_fontname(_TNR[0])
        ax_b.tick_params(axis="y", labelsize=FS, length=0, pad=3)
        ax_b.set_xlim(X_MIN, X_MAX); ax_b.set_ylim(-0.58, n-0.42)
        ax_b.set_xticks([])
        ax_b.grid(axis="x", color="#EBEBEB", linestyle="--", linewidth=0.55, zorder=0)
        for sp in ["top", "right", "bottom"]: ax_b.spines[sp].set_visible(False)

        # ── HSIC λ ramp ──────────────────────────────────────────────────────
        ax_c = fig.add_subplot(gs[2])
        ax_c.set_facecolor("white")
        ax_c.plot(ep_x, lam_y, color="#F57F17", lw=2.0, zorder=3,
                  label=r"HSIC $\lambda$")
        ax_c.axhline(lam_max,   color="#F57F17", ls="--", lw=0.9, alpha=0.50)
        ax_c.axhline(lam_start, color="#F57F17", ls="--", lw=0.9, alpha=0.50)
        ax_c.axvline(hsic_start, color="#888", lw=0.9, ls=":",
                     label=f"ramp start (ep {hsic_start})")
        ax_c.axvline(hsic_end,   color="#555", lw=0.9, ls=":",
                     label=f"ramp end (ep {hsic_end})")
        for _, s, _, _ in PHASES[1:]:
            ax_c.axvline(s, color="#B0BEC5", lw=0.7, ls=":", zorder=1)
        ax3_tr = ax_c.get_yaxis_transform()
        for val, txt, tcol in [(lam_max, "0.55", "#E67E00"),
                                (lam_start, "0.20", "#E67E00"),
                                (0.0, "0", "#999")]:
            ax_c.annotate(txt, xy=(1.03, val), xycoords=ax3_tr,
                          va="center", ha="left",
                          fontsize=FS,
                          color=tcol, annotation_clip=False)
        try:
            best_ckpt, early_stop = _best_stop_from_history("v192")
            _ef = ep_x.astype(float); _lf = lam_y.astype(float)
            if best_ckpt is not None:
                ax_c.scatter(int(best_ckpt),
                             float(np.interp(float(best_ckpt), _ef, _lf)),
                             marker="*", s=110, color="#2E7D32",
                             edgecolors="#1B5E20", lw=0.5, zorder=6, clip_on=False,
                             label=f"Best (ep {best_ckpt})")
            if early_stop is not None:
                ax_c.scatter(int(early_stop),
                             float(np.interp(float(early_stop), _ef, _lf)),
                             marker="X", s=55, color="#2E7D32",
                             edgecolors="#1B5E20", lw=0.8, zorder=6, clip_on=False,
                             label=f"Stop (ep {early_stop})")
        except Exception:
            pass
        ax_c.set_xlim(X_MIN, X_MAX)
        ax_c.set_ylim(-0.04, lam_max + 0.06)
        ax_c.set_yticks([0.0, 0.2, 0.4, 0.55])
        ax_c.set_xticks(XTICKS)
        ax_c.tick_params(axis="both", labelsize=FS)
        for lbl in ax_c.get_xticklabels() + ax_c.get_yticklabels():
            lbl.set_fontname(_TNR[0])
        ax_c.set_xlabel("Epoch", fontsize=FS)
        ax_c.set_ylabel(r"HSIC $\lambda$", fontsize=FS)
        ax_c.yaxis.set_minor_locator(mticker.AutoMinorLocator(2))

        ax_c.grid(alpha=0.18, linestyle="--", linewidth=0.5)
        for sp in ["top", "right"]: ax_c.spines[sp].set_visible(False)

        handles_c, labels_c = ax_c.get_legend_handles_labels()
        leg = fig.legend(handles_c, labels_c,
                          loc="lower center", bbox_to_anchor=(X_MID, -0.10),
                          ncol=3, frameon=True,
                          prop={"family": "Times New Roman", "size": FS},
                          framealpha=0.93, edgecolor="#CFD8DC",
                          handlelength=1.1, labelspacing=0.20,
                          columnspacing=0.7, borderpad=0.30)

        fig.subplots_adjust(left=LEFT, right=RIGHT, top=0.95, bottom=0.10)
        p = os.path.join(SAVE_DIR, "fig2_training_schedule.png")
        fig.savefig(p, dpi=400, bbox_inches="tight", pad_inches=0.04,
                    facecolor="white", bbox_extra_artists=(leg, sup))
        print(f"  ✅  saved → {p}")
        plt.close(fig)

    except Exception as exc:
        import traceback
        print(f"  ⚠️  build_figure_2 failed: {exc}")
        traceback.print_exc()

In [ ]:
def build_figure_3(tag="v192"):
    try:
        df_v = load_beats("val",  tag)
        df_t = load_beats("test", tag)

        sbp_pred_v = _resolve_col(df_v, ("pred_sbp_post", "pred_sbp_cal", "pred_sbp_cal_mean", "pred_sbp"))
        sbp_true_v = _resolve_col(df_v, ("true_sbp",     "true_sbp_eval_mean", "sbp_true"))
        dbp_pred_v = _resolve_col(df_v, ("pred_dbp_post", "pred_dbp_cal", "pred_dbp_cal_mean", "pred_dbp"))
        dbp_true_v = _resolve_col(df_v, ("true_dbp",     "true_dbp_eval_mean", "dbp_true"))

        if df_v.empty or df_t.empty or not sbp_pred_v or not sbp_true_v:
            print("  ⚠️  Fig 3: missing data columns"); return

        sbp_pred_t = _resolve_col(df_t, ("pred_sbp_post", "pred_sbp_cal", "pred_sbp_cal_mean", "pred_sbp")) or sbp_pred_v
        sbp_true_t = _resolve_col(df_t, ("true_sbp",     "true_sbp_eval_mean", "sbp_true")) or sbp_true_v
        dpt        = _resolve_col(df_t, ("pred_dbp_post", "pred_dbp_cal", "pred_dbp_cal_mean", "pred_dbp")) or dbp_pred_v
        dtt        = _resolve_col(df_t, ("true_dbp",     "true_dbp_eval_mean", "dbp_true")) or dbp_true_v

        df_v  = df_v.dropna(subset=[sbp_pred_v, sbp_true_v]).copy()
        df_t  = df_t.dropna(subset=[sbp_pred_t]).copy()
        has_dbp = bool(dbp_pred_v and dbp_true_v)
        df_v2 = df_v.dropna(subset=[dbp_pred_v, dbp_true_v]) if has_dbp else None
        df_t2 = (df_t.dropna(subset=[dpt]) if dtt in df_t.columns else df_t) if has_dbp else None

        lim_s, lim_d = (80, 200), (50, 130)
        
        ba_ylim_s = (-75, 105)
        ba_ylim_d = (-45, 55)

 # 2×2 per figure: [Scatter Val | Scatter Test]
 # [B-A Val | B-A Test ]
        panel_w = 1.6
        panel_h = 1.6
        l, r, t, b = 0.12, 0.03, 0.06, 0.12
        wspace, hspace = 0.40, 0.45
        fig_w = panel_w * 2 / (1 - l - r) + panel_w * wspace
        fig_h = panel_h * 2 / (1 - t - b) + panel_h * hspace

        def _make_2x2(bp, lim, ba_ylim, vtc, vpc, dfv, ttc, tpc, dft, stem):
            with plt.rc_context(_IEEE_RC):
                fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.5))
                fig.subplots_adjust(left=0.14, right=0.97, top=0.93, bottom=0.12,
                                    wspace=0.45, hspace=0.50)
                
                _scatter_panel(axes[0,0], dfv[vtc].values, dfv[vpc].values,
                            C_VAL,  f"Val {bp} Scatter", lim)
                _scatter_panel(axes[0,1], dft[ttc].values, dft[tpc].values,
                            C_TEST, f"Test {bp} Scatter", lim)
                
                _bland_altman_panel(axes[1,0], dfv[vtc].values, dfv[vpc].values,
                            C_VAL,  f"Val {bp} B–A", lim)
                _bland_altman_panel(axes[1,1], dft[ttc].values, dft[tpc].values,
                            C_TEST, f"Test {bp} B–A", lim)
                
                axes[1,0].set_ylim(ba_ylim)
                axes[1,1].set_ylim(ba_ylim)

                _save(fig, stem)

        _make_2x2("SBP", lim_s, ba_ylim_s, sbp_true_v, sbp_pred_v, df_v,
                          sbp_true_t, sbp_pred_t, df_t, "fig3a_sbp_2x2")
        if has_dbp:
            _make_2x2("DBP", lim_d, ba_ylim_d, dbp_true_v, dbp_pred_v, df_v2,
                              dtt, dpt, df_t2, "fig3b_dbp_2x2")

    except Exception as e:
        print(f"  ⚠️  Skipping Fig 3: {e}")
build_figure_3()

In [ ]:
def build_figure_4():
    try:
        EP, MODELS = list(range(10, 170, 10)), _build_all_models()
        _all_corr = [float(v) for _, d in MODELS for v in d.get("corr", [])
                     if v is not None]
        _ylim_lo = max(min(_all_corr) - 0.05, -0.35) if _all_corr else -0.35
        _ylim_hi = min(max(_all_corr) + 0.05,  0.35) if _all_corr else  0.35

        with plt.rc_context(_IEEE_RC):
            fig, ax = plt.subplots(figsize=(_COL1_IN, 3))
            for key, data in MODELS:
                col, ls, lw = STYLE[key]
                arr = _nan(data["corr"])
                if np.all(np.isnan(arr.data if hasattr(arr, "data") else arr)): continue
                ax.plot(EP, arr, color=col, linestyle=ls, linewidth=lw,
                        label=LABEL[key], zorder=3)
                _add_markers_metric(ax, key, data, "corr", EP)

            ax.axhspan(-TARGET_CORR, TARGET_CORR, color="#2CA02C", alpha=0.08, zorder=1)
            ax.axhline(0, color="#333", lw=0.8, alpha=0.4, zorder=2)
            for y in (TARGET_CORR, -TARGET_CORR):
                ax.axhline(y, color="#2CA02C", lw=0.9, ls=":", alpha=0.6)
            _phase_line(ax)
            ax.set_xlabel("Epoch", fontsize=8)
            ax.set_ylabel(r"$r(z_\mathrm{morph},\,z_\mathrm{hemo})$", fontsize=8)
            ax.set_title(r"Latent Correlation $r(z_\mathrm{morph},\,z_\mathrm{hemo})$",
                         fontsize=8, fontweight="bold")
            ax.set_xlim(5, max(EP) + 8); ax.set_ylim(_ylim_lo, _ylim_hi)
            ax.set_xticks(range(10, 165, 30))
            ax.tick_params(labelsize=8)

            mh, eh = _legend_handles()
            ax.legend(handles=mh + eh, loc="upper center",
                bbox_to_anchor=(0.44, -0.3), ncol=2,
                framealpha=0.98, edgecolor="#ccc",
                prop={"family": "Times New Roman", "size": 8},
                columnspacing=0.8, handletextpad=0.4,
                handlelength=1.2, labelspacing=0.3)

            fig.subplots_adjust(bottom=0.52, top=0.88, left=0.18, right=0.97)

            for artist in fig.findobj(plt.Text):
                try:
                    artist.set_fontsize(8)
                    artist.set_fontname("Times New Roman")
                except Exception:
                    pass

            _save(fig, "fig4_corr_trajectory")

    except Exception as e: print(f"  ⚠️  Skipping Fig 4: {e}")

In [ ]:
def build_figure_5(tag="v192"):
    try:
        import numpy as np
        import matplotlib.pyplot as plt
        import matplotlib.patches as mpatches
        from matplotlib.lines import Line2D
        import seaborn as sns
        
        df_test   = load_beats("test", tag, eval_only=True)
        cslow_col = _resolve_col(df_test, ("c_slow", "c_slow_mean"))
        if cslow_col and not df_test.empty:
            df_test = df_test.dropna(subset=[cslow_col])

        epochs_5, icc_vals, vbvw_vals = _load_icc_vbvw_trajectory(tag)
        ov = _load_overall_summary(tag, "test")

        icc_test   = float(ov.get("ICC_C(c_slow)",  np.nan))
        vbvw_test  = float(ov.get("Vb/Vw(c_slow)",  np.nan))
        icc_ci_lo  = float(ov.get("ICC CI lo",       np.nan))
        icc_ci_hi  = float(ov.get("ICC CI hi",       np.nan))
        vbvw_ci_lo = float(ov.get("Vb/Vw CI lo",    np.nan))
        vbvw_ci_hi = float(ov.get("Vb/Vw CI hi",    np.nan))


        leg_prop = {"family": "Times New Roman", "size": 8}
        
        max_ep = max(epochs_5) if epochs_5 else 150

        # ── Pre-compute Vb/Vw ylim ────────────────────────────────────────────
        _vbvw_clean  = [float(v) for v in vbvw_vals if v is not None]
        _vbvw_stable = _vbvw_clean[1:] if len(_vbvw_clean) > 1 else _vbvw_clean
        _spike_val   = _vbvw_clean[0]  if _vbvw_clean else None
        _vbvw_hi_st  = max(_vbvw_stable + [float(vbvw_ci_hi)]) * 1.08 if _vbvw_stable else 300
        _vbvw_lo_st  = max(min(_vbvw_stable + [float(vbvw_ci_lo)]) * 0.92, 0) if _vbvw_stable else 0
        _vbvw_hi_sp  = (_spike_val * 1.10) if _spike_val else _vbvw_hi_st * 1.5
        _vbvw_lo_sp  = (_spike_val * 0.90) if _spike_val else _vbvw_hi_st

        with plt.rc_context(_IEEE_RC):
            fig = plt.figure(figsize=(_COL1_IN, 5.5))
            gs  = fig.add_gridspec(
                2, 2,
                height_ratios=[1.0, 1.4],
                hspace=0.55, wspace=0.38,
                left=0.14, right=0.97,
                top=0.92, bottom=0.08,
            )

            ax_icc = fig.add_subplot(gs[0, 0])
            ax_box = fig.add_subplot(gs[1, :])

            gs_vbvw = gs[0, 1].subgridspec(2, 1, hspace=0.06, height_ratios=[0.28, 0.72])
            ax_vb_top = fig.add_subplot(gs_vbvw[0])
            ax_vb_bot = fig.add_subplot(gs_vbvw[1])

            bbox_style = dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85)

            # ── (a) ICC trajectory ────────────────────────────────────────────
            if epochs_5:
                ax_icc.plot(epochs_5, [float(v) for v in icc_vals],
                            color=C_VAL, marker="o", ms=2.5, lw=1.0,
                            markerfacecolor="white", zorder=3)
            if not np.isnan(icc_test):
                ax_icc.axhline(icc_test, color="#E65100", ls="-", lw=1.2, zorder=2)
                if not (np.isnan(icc_ci_lo) or np.isnan(icc_ci_hi)):
                    ax_icc.axhspan(icc_ci_lo, icc_ci_hi, color="#E65100", alpha=0.15, zorder=1)
                
                ax_icc.text(max_ep, icc_test + 0.0003, f"{icc_test:.4f}", 
                            color="#C62828", fontsize=8, ha="right", va="bottom", fontweight="bold",
                            bbox=bbox_style, zorder=5)

            _icc_clean = [float(v) for v in icc_vals if v is not None]
            _icc_lo    = min(_icc_clean + [icc_test]) if _icc_clean else 0.975
            ax_icc.set_ylim(max(_icc_lo - 0.002, 0.970), 1.002)
            ax_icc.set_xlabel("Epoch", fontsize=8)
            ax_icc.set_ylabel("ICC", fontsize=8)
            ax_icc.set_title(r"(a) ICC($c_\mathrm{slow}$)", fontsize=8, fontweight="bold")
            ax_icc.tick_params(labelsize=7)

            # ── (b) Vb/Vw broken axis ─────────────────────────────────────────
            for ax_vb in (ax_vb_top, ax_vb_bot):
                if epochs_5:
                    ax_vb.plot(epochs_5, [float(v) for v in vbvw_vals],
                               color=C_VAL, marker="s", ms=2.0, lw=1.0,
                               markerfacecolor="white", zorder=3)
                if not np.isnan(vbvw_test):
                    ax_vb.axhline(vbvw_test, color="#E65100", ls="-", lw=1.2, zorder=2)
                    if not np.isnan(vbvw_ci_hi):
                        ax_vb.axhspan(vbvw_ci_lo, vbvw_ci_hi, color="#E65100", alpha=0.15, zorder=1)
                        
            if not np.isnan(vbvw_test):
                ax_vb_bot.text(max_ep, vbvw_test + 15, f"{vbvw_test:.1f}", 
                               color="#C62828", fontsize=8, ha="right", va="bottom", fontweight="bold",
                               bbox=bbox_style, zorder=5)

            ax_vb_top.set_ylim(_vbvw_lo_sp, _vbvw_hi_sp)
            ax_vb_bot.set_ylim(_vbvw_lo_st, _vbvw_hi_st)

            ax_vb_top.spines["bottom"].set_visible(False)
            ax_vb_bot.spines["top"].set_visible(False)
            ax_vb_top.tick_params(bottom=False, labelbottom=False, labelsize=7)
            ax_vb_bot.tick_params(labelsize=7)

            d = 0.015
            kw = dict(transform=ax_vb_top.transAxes, color="k", clip_on=False, lw=0.8)
            ax_vb_top.plot((-d, +d), (-d, +d), **kw)
            ax_vb_top.plot((1-d, 1+d), (-d, +d), **kw)
            kw.update(transform=ax_vb_bot.transAxes)
            ax_vb_bot.plot((-d, +d), (1-d, 1+d), **kw)
            ax_vb_bot.plot((1-d, 1+d), (1-d, 1+d), **kw)

            ax_vb_bot.set_xlabel("Epoch", fontsize=8)
            ax_vb_bot.set_ylabel("$V_b/V_w$", fontsize=8)
            ax_vb_top.set_title(r"(b) $V_b/V_w$ ($c_\mathrm{slow}$)", fontsize=8, fontweight="bold")

            # ── Shared Legend ──
            shared_handles = [
                Line2D([0], [0], color=C_VAL, marker="o", ms=3.5, lw=1.2, markerfacecolor="white", label="Val (train)"),
                Line2D([0], [0], color="#E65100", lw=1.2, label="Test"),
                mpatches.Patch(facecolor="#E65100", alpha=0.15, label="95% CI", edgecolor="none")
            ]
            
            ax_box.legend(handles=shared_handles, prop=leg_prop,
                          loc="lower center", bbox_to_anchor=(0.45, 1.15), ncol=3,
                          framealpha=1.0, edgecolor="#E0E0E0", borderaxespad=0)

            # ── (c) c_slow boxplot by stage ───────────────────────────────────
            if cslow_col and "stage_label" in df_test.columns:
                present = [s for s in STAGE_ORDER if s in df_test["stage_label"].values]
                pal = [STAGE_PALETTE[STAGE_ORDER.index(s)] for s in present]
                sns.boxplot(data=df_test, x="stage_label", y=cslow_col,
                            order=present, palette=pal, ax=ax_box,
                            width=0.45, fliersize=1.5, linewidth=0.9)
                sns.stripplot(data=df_test, x="stage_label", y=cslow_col,
                              order=present, palette=pal, size=1.5, alpha=0.3,
                              ax=ax_box, zorder=2)
                ax_box.set_xlabel("AHA Stage", fontsize=8)
                ax_box.set_ylabel(r"$c_\mathrm{slow}$ Value", fontsize=8)
                
                ax_box.set_title(r"(c) $c_\mathrm{slow}$ vs. Clinical Stage", fontsize=8, fontweight="bold", pad=8)
                
                ax_box.set_xticklabels(present, rotation=15, ha="right", fontsize=8)
                ax_box.tick_params(labelsize=8)
            else:
                ax_box.set_visible(False)

            for artist in fig.findobj(plt.Text):
                try:
                    artist.set_fontsize(8)
                    artist.set_fontname("Times New Roman")
                except Exception:
                    pass

            _save(fig, "fig5_cslow_panel")

    except Exception as e: 
        import traceback
        print(f"  ⚠️  Skipping Fig 5: {e}")
        traceback.print_exc()

In [ ]:
def build_figure_6(tag="v192"):
    try:
        df        = load_beats("test", tag, eval_only=True)   # Fix BUG-4: match eval_window used by ICC scalar
        cslow_col = _resolve_col(df, ("c_slow", "c_slow_mean"))
        if df.empty or not cslow_col or "stage_label" not in df.columns: return

        df = df.dropna(subset=[cslow_col, "stage_label", "subj_id"]).copy()
        df_subj = df.groupby("subj_id").agg(
            c_slow_mean=(cslow_col, "mean"),
            mean_sbp=("true_sbp",  "mean"),
            mean_dbp=("true_dbp",  "mean"),
        ).reset_index()

        def _assign(sbp, dbp):
            stages = []
            for s, d in zip(sbp, dbp):
                if   s >= 140 or d >= 90: stages.append("HTN-2")
                elif s >= 130 or d >= 80: stages.append("HTN-1")
                elif s >= 120:            stages.append("Elevated")
                else:                     stages.append("Normal")
            return stages

        df_subj["stage_label"] = _assign(df_subj["mean_sbp"].values,
                                          df_subj["mean_dbp"].values)

        def _cohens_d(a, b):
            a, b = np.asarray(a, float), np.asarray(b, float)
            n1, n2 = len(a), len(b)
            if n1 < 2 or n2 < 2: return float("nan")
            pooled = np.sqrt(((n1-1)*np.var(a, ddof=1) +
                              (n2-1)*np.var(b, ddof=1)) / (n1+n2-2))
            return (np.mean(a) - np.mean(b)) / (pooled + 1e-12)

        with plt.rc_context(_IEEE_RC):
            fig, ax = plt.subplots(figsize=(_COL1_IN, 3.5))
            present = [s for s in STAGE_ORDER if s in df_subj["stage_label"].values]
            pal = [STAGE_PALETTE[STAGE_ORDER.index(s)] for s in present]

            sns.violinplot(data=df_subj, x="stage_label", y="c_slow_mean",
                           order=present, palette=pal, ax=ax,
                           inner=None, alpha=0.15, linewidth=0)
            sns.swarmplot(data=df_subj, x="stage_label", y="c_slow_mean",
                          order=present, palette=pal, ax=ax,
                          size=2.0, alpha=0.5, zorder=2)
            sns.boxplot(data=df_subj, x="stage_label", y="c_slow_mean",
                        order=present, ax=ax, width=0.22, fliersize=0,
                        linewidth=1.5, zorder=4,
                        boxprops=dict(facecolor="white", edgecolor="black", alpha=0.9),
                        medianprops=dict(color=C_MED, linewidth=1.8))

            y_max = float(df_subj["c_slow_mean"].max())
            y_min = float(df_subj["c_slow_mean"].min())
            y_rng = y_max - y_min
            sp    = max(0.18, y_rng * 0.11)

            adjacent = [("Normal",   "Elevated", 0, 1),
                        ("Elevated", "HTN-1",    1, 2),
                        ("HTN-1",    "HTN-2",    2, 3)]
            y0 = y_max + sp * 0.7
            for tier, (lo, hi, x1, x2) in enumerate(adjacent):
                g_lo = df_subj[df_subj["stage_label"] == lo]["c_slow_mean"].values
                g_hi = df_subj[df_subj["stage_label"] == hi]["c_slow_mean"].values
                if len(g_lo) < 2 or len(g_hi) < 2: continue
                d, d_lo, d_hi = bootstrap_cohens_d_ci(g_hi, g_lo)  # Fix WARN-1: bootstrap CI
                staple_bracket(ax, x1, x2, y0 + tier * sp, d,
                               h=sp * 0.18, fs=7)

            ax.set_ylim(y_min - sp * 0.6, y0 + 3.2 * sp)
            ax.set_xlabel("Hypertension Stage (AHA)", fontsize=8)
            ax.set_ylabel(r"$c_\mathrm{slow}$ Value", fontsize=8)
            ax.set_title(r"$c_\mathrm{slow}$ Distribution by Stage",
                         fontsize=8, fontweight="bold")
            
            ax.set_xticklabels(present, rotation=0, ha="center", fontsize=8)
 # ----------------------------------------
            
            ax.tick_params(labelsize=8)

            for artist in fig.findobj(plt.Text):
                try:
                    artist.set_fontsize(8)
                    artist.set_fontname("Times New Roman")
                except Exception:
                    pass

            fig.tight_layout(pad=0.5)
            _save(fig, "fig6_cslow_distribution")

    except Exception as e: print(f"  ⚠️  Skipping Fig 6: {e}")

In [ ]:
def build_figure_7(mdl_proposed=None, mdl_grl=None):
    try:
        df_t_chance = load_beats("test", "v192", eval_only=True)
        sub_col     = _resolve_col(df_t_chance,
                                   ("subject_id", "subject", "id", "Subject_ID"))
        n_subj      = (df_t_chance[sub_col].nunique()
                       if sub_col and not df_t_chance.empty else 50)
        chance      = 1.0 / n_subj if n_subj > 0 else 0.020

        knn_zh, knn_zm, adv_acc = [np.nan, np.nan], [np.nan, np.nan], [np.nan, np.nan]
        for idx, tag in enumerate(["v192", "v193"]):
            ov = _load_overall_summary(tag, "test")
            knn_zh[idx]  = float(ov.get("acc(z_hemo)",   np.nan))
            knn_zm[idx]  = float(ov.get("acc(z_morph)",  np.nan))
            adv_acc[idx] = float(ov.get("Adversary acc", np.nan))

        leg_prop = {"family": "Times New Roman", "size": 8}

        configs = ["DiBP-Net\n(Proposed)", "DiBP-Net\n(+GRL)"]
        fig, ax = plt.subplots(figsize=(_COL1_IN, 4.75))

 # Global rc for this figure so tick labels / axis labels inherit 8 pt
        with plt.rc_context(_IEEE_RC):
            fig, ax = plt.subplots(figsize=(_COL1_IN, 2.75))
            x, w = np.arange(2), 0.20
            bZH  = ax.bar(x - w, knn_zh, w,
                          label=r"$k$-NN on $z_\mathrm{hemo}$",
                          color="#2CA02C", edgecolor="black", lw=0.5, zorder=3)
            bZM  = ax.bar(x,     knn_zm, w,
                          label=r"$k$-NN on $z_\mathrm{morph}$",
                          color=C_VAL,    edgecolor="black", lw=0.5, zorder=3)
            bADV = ax.bar([1 + w], [adv_acc[1]], w,
                          label="Adversary acc. (GRL)",
                          color=C_TEST,   edgecolor="black", lw=0.5, zorder=3)

            ax.axhline(chance, color="#333", ls="--", lw=1.0, zorder=4,
                       label=f"Chance ({chance:.3f})")

 # "N/A" annotation above DiBP-Net-proposed adversary slot
            ax.text(0 + w, chance + 0.012, "N/A",
                    ha="center", fontsize=8, color="#666",
                    style="italic", fontname=_TNR)

 # Bar-top value labels
            for grp in (bZH, bZM, bADV):
                for r in grp:
                    if r.get_height() > 0.001:
                        ax.text(r.get_x() + r.get_width() / 2,
                                r.get_height() + 0.006,
                                f"{r.get_height():.3f}",
                                ha="center", va="bottom",
                                fontsize=8, fontname=_TNR)

            ax.set_xticks(x)
            ax.set_xticklabels(configs, fontsize=8, fontname=_TNR)
            ax.set_ylabel("Classification Accuracy",
                          fontsize=8, fontname=_TNR)
            ax.set_ylim(0, 0.24)
            ax.set_title("Subject-Identity Leakage",
                         fontweight="bold", fontsize=8, fontname=_TNR)

 # Tick params
            ax.tick_params(axis="both", labelsize=8)

 # Legend below the axes
            ax.legend(prop=leg_prop, loc="upper center",
                      bbox_to_anchor=(0.45, -0.2), ncol=2, framealpha=0.98)

            fig.subplots_adjust(bottom=0.40, top=0.88, left=0.18, right=0.97)
            _save(fig, "fig7_leakage_bar")

    except Exception as e:
        print(f"  ⚠️  Skipping Fig 7: {e}")

In [ ]:
def build_figure_8():
    try:
        EP, MODELS = list(range(10, 170, 10)), _build_all_models()
        leg_prop = {"family": "Times New Roman", "size": 8}
        def _all_finite(metric):
            return [float(v) for _, d in MODELS for v in d.get(metric, [])
                    if v is not None and np.isfinite(float(v))]

        _sbp  = _all_finite("sbp")
        _corr = _all_finite("corr")
        _icc  = _all_finite("icc")
        _cd   = _all_finite("cd")

        ylims = {
            "sbp":  (min(_sbp)  - 0.05, max(_sbp)  + 0.05) if _sbp  else (6.5, 9.0),
            "corr": (min(min(_corr) - 0.03, -TARGET_CORR - 0.02),
                     max(max(_corr) + 0.03,  TARGET_CORR + 0.02)) if _corr else (-0.6, 0.6),
            "icc":  (max(0.0, min(_icc) - 0.002), min(1.005, max(_icc) + 0.002)) if _icc else (0.96, 1.001),
            "cd":   (max(0.0, min(_cd) - 0.2), max(max(_cd) + 0.2, CD_TARGET + 0.2)) if _cd  else (0.0, 7.0),
        }

        PANELS = [
            ("sbp",  "Pre SBP MAE\n(mmHg)",              None,      None, "Pre SBP MAE"),
            ("corr", r"$r(z_\mathrm{morph},\,z_\mathrm{hemo})$", -TARGET_CORR, TARGET_CORR, "Latent Correlation"),
            ("icc",  r"ICC$(c_\mathrm{slow})$",             None,      None, r"ICC$(c_\mathrm{slow})$"),
            ("cd",   r"$C_d$" + "\n(HTN-2 vs Normal)",      CD_TARGET, None, r"Discriminability $C_d$"),
        ]

        mh, eh = _legend_handles()   # model lines + event markers

        with plt.rc_context(_IEEE_RC):
            fig, axes = plt.subplots(4, 1, figsize=(3.5, 9.65), sharex=True)
            
            for ax, (metric, ylabel, tlo, thi, title_str) in zip(axes, PANELS):
                for key, data in MODELS:
                    arr = _nan(data.get(metric, []))
                    if np.all(np.isnan(arr.data if hasattr(arr, "data") else arr)):
                        continue
                    col, ls, lw = STYLE[key]
                    ax.plot(EP, arr, color=col, linestyle=ls, linewidth=lw,
                            label=LABEL[key], zorder=3)
                    _add_markers_metric(ax, key, data, metric, EP)

                if tlo is not None and thi is not None:
                    ax.axhspan(tlo, thi, color="#2CA02C", alpha=0.08, zorder=1)
                    ax.axhline(0, color="#333", lw=0.7, alpha=0.4)
                    for y in (tlo, thi):
                        ax.axhline(y, color="#2CA02C", lw=0.8, ls=":", alpha=0.6)
                elif tlo is not None:
                    ax.axhline(tlo, color="#2CA02C", ls="--", lw=1.0, alpha=0.6)
                _phase_line(ax)

                ax.set_xlim(5, 175)
                ax.set_ylim(ylims[metric])
                ax.set_xticks(range(10, 165, 30))
                ax.set_ylabel(ylabel, fontsize=8, fontname=_TNR[0])
                ax.set_title(title_str, fontsize=8, fontname=_TNR[0], fontweight="bold", pad=3)
                ax.tick_params(axis="both", labelsize=8)

            axes[-1].set_xlabel("Epoch", fontsize=8, fontname=_TNR[0])
            
            fig.legend(
                handles=mh + eh, loc="lower center", bbox_to_anchor=(0.5, 0.025),
                ncol=3, framealpha=0.97, edgecolor="#cccccc", prop=leg_prop,
                handlelength=1.6, handletextpad=0.4, columnspacing=0.8,
                labelspacing=0.25, borderpad=0.4,
            )
            
            fig.tight_layout(pad=0.5, rect=[0, 0.08, 1, 1])
            _save(fig, "fig8_training_dynamics")

    except Exception as e:
        print(f"  ⚠️  Skipping Fig 8: {e}")
build_figure_8()

### Additional-Experiment Definitions

In [ ]:
N_BEATS = ARCH.n_beats
SEQ_LEN = ARCH.seq_len
PHYSIO  = PHYSIO

# ══════════════════════════════════════════════════════════════════════════════
# PATHS and CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════

V192_CKPT_DIR  = "outputs/phase_2_v192/checkpoints"
EXP_OUT_DIR    = os.path.join(OUT_DIR, "exp_additional")
_FIG_DATA = {}   # figure payloads: EXPs populate, the Figure Generation section consumes
os.makedirs(EXP_OUT_DIR, exist_ok=True)

PULSEDB_NPZ_PATH   = "outputs/phase_1/data/pulsedb_tensors.npz"
PULSEDB_CACHE_PATH = "outputs/phase_2_v192/pulsedb_subj_cache.pkl"

EVAL_FRAC            = 0.30
N_MIN_EVAL_PER_STAGE = 60
NOISE_LEVELS         = [0.0, 0.02, 0.05, 0.10, 0.20]
DRIFT_BIN_SIZE       = 3
DRIFT_MAX_BINS       = 6
AGE_SPLIT_YEAR       = 55
MIN_SPLIT_N          = 5
N_BOOT               = 1000
RIDGE_ALPHA          = 10.0
BATCH                = ARCH.batch

STAGE_COLORS = {1: "#27AE60", 2: "#F1C40F", 3: "#E67E22", 4: "#E74C3C"}


# ══════════════════════════════════════════════════════════════════════════════
# INFERENCE HELPERS
# ══════════════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════════════
# DATA ACCESS
# ══════════════════════════════════════════════════════════════════════════════

def _unpack_split(split):
    return split.X, split.Q, split.Ybp, split.subj, split.base

def get_test_arrays():
    return _unpack_split(SPLITS.test)

def get_val_arrays():
    return _unpack_split(SPLITS.val)


def _get_raw_ensemble_transformer(mdl, X, Q, subj_arr, base_arr, batch=BATCH):
    mdl.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            sl      = slice(i, min(i + batch, len(X)))
            x_in    = torch.tensor(X[sl], dtype=torch.float32).to(DEVICE)
            subj_in = torch.tensor(subj_arr[sl], dtype=torch.long).to(DEVICE)
            base_in = torch.tensor(base_arr[sl], dtype=torch.float32).to(DEVICE)
            out = mdl(x_in, subj_in, base_in)
            preds.append(out.cpu().numpy())
    return np.concatenate(preds)

In [ ]:
# SHARED INFERENCE HELPERS
# ══════════════════════════════════════════════════════════════════════════════



def _infer_forward(mdl, X, Q, subj_arr, base_arr, batch=BATCH):
    mdl.eval()
    pop_sig = mdl.pop_sig.item()
    pop_mu  = mdl.pop_mu.item()
    out_keys = ["abp_pred", "z_hemo", "z_morph", "bp_direct", "bp_wave", "bp_ens",
                "R", "C", "RC", "c_slow", "gate", "htn_logit", "aux_logit",
                "sbp_wk", "dbp_wk"]
    acc = {k: [] for k in out_keys}
    with torch.no_grad():
        for i in range(0, len(X), batch):
            sl  = slice(i, min(i + batch, len(X)))
            q_c = torch.tensor(Q[sl, ARCH.n_beats // 2]).to(DEVICE)
            if q_c.dim() == 3:
                q_c = q_c.squeeze(1)
            raw = mdl(torch.tensor(X[sl]).to(DEVICE),
                      torch.tensor(subj_arr[sl], dtype=torch.long).to(DEVICE),
                      torch.tensor(base_arr[sl]).to(DEVICE),
                      q_c, 0.0)
            abp_mm = raw[0].cpu().numpy() * pop_sig + pop_mu
            acc["abp_pred"].append(raw[0].cpu().numpy())
            acc["z_hemo"].append(raw[4].cpu().numpy())
            acc["z_morph"].append(raw[18].cpu().numpy())
            acc["bp_direct"].append(raw[10].cpu().numpy())
            acc["bp_wave"].append(raw[11].cpu().numpy())
            acc["bp_ens"].append(raw[12].cpu().numpy())
            acc["R"].append(raw[7].cpu().numpy())
            acc["C"].append(raw[8].cpu().numpy())
            acc["RC"].append(raw[19].cpu().numpy())
            acc["c_slow"].append(raw[20].cpu().numpy())
            acc["gate"].append(raw[14].cpu().numpy().squeeze())
            acc["htn_logit"].append(raw[6].cpu().numpy())
            acc["aux_logit"].append(raw[13].cpu().numpy())
            acc["sbp_wk"].append(_soft_max_np(abp_mm))
            acc["dbp_wk"].append(_soft_min_np(abp_mm))
    return {k: np.concatenate(v) for k, v in acc.items()}


def _get_raw_ensemble(mdl, X, Q, subj_arr, base_arr):
 # SSoT: raw ensemble = model's bp_ens directly, matching get_raw_predictions()
 # (the calibration section, the main table). Was a re-blend LW.ens_bp_weight*bp_ens +
 # LW.ens_wk_weight*WK; unified to bp_ens per the raw-ensemble decision.
    return _infer_forward(mdl, X, Q, subj_arr, base_arr)["bp_ens"]


def _build_ridge_feats_10(mdl, X, Q, subj_arr, base_arr):
    fwd    = _infer_forward(mdl, X, Q, subj_arr, base_arr)
    abp_mm = fwd["abp_pred"] * mdl.pop_sig.item() + mdl.pop_mu.item()
    return _stack_ridge_feats_10(fwd["bp_direct"], fwd["bp_wave"],
                                 _soft_max_np(abp_mm), _soft_min_np(abp_mm),
                                 fwd["R"], fwd["C"], base_arr)


# _build_ridge_feats_5: single source of truth defined in the calibration utilities (Calibration Utilities,
# cell defining CalibState / fit_ridge_on_val) — already in scope here, not redefined.


In [ ]:
# CALIBRATION HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _calib_get(c, *keys):
    for k in keys:
        v = c.get(k) if isinstance(c, dict) else getattr(c, k, None)
        if v is not None:
            return v
    return None


def fit_ridge_iso(feats, Ybp, alpha=RIDGE_ALPHA):
    ridge_s = Ridge(alpha=alpha).fit(feats, Ybp[:, 0])
    ridge_d = Ridge(alpha=alpha).fit(feats, Ybp[:, 1])
    rp      = np.stack([ridge_s.predict(feats), ridge_d.predict(feats)], axis=1)
    iso_s   = IsotonicRegression(out_of_bounds="clip").fit(
        rp[np.argsort(rp[:, 0]), 0], Ybp[np.argsort(rp[:, 0]), 0])
    iso_d   = IsotonicRegression(out_of_bounds="clip").fit(
        rp[np.argsort(rp[:, 1]), 1], Ybp[np.argsort(rp[:, 1]), 1])
    return ridge_s, ridge_d, iso_s, iso_d


def apply_ridge_iso(feats, ridge_s, ridge_d, iso_s, iso_d):
    rp      = np.stack([ridge_s.predict(feats), ridge_d.predict(feats)], axis=1)
    out     = rp.copy()
    out[:,0] = iso_s.predict(rp[:, 0])
    out[:,1] = iso_d.predict(rp[:, 1])
    return out.astype(np.float32)


def run_subject_bias_correction(primary, Ybp, subj_arr,
                                ref_beats=None, ols_gap=None,
                                eval_min=None, min_beats=None):
    ref_beats  = ref_beats  or CALIB.ref_beats
    ols_gap    = ols_gap    or CALIB.ols_gap
    eval_min   = eval_min   or CALIB.eval_min
    min_beats  = min_beats  or CALIB.min_beats

    cal_pred  = primary.copy()
    eval_mask = np.zeros(len(primary), dtype=bool)

    for s in np.unique(subj_arr):
        idx = np.where(subj_arr == s)[0]
        n   = len(idx)
        if n < min_beats:
            continue
        n_ref = (ref_beats if n >= ref_beats + ols_gap + eval_min
                 else max(1, n - ols_gap - eval_min))

 # [FIX-LEAK] positional ref/eval split with an explicit ols_gap
 # between them -- matches apply_subject_calibration() (the evaluation section EVALUATION).
 # Previously this ranked beats by SBP-representativeness (closest to
 # the subject's own mean SBP) with NO gap enforced, which hand-picks
 # an easier-than-real-deployment reference set and lets ref/eval
 # beats sit adjacent. That inflated Subj-Cal accuracy in every Part 5+
 # experiment relative to the Master Table's causal split.
        ref_idx  = idx[:n_ref]
        eval_idx = idx[n_ref + ols_gap:]

        if len(eval_idx) < eval_min:
            continue

        cal_pred[eval_idx, 0] = primary[eval_idx, 0] + float(
            np.median(Ybp[ref_idx, 0] - primary[ref_idx, 0]))
        cal_pred[eval_idx, 1] = primary[eval_idx, 1] + float(
            np.median(Ybp[ref_idx, 1] - primary[ref_idx, 1]))
        eval_mask[eval_idx] = True

    return cal_pred, eval_mask


def full_pipeline_test(mdl, calib_dict):
    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    _build_subj_cache(mdl, X_te, Q_te, subj_te, base_te)
    feats10 = _build_ridge_feats_10(mdl, X_te, Q_te, subj_te, base_te)
    r_s = _calib_get(calib_dict, "ridge_sbp")
    r_d = _calib_get(calib_dict, "ridge_dbp")
    i_s = _calib_get(calib_dict, "iso_s")
    i_d = _calib_get(calib_dict, "iso_d")
    if all(v is not None for v in [r_s, r_d, i_s, i_d]):
        primary = apply_ridge_iso(feats10, r_s, r_d, i_s, i_d)
    else:
        print("  [WARN] No Ridge/Iso — using raw ensemble")
        primary = _get_raw_ensemble(mdl, X_te, Q_te, subj_te, base_te)
    cal_pred, eval_mask = run_subject_bias_correction(primary, Ybp_te, subj_te)
    fwd = _infer_forward(mdl, X_te, Q_te, subj_te, base_te)
    return primary, cal_pred, eval_mask, fwd


# ══════════════════════════════════════════════════════════════════════════════
# SHARED STATISTICS HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def mae(pred, true, col):
    return float(np.abs(pred[:, col] - true[:, col]).mean())


def eval_metrics(primary, cal_pred, Ybp, subj_arr, eval_mask, raw=None):
    eidx      = np.where(eval_mask)[0]
    beat_pre  = np.abs(primary[eidx]  - Ybp[eidx]).mean(0)
    beat_post = np.abs(cal_pred[eidx] - Ybp[eidx]).mean(0)
    sbp_pre,  dbp_pre  = subj_mae(primary,  Ybp, subj_arr, eval_mask)
    sbp_post, dbp_post = subj_mae(cal_pred, Ybp, subj_arr, eval_mask)

    out = dict(sbp_beat_pre=float(beat_pre[0]),  sbp_beat_post=float(beat_post[0]),
               dbp_beat_pre=float(beat_pre[1]),  dbp_beat_post=float(beat_post[1]),
               sbp_subj_pre=sbp_pre,             sbp_subj_post=sbp_post,
               dbp_subj_pre=dbp_pre,             dbp_subj_post=dbp_post)

 # [ADD] Raw (pre-Ridge/Iso) metrics — computed only when raw is supplied,
 # so existing callers that don't pass raw see identical behavior as before.
    if raw is not None:
        beat_raw = np.abs(raw[eidx] - Ybp[eidx]).mean(0)
        sbp_subj_raw, dbp_subj_raw = subj_mae(raw, Ybp, subj_arr, eval_mask)
        out.update(sbp_beat_raw=float(beat_raw[0]), dbp_beat_raw=float(beat_raw[1]),
                   sbp_subj_raw=sbp_subj_raw,       dbp_subj_raw=dbp_subj_raw)
    return out


def partial_corr(x, y, covariates):
    cov = (np.column_stack(covariates) if len(covariates) > 1
           else np.array(covariates[0]).reshape(-1, 1))
    lr = LinearRegression()
    return pearsonr(x - lr.fit(cov, x).predict(cov),
                    y - lr.fit(cov, y).predict(cov))


def bootstrap_ci(values, func=np.mean, n_boot=N_BOOT, seed=42):
    rng  = np.random.default_rng(seed)
    boot = [func(rng.choice(values, len(values), replace=True)) for _ in range(n_boot)]
    return float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))


def norm_wss(vals, subj_arr):
    g = vals.std()
    if g < 1e-9:
        return np.nan
    stds = [vals[subj_arr == s].std(ddof=1) / g
            for s in np.unique(subj_arr) if (subj_arr == s).sum() > 1]
    return float(np.mean(stds)) if stds else np.nan


def stability_ratio_with_ci(vals_ref, vals_comp, subj_arr, n_boot=N_BOOT, seed=42):
    ratio = norm_wss(vals_comp, subj_arr) / (norm_wss(vals_ref, subj_arr) + 1e-9)
    rng   = np.random.default_rng(seed)
    subjs = np.unique(subj_arr)
    boot  = []
    for _ in range(n_boot):
        chosen = rng.choice(subjs, len(subjs), replace=True)
        idx    = np.concatenate([np.where(subj_arr == s)[0] for s in chosen])
        r_ref  = norm_wss(vals_ref[idx],  subj_arr[idx])
        r_comp = norm_wss(vals_comp[idx], subj_arr[idx])
        boot.append(r_comp / (r_ref + 1e-9))
    return float(ratio), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))


def fisher_ci(r, n, ci=0.95):
    z  = np.arctanh(np.clip(r, -0.9999, 0.9999))
    se = 1.0 / np.sqrt(max(n - 3, 1))
    zc = scipy_norm.ppf((1 + ci) / 2)
    return float(np.tanh(z - zc * se)), float(np.tanh(z + zc * se))


def power_twosample(n1, n2, d, alpha=0.05):
    se  = np.sqrt(1.0 / n1 + 1.0 / n2)
    z_a = scipy_norm.ppf(1.0 - alpha / 2.0)
    ncp = d / se
    return float(1.0 - scipy_norm.cdf(z_a - ncp) + scipy_norm.cdf(-z_a - ncp))


def aami_bhs(pred, true, label=""):
    err  = pred - true
    me   = err.mean(); sd = err.std(); mae_v = np.abs(err).mean()
    p5   = (np.abs(err) <= 5).mean()  * 100
    p10  = (np.abs(err) <= 10).mean() * 100
    p15  = (np.abs(err) <= 15).mean() * 100
    aami = "PASS" if abs(me) <= 5 and sd <= 8 else "FAIL"
    bhs = _bhs_grade(p5, p10, p15)   # SSoT (cell 3)
    print(f"  ── {label} ──")
    print(f"     MAE={mae_v:.2f}  ME={me:+.2f}  SD={sd:.2f}  "
          f"AAMI:{aami}  BHS:{bhs}  (%5/{p5:.1f} %10/{p10:.1f} %15/{p15:.1f})")
    return dict(label=label, mae=mae_v, me=me, sd=sd,
                p5=p5, p10=p10, p15=p15, aami=aami, bhs=bhs)


def print_metric_table(title, rows, col_names):
    keys = [("SBP Beat Raw (mmHg)",         "sbp_beat_raw"),
            ("SBP Beat Global-Cal (mmHg)",  "sbp_beat_pre"),
            ("SBP Beat Subj-Cal (mmHg)",    "sbp_beat_post"),
            ("SBP Subj Raw (mmHg)",         "sbp_subj_raw"),
            ("SBP Subj Global-Cal (mmHg)",  "sbp_subj_pre"),
            ("SBP Subj Subj-Cal (mmHg)",    "sbp_subj_post"),
            ("DBP Beat Raw (mmHg)",         "dbp_beat_raw"),
            ("DBP Beat Global-Cal (mmHg)",  "dbp_beat_pre"),
            ("DBP Beat Subj-Cal (mmHg)",    "dbp_beat_post"),
            ("DBP Subj Raw (mmHg)",         "dbp_subj_raw"),
            ("DBP Subj Global-Cal (mmHg)",  "dbp_subj_pre"),
            ("DBP Subj Subj-Cal (mmHg)",    "dbp_subj_post")]
    def _f(d, k):
        v = d.get(k, np.nan) if d else np.nan
        return f"{'—':>12}" if (v is None or np.isnan(v)) else f"{v:>12.2f}"
    print(f"\n  {title}")
    print(f"  {'Metric':<35}" + "".join(f" {c:>12}" for c in col_names))
    print(f"  {'─'*35}" + "".join(f" {'─'*12}" for _ in col_names))
    for label, key in keys:
        print(f"  {label:<35}" + "".join(f" {_f(d, key)}" for d in rows))


In [ ]:
# EXTERNAL DATASET LOADERS (PulseDB + phase-1 datasets)
# ══════════════════════════════════════════════════════════════════════════════
def build_windows(X_ppg, Ybp, subj_ids, window_size=None, beat_cap=20, seed=42,
                   Yabp=None, guarantee_first_n=None):
    """
    [FIX] guarantee_first_n: number of earliest windows (in real time) that must always
    be included regardless of the random draw -- prevents rng.choice() from dropping the
    true start, which would leave run_subject_bias_correction's idx[:ref_beats] and _build
    ref set no longer the true first beats (the ref-beat audit found:
    97.4% of PulseDB subjects have more windows than beat_cap, so the first beat can drop).
    default = CALIB.ref_beats + CALIB.ols_gap (the count calibration needs as a true
    reference before reaching the eval beats).
    """
    window_size = window_size or (ARCH.n_beats + 1)
    guarantee_first_n = guarantee_first_n or (CALIB.ref_beats + CALIB.ols_gap)  # 3+1=4

    if X_ppg.ndim == 3:
        X_ppg = X_ppg[:, 0, :]

    order = np.argsort(subj_ids, kind="stable")
    sorted_subj = subj_ids[order]
    uniq, starts, counts = np.unique(sorted_subj, return_index=True, return_counts=True)

    rng = np.random.default_rng(seed)
    c   = window_size // 2
    
    X_out, Q_out, Y_out, S_out, BTRUE_out = [], [], [], [], []

    for sid, st, cnt in zip(uniq, starts, counts):
        if cnt < window_size:
            continue
        idx = np.sort(order[st:st + cnt])   # true chronological beat order
        xs = np.asarray(X_ppg[idx], dtype=np.float32)
        ys = np.asarray(Ybp[idx],   dtype=np.float32)

        if Yabp is not None:
            wave_first = np.asarray(Yabp[idx[0]], dtype=np.float32)
            wave_first_mm = wave_first * SPLITS.pop_sig + SPLITS.pop_mu
            base_true_first_beat = float((wave_first_mm.max() + wave_first_mm.min()) / 2.0)
        else:
            base_true_first_beat = float((ys[0, 0] + ys[0, 1]) / 2.0)

        d    = np.gradient(xs, axis=-1)
        q    = np.maximum(d, 0.0)
        area = np.trapezoid(q, axis=-1)[:, np.newaxis]
        area = np.where(area < 1e-6, 1e-6, area)
        q_all = (q / area * 8.0).astype(np.float32)

        n_wins = cnt - window_size + 1
        win_starts = np.arange(n_wins)

        if n_wins > beat_cap:
            n_guarantee = min(guarantee_first_n, n_wins)
            guaranteed  = np.arange(n_guarantee)
            remaining_budget = beat_cap - n_guarantee
            if remaining_budget > 0:
                remaining_pool = np.arange(n_guarantee, n_wins)
                extra = (rng.choice(remaining_pool, remaining_budget, replace=False)
                          if len(remaining_pool) > remaining_budget else remaining_pool)
                win_starts = np.concatenate([guaranteed, extra])
            else:
                win_starts = guaranteed
            win_starts = np.sort(win_starts)

        for s in win_starts:
            X_out.append(xs[s:s + window_size])
            Q_out.append(q_all[s:s + window_size])
            Y_out.append(ys[s + c])
            S_out.append(int(sid))
            BTRUE_out.append(base_true_first_beat)

    if not X_out:
        return None
        
    return dict(
        X    = np.stack(X_out).astype(np.float32),
        Q    = np.stack(Q_out).astype(np.float32),
        Ybp  = np.stack(Y_out).astype(np.float32),
        subj = np.array(S_out, dtype=np.int64),
        base_true_first_beat = np.array(BTRUE_out, dtype=np.float32),
    )


def _normalize_base_per_subject(base_raw_mmhg, pop_mu, pop_sig):
    """z-score a raw-mmHg per-subject-constant base array."""
    return ((base_raw_mmhg - pop_mu) / pop_sig).astype(np.float32)

def load_external(tag, beat_cap=20):
    path, name = _EXT_PATHS[tag]
    if not os.path.exists(path):
        print(f"  [{tag}] NOT FOUND: {path}"); return None

    raw  = np.load(path, allow_pickle=True, mmap_mode="r")
    Xppg = raw["X_ppg"]
    Ybp  = raw["Y_bp"].astype(np.float32)
    sraw = raw["subject_ids"]

 # [FIX] load Y_abp too, when present (Sensors/BCG/PulseDB have it;
 # Hospital doesn't -- PPG-only dataset, no waveform to fall back on)
    Yabp_full = raw["Y_abp"].astype(np.float32) if "Y_abp" in raw else None
    if Yabp_full is None:
        print(f"  [{tag}] no Y_abp in source npz — base will use Ybp-label proxy (Hospital-style)")

    offset   = _EXT_OFFSETS[tag]
    uniq     = np.unique(sraw)
    subj_int = (np.searchsorted(uniq, sraw) + offset).astype(np.int64)

    print(f"  Building windows [{tag}]: {len(Xppg):,} beats, {len(uniq):,} subjects", end=" … ", flush=True)
    w = build_windows(Xppg, Ybp, subj_int, window_size=ARCH.n_beats + 1,
                       beat_cap=beat_cap, Yabp=Yabp_full)
    if w is None:
        print("too few beats/subj — skip"); return None
    n_subj = len(np.unique(w["subj"]))
    print(f"→ {len(w['X']):,} windows, {n_subj:,} subj usable")

    base_fixed = _normalize_base_per_subject(w["base_true_first_beat"],
                                         SPLITS.pop_mu, SPLITS.pop_sig)
    
    return dict(X_eval=w["X"], Q_eval=w["Q"], Ybp_eval=w["Ybp"],
                subj_eval=w["subj"], base_eval=base_fixed, name=name, tag=tag,
                used_exact_waveform_base=(Yabp_full is not None))   # for auditing later


def load_pulsedb(n_beats_cap: int = 20):
    """Load all PulseDB subjects for zero-shot evaluation.

    Uses all available subjects (no UCI-count cap, no fit/eval split).
    Calibration is applied zero-shot from UCI-fitted weights.
    Builds REAL 11-beat consecutive windows (see the window-builder header for the bug fix).
    n_beats_cap: max independent windows sampled per subject (memory guard).
    """
    if not os.path.exists(PULSEDB_NPZ_PATH):
        print(f"  [SKIP] PulseDB not found: {PULSEDB_NPZ_PATH}")
        return None

 # Prefer the uncompressed .npy folder (produced once by presplit_pulsedb.py):
 # compressed npz cannot mmap -> numpy loads the full ~8 GB X_ppg into RAM and
 # OOM-crashes the kernel. Uncompressed .npy mmaps for real, so only the
 # sampled windows (~320 MB) ever hit RAM.
    _UNCOMP = os.path.join(os.path.dirname(PULSEDB_NPZ_PATH), "pulsedb_uncompressed")
    if os.path.isdir(_UNCOMP) and os.path.exists(os.path.join(_UNCOMP, "X_ppg.npy")):
        _ld = lambda k: np.load(os.path.join(_UNCOMP, k + ".npy"),
                                mmap_mode="r", allow_pickle=True)
        X_raw      = _ld("X_ppg")                       # stays memory-mapped (big)
        Ybp_raw    = np.array(_ld("Y_bp"))              # small -> materialise
        subj_raw   = np.array(_ld("subject_ids"))
        age_raw    = np.array(_ld("age"),    dtype=np.float32)
        height_raw = np.array(_ld("height"), dtype=np.float32)
        weight_raw = np.array(_ld("weight"), dtype=np.float32)
        gender_raw = np.array(_ld("gender"))
        
        Yabp_raw = _ld("Y_abp") if os.path.exists(os.path.join(_UNCOMP, "Y_abp.npy")) else None
        print("  [PulseDB] using uncompressed mmap folder (OOM-safe)")
    else:
        raw        = np.load(PULSEDB_NPZ_PATH, allow_pickle=True, mmap_mode="r")
        X_raw      = raw["X_ppg"]
        Ybp_raw    = raw["Y_bp"]
        subj_raw   = raw["subject_ids"]
        age_raw    = np.array(raw["age"],    dtype=np.float32)
        height_raw = np.array(raw["height"], dtype=np.float32)
        weight_raw = np.array(raw["weight"], dtype=np.float32)
        gender_raw = np.array(raw["gender"])
        
        Yabp_raw = raw["Y_abp"] if "Y_abp" in raw else None
        print("  [PulseDB] compressed npz (may OOM -- run presplit_pulsedb.py to avoid)")

    unique_orig = np.unique(subj_raw)
    subj_int    = (np.searchsorted(unique_orig, subj_raw) + 99999).astype(np.int64)

    n_uci_subj  = len(np.unique(SPLITS.test.subj))
    n_uci_beats = int(np.median(
        [(SPLITS.test.subj == s).sum() for s in np.unique(SPLITS.test.subj)]))
    print(f"  PulseDB: {len(X_raw):,} beats, {len(unique_orig):,} subjects")
    print(f"  UCI TEST reference: {n_uci_subj} subj, median {n_uci_beats} beats/subj")
    print(f"  Mode: all PulseDB subjects → eval (zero-shot, real 11-beat windows, cap={n_beats_cap})")

 # Demographic lookup keyed by remapped subject id (first occurrence)
    _, first_idx = np.unique(subj_int, return_index=True)
    demo = {}
    for i in first_idx:
        g = gender_raw[i]
        if isinstance(g, (str, bytes, np.str_, np.bytes_)):
            g_i = int(str(g) == "1")
        else:
            g_f = float(g)
            g_i = int(g_f) if not np.isnan(g_f) else -1   # -1 = unknown/missing
        demo[int(subj_int[i])] = (float(age_raw[i]), float(height_raw[i]),
                                   float(weight_raw[i]), g_i)

    windows = build_windows(X_raw, Ybp_raw, subj_int,
                            window_size=ARCH.n_beats + 1, beat_cap=n_beats_cap,
                            Yabp=Yabp_raw)
    if windows is None:
        print("  [SKIP] no PulseDB subject has enough consecutive beats")
        return None

    base_fixed = _normalize_base_per_subject(windows["base_true_first_beat"],
                                         SPLITS.pop_mu, SPLITS.pop_sig)
                                         
    pdb = dict(X_eval=windows["X"], Q_eval=windows["Q"], Ybp_eval=windows["Ybp"],
               subj_eval=windows["subj"], base_eval=base_fixed,
               name="PulseDB", tag="pulsedb",
               used_exact_waveform_base=(Yabp_raw is not None))
               
    pdb["age_eval"]    = np.array([demo[s][0] for s in pdb["subj_eval"]], dtype=np.float32)
    pdb["height_eval"] = np.array([demo[s][1] for s in pdb["subj_eval"]], dtype=np.float32)
    pdb["weight_eval"] = np.array([demo[s][2] for s in pdb["subj_eval"]], dtype=np.float32)
    pdb["gender_eval"] = np.array([demo[s][3] for s in pdb["subj_eval"]], dtype=np.int64)
    pdb["n_uci_subj"]  = n_uci_subj
    pdb["n_uci_beats"] = n_uci_beats
    n_subj_used = len(np.unique(pdb["subj_eval"]))
    print(f"  pdb_eval: {len(pdb['X_eval']):,} windows, {n_subj_used:,} subjects")
    return pdb


# ── Generic loader for the other phase-1 external datasets ─────────────────
_EXT_PATHS = {
    "sensors":  ("outputs/phase_1/data/sensors_tensors.npz",  "Multi-Sensor PPG"),
    "bcg":      ("outputs/phase_1/data/bcg_tensors.npz",      "BCG"),
    "hospital": ("outputs/phase_1/data/hospital_tensors.npz", "Hospital PPG"),
}
_EXT_OFFSETS = {"sensors": 200_000, "bcg": 300_000, "hospital": 400_000}

def load_all_external(beat_cap=20):
    """Load PulseDB + every other phase-1 external dataset for zero-shot eval.
    Returns {tag: data_dict} — datasets with too few beats/subject are omitted.
    """
    result = {"pulsedb": load_pulsedb(beat_cap)}
    for tag in _EXT_PATHS:
        result[tag] = load_external(tag, beat_cap)
    return {k: v for k, v in result.items() if v is not None}



# ══════════════════════════════════════════════════════════════════════════════
# UNIFORM EXTERNAL-DATASET EVALUATION HELPER
# ══════════════════════════════════════════════════════════════════════════════

def _get_primary_cal_mask(d):
    """Zero-shot UCI Ridge+Iso + subject-bias-correction for one external
    dataset dict `d` (from load_pulsedb/load_external). Returns (primary, cal, mask)."""
    r_s   = _calib_get(calib_proposed, "ridge_sbp")
    r_d   = _calib_get(calib_proposed, "ridge_dbp")
    iso_s = _calib_get(calib_proposed, "iso_s")
    iso_d = _calib_get(calib_proposed, "iso_d")
    _build_subj_cache(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])
    feats   = _build_ridge_feats_10(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])
    primary = apply_ridge_iso(feats, r_s, r_d, iso_s, iso_d)
    cal, mask = run_subject_bias_correction(primary, d["Ybp_eval"], d["subj_eval"])
    return primary, cal, mask


def _iter_ext_datasets(ext_dict):
    """Yield (display_name, tag, data_dict) for every loaded external dataset,
    PulseDB first, in a stable order."""
    order = ["pulsedb", "sensors", "bcg", "hospital"]
    for tag in order:
        d = ext_dict.get(tag)
        if d is not None:
            yield d.get("name", tag), tag, d


In [ ]:
# Calibration feature ablation
# ══════════════════════════════════════════════════════════════════════════════

def run_exp1():
    print("\n" + "─" * 70)
    print("  Feature-matched 5-feature vs 10-feature Ridge  [TEST set]")
    print("─" * 70)

    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    X_va, Q_va, Ybp_va, subj_va, base_va = get_val_arrays()

    _build_subj_cache(model, X_va, Q_va, subj_va, base_va)
    raw_va    = _get_raw_ensemble(model, X_va, Q_va, subj_va, base_va)
    r5_s, r5_d, iso5_s, iso5_d = fit_ridge_iso(_build_ridge_feats_5(raw_va, base_va), Ybp_va)

 # Raw stage = truly-raw: empty subject cache → pop_mean embedding (zero
 # personalization), matching the Master Table's "Raw" definition .
    model._subj_z_cache = {}
    raw_pop   = _get_raw_ensemble(model, X_te, Q_te, subj_te, base_te)

 # Pre/Post path uses the personalized 3-beat cache (Ridge+Iso input).
    _build_subj_cache(model, X_te, Q_te, subj_te, base_te)
    raw_te    = _get_raw_ensemble(model, X_te, Q_te, subj_te, base_te)
    primary5  = apply_ridge_iso(_build_ridge_feats_5(raw_te, base_te),
                                r5_s, r5_d, iso5_s, iso5_d)

    cal5, mask5 = run_subject_bias_correction(primary5, Ybp_te, subj_te)
    metrics5    = eval_metrics(primary5, cal5, Ybp_te, subj_te, mask5, raw=raw_pop)

    primary10, cal10, mask10, _ = full_pipeline_test(model, calib_proposed)
    metrics10 = eval_metrics(primary10, cal10, Ybp_te, subj_te, mask10, raw=raw_pop)

    print_metric_table("Calibration feature ablation [TEST set]",
                        [metrics10, metrics5], ["10-feat (paper)", "5-feat (matched)"])
    delta = metrics5["sbp_beat_post"] - metrics10["sbp_beat_post"]
    print(f"\n  → Physics-feature Δ SBP Beat Post: {delta:+.2f} mmHg")
    print(f"  → {'Negligible' if abs(delta) < 1.0 else 'Meaningful'} difference — "
          f"physics features add interpretability{'.' if abs(delta) < 1.0 else ', not just shortcut.'}")
    print("DONE ✓")
    return dict(metrics10=metrics10, metrics5=metrics5, delta_sbp=delta)

In [ ]:
# (UNIFIED) : Zero-Shot External Validation
# ══════════════════════════════════════════════════════════════════════════════

def run_exp2_unified(pdb, ext_dict):
    print("\n" + "─" * 80)
    print("  Zero-shot external validation  [UCI → ALL EXTERNALS]")
    print("─" * 80)
    
    print("  Loading UCI-fitted Ridge+Iso (zero-shot transfer)...")
    r_s   = _calib_get(calib_proposed, "ridge_sbp")
    r_d   = _calib_get(calib_proposed, "ridge_dbp")
    iso_s = _calib_get(calib_proposed, "iso_s")
    iso_d = _calib_get(calib_proposed, "iso_d")
    
    if not all([r_s, r_d, iso_s, iso_d]):
        print("  [SKIP] calib_proposed incomplete")
        return None

    all_metrics = []
    all_names   = []
    extra_stats = []
    results     = {}

 # 2. UCI TEST baseline
    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    model._subj_z_cache = {}   # Raw = truly-raw: pop_mean, no personalization (matches the master evaluation)
    raw_uci = _get_raw_ensemble(model, X_te, Q_te, subj_te, base_te)
    primary_uci, cal_uci, mask_uci, _ = full_pipeline_test(model, calib_proposed)  # rebuilds personalized cache for Pre
    metrics_uci = eval_metrics(primary_uci, cal_uci, Ybp_te, subj_te, mask_uci, raw=raw_uci)  # [ADD raw=]
    
    all_metrics.append(metrics_uci)
    all_names.append("UCI (TEST)")
    results["uci"] = dict(**metrics_uci, primary=primary_uci, cal=cal_uci, mask=mask_uci, raw=raw_uci)

    externals = []
    if pdb is not None:
        externals.append(("PulseDB", pdb))
    if ext_dict is not None:
        for tag in ("sensors", "bcg", "hospital"):
            if ext_dict.get(tag):
                externals.append((ext_dict[tag]["name"], ext_dict[tag]))

    for name, d in externals:
        model._subj_z_cache = {}   # Raw = truly-raw: pop_mean
        raw     = _get_raw_ensemble(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])
        _build_subj_cache(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])  # personalized for Pre/Post + c_slow
        feats   = _build_ridge_feats_10(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])
        primary = apply_ridge_iso(feats, r_s, r_d, iso_s, iso_d)
        cal, mask = run_subject_bias_correction(primary, d["Ybp_eval"], d["subj_eval"])

        eidx = np.where(mask)[0]
        if len(eidx) == 0:
            print(f"  [{name}] n_eval=0 — insufficient reference beats per subject, skip")
            continue

        metrics = eval_metrics(primary, cal, d["Ybp_eval"], d["subj_eval"], mask, raw=raw)  # [ADD raw=]
        all_metrics.append(metrics)
        all_names.append(name)

        cs = _infer_forward(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])["c_slow"]
        r_cs, p_cs = pearsonr(cs, d["Ybp_eval"][:, 0])

        d_bias = primary[eidx, 0] - d["Ybp_eval"][eidx, 0]
        bias_mean = d_bias.mean()
        z = PHYSIO.loa_z_score
        loa_l = bias_mean - z * d_bias.std()
        loa_h = bias_mean + z * d_bias.std()

        extra_stats.append((name, r_cs, p_cs, bias_mean, loa_l, loa_h))
        results[name] = dict(**metrics, primary=primary, cal=cal, mask=mask, cs=cs, raw=raw)  # [ADD raw=]

    print("\n")
    print_metric_table("Zero-shot external validation", all_metrics, all_names)

    _p = os.path.join(EXP_OUT_DIR, "exp2_zeroshot_external.csv")
    pd.DataFrame(all_metrics, index=all_names).rename_axis("Dataset").to_csv(_p)
    print(f"\n  Saved \u2192 {_p}")

    print("\n  [Additional Statistics]")
    for name, r_cs, p_cs, bias, loa_l, loa_h in extra_stats:
        print(f"  {name:<16} | c_slow vs SBP: r={r_cs:+.3f} (p={p_cs:.1e}) | B-A SBP Bias: {bias:+.2f} [{loa_l:+.2f}, {loa_h:+.2f}] mmHg")

    print("\nDONE ✓")
    return results

In [ ]:
# the zero-shot validation (70% IN-DOMAIN REFIT VARIANT) : same table as the zero-shot validation
# ══════════════════════════════════════════════════════════════════════════════

def run_exp2_refit70(pdb, ext_dict, fit_frac=0.7, seed=42):
    print("\n" + "─" * 80)
    print(f"  In-domain refit ( {int(fit_frac*100)}%): Per-Dataset Ridge+Iso Refit  [held-out {int((1-fit_frac)*100)}% eval]")
    print("─" * 80)

    all_metrics = []
    all_names   = []
    extra_stats = []
    results     = {}
    rng = np.random.RandomState(seed)

    # ── 1. UCI TEST — reference row, unchanged (UCI already IS the fit domain) ──
    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    model._subj_z_cache = {}   # Raw = truly-raw: pop_mean (matches the master evaluation)
    raw_uci = _get_raw_ensemble(model, X_te, Q_te, subj_te, base_te)
    primary_uci, cal_uci, mask_uci, _ = full_pipeline_test(model, calib_proposed)  # rebuilds personalized cache for Pre
    metrics_uci = eval_metrics(primary_uci, cal_uci, Ybp_te, subj_te, mask_uci, raw=raw_uci)

    all_metrics.append(metrics_uci)
    all_names.append("UCI (TEST)")
    results["uci"] = dict(**metrics_uci, primary=primary_uci, cal=cal_uci, mask=mask_uci, raw=raw_uci)

    externals = []
    if pdb is not None:
        externals.append(("PulseDB", pdb))
    if ext_dict is not None:
        for tag in ("sensors", "bcg", "hospital"):
            if ext_dict.get(tag):
                externals.append((ext_dict[tag]["name"], ext_dict[tag]))

    # ── 2. Loop external datasets: split subjects, refit, eval on held-out ──
    for name, d in externals:
        subj = d["subj_eval"]
        Ybp  = d["Ybp_eval"]
        
        uniq_subj = np.unique(subj)
        if len(uniq_subj) < 10:
            print(f"  [{name}] only {len(uniq_subj)} subjects — skip")
            continue

        rng.shuffle(uniq_subj)
        n_fit = max(5, int(len(uniq_subj) * fit_frac))
        fit_subj  = set(uniq_subj[:n_fit])
        eval_subj = set(uniq_subj[n_fit:])
        fit_mask  = np.array([s in fit_subj  for s in subj])
        eval_mask_subj = np.array([s in eval_subj for s in subj])

        model._subj_z_cache = {}   # Raw = truly-raw: pop_mean
        raw_full = _get_raw_ensemble(model, d["X_eval"], d["Q_eval"], subj, d["base_eval"])
        _build_subj_cache(model, d["X_eval"], d["Q_eval"], subj, d["base_eval"])  # personalized for feats/Pre + c_slow
        feats = _build_ridge_feats_10(model, d["X_eval"], d["Q_eval"], subj, d["base_eval"])

 # slice everything down to the held-out eval subjects for fair comparison
        feats_e = feats[eval_mask_subj]
        Ybp_e   = Ybp[eval_mask_subj]
        subj_e  = subj[eval_mask_subj]
        raw_e   = raw_full[eval_mask_subj]

 # --- refit Ridge+Iso on THIS dataset's own fit_subj (70%) ---
        r_s1, r_d1, iso_s1, iso_d1 = fit_ridge_iso(feats[fit_mask], Ybp[fit_mask])

        primary = apply_ridge_iso(feats_e, r_s1, r_d1, iso_s1, iso_d1)
        cal, mask = run_subject_bias_correction(primary, Ybp_e, subj_e)

        eidx = np.where(mask)[0]
        if len(eidx) == 0:
            print(f"  [{name}] n_eval=0 after 70/30 split — insufficient reference beats per subject, skip")
            continue

        metrics = eval_metrics(primary, cal, Ybp_e, subj_e, mask, raw=raw_e)
        all_metrics.append(metrics)
        all_names.append(name)

        cs = _infer_forward(model, d["X_eval"][eval_mask_subj], d["Q_eval"][eval_mask_subj],
                             subj_e, d["base_eval"][eval_mask_subj])["c_slow"]
        r_cs, p_cs = pearsonr(cs, Ybp_e[:, 0])

        d_bias = primary[eidx, 0] - Ybp_e[eidx, 0]
        bias_mean = d_bias.mean()
        z = PHYSIO.loa_z_score
        loa_l = bias_mean - z * d_bias.std()
        loa_h = bias_mean + z * d_bias.std()

        extra_stats.append((name, r_cs, p_cs, bias_mean, loa_l, loa_h))
        results[name] = dict(**metrics, primary=primary, cal=cal, mask=mask, cs=cs, raw=raw_e,
                             n_fit_subj=n_fit, n_eval_subj=len(eval_subj))

        print(f"  [{name}] n_subj_fit={n_fit}/{len(uniq_subj)}  n_subj_eval={len(eval_subj)}")

    print("\n")
    _p = os.path.join(EXP_OUT_DIR, "exp2_refit70.csv")
    pd.DataFrame(all_metrics, index=all_names).rename_axis("Dataset").to_csv(_p)
    print(f"\n  Saved \u2192 {_p}")

    print_metric_table(f"In-domain refit ({int(fit_frac*100)}% fit / {int((1-fit_frac)*100)}% eval)",
                        all_metrics, all_names)

    print("\n  [Additional Statistics]")
    for name, r_cs, p_cs, bias, loa_l, loa_h in extra_stats:
        print(f"  {name:<16} | c_slow vs SBP: r={r_cs:+.3f} (p={p_cs:.1e}) | B-A SBP Bias: {bias:+.2f} [{loa_l:+.2f}, {loa_h:+.2f}] mmHg")

    print("\nDONE ✓")
    return results

In [ ]:
# Ablation artifact check and latent stability
# ══════════════════════════════════════════════════════════════════════════════

def run_ablation_and_icc():
    print("\n" + "─" * 70)
    print("  Ablation Artifact & ICC/Stability  [TEST set]")
    print("─" * 70)

    if mdl_ablA is None:
        print("  [SKIP] mdl_ablA not loaded"); return None

    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    sbp_te = Ybp_te[:, 0]
    dbp_te = Ybp_te[:, 1]

    _build_subj_cache(model,    X_te, Q_te, subj_te, base_te)
    _build_subj_cache(mdl_ablA, X_te, Q_te, subj_te, base_te)

    _, _, eval_mask,   fwd_prop = full_pipeline_test(model,    calib_proposed)
    _, _, eval_mask_a, fwd_ablA = full_pipeline_test(mdl_ablA, calib_proposed)

 # [F1] all downstream arrays are masked from here
    cs_prop   = fwd_prop["c_slow"][eval_mask]
    cs_ablA   = fwd_ablA["c_slow"][eval_mask_a]
    subj_te_e = subj_te[eval_mask]
    sbp_te_e  = sbp_te[eval_mask]
    dbp_te_e  = dbp_te[eval_mask]
    subjs_u   = np.unique(subj_te_e)
    stages_te = assign_stages(sbp_te_e, dbp_te_e)

    # ── artifact check ────────────────────────────────────────────────
    def _subj_mean(cs):
        mc = np.array([cs[subj_te_e == s].mean() for s in subjs_u])
        ms = np.array([sbp_te_e[subj_te_e == s].mean() for s in subjs_u])
        return pearsonr(mc, ms)

    def _cd_h2n_groups(cs):
 # Fix: use assign_stages(mean_sbp, mean_dbp) to match the bootstrap staging staging
        _sl = np.unique(subj_te_e)
        _sc = np.array([cs[subj_te_e == s].mean() for s in _sl])
        _ms = np.array([sbp_te_e[subj_te_e == s].mean() for s in _sl])
        _md = np.array([dbp_te_e[subj_te_e == s].mean() for s in _sl])
        _st = assign_stages(_ms, _md)
        return _sc[_st == 4], _sc[_st == 1]  # (htn2, normal)

    r_prop, _ = _subj_mean(cs_prop)
    r_ablA, _ = _subj_mean(cs_ablA)

 # [FIX-CI] Vb/Vw and Cohen's d now bootstrap CI (n=N_BOOT), reusing the
 # exact same bootstrap helpers the Master Table uses (the evaluation section run_full_evaluation), so the two
 # tables are statistically comparable once the leakage fixes above make
 # their inputs comparable too.
    vbvw_prop, vbvw_prop_lo, vbvw_prop_hi = _bootstrap_vbvw(cs_prop, subj_te_e, N_BOOT)
    vbvw_ablA, vbvw_ablA_lo, vbvw_ablA_hi = _bootstrap_vbvw(cs_ablA, subj_te_e, N_BOOT)

    htn2_prop, norm_prop = _cd_h2n_groups(cs_prop)
    htn2_ablA, norm_ablA = _cd_h2n_groups(cs_ablA)
    cd_prop, cd_prop_lo, cd_prop_hi = bootstrap_cohens_d_ci(htn2_prop, norm_prop, N_BOOT)
    cd_ablA, cd_ablA_lo, cd_ablA_hi = bootstrap_cohens_d_ci(htn2_ablA, norm_ablA, N_BOOT)

    is_artifact = r_ablA > 0.70 and cd_ablA > cd_prop

    def _ci(v, lo, hi, fmt="{:.1f}"):
        return f"{fmt.format(v)} [{fmt.format(lo)},{fmt.format(hi)}]"

    print(f"\n  {'Metric':<35} {'Proposed':>26} {'Ablation A':>26}")
    print(f"  {'─' * 90}")
    print(f"  {'Vb/Vw':<35} "
          f"{_ci(vbvw_prop, vbvw_prop_lo, vbvw_prop_hi):>26} "
          f"{_ci(vbvw_ablA, vbvw_ablA_lo, vbvw_ablA_hi):>26}")
    print(f"  {'Cohen d (HTN-2 vs Normal)':<35} "
          f"{_ci(cd_prop, cd_prop_lo, cd_prop_hi, '{:.3f}'):>26} "
          f"{_ci(cd_ablA, cd_ablA_lo, cd_ablA_hi, '{:.3f}'):>26}")
    print(f"  {'r(c_slow, mean-SBP)':<35} {r_prop:>26.3f} {r_ablA:>26.3f}")
    print(f"\n  Ablation A artifact: "
          f"{'YES (c_slow collapses to mean-SBP)' if is_artifact else 'NO (genuine)'}")

 # Split into 2 independent 3.5in single-panel figures (was 1x2 subplot
 # sharing a 7.0in total width) with consistent IEEE 8pt styling.
 # (ablA c_slow stage histograms -> Figure Generation section: plot_ablation_icc)

    # ── ICC + stability ratio ─────────────────────────────────────────
    icc_cs,  lo_cs,  hi_cs  = bootstrap_icc_ci(cs_prop,   subj_te_e)
    icc_sbp, lo_sbp, hi_sbp = bootstrap_icc_ci(sbp_te_e,  subj_te_e)
    icc_dbp, lo_dbp, hi_dbp = bootstrap_icc_ci(dbp_te_e,  subj_te_e)
    ratio_sbp, r_lo, r_hi   = stability_ratio_with_ci(cs_prop, sbp_te_e, subj_te_e)
    ratio_dbp, _, _         = stability_ratio_with_ci(cs_prop, dbp_te_e, subj_te_e)

    cs_g  = cs_prop.std(); sbp_g = sbp_te_e.std()
    nstd_cs, nstd_sbp = [], []  # [F2]
    for s in subjs_u:
        m = subj_te_e == s
        if m.sum() < 2: continue
        nstd_cs.append(cs_prop[m].std()   / (cs_g  + 1e-9))
        nstd_sbp.append(sbp_te_e[m].std() / (sbp_g + 1e-9))
    nstd_cs  = np.array(nstd_cs); nstd_sbp = np.array(nstd_sbp)
    t_stat, p_tt = ttest_rel(nstd_cs, nstd_sbp)
    pct_stable   = (nstd_cs < nstd_sbp).mean() * 100

    print(f"\n  {'Variable':<12}  {'ICC':>8}  {'95% CI':>20}  Stability")
    print(f"  {'─' * 55}")
    for name, icc, lo, hi, ratio in [
            ("c_slow", icc_cs,  lo_cs,  hi_cs,  None),
            ("SBP",    icc_sbp, lo_sbp, hi_sbp, ratio_sbp),
            ("DBP",    icc_dbp, lo_dbp, hi_dbp, ratio_dbp)]:
        ratio_s = "(reference)" if ratio is None else f"{ratio:.2f}× more variable"
        print(f"  {name:<12}  {icc:>8.4f}  [{lo:.4f}, {hi:.4f}]  {ratio_s}")
    print(f"\n  pop-norm stability ratio SBP/c_slow: {ratio_sbp:.2f}× [{r_lo:.2f},{r_hi:.2f}]")
    print(f"  paired t-test: t={t_stat:.3f}, p={p_tt:.3e}  "
          f"({pct_stable:.0f}% subjects c_slow more stable)")

    records_icc = [dict(var="c_slow", icc=icc_cs,  lo=lo_cs,  hi=hi_cs),
                   dict(var="SBP",    icc=icc_sbp, lo=lo_sbp, hi=hi_sbp),
                   dict(var="DBP",    icc=icc_dbp, lo=lo_dbp, hi=hi_dbp)]
    _FIG_DATA["ablation_icc"] = dict(cs_prop=cs_prop, cs_ablA=cs_ablA,
                                     stages_te=stages_te, records_icc=records_icc)

    print("DONE ✓")
 # [F4] return masked versions only
    return dict(
        cs_prop=cs_prop,   cs_ablA=cs_ablA,
        sbp_te=sbp_te_e,   dbp_te=dbp_te_e,   subj_te=subj_te_e,
        eval_mask=eval_mask,
        vbvw_prop=vbvw_prop, vbvw_ablA=vbvw_ablA,
        vbvw_prop_ci=(vbvw_prop_lo, vbvw_prop_hi),
        vbvw_ablA_ci=(vbvw_ablA_lo, vbvw_ablA_hi),
        cd_prop=cd_prop,     cd_ablA=cd_ablA,
        cd_prop_ci=(cd_prop_lo, cd_prop_hi),
        cd_ablA_ci=(cd_ablA_lo, cd_ablA_hi),
        r_prop=r_prop,       r_ablA=r_ablA,
        is_artifact=is_artifact,
        icc_cs=icc_cs,  lo_cs=lo_cs,   hi_cs=hi_cs,
        icc_sbp=icc_sbp, lo_sbp=lo_sbp, hi_sbp=hi_sbp,
        ratio_sbp=ratio_sbp, r_lo=r_lo, r_hi=r_hi,
        p_tt=p_tt, pct_stable=pct_stable,
    )


# ══════════════════════════════════════════════════════════════════════════════
# Statistical significance (Wilcoxon)
# ══════════════════════════════════════════════════════════════════════════════

def run_exp5():
    print("\n" + "─" * 70)
    print("  Statistical Significance  [TEST set]")
    print("─" * 70)

    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    primary_m, cal_m, mask_m, _ = full_pipeline_test(model, calib_proposed)
    subjs_eval = np.array([s for s in np.unique(subj_te) if mask_m[subj_te == s].any()])
    subj_mae_m = {s: float(np.abs(cal_m[(subj_te==s)&mask_m, 0]
                                   - Ybp_te[(subj_te==s)&mask_m, 0]).mean())
                  for s in subjs_eval}
    vals_m = np.array(list(subj_mae_m.values()))
    lo_m, hi_m = bootstrap_ci(vals_m)
    print(f"  DiBP-Net SBP Subj Post: {vals_m.mean():.2f} [CI:{lo_m:.2f}–{hi_m:.2f}] mmHg")

    wilcoxon_p = None
    if mdl_transformer is not None and calib_transformer:
        r_s = _calib_get(calib_transformer, "ridge_s", "ridge_sbp")
        r_d = _calib_get(calib_transformer, "ridge_d", "ridge_dbp")
        i_s = _calib_get(calib_transformer, "iso_s")
        i_d = _calib_get(calib_transformer, "iso_d")
        raw_t    = _get_raw_ensemble_transformer(mdl_transformer, X_te, Q_te, subj_te, base_te)
        primary_t = apply_ridge_iso(_build_ridge_feats_5(raw_t, base_te), r_s, r_d, i_s, i_d)
        cal_t, mask_t = run_subject_bias_correction(primary_t, Ybp_te, subj_te)
        subj_mae_t = {s: float(np.abs(cal_t[(subj_te==s)&mask_t, 0]
                                       - Ybp_te[(subj_te==s)&mask_t, 0]).mean())
                      for s in np.unique(subj_te) if mask_t[subj_te==s].any()}
        common = np.intersect1d(list(subj_mae_m.keys()), list(subj_mae_t.keys()))
        _, wilcoxon_p = scipy_stats.wilcoxon(
            [subj_mae_m[s] for s in common], [subj_mae_t[s] for s in common],
            alternative="two-sided")
        print(f"  Wilcoxon DiBP-Net vs Transformer: p = {wilcoxon_p:.3e} "
              f"({'sig' if wilcoxon_p < 0.05 else 'n.s.'})")
    else:
        print("  [SKIP Wilcoxon] Transformer not loaded")

    print("DONE ✓")
    return dict(sbp_mean=float(vals_m.mean()), sbp_lo=lo_m,
                sbp_hi=hi_m, wilcoxon_p=wilcoxon_p)


# ══════════════════════════════════════════════════════════════════════════════
# Temporal stability
# ══════════════════════════════════════════════════════════════════════════════

def run_temporal_stability():
    print("\n" + "─" * 70)
    print("  Temporal Calibration Stability  [TEST set]")
    print("─" * 70)

    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    primary_m, cal_m, mask_m, _ = full_pipeline_test(model, calib_proposed)

    beat_pos = np.full(len(subj_te), -1, dtype=int)
    for s in np.unique(subj_te):
        for j, i in enumerate(np.where(subj_te == s)[0]):
            beat_pos[i] = j

    bps  = np.array([(subj_te == s).sum() for s in np.unique(subj_te)])
    p_95 = int(np.percentile(bps, 95))
    if   p_95 <= 6:  bins6, lbls6 = [0,3,6],        ["Beats 1–3","Beats 4–6"]
    elif p_95 <= 10: bins6, lbls6 = [0,3,6,10],     ["Beats 1–3","Beats 4–6","Beats 7–10"]
    else:            bins6, lbls6 = [0,3,6,10,p_95], ["Beats 1–3","Beats 4–6","Beats 7–10",
                                                       f"Beats 11–{p_95}"]

    rows6 = []
    print(f"\n  Beat-position stability (bins: {lbls6})")
    for lo, hi, lbl in zip(bins6[:-1], bins6[1:], lbls6):
        m = mask_m & (beat_pos >= lo) & (beat_pos < hi)
        if not m.any(): continue
        rows6.append(dict(bin_label=lbl,
                          sbp_mae=round(mae(cal_m[m], Ybp_te[m], 0), 3),
                          dbp_mae=round(mae(cal_m[m], Ybp_te[m], 1), 3),
                          n=int(m.sum())))
        print(f"  {lbl:<14} n={m.sum():>5}  SBP={rows6[-1]['sbp_mae']:.3f}  "
              f"DBP={rows6[-1]['dbp_mae']:.3f}")

    n_ref = CALIB.ref_beats
    rows_b, base_sbp_b = [], None
    print(f"\n  Temporal drift (bin_size={DRIFT_BIN_SIZE} beats, ref={n_ref})")
    for b in range(DRIFT_MAX_BINS):
        lo_b = n_ref + b * DRIFT_BIN_SIZE
        hi_b = lo_b + DRIFT_BIN_SIZE
        m_b  = mask_m & (beat_pos >= lo_b) & (beat_pos < hi_b)
        if not m_b.any(): continue
        sbp_m = mae(cal_m[m_b], Ybp_te[m_b], 0)
        dbp_m = mae(cal_m[m_b], Ybp_te[m_b], 1)
        if base_sbp_b is None: base_sbp_b = sbp_m
        rows_b.append(dict(bin=b+1, beat_range=f"beats {lo_b+1}–{hi_b}",
                           n_beats=int(m_b.sum()),
                           n_subj=int(len(np.unique(subj_te[m_b]))),
                           sbp_mae=round(sbp_m, 3), dbp_mae=round(dbp_m, 3),
                           delta_sbp=round(sbp_m - base_sbp_b, 3)))
        print(f"  Bin {b+1}  {rows_b[-1]['beat_range']:<14}  "
              f"N={m_b.sum():>5}  SBP={sbp_m:.3f}  ΔSBP={rows_b[-1]['delta_sbp']:+.3f}")

    df_b = pd.DataFrame(rows_b)
    if len(df_b) >= 2:
        slope, _, r, p, _ = scipy_stats.linregress(df_b["bin"], df_b["sbp_mae"])
        print(f"  Linear drift slope: {slope:+.4f} mmHg/bin  r={r:.3f}, p={p:.3f}")

    df_b.to_csv(os.path.join(EXP_OUT_DIR, "exp_b_drift.csv"), index=False)
    if not df_b.empty:
 # Split into 2 independent 3.5in single-panel figures.
        _FIG_DATA["exp_b_drift"] = dict(df_b=df_b.copy())

    print("DONE ✓")
    return dict(rows6=rows6, df_b=df_b)



In [ ]:
# PPG noise robustness
# ══════════════════════════════════════════════════════════════════════════════
def run_exp4():
    print("\n" + "─" * 70)
    print("  PPG Noise Robustness  [TEST set]")
    print("─" * 70)

    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    rng = np.random.default_rng(42)
    results = []
    base_sbp_pre = None

    ridge_s = _calib_get(calib_proposed, "ridge_sbp")
    ridge_d = _calib_get(calib_proposed, "ridge_dbp")
    iso_s   = _calib_get(calib_proposed, "iso_s")
    iso_d   = _calib_get(calib_proposed, "iso_d")

    for sigma in NOISE_LEVELS:
        X_noisy = (X_te if sigma == 0.0
                   else np.clip(X_te + rng.normal(0, sigma, X_te.shape).astype(np.float32),
                                0.0, 1.0))
        _build_subj_cache(model, X_noisy, Q_te, subj_te, base_te)
        feats10   = _build_ridge_feats_10(model, X_noisy, Q_te, subj_te, base_te)
        primary_n = apply_ridge_iso(feats10, ridge_s, ridge_d, iso_s, iso_d)
        cal_n, mask_n = run_subject_bias_correction(primary_n, Ybp_te, subj_te)
        m_n = eval_metrics(primary_n, cal_n, Ybp_te, subj_te, mask_n)

        if base_sbp_pre is None:
            base_sbp_pre = m_n["sbp_beat_pre"]

        row = {**m_n, "sigma": sigma,
               "sbp_delta_beat_pre": m_n["sbp_beat_pre"] - base_sbp_pre}
        results.append(row)
        print(f"  σ={sigma:.2f}  BeatGlobalCal={m_n['sbp_beat_pre']:.2f}  "
            f"BeatSubjCal={m_n['sbp_beat_post']:.2f}  "
            f"SubjSubjCal={m_n['sbp_subj_post']:.2f}  "
            f"ΔSBP={row['sbp_delta_beat_pre']:+.2f}")

    df = pd.DataFrame(results)
    df.to_csv(os.path.join(EXP_OUT_DIR, "noise_robustness.csv"), index=False)
    return df

In [ ]:
csv_file_path = os.path.join(EXP_OUT_DIR, "noise_robustness.csv")
def plot_noise_robustness_from_csv(
    csv_path: str, 
    save_dir: str, 
    filename: str = "noise_robustness.png",
    ieee_rc: dict = None, 
    fig_width: float = 3.5):
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"CSV file not found at: {csv_path}")
        return

    if ieee_rc is None:
        ieee_rc = {}
        
    os.makedirs(save_dir, exist_ok=True)

    # ── Configuration ──
    C_VAL_, C_TEST_ = "#1D4E89", "#C53030"
    
    proxy_handles = [
        Line2D([0], [0], marker="s",  color="dimgray", lw=1.0, ms=3,
               linestyle="-",  label="Beat, Global-Cal"),
        Line2D([0], [0], marker="^",  color="dimgray", lw=1.0, ms=3,
               linestyle="-",  label="Beat, Subj-Cal"),
        Line2D([0], [0], marker="D",  color="dimgray", lw=1.0, ms=3,
               linestyle="--", label="Subject, Subj-Cal"),
    ]
    
    bp_cfg = [
        ("sbp", "SBP", C_VAL_,  "#2196F3", "#00BCD4", "noise_robustness_sbp"),
        ("dbp", "DBP", C_TEST_, "#F44336", "#FF4081", "noise_robustness_dbp"),
    ]

    # ── Plotting ──
    with plt.rc_context(ieee_rc):
        fig, axes = plt.subplots(1, 2, figsize=(fig_width, 2.5))
        
        for ax, (bp, label, c0, c1, c2, stem) in zip(axes, bp_cfg):
            ax.plot(df["sigma"], df[f"{bp}_beat_pre"],  "s-", color=c1, lw=1.0, ms=3)
            ax.plot(df["sigma"], df[f"{bp}_beat_post"], "^-", color=c2, lw=1.0, ms=3)
            ax.plot(df["sigma"], df[f"{bp}_subj_post"], "D--", color=c0, lw=1.0, ms=3)
            
            ax.set_ylabel(f"{label} MAE (mmHg)", fontsize=8)
            ax.set_xlabel(r"PPG Noise $\sigma$", fontsize=8)
            ax.set_title(f"{label} Noise Robustness", fontsize=8, fontweight="bold")
            ax.tick_params(labelsize=8)

        fig.legend(
            handles=proxy_handles, loc="upper center", bbox_to_anchor=(0.5, 0.1),
            ncol=3, fontsize=8, framealpha=0.9, handlelength=1.5,
            handletextpad=0.4, columnspacing=0.8,
        )
        
        fig.tight_layout(pad=0.5, rect=[0, 0.10, 1, 1])
        
        for artist in fig.findobj(plt.Text):
            try:
                artist.set_fontsize(8)
                artist.set_fontname("Times New Roman")
            except Exception:
                pass
                
        # ── Saving ──
        p = os.path.join(save_dir, filename)
        fig.savefig(p, bbox_inches="tight", pad_inches=0.05,
                    facecolor=fig.get_facecolor())
        print(f"  ✅  saved → {p}")
        plt.close(fig)
        
    print("DONE ✓")

In [ ]:
# AAMI/BHS compliance
# ══════════════════════════════════════════════════════════════════════════════

def run_exp7(ext_dict=None):
    print("\n" + "─" * 70)
    print("  AAMI / BHS Standards  [TEST + ALL EXTERNAL sets]")
    print("─" * 70)

    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    primary_m, cal_m, mask_m, _ = full_pipeline_test(model, calib_proposed)
    eidx = np.where(mask_m)[0]
    rows = []

    for col, tag in [(0, "SBP"), (1, "DBP")]:
        rows.append(aami_bhs(primary_m[eidx, col], Ybp_te[eidx, col],
                             f"{tag} Pre  (Beat, UCI-TEST)"))
        rows.append(aami_bhs(cal_m[eidx, col],     Ybp_te[eidx, col],
                             f"{tag} Post (Beat, UCI-TEST)"))

    subjs_eval = np.array([s for s in np.unique(subj_te) if mask_m[subj_te == s].any()])
    for col, tag in [(0, "SBP"), (1, "DBP")]:
        pre_s  = np.array([primary_m[(subj_te==s)&mask_m, col].mean() for s in subjs_eval])
        post_s = np.array([cal_m    [(subj_te==s)&mask_m, col].mean() for s in subjs_eval])
        true_s = np.array([Ybp_te   [(subj_te==s)&mask_m, col].mean() for s in subjs_eval])
        rows.append(aami_bhs(pre_s,  true_s, f"{tag} Pre  (Subject, UCI-TEST)"))
        rows.append(aami_bhs(post_s, true_s, f"{tag} Post (Subject, UCI-TEST)"))

    for name, tag, d in _iter_ext_datasets(ext_dict or {}):
        primary_e, cal_e, mask_e = _get_primary_cal_mask(d)
        eidx_e = np.where(mask_e)[0]
        if len(eidx_e) == 0:
            print(f"  [{name}] n_eval=0 — insufficient reference beats per subject, skip")
            continue
        for col, tag2 in [(0, "SBP"), (1, "DBP")]:
            rows.append(aami_bhs(primary_e[eidx_e, col], d["Ybp_eval"][eidx_e, col],
                                  f"{tag2} Pre  (Beat, {name}-EXTERNAL)"))
            rows.append(aami_bhs(cal_e[eidx_e, col],     d["Ybp_eval"][eidx_e, col],
                                  f"{tag2} Post (Beat, {name}-EXTERNAL)"))

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(EXP_OUT_DIR, "aami_bhs_table.csv"), index=False)
    print("DONE ✓"); return df


# ══════════════════════════════════════════════════════════════════════════════
# PulseDB subgroup and HTN-stratified calibration
# ══════════════════════════════════════════════════════════════════════════════

def run_exp8_subgroup(pdb, exp2_result):
    print("\n" + "─" * 70)
    print("  PulseDB Subgroup Analysis  [EXTERNAL pdb_eval]")
    print("─" * 70)
    if pdb is None or exp2_result is None:
        print("  [SKIP]"); return None

    pdb_res = exp2_result.get("PulseDB") if isinstance(exp2_result, dict) else None
    if pdb_res is not None:
        primary_pdb = pdb_res["primary"]
        cal_pdb     = pdb_res["cal"]
        mask_pdb    = pdb_res["mask"]
    elif "primary_pdb" in exp2_result:
        primary_pdb = exp2_result["primary_pdb"]
        cal_pdb     = exp2_result["cal_pdb"]
        mask_pdb    = exp2_result["mask_pdb"]
    else:
        print("  [SKIP] PulseDB not present in exp2_result "
              "(dataset was skipped upstream, or an unexpected result object was passed)")
        return None
    
    Ybp_eval    = pdb["Ybp_eval"]
    age_eval    = pdb["age_eval"]
    subj_eval   = pdb["subj_eval"]
    bmi_eval    = pdb["weight_eval"] / (pdb["height_eval"] / 100.0) ** 2
    
    rows = []   # collected for the table printed at the end

    def _sg(mask, label):
        m = mask & mask_pdb
        if m.sum() < 10:
            print(f"  {label:<26} [n={m.sum()} < 10, skip]")
            return
        
        beat_g = np.abs(primary_pdb[m] - Ybp_eval[m]).mean(0)
        beat_s = np.abs(cal_pdb[m]     - Ybp_eval[m]).mean(0)
        subj_g = subj_mae(primary_pdb, Ybp_eval, subj_eval, eval_mask=m)
        subj_s = subj_mae(cal_pdb,     Ybp_eval, subj_eval, eval_mask=m)
        
        rows.append(dict(
            label=label, n=int(m.sum()),
            sbp_beat_g=beat_g[0], sbp_beat_s=beat_s[0],
            sbp_subj_g=subj_g[0], sbp_subj_s=subj_s[0],
            dbp_beat_g=beat_g[1], dbp_beat_s=beat_s[1],
            dbp_subj_g=subj_g[1], dbp_subj_s=subj_s[1],
        ))

    _sg(np.ones(len(Ybp_eval), dtype=bool), "Overall")
    for lo, hi, lbl in [(0,40,"Age <40"),(40,60,"Age 40–59"),(60,150,"Age ≥60")]:
        _sg((age_eval >= lo) & (age_eval < hi), lbl)
    stages_eval = assign_stages(Ybp_eval[:, 0], Ybp_eval[:, 1])
    for sid in [1, 2, 3, 4]:
        _sg(stages_eval == sid, f"Stage {STAGE_LABELS[sid]}")
    valid_bmi = (bmi_eval > 10) & (bmi_eval < 60)
    for lo, hi, lbl in [(0,18.5,"BMI <18.5"),(18.5,25,"BMI 18.5–24.9"),
                        (25,30,"BMI 25–29.9"),(30,99,"BMI ≥30")]:
        _sg(valid_bmi & (bmi_eval >= lo) & (bmi_eval < hi), lbl)
        
 # Full Beat + Subject, Global-Cal + Subj-Cal table (same G/S convention as
    _p = os.path.join(EXP_OUT_DIR, "exp8_pulsedb_subgroup.csv")
    pd.DataFrame(rows).to_csv(_p, index=False)
    print(f"\n  Saved \u2192 {_p}")

 # print_metric_table / Master Table -- avoids the earlier Pre/Post
 # ambiguity and now also reports subject-level MAE, not beat-level only).
    print("\n  " + "─" * 118)
    print(f"  {'Subgroup':<26} {'n':>6} "
          f"{'SBP Beat G':>10} {'SBP Beat S':>10} {'SBP Subj G':>10} {'SBP Subj S':>10} "
          f"{'DBP Beat G':>10} {'DBP Beat S':>10} {'DBP Subj G':>10} {'DBP Subj S':>10}")
    print("  " + "─" * 118)
    for r in rows:
        print(f"  {r['label']:<26} {r['n']:>6} "
              f"{r['sbp_beat_g']:>10.2f} {r['sbp_beat_s']:>10.2f} "
              f"{r['sbp_subj_g']:>10.2f} {r['sbp_subj_s']:>10.2f} "
              f"{r['dbp_beat_g']:>10.2f} {r['dbp_beat_s']:>10.2f} "
              f"{r['dbp_subj_g']:>10.2f} {r['dbp_subj_s']:>10.2f}")
    print("  " + "─" * 118)
    print("DONE ✓")
    
    return rows


def run_exp_c_htn_cal():
    print("\n" + "─" * 70)
    print("  HTN-Stage-Stratified Calibration  [TEST set]")
    print("─" * 70)

    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    _build_subj_cache(model, X_te, Q_te, subj_te, base_te)
    primary = apply_ridge_iso(
        _build_ridge_feats_10(model, X_te, Q_te, subj_te, base_te),
        _calib_get(calib_proposed, "ridge_sbp"),
        _calib_get(calib_proposed, "ridge_dbp"),
        _calib_get(calib_proposed, "iso_s"),
        _calib_get(calib_proposed, "iso_d"))

    n_ref, ols_gap, eval_min = CALIB.ref_beats, CALIB.ols_gap, CALIB.eval_min
    cal_c1 = primary.copy(); cal_c2 = primary.copy()
    eval_mask = np.zeros(len(primary), dtype=bool)
    biases_s, biases_d, subj_stage_map = {}, {}, {}

    for s in np.unique(subj_te):
        idx = np.where(subj_te == s)[0]; n = len(idx)
        if n < CALIB.min_beats: continue
        n_r     = n_ref if n >= n_ref + ols_gap + eval_min else max(1, n - ols_gap - eval_min)
        ref_idx = idx[:n_r]; eval_idx = idx[n_r + ols_gap:]
        if len(eval_idx) < eval_min: continue
        stage = int(assign_stages(np.array([Ybp_te[ref_idx, 0].mean()]),
                                  np.array([Ybp_te[ref_idx, 1].mean()]))[0])
        b_s = float(np.median(Ybp_te[ref_idx, 0] - primary[ref_idx, 0]))
        b_d = float(np.median(Ybp_te[ref_idx, 1] - primary[ref_idx, 1]))
        biases_s[s] = b_s; biases_d[s] = b_d; subj_stage_map[s] = stage
        eval_mask[eval_idx] = True
        cal_c1[eval_idx, 0] = primary[eval_idx, 0] + b_s
        cal_c1[eval_idx, 1] = primary[eval_idx, 1] + b_d

    for st in [1, 2, 3, 4]:
        s_in = [s for s, sg in subj_stage_map.items() if sg == st]
        if not s_in: continue
        med_s = float(np.median([biases_s[s] for s in s_in]))
        med_d = float(np.median([biases_d[s] for s in s_in]))
        for s in s_in:
            idx = np.where(subj_te == s)[0]; n = len(idx)
            n_r = (n_ref if n >= n_ref + ols_gap + eval_min
                   else max(1, n - ols_gap - eval_min))
            ei = idx[n_r + ols_gap:]
            if len(ei) >= eval_min:
                cal_c2[ei, 0] = primary[ei, 0] + med_s
                cal_c2[ei, 1] = primary[ei, 1] + med_d

    eidx = np.where(eval_mask)[0]; records = []
    print(f"\n  {'Stage':<12}  {'N':>5}  {'C1 SBP':>8}  {'C2 SBP':>8}  {'Δ':>8}")
    print(f"  {'─'*48}")
    for st in [1, 2, 3, 4]:
        s_s    = [s for s, sg in subj_stage_map.items() if sg == st]
        s_eidx = np.array([i for i in eidx if subj_te[i] in set(s_s)])
        if len(s_eidx) == 0: continue
        c1 = mae(cal_c1[s_eidx], Ybp_te[s_eidx], 0)
        c2 = mae(cal_c2[s_eidx], Ybp_te[s_eidx], 0)
        print(f"  {STAGE_LABELS[st]:<12}  {len(s_eidx):>5}  "
              f"{c1:>8.3f}  {c2:>8.3f}  {c2-c1:>+8.3f}")
        for method, v in [("C1", c1), ("C2", c2)]:
            records.append(dict(stage=st, stage_label=STAGE_LABELS[st],
                                method=method, n_eval=len(s_eidx), sbp_mae=round(v, 3)))
    c1_all = mae(cal_c1[eidx], Ybp_te[eidx], 0)
    c2_all = mae(cal_c2[eidx], Ybp_te[eidx], 0)
    print(f"  {'Overall':<12}  {len(eidx):>5}  {c1_all:>8.3f}  {c2_all:>8.3f}  "
          f"{c2_all-c1_all:>+8.3f}")

    diag_rows = []
    for st in [1, 2, 3, 4]:
        s_in = [s for s, sg in subj_stage_map.items() if sg == st]
        if not s_in: continue
        bs   = np.array([biases_s[s] for s in s_in])
        flag = "⚠ HIGH" if bs.std() > 5.0 else "~ Mod" if bs.std() > 2.5 else "✓ Low"
        diag_rows.append(dict(stage=st, stage_label=STAGE_LABELS[st], n_subj=len(s_in),
                               bias_mean=round(float(bs.mean()), 3),
                               bias_std=round(float(bs.std()), 3), flag=flag))
        print(f"  Bias-std {STAGE_LABELS[st]}: {bs.std():.2f} mmHg  {flag}")

    df_c, df_diag = pd.DataFrame(records), pd.DataFrame(diag_rows)
    b_range = f"{df_diag['bias_std'].min():.1f}–{df_diag['bias_std'].max():.1f}"
    print(f"\n  Within-stage bias std range: {b_range} mmHg")
    print("  → BP stage alone insufficient; individual calibration necessary")
    df_c.to_csv(os.path.join(EXP_OUT_DIR, "exp_c_htn_cal.csv"), index=False)
    df_diag.to_csv(os.path.join(EXP_OUT_DIR, "exp_c_bias_std_diagnostic.csv"), index=False)
    print("DONE ✓")
    return df_c, df_diag

In [ ]:
# UMAP domain overlap
# ══════════════════════════════════════════════════════════════════════════════

def run_exp9(ext_dict):
    import umap
    if not ext_dict:
        print("  [SKIP] no external datasets available"); return None

    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    fwd_uci = _infer_forward(model, X_te, Q_te, subj_te, base_te)
    zh_uci  = fwd_uci["z_hemo"]

    domain_names  = ["UCI-TEST"]
    domain_arrays = [zh_uci]
    sbp_arrays    = [Ybp_te[:, 0]]

    for name, tag, d in _iter_ext_datasets(ext_dict):
        fwd_ext = _infer_forward(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])
        domain_names.append(f"{name}-EXTERNAL")
        domain_arrays.append(fwd_ext["z_hemo"])
        sbp_arrays.append(d["Ybp_eval"][:, 0])

    Z_all   = np.vstack(domain_arrays)
    domain  = np.concatenate([np.full(len(a), i) for i, a in enumerate(domain_arrays)])
    sbp_all = np.concatenate(sbp_arrays)
    stage_all = np.where(sbp_all >= 140, 3, np.where(sbp_all >= 130, 2,
                np.where(sbp_all >= 120, 1, 0)))

    reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42)
    Z2      = reducer.fit_transform(Z_all)
    centroid     = Z2.mean(axis=0)
    dist         = np.linalg.norm(Z2 - centroid, axis=1)
    outlier_mask = dist > (dist.mean() + 3 * dist.std())
    n_outliers   = outlier_mask.sum()

    n_dom = len(domain_names)
    markers = ["o", "^", "s", "D", "v"][:n_dom]
    colors  = ["#2196F3", "#E64A19", "#4CAF50", "#9C27B0", "#FF9800"][:n_dom]

 # Split into 2 independent 3.5in single-panel figures (was 1x2 subplot
 # sharing a 7.16in total width) with consistent IEEE 8pt styling.
    n_str = ", ".join(f"{n}={len(a)}" for n, a in zip(domain_names, domain_arrays))
    _FIG_DATA["exp9_umap"] = dict(markers=markers, colors=colors,
        domain_names=domain_names, domain=domain, outlier_mask=outlier_mask,
        Z2=Z2, n_outliers=n_outliers, stage_all=stage_all)

    print("\n" + "─" * 70)
    print("  UMAP Domain Overlap  [UCI vs ALL EXTERNAL]")
    print("─" * 70)
    c_uci = Z2[domain == 0].mean(axis=0)
    records = []
    for d_idx in range(1, n_dom):
        c_ext = Z2[domain == d_idx].mean(axis=0)
        cent_dist = float(np.linalg.norm(c_uci - c_ext))
        intra     = float((np.linalg.norm(Z2[domain==0]-c_uci, axis=1).mean() +
                           np.linalg.norm(Z2[domain==d_idx]-c_ext, axis=1).mean()) / 2)
        overlap = cent_dist / max(intra, 1e-6)
        print(f"  [{domain_names[d_idx]}] Centroid distance: {cent_dist:.3f}  "
              f"Intra-domain spread: {intra:.3f}  Overlap ratio: {overlap:.3f}")
        records.append(dict(dataset=domain_names[d_idx], centroid_dist=round(cent_dist, 4),
                            intra_spread=round(intra, 4), overlap_ratio=round(overlap, 4)))
    print(f"  Outlier cluster: n={n_outliers} "
          f"({'present — investigate' if n_outliers > 0 else 'none'})")
    pd.DataFrame(records).to_csv(os.path.join(EXP_OUT_DIR, "exp_9_umap_overlap.csv"), index=False)
    print("DONE ✓")
    return pd.DataFrame(records)


# ══════════════════════════════════════════════════════════════════════════════
# Partial correlation and age sensitivity
# Fixes: [F5] use masked arrays from ablation_result
# [F6] compute ai_te from full X_te then mask
# ══════════════════════════════════════════════════════════════════════════════

def run_partial_corr_and_age(ablation_result, pdb=None):
    print("\n" + "─" * 70)
    print("  Partial Correlation & Age")
    print("─" * 70)

 # [F5] pull masked arrays — do NOT call get_test_arrays() for indexing
    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    cs_prop   = ablation_result["cs_prop"]
    subj_te_e = ablation_result["subj_te"]
    sbp_te_e  = ablation_result["sbp_te"]
    dbp_te_e  = ablation_result["dbp_te"]
    eval_mask = ablation_result["eval_mask"]

    subjs_u  = np.unique(subj_te_e)
    mean_cs  = np.array([cs_prop[subj_te_e == s].mean()  for s in subjs_u])
    mean_sbp = np.array([sbp_te_e[subj_te_e == s].mean() for s in subjs_u])
    mean_dbp = np.array([dbp_te_e[subj_te_e == s].mean() for s in subjs_u])
    mean_pp  = mean_sbp - mean_dbp
    stage_s  = assign_stages(mean_sbp, mean_dbp)
    htn_bin  = (stage_s >= 3).astype(int)

    # ── partial correlation ───────────────────────────────────────────────────────────────
    r_raw,                _ = pearsonr(mean_cs, mean_sbp)
    r_ctrl_dbp,           _ = partial_corr(mean_cs, mean_sbp, [mean_dbp])
    r_ctrl_pp,            _ = partial_corr(mean_cs, mean_sbp, [mean_pp])
    r_ctrl_both, p_ctrl_both = partial_corr(mean_cs, mean_sbp, [mean_dbp, mean_pp])

    scl = StandardScaler()
    cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    auc_sbp_only = cross_val_score(LogisticRegression(max_iter=1000),
                                   scl.fit_transform(mean_sbp.reshape(-1, 1)),
                                   htn_bin, cv=cv5, scoring="roc_auc").mean()
    auc_sbp_cs   = cross_val_score(LogisticRegression(max_iter=1000),
                                   scl.fit_transform(np.column_stack([mean_sbp, mean_cs])),
                                   htn_bin, cv=cv5, scoring="roc_auc").mean()

    print(f"\n  Partial correlation analysis — UCI TEST (N={len(subjs_u)} subjects)")
    for label, r in [("r(c_slow,SBP) raw",    r_raw),
                     ("r(c_slow,SBP)|DBP",    r_ctrl_dbp),
                     ("r(c_slow,SBP)|PP",     r_ctrl_pp),
                     ("r(c_slow,SBP)|DBP+PP", r_ctrl_both)]:
        print(f"  {label:<35} {r:>+.4f}")
    print(f"  AUC: SBP={auc_sbp_only:.3f}  SBP+c_slow={auc_sbp_cs:.3f}  "
          f"Δ={auc_sbp_cs - auc_sbp_only:+.3f}")

 # TEST-B
    lr_b     = LinearRegression().fit(np.column_stack([mean_sbp, mean_dbp]), mean_cs)
    cs_resid = mean_cs - lr_b.predict(np.column_stack([mean_sbp, mean_dbp]))
    auc_resid  = cross_val_score(LogisticRegression(max_iter=1000),
                                  scl.fit_transform(cs_resid.reshape(-1, 1)),
                                  htn_bin, cv=cv5, scoring="roc_auc").mean()
    auc_both_b = cross_val_score(LogisticRegression(max_iter=1000),
                                  scl.fit_transform(np.column_stack([mean_sbp, cs_resid])),
                                  htn_bin, cv=cv5, scoring="roc_auc").mean()
    print(f"  Residual AUC: cslow_resid={auc_resid:.3f}  "
          f"SBP+resid={auc_both_b:.3f}  Δ={auc_both_b - auc_sbp_only:+.3f}")

    # ── TEST-D ───────────────────────────────────────────────────────────────
    def _aug_index(ppg_batch):
        ais = []
        for ppg in ppg_batch:
            peaks, _ = find_peaks(ppg, distance=15, prominence=0.05)
            ais.append(float(ppg[peaks[1]] / (ppg[peaks[0]] + 1e-6))
                       if len(peaks) >= 2 else np.nan)
        return np.array(ais)

 # [F6] compute on full X_te then mask to match subj_te_e
    ai_te   = _aug_index(X_te[:, 0, :])
    ai_te_e = ai_te[eval_mask]
    mean_ai = np.array([
        ai_te_e[(subj_te_e == s) & ~np.isnan(ai_te_e)].mean()
        if ((subj_te_e == s) & ~np.isnan(ai_te_e)).any() else np.nan
        for s in subjs_u])
    ok = ~np.isnan(mean_ai)
    r_cs_ai,  _ = pearsonr(mean_cs[ok],  mean_ai[ok])
    r_sbp_ai, _ = pearsonr(mean_sbp[ok], mean_ai[ok])
    print(f"  Augmentation-index correlation: r(c_slow,AugIdx)={r_cs_ai:.3f}  r(SBP,AugIdx)={r_sbp_ai:.3f}")

    # ── age sensitivity ────────────────────────────────────────────────────────────────
    records_e = []
    if pdb is not None:
        fwd_pdb = _infer_forward(model, pdb["X_eval"], pdb["Q_eval"],
                                 pdb["subj_eval"], pdb["base_eval"])
        cs_pdb  = fwd_pdb["c_slow"]
        sbp_pdb = pdb["Ybp_eval"][:, 0]
        dbp_pdb = pdb["Ybp_eval"][:, 1]
        pp_pdb  = sbp_pdb - dbp_pdb
        age_pdb = pdb["age_eval"]
        n_pdb   = len(cs_pdb)

        def _add(label, r, p, cov):
            stars = ("***" if p < 0.001 else "**" if p < 0.01
                     else "*" if p < 0.05 else "ns")
            print(f"  {label:<50} r={r:>+.4f} {stars}")
            records_e.append(dict(label=label, covariates=cov,
                                  partial_r=round(r, 4), p_value=round(p, 6)))

        r0a, p0a = pearsonr(cs_pdb, age_pdb)
        r0s, p0s = pearsonr(cs_pdb, sbp_pdb)
        r0d, p0d = pearsonr(cs_pdb, dbp_pdb)
        _add("r(c_slow, age) zero-order",         r0a, p0a, "none")
        _add("r(c_slow, SBP) zero-order",         r0s, p0s, "none")
        _add("r(c_slow, DBP) zero-order",         r0d, p0d, "none")
        r1, p1 = partial_corr(cs_pdb, age_pdb, [sbp_pdb, dbp_pdb, pp_pdb])
        r2, p2 = partial_corr(cs_pdb, sbp_pdb, [age_pdb])
        r3, p3 = partial_corr(cs_pdb, dbp_pdb, [age_pdb])
        _add("partial r(c_slow, age|SBP,DBP,PP)", r1, p1, "SBP,DBP,PP")
        _add("partial r(c_slow, SBP|age)",        r2, p2, "age")
        _add("partial r(c_slow, DBP|age)",        r3, p3, "age")

 # TEST-C
        subjs_p = np.unique(pdb["subj_eval"])
        mc_p    = np.array([cs_pdb[pdb["subj_eval"] == s].mean()  for s in subjs_p])
        ms_p    = np.array([sbp_pdb[pdb["subj_eval"] == s].mean() for s in subjs_p])
        ma_p    = np.array([age_pdb[pdb["subj_eval"] == s].mean() for s in subjs_p])
        valid   = ~np.isnan(ma_p)
        r_cs_age,  _ = pearsonr(mc_p[valid], ma_p[valid])
        r_sbp_age, _ = pearsonr(ms_p[valid], ma_p[valid])
        print(f"\n  Age correlation: r(c_slow,age)={r_cs_age:.3f}  r(SBP,age)={r_sbp_age:.3f}")
        print(f"  → c_slow "
              f"{'tracks age MORE' if abs(r_cs_age) > abs(r_sbp_age) else 'tracks age LESS'}"
              f" than SBP")

 # Forest plot (single panel; width standardized to 3.5in IEEE column
 # width, height still scales with the number of records)
        _FIG_DATA["exp_e_forest"] = dict(records_e=records_e, n_pdb=n_pdb)

        pd.DataFrame(records_e).to_csv(
            os.path.join(EXP_OUT_DIR, "exp_e_partial_corr.csv"), index=False)

    print("DONE ✓")
    return dict(r_ctrl_both=r_ctrl_both, p_ctrl_both=p_ctrl_both,
                auc_sbp_only=auc_sbp_only, auc_sbp_cs=auc_sbp_cs,
                auc_resid=auc_resid, r_cs_ai=r_cs_ai, r_sbp_ai=r_sbp_ai,
                records_e=records_e)

In [ ]:
# Bootstrap confidence intervals
# ══════════════════════════════════════════════════════════════════════════════

def run_exp_d_bootstrap(ablation_result, pdb=None):
    print("\n" + "─" * 70)
    print(f"  Bootstrap 95% CI  (n_boot={N_BOOT})  [TEST + EXTERNAL]")
    print("─" * 70)
 
    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    primary_m, cal_m, mask_m, _ = full_pipeline_test(model, calib_proposed)
 
 # pull masked arrays — cs_prop is already eval-masked (from run_ablation_and_icc)
    cs_prop   = ablation_result["cs_prop"]
    subj_te_e = ablation_result["subj_te"]   # eval-masked subject IDs
    rng       = np.random.default_rng(42)
 
    records = []
    def _add(name, point, lo, hi, unit=""):
        records.append(dict(metric=name, point=round(point, 4),
                            ci_lo=round(lo, 4), ci_hi=round(hi, 4), unit=unit))
        print(f"  {name:<52} {point:>8.4f}  [{lo:.4f}, {hi:.4f}] {unit}")
 
    eidx = np.where(mask_m)[0]
 
    # ── Beat-level MAE ────────────────────────────────────────────────────
    for col, tag in [(0, "SBP"), (1, "DBP")]:
        err = np.abs(cal_m[eidx, col] - Ybp_te[eidx, col])
        lo, hi = bootstrap_ci(err, np.mean)
        _add(f"{tag} Beat Post MAE", err.mean(), lo, hi, "mmHg")
 
    # ── Subject-level MAE ─────────────────────────────────────────────────
    subjs_eval = np.array([s for s in np.unique(subj_te) if mask_m[subj_te == s].any()])
    print(f"\n  Subject-level (N_eval_subj={len(subjs_eval)})")
    for col, tag in [(0, "SBP"), (1, "DBP")]:
        post_s = np.array([cal_m [(subj_te == s) & mask_m, col].mean() for s in subjs_eval])
        true_s = np.array([Ybp_te[(subj_te == s) & mask_m, col].mean() for s in subjs_eval])
        err_s  = np.abs(post_s - true_s)
        lo_mae, hi_mae = bootstrap_ci(err_s, np.mean)
        _add(f"{tag} Subject Post MAE", err_s.mean(), lo_mae, hi_mae, "mmHg")
        err_signed = post_s - true_s
        sd_pt  = float(err_signed.std(ddof=1))
        me_pt  = float(err_signed.mean())
        boot_sd = [float(rng.choice(err_signed, len(err_signed), replace=True).std(ddof=1))
                   for _ in range(N_BOOT)]
        _add(f"{tag} Subject Post SD (AAMI)", sd_pt,
             float(np.percentile(boot_sd, 2.5)), float(np.percentile(boot_sd, 97.5)), "mmHg")
        aami_ok = sd_pt <= 8.0 and abs(me_pt) <= 5.0
        print(f"  → {tag} ME={me_pt:+.4f}  SD={sd_pt:.4f}  "
              f"AAMI: {'✅ PASS' if aami_ok else '⚠ FAIL'}")
 
    # ── ICC — BUG #3 already fixed by patched bootstrap_icc_ci() ──────────────
 # Uses bootstrap_icc_ci() → correct block-bootstrap, wider CI
    icc_pt, icc_lo, icc_hi = bootstrap_icc_ci(cs_prop, subj_te_e)
    _add("ICC(c_slow)", icc_pt, icc_lo, icc_hi)
 
    # ── Cohen's d + Vb/Vw ────────────────────────────────────────────────
    subjs_u = np.unique(subj_te_e)
    s_cs  = np.array([cs_prop[subj_te_e == s].mean() for s in subjs_u])
 
    s_sbp = np.array([Ybp_te[(subj_te == s) & mask_m, 0].mean() for s in subjs_u])  # BUG #5 FIX
    s_dbp = np.array([Ybp_te[(subj_te == s) & mask_m, 1].mean() for s in subjs_u])  # BUG #5 FIX
    s_st  = assign_stages(s_sbp, s_dbp)
    n_idx = np.where(s_st == 1)[0]
    h_idx = np.where(s_st == 4)[0]
 
    d_pt, d_lo, d_hi = bootstrap_cohens_d_ci(
        s_cs[h_idx], s_cs[n_idx], n_boot=N_BOOT, seed=42)
    _add("Cohen's d (HTN-2 vs Normal, c_slow)", d_pt, d_lo, d_hi)
 
    vbvw_pt, vbvw_lo, vbvw_hi = bootstrap_vbvw_ci(cs_prop, subj_te_e, n_boot=N_BOOT, seed=42)
    _add("Vb/Vw (c_slow)", vbvw_pt, vbvw_lo, vbvw_hi)
 
    # ── PulseDB external ──────────────────────────────────────────────────
    if pdb is not None:
        fwd_pdb = _infer_forward(model, pdb["X_eval"], pdb["Q_eval"],
                                 pdb["subj_eval"], pdb["base_eval"])
        cs_pdb  = fwd_pdb["c_slow"]
        sbp_pdb = pdb["Ybp_eval"][:, 0]
        age_pdb = pdb["age_eval"]
        n_pdb   = len(cs_pdb)
        r_age, _ = pearsonr(cs_pdb, age_pdb)
        r_sbp, _ = pearsonr(sbp_pdb, age_pdb)
        lo_a, hi_a = fisher_ci(r_age, n_pdb)
        lo_s, hi_s = fisher_ci(r_sbp, n_pdb)
        _add("r(c_slow, age) PulseDB-EXTERNAL", r_age, lo_a, hi_a)
        _add("r(SBP, age) PulseDB-EXTERNAL",    r_sbp, lo_s, hi_s)
 
    df_d = pd.DataFrame(records)
    df_d.to_csv(os.path.join(EXP_OUT_DIR, "exp_d_bootstrap_ci.csv"), index=False)
    print("DONE ✓")
    return df_d


# ══════════════════════════════════════════════════════════════════════════════
# Intra-stage and cross-dataset comparison
# ══════════════════════════════════════════════════════════════════════════════

def run_exp_f_intra_stage(pdb):
    print("\n" + "─" * 70)
    print("  Intra-stage Age Phenotype  [EXTERNAL pdb_eval]")
    print("─" * 70)
    if pdb is None:
        print("  [SKIP]"); return None

    fwd_pdb = _infer_forward(model, pdb["X_eval"], pdb["Q_eval"],
                             pdb["subj_eval"], pdb["base_eval"])
    cs_pdb = fwd_pdb["c_slow"]
    age_e  = pdb["age_eval"]
    Ybp_e  = pdb["Ybp_eval"]
    subj_e = pdb["subj_eval"]
    subjs  = np.unique(subj_e)
    s_cs   = np.array([cs_pdb[subj_e == s].mean() for s in subjs])
    s_age  = np.array([age_e[subj_e  == s].mean() for s in subjs])
    s_sbp  = np.array([Ybp_e[subj_e  == s, 0].mean() for s in subjs])
    s_dbp  = np.array([Ybp_e[subj_e  == s, 1].mean() for s in subjs])
    s_st   = assign_stages(s_sbp, s_dbp)

    records, power_rows = [], []
    print(f"  {'Stage':<12}  {'Split':>5}  {'n_y':>5}  {'n_o':>5}  "
          f"{'d':>7}  {'MW-p':>12}  {'power':>8}")
    print(f"  {'─'*62}")
    for st in [1, 2, 3, 4]:
        m = s_st == st
        if m.sum() < 10: continue
        split = AGE_SPLIT_YEAR
        young = s_cs[m & (s_age < split)]; old = s_cs[m & (s_age >= split)]
        if len(young) < MIN_SPLIT_N or len(old) < MIN_SPLIT_N:
            split = float(np.median(s_age[m]))
            young = s_cs[m & (s_age < split)]; old = s_cs[m & (s_age >= split)]
        if len(young) < MIN_SPLIT_N or len(old) < MIN_SPLIT_N:
            print(f"  {STAGE_LABELS[st]:<12}  [SKIP]"); continue
        d    = cohens_d(old, young)
        _, p = mannwhitneyu(old, young, alternative="two-sided")
        pwr  = power_twosample(len(young), len(old), d)
        stars = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns"
        print(f"  {STAGE_LABELS[st]:<12}  {split:>5.0f}  {len(young):>5}  {len(old):>5}  "
              f"{d:>7.3f}  {p:>12.4e} {stars}  {pwr:>8.3f}")
        for grp, arr in [("Young", young), ("Old", old)]:
            records.append(dict(stage=st, stage_label=STAGE_LABELS[st], age_group=grp,
                                n=len(arr), split_threshold=split,
                                cs_mean=round(float(arr.mean()), 4),
                                cs_std=round(float(arr.std()), 4),
                                cohens_d=round(d, 4), mw_p=round(p, 6)))
        if pwr < 0.80:
            print(f"  → ⚠ Underpowered (power={pwr:.2f}) — interpret cautiously")
        power_rows.append(dict(stage=STAGE_LABELS[st], n_young=len(young), n_old=len(old),
                               split=split, cohens_d=round(d, 4),
                               mw_p=round(p, 6), power=round(pwr, 3)))

    df_f     = pd.DataFrame(records)
    df_power = pd.DataFrame(power_rows)
    df_f.to_csv(os.path.join(EXP_OUT_DIR, "exp_f_intra_stage.csv"), index=False)
    df_power.to_csv(os.path.join(EXP_OUT_DIR, "exp_f_power.csv"), index=False)
    print("DONE ✓"); return df_f, df_power


def run_exp_h_cross_dataset(ablation_result, ext_dict=None):
    print("\n" + "─" * 70)
    print("  Cross-Dataset Phenotype Consistency  [TEST + ALL EXTERNAL]")
    print("─" * 70)
    if not ext_dict:
        print("  [SKIP]"); return None

    # ── [FIX] Pull already-masked arrays from ablation_result ──────────────
    cs_uci  = ablation_result["cs_prop"]     # shape: (n_eval_beats,)
    subj_te = ablation_result["subj_te"]     # shape: (n_eval_beats,) — already masked
    sbp_uci = ablation_result["sbp_te"]      # shape: (n_eval_beats,)
    # ───────────────────────────────────────────────────────────────────────

    subjs_u = np.unique(subj_te)
    mc_u    = np.array([cs_uci[subj_te == s].mean()  for s in subjs_u])
    ms_u    = np.array([sbp_uci[subj_te == s].mean() for s in subjs_u])
    st_u    = assign_stages(ms_u, np.array([ablation_result["dbp_te"][subj_te == s].mean()
                                             for s in subjs_u]))

    records = []
    results = {}

    for name, tag, d in _iter_ext_datasets(ext_dict):
        fwd_ext = _infer_forward(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])
        cs_ext  = fwd_ext["c_slow"]
        sbp_ext = d["Ybp_eval"][:, 0]; dbp_ext = d["Ybp_eval"][:, 1]

        subjs_e = np.unique(d["subj_eval"])
        mc_e    = np.array([cs_ext[d["subj_eval"] == s].mean()  for s in subjs_e])
        ms_e    = np.array([sbp_ext[d["subj_eval"] == s].mean() for s in subjs_e])
        st_e    = assign_stages(ms_e, np.array([dbp_ext[d["subj_eval"] == s].mean()
                                                 for s in subjs_e]))

        r_u, _ = pearsonr(mc_u, ms_u); r_e, _ = pearsonr(mc_e, ms_e)
        d_u    = cohens_d(mc_u[st_u == 4], mc_u[st_u == 1])
        d_e    = (cohens_d(mc_e[st_e == 4], mc_e[st_e == 1])
                  if (st_e == 4).sum() >= 5 and (st_e == 1).sum() >= 5 else np.nan)

        print(f"\n  [{name}]")
        print(f"  r(c_slow,SBP): UCI-TEST={r_u:+.4f}  {name}={r_e:+.4f}  "
              f"Consistent: {'✓' if np.sign(r_u)==np.sign(r_e) else '✗ Sign reversal'}")
        print(f"  Cohen d: UCI={d_u:.4f}  {name}={d_e:.4f}")

        beat_r_u, _ = pearsonr(cs_uci, sbp_uci)
        beat_r_e, _ = pearsonr(cs_ext, sbp_ext)
        mean_diff   = abs(cs_uci.mean() - cs_ext.mean()) / cs_uci.std()
        h1 = "POSSIBLE"  if mean_diff > 0.5 else "UNLIKELY"
        h2 = "CONFIRMED" if np.sign(beat_r_e) != np.sign(r_e) else "UNLIKELY"
        print(f"  H1 (distribution shift): {h1}  (mean_diff/std={mean_diff:.2f})")
        print(f"  H2 (aggregation reversal): {h2}  "
              f"(beat-r: UCI={beat_r_u:+.4f} {name}={beat_r_e:+.4f})")

        records.append(dict(dataset=name, tag=tag, r_uci=round(r_u, 4), r_ext=round(r_e, 4),
                            d_uci=round(d_u, 4), d_ext=round(d_e, 4) if not np.isnan(d_e) else None,
                            beat_r_uci=round(beat_r_u, 4), beat_r_ext=round(beat_r_e, 4),
                            mean_diff_std=round(mean_diff, 4), h1=h1, h2=h2,
                            sign_reversal=bool(np.sign(r_u) != np.sign(r_e))))
        results[tag] = dict(r_u=r_u, r_p=r_e, d_u=d_u, d_p=d_e, h1=h1, h2=h2)

    df_h = pd.DataFrame(records)
    df_h.to_csv(os.path.join(EXP_OUT_DIR, "exp_h_cross_dataset_summary.csv"), index=False)
    n_reversals = int(df_h["sign_reversal"].sum())
    print(f"\n  → {n_reversals}/{len(df_h)} external datasets show sign reversal vs UCI.")
    if n_reversals > 0:
        print("  → RECOMMENDATION: exclude affected datasets' r(c_slow,SBP) claims from Main Paper "
              "until reversal explained (see the cross-dataset diagnostic cell for PulseDB's own resolution).")
    print("DONE ✓")
    return df_h


### Summary Table

An optional all-in-one runner; the cells below run each experiment individually instead.

In [ ]:
# Split-effect control
# ══════════════════════════════════════════════════════════════════════════════

def build_paper_summary(df_d, records_e, ablation_result, partial_result):
    print("\n" + "─" * 70)
    print("  SEC-F: Unified Paper-Ready Summary Table")
    print("─" * 70)

    def _get(name, col="point"):
        rows = df_d[df_d["metric"].str.contains(name, regex=False)]
        return float(rows.iloc[0][col]) if not rows.empty else np.nan

    def _ci(name):
        rows = df_d[df_d["metric"].str.contains(name, regex=False)]
        return (f"[{rows.iloc[0]['ci_lo']:.4f}, {rows.iloc[0]['ci_hi']:.4f}]"
                if not rows.empty else "—")

    ar = ablation_result
    _age_rec_ps   = next((r for r in records_e
                          if "SBP,DBP,PP" in r.get("covariates", "")), None)
    partial_r_age = float(_age_rec_ps["partial_r"]) if _age_rec_ps else np.nan
    _p_age_ps     = float(_age_rec_ps["p_value"]) if _age_rec_ps else float("nan")
    _age_claim    = ("Age sensitivity"
                     if (not np.isnan(_p_age_ps)) and (_p_age_ps < 0.05)
                     else "Weak age dependence")

    claims = [
        dict(Claim="Temporal stability", Metric="ICC(c_slow)",
             Value=f"{ar['icc_cs']:.4f}", CI=_ci("ICC"),
             Comparator=f"ICC(SBP)={ar['icc_sbp']:.4f}", EXP="EXP-G/EXP-D"),
        dict(Claim="Temporal stability", Metric="Stability ratio",
             Value=f"{ar['ratio_sbp']:.2f}×", CI=f"[{ar['r_lo']:.2f},{ar['r_hi']:.2f}]",
             Comparator=f"{ar['pct_stable']:.0f}% subj, p={ar['p_tt']:.2e}", EXP="EXP-G"),
        dict(Claim="HTN separability", Metric="Cohen's d (HTN-2 vs Normal)",
             Value=f"{_get('Cohen'):.4f}", CI=_ci("Cohen"), Comparator="—", EXP="EXP-D"),
        dict(Claim="HTN separability", Metric="Vb/Vw ratio",
             Value=f"{_get('Vb/Vw'):.2f}", CI=_ci("Vb/Vw"), Comparator="—", EXP="EXP-D"),
        dict(Claim=_age_claim, Metric="partial r(c_slow,age|SBP,DBP,PP)",
             Value=f"{partial_r_age:.4f}", CI="Forest plot (EXP-E)",
             Comparator=f"p={_p_age_ps:.3f}", EXP="EXP-E"),
        dict(Claim="Calibration perf.", Metric="SBP Beat Post MAE",
             Value=f"{_get('SBP Beat'):.2f} mmHg", CI=_ci("SBP Beat"),
             Comparator="—", EXP="EXP-D"),
        dict(Claim="Calibration perf.", Metric="DBP Beat Post MAE",
             Value=f"{_get('DBP Beat'):.2f} mmHg", CI=_ci("DBP Beat"),
             Comparator="—", EXP="EXP-D"),
    ]

    df_paper = pd.DataFrame(claims)
    print(f"\n  {'Claim':<22}  {'Metric':<40}  {'Value':>12}  {'CI':>24}  EXP")
    print(f"  {'─'*102}")
    for row in claims:
        print(f"  {row['Claim']:<22}  {row['Metric']:<40}  "
              f"{row['Value']:>12}  {row['CI']:>24}  {row['EXP']}")
    df_paper.to_csv(os.path.join(EXP_OUT_DIR, "paper_summary_table.csv"), index=False)
    print(f"\n  Saved → paper_summary_table.csv")
    return df_paper

# Model Loading and Training

Single entry point: loads from checkpoints if present, trains if missing, and produces the model registry.

In [ ]:
_loaded = initialize_all_models()
globals().update(_loaded)
MODEL_REGISTRY = _loaded["MODEL_REGISTRY"]
HIST_REGISTRY  = _loaded["HIST_REGISTRY"]

# Export and Validation

In [ ]:
N_BOOT = 1000
_cell3_results = run_cell3_evaluation(N_BOOT)
master_df, all_master_rows, stage_results = (
    _cell3_results["master_df"], _cell3_results["all_master_rows"], _cell3_results["stage_results"]
)

# Figures

### Figure Generation

In [ ]:
print("[4-F] build_figure_0 / print_stage_table / print_dataset_stats defined (unified staging)")
build_figure_0()
print_stage_table()
print_dataset_stats()

In [ ]:
print("[4-G] build_figure_1 defined — run: build_figure_1(mdl=model)")
build_figure_1()

In [ ]:
# [run]
print("[4-H] build_figure_2 defined — run: build_figure_2()")
build_figure_2()

In [ ]:
# [run]
print("[4-I] build_figure_3 defined — run: build_figure_3(tag='v192')")
build_figure_3(tag="v192")

In [ ]:
# [run]
print("[4-J] build_figure_4 defined — run: build_figure_4()")
build_figure_4()

In [ ]:
# [run]
print("[4-K] build_figure_5 defined — run: build_figure_5(tag='v192')")
build_figure_5(tag="v192")

In [ ]:
# [run]
print("[4-L] build_figure_6 defined — run: build_figure_6(tag='v192')")
build_figure_6(tag="v192")

In [ ]:
# [run]
print("[4-M] build_figure_7 defined — run: build_figure_7()")
build_figure_7()

In [ ]:
# [run]
print("[4-N] build_figure_8 defined (single vertical 4-panel figure) - run: build_figure_8()")
build_figure_8()

# Additional Experiments

In [ ]:
calib_proposed = calib_v192   # reuse v192 ModelCalib loaded in the model-loading part

print(f"    calib_proposed  : {'OK' if calib_proposed else 'MISSING'}")
print(f"    proposed        : {'OK' if model          else 'MISSING'}")
print(f"    ablA            : {'OK' if mdl_ablA       else 'MISSING'}")
print(f"    transformer     : {'OK' if mdl_transformer else 'MISSING'}")
print("\n" + "=" * 70)
print("  AUC SIGNIFICANCE TEST — DiBP-Net vs Transformer")
print("=" * 70)

# ─────────────────────────────────────────────────────────────────────────────
# DeLong test (Sun & Xu, 2014 — numerically stable)
# ─────────────────────────────────────────────────────────────────────────────
def delong_test(y_true, score_a, score_b):
    """
    Paired DeLong test for two correlated AUCs.
    Returns: auc_a, auc_b, z, p_two_sided
    """
    y = y_true.astype(int)
    m = y.sum()
    n = len(y) - m

    pos_a = score_a[y == 1]; neg_a = score_a[y == 0]
    pos_b = score_b[y == 1]; neg_b = score_b[y == 0]

    V10_a = np.array([(pos_a[i] > neg_a).mean() + 0.5 * (pos_a[i] == neg_a).mean()
                      for i in range(m)])
    V01_a = np.array([(neg_a[j] < pos_a).mean() + 0.5 * (neg_a[j] == pos_a).mean()
                      for j in range(n)])
    V10_b = np.array([(pos_b[i] > neg_b).mean() + 0.5 * (pos_b[i] == neg_b).mean()
                      for i in range(m)])
    V01_b = np.array([(neg_b[j] < pos_b).mean() + 0.5 * (neg_b[j] == pos_b).mean()
                      for j in range(n)])

    auc_a = V10_a.mean()
    auc_b = V10_b.mean()

    var_a  = np.var(V10_a, ddof=1) / m + np.var(V01_a, ddof=1) / n
    var_b  = np.var(V10_b, ddof=1) / m + np.var(V01_b, ddof=1) / n
    cov_ab = (np.cov(V10_a, V10_b, ddof=1)[0, 1] / m +
              np.cov(V01_a, V01_b, ddof=1)[0, 1] / n)

    var_diff = max(var_a + var_b - 2 * cov_ab, 1e-12)
    z = (auc_a - auc_b) / np.sqrt(var_diff)
    p = 2 * (1 - scipy_norm.cdf(abs(z)))
    return auc_a, auc_b, z, p

def _get_htn_logits(mdl, X, Q, subj_arr, base_arr, batch=BATCH):
    """Return (htn_logit, aux_logit) as numpy arrays."""
    mdl.eval()
    htn_out, aux_out = [], []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            sl  = slice(i, min(i + batch, len(X)))
            q_c = torch.tensor(Q[sl, ARCH.n_beats // 2]).to(DEVICE)
            if q_c.dim() == 3: q_c = q_c.squeeze(1)
            raw = mdl(
                torch.tensor(X[sl]).to(DEVICE),
                torch.tensor(subj_arr[sl], dtype=torch.long).to(DEVICE),
                torch.tensor(base_arr[sl]).to(DEVICE),
                q_c, 0.0,
            )
            htn_out.append(raw[6].cpu().numpy())   # index 6 = htn_logit
            aux_out.append(raw[13].cpu().numpy())  # index 13 = aux_logit
    return np.concatenate(htn_out), np.concatenate(aux_out)

# ─────────────────────────────────────────────────────────────────────────────
# 1. Get DiBP-Net scores
# ─────────────────────────────────────────────────────────────────────────────
print("\n  Computing DiBP-Net classification scores …")
if 'Ybp_te' not in dir() or Ybp_te is None:
    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    print("  [auto-loaded test arrays]")

htn_te = ((Ybp_te[:, 0] >= HTN_SBP) | (Ybp_te[:, 1] >= HTN_DBP)).astype(np.float32)

import __main__ as _m

_HTN_MAIN_W = getattr(_m.LW, "htn_main_weight", 0.60)
_HTN_AUX_W  = getattr(_m.LW, "htn_aux_weight",  0.40)

_tmp_calib = _m.ModelCalib(
    ridge_sbp = _calib_get(calib_proposed, "ridge_sbp"),
    ridge_dbp = _calib_get(calib_proposed, "ridge_dbp"),
    iso_s     = _calib_get(calib_proposed, "iso_s"),
    iso_d     = _calib_get(calib_proposed, "iso_d"),
)

with _m._calib_ctx(_tmp_calib):
    _build_subj_cache(model, X_te, Q_te, subj_te, base_te)
    htn_logit, aux_logit = _get_htn_logits(model, X_te, Q_te, subj_te, base_te)

sc_mcnhm = _HTN_MAIN_W * htn_logit.squeeze() + _HTN_AUX_W * aux_logit.squeeze()
yt = htn_te

_, eval_mask = apply_subject_calibration(np.zeros((len(subj_te), 2)), Ybp_te, subj_te)
eidx = np.where(eval_mask)[0]
yt, sc_mcnhm = yt[eidx], sc_mcnhm[eidx]

auc_mcnhm = roc_auc_score(yt, sc_mcnhm)
print(f"  DiBP-Net AUC (logit combo, eval_window n={len(eidx)}/{len(htn_te)}): {auc_mcnhm:.4f}  "
      f"(HTN pos: {int(yt.sum())}/{len(yt)} beats)")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Get Transformer scores
# ─────────────────────────────────────────────────────────────────────────────
print("\n  Computing Transformer classification scores …")

if mdl_transformer is None or not calib_transformer:
    print("  ⚠️  mdl_transformer not found — cannot run AUC comparison.")
    print("  → Paper claim p=8.0×10⁻³ cannot be verified.")
    print("  → Recommend: remove Wilcoxon p from paper text.")
    auc_trans = None
else:
    r_s = _calib_get(calib_transformer, "ridge_sbp", "ridge_s")
    r_d = _calib_get(calib_transformer, "ridge_dbp", "ridge_d")
    i_s = _calib_get(calib_transformer, "iso_s")
    i_d = _calib_get(calib_transformer, "iso_d")

    t_raw     = _get_raw_ensemble_transformer(mdl_transformer, X_te, Q_te, subj_te, base_te)
    t_feats   = _build_ridge_feats_5(t_raw, base_te)
    t_primary = apply_ridge_iso(t_feats, r_s, r_d, i_s, i_d)

    cls_head = _calib_get(calib_transformer, "cls_head")
    if cls_head is not None:
        sc_trans = _cls_score_baseline(cls_head, t_primary, t_feats)
        if np.all(np.isnan(sc_trans)):
            print("  ⚠️  cls_head scoring failed — falling back to SBP regression proxy")
            sc_trans = t_primary[:, 0]
    else:
        print("  ⚠️  calib_transformer has no cls_head — using SBP regression proxy "
              "(NOT consistent with master table methodology)")
        sc_trans = t_primary[:, 0]

    sc_trans = sc_trans[eidx]
    auc_trans = roc_auc_score(yt, sc_trans)
    print(f"  Transformer AUC (cls_head, eval_window n={len(eidx)}/{len(htn_te)}): {auc_trans:.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. Statistical Tests
# ─────────────────────────────────────────────────────────────────────────────
if auc_trans is not None:
    print("\n" + "─" * 70)
    print("  Statistical Tests")
    print("─" * 70)
    print(f"  (all tests run on the eval_window subset, n={len(eidx)}/{len(htn_te)} beats — "
          "same subset used for every column in master_ablation_summary.csv)")

    subj_te_p = subj_te[eidx]

 # --- Test A: DeLong ---
    auc_a, auc_b, z_dl, p_dl = delong_test(yt.astype(int), sc_mcnhm, sc_trans)
    print(f"\n  [A] DeLong test (paired, exact):")
    print(f"      DiBP-Net AUC  = {auc_a:.4f}")
    print(f"      Trans AUC  = {auc_b:.4f}")
    print(f"      ΔAUC       = {auc_a - auc_b:+.4f}")
    print(f"      z = {z_dl:.4f}   p = {p_dl:.4e}  "
          f"({'significant' if p_dl < 0.05 else 'NOT significant'} at α=0.05)")

 # --- Test B: Bootstrap CI on ΔAUC ---
    print(f"\n  [B] Bootstrap CI on ΔAUC (n_boot=1000) …")
    rng = np.random.default_rng(42)
    n_obs = len(yt)
    delta_boots = []
    for _ in range(1000):
        idx  = rng.integers(0, n_obs, n_obs)
        yt_b = yt[idx].astype(int)
        if len(np.unique(yt_b)) < 2:
            continue
        try:
            delta_boots.append(roc_auc_score(yt_b, sc_mcnhm[idx]) -
                                roc_auc_score(yt_b, sc_trans[idx]))
        except Exception:
            pass
    delta_boots = np.array(delta_boots)
    ci_lo = np.percentile(delta_boots, 2.5)
    ci_hi = np.percentile(delta_boots, 97.5)
    zero_in_ci = ci_lo <= 0 <= ci_hi
    print(f"      ΔAUC = {delta_boots.mean():+.4f}  95% CI: [{ci_lo:+.4f}, {ci_hi:+.4f}]")
    print(f"      CI includes 0: {zero_in_ci}  "
          f"→ {'NOT significant' if zero_in_ci else 'significant'}")

 # --- Test C: Wilcoxon on subject-level AUC proxy ---
    print(f"\n  [C] Subject-level Wilcoxon (subj with HTN mix, n beats ≥ 4) …")
    subj_auc_m, subj_auc_t = [], []
    valid_subjs = 0
    for s in np.unique(subj_te_p):
        mask = subj_te_p == s
        yt_s = yt[mask].astype(int)
        if len(np.unique(yt_s)) < 2 or mask.sum() < 4:
            continue
        try:
            subj_auc_m.append(roc_auc_score(yt_s, sc_mcnhm[mask]))
            subj_auc_t.append(roc_auc_score(yt_s, sc_trans[mask]))
            valid_subjs += 1
        except Exception:
            pass

    if len(subj_auc_m) >= 10:
        _, p_wilcox = wilcoxon(subj_auc_m, subj_auc_t, alternative="two-sided")
        print(f"      Valid subjects: {valid_subjs}")
        print(f"      DiBP-Net mean subj-AUC: {np.mean(subj_auc_m):.4f}")
        print(f"      Trans mean subj-AUC: {np.mean(subj_auc_t):.4f}")
        print(f"      Wilcoxon p = {p_wilcox:.4e}  "
              f"({'significant' if p_wilcox < 0.05 else 'NOT significant'})")
    else:
        p_wilcox = None
        print(f"      ⚠️  Too few subjects with HTN mix (n={valid_subjs}) — skip")

    # ─────────────────────────────────────────────────────────────────────────
 # 4. Variance sanity check
    # ─────────────────────────────────────────────────────────────────────────
    y_int = yt.astype(int)
    m_pos = y_int.sum(); n_neg = len(y_int) - m_pos

    pos_m = sc_mcnhm[y_int == 1]; neg_m = sc_mcnhm[y_int == 0]
    pos_t = sc_trans[y_int == 1];  neg_t = sc_trans[y_int == 0]

    V10_m = np.array([(pos_m[i] > neg_m).mean() + 0.5*(pos_m[i]==neg_m).mean()
                      for i in range(m_pos)])
    V01_m = np.array([(neg_m[j] < pos_m).mean() + 0.5*(neg_m[j]==pos_m).mean()
                      for j in range(n_neg)])
    V10_t = np.array([(pos_t[i] > neg_t).mean() + 0.5*(pos_t[i]==neg_t).mean()
                      for i in range(m_pos)])
    V01_t = np.array([(neg_t[j] < pos_t).mean() + 0.5*(neg_t[j]==pos_t).mean()
                      for j in range(n_neg)])

    var_m    = np.var(V10_m, ddof=1)/m_pos + np.var(V01_m, ddof=1)/n_neg
    var_t    = np.var(V10_t, ddof=1)/m_pos + np.var(V01_t, ddof=1)/n_neg
    cov_mt   = (np.cov(V10_m, V10_t, ddof=1)[0,1]/m_pos +
                np.cov(V01_m, V01_t, ddof=1)[0,1]/n_neg)
    var_diff = var_m + var_t - 2*cov_mt

    print(f"\n  Variance sanity: m={m_pos}, n={n_neg}")
    print(f"  var(DiBP-Net) = {var_m:.6e}  var(Trans) = {var_t:.6e}")
    print(f"  cov        = {cov_mt:.6e}  var(ΔAUC) = {var_diff:.6e}")
    print(f"  cov/var_m ratio = {cov_mt/var_m:.4f}  (>1 -> cov overestimate)")

    # ─────────────────────────────────────────────────────────────────────────
 # 5. Summary + Paper verdict
    # ─────────────────────────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("  VERDICT FOR PAPER")
    print("=" * 70)
    print(f"  Original paper claim: Wilcoxon vs. Transformer p = 8.0×10⁻³")
    print(f"  SBP MAE Wilcoxon test:  p = 0.131  (NOT significant)")
    print(f"  DeLong AUC test:            p = {p_dl:.4e}  "
          f"({'sig' if p_dl < 0.05 else 'NOT sig'})")
    print(f"  Bootstrap ΔAUC CI:          [{ci_lo:+.4f}, {ci_hi:+.4f}]  "
          f"({'sig' if not zero_in_ci else 'NOT sig'})")
    if p_wilcox is not None:
        print(f"  Subject Wilcoxon AUC:       p = {p_wilcox:.4e}  "
              f"({'sig' if p_wilcox < 0.05 else 'NOT sig'})")

    sig_count = sum([
        p_dl < 0.05,
        not zero_in_ci,
        (p_wilcox < 0.05) if p_wilcox is not None else False,
    ])

    print()
    if sig_count == 0:
        print("  → ALL TESTS: NOT significant")
        print("  ACTION: remove p=8.0e-3 from the paper")
        print("  replace with: DiBPNet achieves AUC = {:.3f}, "
              "comparable to Transformer ({:.3f}; ΔAUC = {:+.3f}, "
              "p = {:.2f}, DeLong test)".format(auc_a, auc_b, auc_a - auc_b, p_dl))
        paper_action = "REMOVE_CLAIM"
    elif sig_count >= 2:
        print("  → MAJORITY TESTS: significant")
        print("  ACTION: fix the p-value in the text to match DeLong")
        paper_action = "UPDATE_PVALUE"
    else:
        print("  → MIXED results")
        print("  ACTION: report DeLong only (most appropriate for AUC comparison)")
        paper_action = "USE_DELONG_ONLY"

    rows = [
        {"test": "DeLong", "auc_mcnhm": auc_a, "auc_trans": auc_b,
         "delta": auc_a - auc_b, "p": p_dl, "significant": p_dl < 0.05},
        {"test": "Bootstrap_ΔAUC_CI_lo", "auc_mcnhm": float("nan"),
         "auc_trans": float("nan"), "delta": ci_lo, "p": float("nan"),
         "significant": float("nan")},
        {"test": "Bootstrap_ΔAUC_CI_hi", "auc_mcnhm": float("nan"),
         "auc_trans": float("nan"), "delta": ci_hi, "p": float("nan"),
         "significant": float("nan")},
    ]
    if p_wilcox is not None:
        rows.append({"test": "Wilcoxon_subj_AUC",
                     "auc_mcnhm": np.mean(subj_auc_m),
                     "auc_trans": np.mean(subj_auc_t),
                     "delta": np.mean(subj_auc_m) - np.mean(subj_auc_t),
                     "p": p_wilcox, "significant": p_wilcox < 0.05})

    out_csv = f"{EXP_OUT_DIR}/exp_auc_significance.csv"
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"\n  CSV saved → {out_csv}")
    print(f"  Paper action: {paper_action}")

print("\n  ✅  AUC SIGNIFICANCE TEST DONE")
print("=" * 70)

### Load External Datasets

PulseDB, Multi-Sensor PPG, Hospital PPG, and BCG, evaluated zero-shot.

In [ ]:
ext_dict = load_all_external()
pdb      = ext_dict.get("pulsedb")

### Calibration Feature Ablation

Compares the 5-feature and 10-feature Ridge calibrators.

In [ ]:
r1 = run_exp1()

### External Zero-Shot Validation

Evaluates the UCI-trained model on the external cohorts without refitting.

In [ ]:
r2_all = run_exp2_unified(pdb, ext_dict)

In [ ]:
r2_refit70 = run_exp2_refit70(pdb, ext_dict)

In [ ]:
# 1. TEMPORAL DRIFT ANALYSIS (Proof of Physiological Stabilization)
# ══════════════════════════════════════════════════════════════════════════════
def analyze_temporal_trajectory(name, d, max_steps=15):
    ybp = d["Ybp_eval"]
    subj = d["subj_eval"]
    
    step_sbp = {i: [] for i in range(max_steps)}
    step_dbp = {i: [] for i in range(max_steps)}
    uniq_subj = np.unique(subj)
    
    for s in uniq_subj:
        idx = np.where(subj == s)[0]
        for j, global_idx in enumerate(idx):
            if j < max_steps:
                step_sbp[j].append(ybp[global_idx, 0])
                step_dbp[j].append(ybp[global_idx, 1])

    print(f"\n  Temporal Label Trajectory ({name} Dataset)")
    print("  " + "─" * 65)
    print(f"  {'Step':>4} | {'N Subj':>6} | {'Median SBP (mmHg)':>17} | {'Median DBP (mmHg)':>17}")
    print("  " + "─" * 65)
    
    for i in range(max_steps):
        if len(step_sbp[i]) < 5: 
            break
        n = len(step_sbp[i])
        m_sbp = np.median(step_sbp[i])
        m_dbp = np.median(step_dbp[i])
        print(f"  {i:>4d} | {n:>6d} | {m_sbp:>17.2f} | {m_dbp:>17.2f}")
    print("  " + "─" * 65)

if pdb is not None:
        analyze_temporal_trajectory("PulseDB", pdb, max_steps=15)
if ext_dict is not None:
    for tag in ("sensors", "bcg", "hospital"):
         if ext_dict.get(tag) and len(ext_dict[tag]["subj_eval"]) > 0:
            analyze_temporal_trajectory(ext_dict[tag]["name"], ext_dict[tag], max_steps=15)

In [ ]:
def test_multi_point_calibration(primary, ybp, subj, dataset_name="Dataset", ols_gap=5):
    """
    Main function to compute and compare multi-point calibration.
    """
    print("\n" + "─" * 70)
    print(f"  Multi-Point Calibration Analysis: {dataset_name}")
    print("─" * 70)
    
    strategies = {
        "1-Point (1st Beat)": lambda diff: np.median(diff[:1], axis=0),
        "3-Point Median"    : lambda diff: np.median(diff[:3], axis=0),
        "5-Point Median"    : lambda diff: np.median(diff[:5], axis=0),
        "5-Point Mean"      : lambda diff: np.mean(diff[:5], axis=0),
    }
    
    uniq_subj = np.unique(subj)
    max_ref = 5
    cal_preds = {k: primary.copy() for k in strategies.keys()}
    eval_mask = np.zeros(len(primary), dtype=bool)
    n_valid_subj = 0
    
    for s in uniq_subj:
        idx = np.where(subj == s)[0]
        if len(idx) < max_ref + ols_gap + 1:
            continue  
            
        n_valid_subj += 1
        ref_idx = idx[:max_ref]
        eval_idx = idx[max_ref + ols_gap:]
        eval_mask[eval_idx] = True
        
        diff = ybp[ref_idx] - primary[ref_idx]
        
        for name, func in strategies.items():
            bias = func(diff)
            cal_preds[name][eval_idx, 0] = primary[eval_idx, 0] + bias[0]
            cal_preds[name][eval_idx, 1] = primary[eval_idx, 1] + bias[1]
            
    if n_valid_subj == 0:
        print("  [SKIP] Not enough valid subjects with > 10 beats for this test.")
        return None
        
    print(f"  Valid Subjects: {n_valid_subj} | Eval Beats: {eval_mask.sum()}")
    print("  " + "─" * 65)
    print(f"  {'Calibration Strategy':<22} | {'SBP MAE':>10} | {'DBP MAE':>10} | {'Δ SBP (vs 1-Pt)':>15}")
    print("  " + "─" * 65)
    
    mae_sbp_1pt = np.mean(np.abs(cal_preds["1-Point (1st Beat)"][eval_mask, 0] - ybp[eval_mask, 0]))
    
    records = []
    for name in strategies.keys():
        mae_sbp = np.mean(np.abs(cal_preds[name][eval_mask, 0] - ybp[eval_mask, 0]))
        mae_dbp = np.mean(np.abs(cal_preds[name][eval_mask, 1] - ybp[eval_mask, 1]))
        
        delta = mae_sbp - mae_sbp_1pt if name != "1-Point (1st Beat)" else 0.0
        delta_str = f"{delta:+.2f}" if name != "1-Point (1st Beat)" else "-"
        
        print(f"  {name:<22} | {mae_sbp:>10.2f} | {mae_dbp:>10.2f} | {delta_str:>15}")
        records.append({"Dataset": dataset_name, "Strategy": name, "SBP_MAE": round(float(mae_sbp), 2), "DBP_MAE": round(float(mae_dbp), 2)})
        
    print("  " + "─" * 65)
    return records

def run_all_multi_point_calibrations(pdb, ext_dict, r2_zero):
    """
    Loops over every dataset already run in r2_zero.
    """
    print("=" * 80)
    print("  🚀 MULTI-POINT CALIBRATION ANALYSIS (ALL DATASETS)")
    print("=" * 80)
    
    all_results = []
    
 # 1. UCI (TEST)
    if "uci" in r2_all:
        X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
        res = test_multi_point_calibration(r2_zero["uci"]["primary"], Ybp_te, subj_te, "UCI (TEST)")
        if res: all_results.extend(res)

    externals = []
    if pdb is not None:
        externals.append(("PulseDB", pdb))
    if ext_dict is not None:
        for tag in ("sensors", "bcg", "hospital"):
            if ext_dict.get(tag):
                externals.append((ext_dict[tag]["name"], ext_dict[tag]))

    for name, d in externals:
        if name in r2_zero:
            res = test_multi_point_calibration(r2_zero[name]["primary"], d["Ybp_eval"], d["subj_eval"], name)
            if res: all_results.extend(res)
        else:
            print(f"\n  [SKIP] {name} - Not found in Zero-Shot results.")
            
    df_all = pd.DataFrame(all_results)
    return df_all

df_multi_cal = run_all_multi_point_calibrations(pdb, ext_dict, r2_all)

In [ ]:
def test_periodic_calibration(primary, ybp, subj, dataset_name="Dataset", model_name="Default", update_interval=15):
    """
    Helper to test periodic / continuous calibration for one model and one dataset.
    """
    uniq_subj = np.unique(subj)
    
    preds_static = []
    preds_periodic = []
    preds_rolling = []
    ybp_eval_list = []
    
    n_valid_subj = 0
    
    rb  = CALIB.ref_beats      # 3
    gap = CALIB.ols_gap        # 1

    for s in uniq_subj:
        idx = np.where(subj == s)[0]

        if len(idx) < 11:
            continue

        n_valid_subj += 1

        init_ref    = idx[0:rb]
        static_bias = np.median(ybp[init_ref] - primary[init_ref], axis=0)

        periodic_bias = static_bias.copy()

        for i in range(rb + gap, len(idx)):
            global_i = idx[i]

 # [Strategy 1] Static (single t=0 reference)
            pred_s = primary[global_i] + static_bias

            if i > rb + gap and i % update_interval == 0:
                recent_ref    = idx[i-gap-rb : i-gap]
                periodic_bias = np.median(ybp[recent_ref] - primary[recent_ref], axis=0)
            pred_p = primary[global_i] + periodic_bias

 # [Strategy 3] Continuous (rolling rb-beat median + gap)
            roll_ref     = idx[i-gap-rb : i-gap]
            rolling_bias = np.median(ybp[roll_ref] - primary[roll_ref], axis=0)
            pred_r = primary[global_i] + rolling_bias

            preds_static.append(pred_s)
            preds_periodic.append(pred_p)
            preds_rolling.append(pred_r)
            ybp_eval_list.append(ybp[global_i])
            
    if len(preds_static) == 0:
        return []
        
    preds_static = np.array(preds_static)
    preds_periodic = np.array(preds_periodic)
    preds_rolling = np.array(preds_rolling)
    y_true = np.array(ybp_eval_list)
    
    mae_s_sbp = np.mean(np.abs(preds_static[:, 0] - y_true[:, 0]))
    mae_s_dbp = np.mean(np.abs(preds_static[:, 1] - y_true[:, 1]))
    
    mae_p_sbp = np.mean(np.abs(preds_periodic[:, 0] - y_true[:, 0]))
    mae_p_dbp = np.mean(np.abs(preds_periodic[:, 1] - y_true[:, 1]))
    
    mae_r_sbp = np.mean(np.abs(preds_rolling[:, 0] - y_true[:, 0]))
    mae_r_dbp = np.mean(np.abs(preds_rolling[:, 1] - y_true[:, 1]))
    
    drop_p = (mae_s_sbp - mae_p_sbp) / mae_s_sbp * 100
    drop_r = (mae_s_sbp - mae_r_sbp) / mae_s_sbp * 100
    
    print(f"  [{model_name}] {dataset_name:<16} | Subj: {n_valid_subj:>4} | Eval Beats: {len(y_true):>6}")
    print(f"    - Static     SBP: {mae_s_sbp:>6.2f}  DBP: {mae_s_dbp:>6.2f}")
    print(f"    - Periodic   SBP: {mae_p_sbp:>6.2f} (Drop {drop_p:>5.1f}%)")
    print(f"    - Continuous SBP: {mae_r_sbp:>6.2f} (Drop {drop_r:>5.1f}%)")
    
    return [
        {"Model": model_name, "Dataset": dataset_name, "Strategy": "1. Static (t=0 Only)", "SBP_MAE": round(float(mae_s_sbp), 2), "DBP_MAE": round(float(mae_s_dbp), 2), "SBP_Drop_%": 0.0},
        {"Model": model_name, "Dataset": dataset_name, "Strategy": "2. Periodic (Re-cal 15s)", "SBP_MAE": round(float(mae_p_sbp), 2), "DBP_MAE": round(float(mae_p_dbp), 2), "SBP_Drop_%": round(float(drop_p), 1)},
        {"Model": model_name, "Dataset": dataset_name, "Strategy": "3. Continuous (Rolling)", "SBP_MAE": round(float(mae_r_sbp), 2), "DBP_MAE": round(float(mae_r_dbp), 2), "SBP_Drop_%": round(float(drop_r), 1)}
    ]

def run_all_models_periodic(pdb, ext_dict, r2_all):
    """
    Loops over every model and dataset in r2_all to address temporal drift.
    """
    print("=" * 85)
    print("  🚀 PERIODIC CALIBRATION ANALYSIS (ALL MODELS & ALL DATASETS)")
    print("=" * 85)
    
    all_results = []
    
    if "uci" in r2_all or "PulseDB" in r2_all:
        r2_all = {"Main_Model": r2_all}
        
    for model_name, model_results in r2_all.items():
        print(f"\n  " + "─" * 81)
        print(f"  Model: {model_name.upper()}")
        print("  " + "─" * 81)
        
 # 1. UCI (TEST)
        if "uci" in model_results:
            X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
            res = test_periodic_calibration(model_results["uci"]["primary"], Ybp_te, subj_te, "UCI (TEST)", model_name)
            if res: all_results.extend(res)

        externals = []
        if pdb is not None:
            externals.append(("PulseDB", pdb))
        if ext_dict is not None:
            for tag in ("sensors", "bcg", "hospital"):
                if ext_dict.get(tag):
                    externals.append((ext_dict[tag]["name"], ext_dict[tag]))

        for name, d in externals:
            if name in model_results:
                res = test_periodic_calibration(model_results[name]["primary"], d["Ybp_eval"], d["subj_eval"], name, model_name)
                if res: all_results.extend(res)
            else:
                print(f"  [{model_name}] {name:<16} | [SKIP] Not found in zero-shot results.")
                
    df_all = pd.DataFrame(all_results)

    _p = os.path.join(EXP_OUT_DIR, "exp_periodic_recalibration.csv")
    df_all.to_csv(_p, index=False)
    print(f"\n  Saved \u2192 {_p}")

    print("\n" + "=" * 85)
    print("  ✅ PERIODIC ANALYSIS COMPLETE. DATA GATHERED SUCCESSFULLY.")
    print("=" * 85)
    
    return df_all

df_periodic_cal_all = run_all_models_periodic(pdb, ext_dict, r2_all)

### Ablation and Latent Stability

Architecture ablation and the ICC/stability diagnostics of the latent.

In [ ]:
abl = run_ablation_and_icc()

### PPG Noise Robustness

Inference-time Gaussian-noise sensitivity of the calibrated predictions.

In [ ]:
df_noise = run_exp4()

In [ ]:
plot_noise_robustness_from_csv(
    csv_path=csv_file_path, 
    save_dir=SAVE_DIR, 
    filename="noise_robustness.png",
    ieee_rc=_IEEE_RC,
    fig_width=_COL1_IN
)

### Statistical Significance

Wilcoxon signed-rank test of SBP-MAE between models.

In [ ]:
r5 = run_exp5()

### Temporal Stability

In [ ]:
run_temporal_stability();

### AAMI/BHS Compliance

In [ ]:
df_aami = run_exp7(ext_dict)

### PulseDB Subgroup Analysis

By age, blood-pressure stage, and BMI.

In [ ]:
run_exp8_subgroup(pdb, r2_all);

### HTN-Stage-Stratified Calibration

In [ ]:
df_c, df_diag = run_exp_c_htn_cal()

### UMAP Domain Overlap

UCI and the four external datasets in a shared latent projection.

In [ ]:
run_exp9(ext_dict);

### Partial Correlation and Age Sensitivity

In [ ]:
rpart = run_partial_corr_and_age(abl, pdb);

### Bootstrap Confidence Intervals

In [ ]:
df_d = run_exp_d_bootstrap(abl, pdb)

### Intra-Stage Analysis

In [ ]:
run_exp_f_intra_stage(pdb);

### Cross-Dataset Comparison

In [ ]:
df_h = run_exp_h_cross_dataset(abl, ext_dict)

### Population-Level c_slow Analysis

A population-level view of c_slow stability across hypertension stages, using the test set intentionally.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Fig X – c_slow within-subject stability vs. SBP stability (FAST TRACK)
# ══════════════════════════════════════════════════════════════════════════════

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

print("Extracting features (Fast Track)...")
X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
_, _, eval_mask, fwd_prop = full_pipeline_test(model, calib_proposed)

cs_all   = fwd_prop["c_slow"][eval_mask]
sbp_all  = Ybp_te[eval_mask, 0]
dbp_all  = Ybp_te[eval_mask, 1]
subj_all = subj_te[eval_mask]

stage_all = assign_stages(sbp_all, dbp_all)

MIN_BEATS = 2

# ── 1. Per-subject stability ──────────────────────────────────────────────────
sigma_global_cs  = float(np.std(cs_all, ddof=1))
sigma_global_sbp = float(np.std(sbp_all, ddof=1))

records = []
for sid in np.unique(subj_all):
    m = subj_all == sid
    if m.sum() < MIN_BEATS:
        continue

    cs_norm  = float(np.std(cs_all[m], ddof=1)) / (sigma_global_cs + 1e-9)
    sbp_norm = float(np.std(sbp_all[m], ddof=1)) / (sigma_global_sbp + 1e-9)
    stage_mode = int(np.bincount(stage_all[m]).argmax())

    records.append({
        "subj_id": sid,
        "cs_norm": cs_norm,
        "sbp_norm": sbp_norm,
        "stage": stage_mode
    })

df_subj = pd.DataFrame(records)
N_subj = len(df_subj)
frac_below = float((df_subj["cs_norm"] < df_subj["sbp_norm"]).mean())

print(f"Records (N >= 2): {N_subj:,}")
print(f"Below diagonal (c_slow more stable): {frac_below*100:.1f}%")


# ── 2. Plotting ───────────────────────────────────────────────────────────────

STAGE_MAP = {1: "Normal", 2: "Elevated", 3: "HTN-1", 4: "HTN-2"}

try:
    STAGE_PALETTE_MAP = dict(zip(STAGE_ORDER, STAGE_PALETTE))
    STAGE_ORDER_LIST = STAGE_ORDER
except NameError:
    STAGE_ORDER_LIST = ["Normal", "Elevated", "HTN-1", "HTN-2"]
    STAGE_PALETTE = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]
    STAGE_PALETTE_MAP = dict(zip(STAGE_ORDER_LIST, STAGE_PALETTE))

FALLBACK_C = "#888888"
_FIG_H_IN = 3.5

with plt.rc_context(_IEEE_RC):
    fig, ax = plt.subplots(figsize=(_COL1_IN, 3.5))

    for stage_id, label in STAGE_MAP.items():
        mask = df_subj["stage"] == stage_id
        ax.scatter(df_subj.loc[mask, "sbp_norm"], df_subj.loc[mask, "cs_norm"],
                   c=STAGE_PALETTE_MAP.get(label, FALLBACK_C),
                   s=6, alpha=0.6, linewidths=0, rasterized=True, zorder=2, label=label)

    lim_max = float(max(df_subj["sbp_norm"].quantile(0.995), df_subj["cs_norm"].quantile(0.995))) * 1.03
    diag = np.array([0.0, lim_max])
    ax.plot(diag, diag, color="#444444", lw=1.0, ls="--", zorder=3)

    ax.text(0.04, 0.96, 
            rf"$\mathbf{{{frac_below*100:.1f}\%}}$ below diagonal" + "\n" + r"($c_{\rm slow}$ more stable)",
            transform=ax.transAxes, ha="left", va="top", fontsize=8, linespacing=1.4,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#DDDDDD", lw=0.5, alpha=0.9))

    stage_handles = [Line2D([0], [0], marker="o", color="none",
                     markerfacecolor=STAGE_PALETTE_MAP.get(lbl, FALLBACK_C),
                     markersize=3.5, alpha=0.85, label=lbl, markeredgewidth=0)
                     for lbl in STAGE_ORDER_LIST if lbl in STAGE_PALETTE_MAP]

    diag_handle = Line2D([0], [0], color="#444444", lw=1.0, ls="--", label="Equal stability")

    ax.legend(handles=stage_handles + [diag_handle], fontsize=8, loc="upper right",
              framealpha=0.90, edgecolor="#BBBBBB", handlelength=1.0, handletextpad=0.4,
              labelspacing=0.20, prop={"family": "Times New Roman", "size": 8})

    ax.set_xlim(0, lim_max)
    ax.set_ylim(0, lim_max)
    ax.set_xlabel(r"SBP within-$\sigma$ / global $\sigma$", fontsize=8)
    ax.set_ylabel(r"$c_{\rm slow}$ within-$\sigma$ / global $\sigma$", fontsize=8)
    ax.set_title(f"Within-Subject Temporal Stability\n($N={N_subj:,}$)", fontsize=8, fontweight="bold")
    ax.tick_params(labelsize=8)

    for artist in fig.findobj(plt.Text):
        try:
            artist.set_fontsize(8)
            artist.set_fontname("Times New Roman")
        except Exception:
            pass

    fig.tight_layout(pad=0.3)
    save_dir = globals().get("FIG_DIR", "outputs/phase_2_v192/figures")
    os.makedirs(save_dir, exist_ok=True)
    out_path = os.path.join(save_dir, "cslow_stability_scatter.png")
    fig.savefig(out_path, dpi=400, bbox_inches="tight", pad_inches=0.05)
    plt.show()
    print(f"✓ Saved → {out_path}")

### Summary Table and Export

Paper table, spreadsheet export, and key numbers.

In [ ]:
records_e = rpart.get("records_e", []) if rpart else []
df_paper  = build_paper_summary(df_d, records_e, abl, rpart)

xl_path = os.path.join(EXP_OUT_DIR, "all_experiments_v5.xlsx")
sheets  = {"EXP-D_Bootstrap_CI":    df_d,
           "EXP-C_HTN_Cal":         df_c,
           "EXP-C_Bias_Diagnostic": df_diag,
           "EXP-7_AAMI_BHS":        df_aami,
           "SUMMARY_paper":         df_paper}
with pd.ExcelWriter(xl_path, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        if df is not None:
            df.to_excel(writer, sheet_name=sheet_name, index=False)
print(f"\n  ✅  Excel → {xl_path}")

print("\n" + "=" * 70); print("  KEY NUMBERS"); print("=" * 70)
if r1:
    print(f"  Physics-feature Δ SBP Beat Post: {r1['delta_sbp']:+.2f} mmHg")
if r2_all and "PulseDB" in r2_all:
    p = r2_all["PulseDB"]
    print(f"  PulseDB SBP Beat Post: {p['sbp_beat_post']:.2f} mmHg  "
          f"(UCI: {r2_all['uci']['sbp_beat_post']:.2f})")
r2x = locals().get("r2x")
if r2x:
    for tag, m in r2x.items():
        print(f"  {m['name']:<18s} SBP Beat Post: {m['sbp_beat_post']:.2f} mmHg")
if abl:
    print(f"  ICC(c_slow)={abl['icc_cs']:.4f}  ICC(SBP)={abl['icc_sbp']:.4f}  "
          f"stability={abl['ratio_sbp']:.2f}× [{abl['r_lo']:.2f},{abl['r_hi']:.2f}]")
if rpart:
    print(f"  Partial r(c_slow,SBP|DBP+PP)={rpart['r_ctrl_both']:.3f}  "
          f"ΔAUC={rpart['auc_sbp_cs']-rpart['auc_sbp_only']:+.3f}")

sbp_subj_row = df_aami[df_aami["label"].str.contains(
    "SBP Post.*Subject.*UCI", regex=True)]
if not sbp_subj_row.empty:
    rs   = sbp_subj_row.iloc[0]
    flag = "⚠ >8 mmHg" if rs["sd"] > 8.0 else "✅"
    print(f"  SBP Subject Post: ME={rs['me']:+.2f}  SD={rs['sd']:.2f}  "
          f"{rs['aami']}  Grade {rs['bhs']}  {flag}")

if abl and rpart:
    _age_rec   = next((r for r in rpart.get('records_e', [])
                       if "SBP,DBP,PP" in r.get("covariates", "")), None)
    _p_age     = float(_age_rec["p_value"]) if _age_rec else float("nan")
    _age_sig   = (not np.isnan(_p_age)) and (_p_age < 0.05)
    _age_tick  = "✓" if _age_sig else "~"
    _age_label = "Age-sensitive" if _age_sig else "Weak age dependence"
    print(f"""
  ── c_slow narrative ──────────────────────────────────────────────────
  ✓ Temporally stable (ICC={abl['icc_cs']:.4f}, {abl['ratio_sbp']:.2f}× more stable)
  ✓ HTN-stage separable (Cohen d={abl['cd_prop']:.3f})
  {_age_tick} {_age_label} (partial r(c_slow,age|SBP,DBP,PP) p={_p_age:.3f})
  ✗ Independent phenotype (partial r|DBP+PP={rpart['r_ctrl_both']:.3f}) — weak
  ✗ EXP-H: sign reversal unresolved — exclude from Main Paper

  ── Manual items still needed ─────────────────────────────────────────
  [1] EXP-H: interpret figures → write manuscript sentence
  [2] EXP-F: add underpowered-stage caveats for Normal/Elevated
  [3] EXP-C: add negative result (C2≥C1) to Discussion
  [4] Add exp_e_partial_corr_forest.png as Figure X
""")
print("=" * 70)
print("  ALL EXPERIMENTS COMPLETE ✓")
print("=" * 70)

# Extended Physics and Calibration Experiments

Physics-parameter consistency, domain-gap diagnostics, calibration-robustness candidates, and reviewer-response baselines.

## Definitions

In [ ]:
# Physics Parameter Consistency (R and C)
# ══════════════════════════════════════════════════════════════════════════════

def run_exp_i_physics_consistency(ext_dict):
    """
    Compare the distributions of R and C between UCI (source) and every
    external dataset available (PulseDB, sensors, bcg, hospital) to check whether the
    physics parameters are consistent (generalizable) across domains,
    even when c_slow undergoes a domain shift.
    """
    print("\n" + "─" * 70)
    print("  Physics Parameter Consistency (R & C)  [TEST vs ALL EXTERNAL]")
    print("─" * 70)
    if not ext_dict:
        print("  [SKIP] no external datasets available"); return None

    # ── 1. Extract R, C for UCI TEST (reference domain) ───────────────────────
    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    fwd_uci = _infer_forward(model, X_te, Q_te, subj_te, base_te)
    R_uci = fwd_uci["R"]; C_uci = fwd_uci["C"]; tau_uci = R_uci * C_uci

    def _stats(arr_uci, arr_ext, name, ds_label):
        m_u, s_u = arr_uci.mean(), arr_uci.std()
        m_e, s_e = arr_ext.mean(), arr_ext.std()
        d = cohens_d(arr_ext, arr_uci)
        print(f"  [{ds_label:<12}] {name:<12} | UCI: {m_u:.3f} ± {s_u:.3f}  |  "
              f"{ds_label}: {m_e:.3f} ± {s_e:.3f}  |  Δ Cohen's d = {d:+.3f}")
        return dict(dataset=ds_label, parameter=name,
                    uci_mean=round(float(m_u), 3), uci_std=round(float(s_u), 3),
                    ext_mean=round(float(m_e), 3), ext_std=round(float(s_e), 3),
                    cohens_d=round(float(d), 3))

    records = []
    box_R = {"UCI (TEST)": R_uci}
    box_C = {"UCI (TEST)": C_uci}

    for name, tag, d in _iter_ext_datasets(ext_dict):
        fwd_ext = _infer_forward(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])
        R_ext = fwd_ext["R"]; C_ext = fwd_ext["C"]; tau_ext = R_ext * C_ext

        print(f"\n  {'Parameter':<12} | {'UCI TEST':^18} | {name:^18} | {'Shift Size'}")
        print(f"  {'─'*70}")
        records.append(_stats(R_uci,   R_ext,   "R (Resistance)", name))
        records.append(_stats(C_uci,   C_ext,   "C (Compliance)", name))
        records.append(_stats(tau_uci, tau_ext, "τ = R×C",        name))

        box_R[name] = R_ext
        box_C[name] = C_ext

    print(f"\n  ⚠️  CAVEAT — raw C (Compliance) sign is NOT guaranteed to be preserved")
    print(f"     across datasets (see per-dataset Cohen's d above): only τ=R×C is")
    print(f"     directly supervised by the Windkessel ODE loss, so prefer τ (or R")
    print(f"     alone) over raw C for any cross-dataset interpretability claim.")

    df_physio = pd.DataFrame(records)
    df_physio.to_csv(os.path.join(EXP_OUT_DIR, "exp_i_physics_consistency.csv"), index=False)

 # Plotting moved to the Figure Generation section (decoupled).
    _FIG_DATA["exp_i"] = dict(box_R=box_R, box_C=box_C)

    print("DONE ✓")
    return df_physio

In [ ]:
# Comprehensive HTN Stage Analysis (R, C, c_slow)
# ══════════════════════════════════════════════════════════════════════════════

def run_exp_j_comprehensive_stage_analysis(ext_dict):
    """
    Analyse how well R, C, and c_slow separate HTN stages,
    comparing UCI against every available external dataset.
    """
    print("\n" + "─" * 70)
    print("  Comprehensive HTN Stage Analysis (R, C, c_slow)")
    print("         [UCI TEST vs ALL EXTERNAL]")
    print("─" * 70)
    if not ext_dict:
        print("  [SKIP] no external datasets available"); return None

    # ── 1. UCI reference ──────────────────────────────────────────────────────
    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    fwd_uci = _infer_forward(model, X_te, Q_te, subj_te, base_te)

    def _subject_agg(fwd, Ybp, subj):
        subjs = np.unique(subj)
        s_R   = np.array([fwd["R"][subj == s].mean()  for s in subjs])
        s_C   = np.array([fwd["C"][subj == s].mean()  for s in subjs])
        s_cs  = np.array([fwd["c_slow"][subj == s].mean() for s in subjs])
        s_sbp = np.array([Ybp[subj == s, 0].mean() for s in subjs])
        s_dbp = np.array([Ybp[subj == s, 1].mean() for s in subjs])
        s_st  = assign_stages(s_sbp, s_dbp)
        return s_R, s_C, s_cs, s_sbp, s_st

    uci_R, uci_C, uci_cs, uci_sbp, uci_st = _subject_agg(fwd_uci, Ybp_te, subj_te)
    uci_tau = uci_R * uci_C

    records = []
    pairs = [(1, 2), (1, 3), (1, 4), (2, 3), (3, 4)]

    def _analyze(dataset_name, param_name, s_vals, s_sbp, s_st, expected_trend=None):
 # Table XI: r (Fisher-z 95% CI) + Cohen's d (subject-bootstrap 95% CI) per
 # stage-pair. Absorbs the former standalone run_exp_p4_table_xi_ci (SSoT).
        print(f"\n  [{dataset_name}] {param_name}")
        n = len(s_vals)
        r_val, p_val = pearsonr(s_vals, s_sbp)
        r_lo, r_hi   = fisher_ci(r_val, n)
        stars_r = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
        print(f"    Corr with SBP: r = {r_val:>+5.3f} [{r_lo:+.3f}, {r_hi:+.3f}] | p = {p_val:.2e} | {stars_r}")
        print(f"    {'Comparison':<20} | {'Cohen d [95% CI]':>24} | {'p-value (MW)':>13} | {'Sig'}")
        print(f"    {'-'*74}")
        for st1, st2 in pairs:
            g1 = s_vals[s_st == st2]; g2 = s_vals[s_st == st1]
            if len(g1) < 2 or len(g2) < 2:
                continue
            d, d_lo, d_hi = bootstrap_cohens_d_ci(g1, g2)
            _, p = mannwhitneyu(g1, g2, alternative="two-sided")
            stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
            trend_ok = ""
            if expected_trend is not None and p < 0.05:
                trend_ok = "✓" if np.sign(d) == np.sign(expected_trend) else "✗"
            comp_lbl = f"{STAGE_LABELS[st2]} vs {STAGE_LABELS[st1]}"
            print(f"    {comp_lbl:<20} | {d:>+7.3f} [{d_lo:+.3f},{d_hi:+.3f}] | {p:>13.2e} | {stars:<3} {trend_ok}")
            records.append({
                "Dataset": dataset_name, "Parameter": param_name,
                "Stage_High": STAGE_LABELS[st2], "Stage_Low": STAGE_LABELS[st1],
                "Cohen_d": round(d, 4), "d_ci_lo": round(d_lo, 4), "d_ci_hi": round(d_hi, 4),
                "p_value": p, "Significance": stars,
                "Corr_with_SBP": round(r_val, 4),
                "r_ci_lo": round(r_lo, 4), "r_ci_hi": round(r_hi, 4),
            })

    _analyze("UCI TEST", "R (Resistance)", uci_R,   uci_sbp, uci_st, expected_trend=1)
    _analyze("UCI TEST", "C (Compliance)", uci_C,   uci_sbp, uci_st, expected_trend=-1)
    _analyze("UCI TEST", "tau=R*C",        uci_tau, uci_sbp, uci_st, expected_trend=None)
    _analyze("UCI TEST", "c_slow",         uci_cs,  uci_sbp, uci_st, expected_trend=None)

    for name, tag, d in _iter_ext_datasets(ext_dict):
        fwd_ext = _infer_forward(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])
        ext_R, ext_C, ext_cs, ext_sbp, ext_st = _subject_agg(fwd_ext, d["Ybp_eval"], d["subj_eval"])
        ext_tau = ext_R * ext_C

        _analyze(f"{name} EXTERNAL", "R (Resistance)", ext_R,   ext_sbp, ext_st, expected_trend=1)
        _analyze(f"{name} EXTERNAL", "C (Compliance)", ext_C,   ext_sbp, ext_st, expected_trend=-1)
        _analyze(f"{name} EXTERNAL", "tau=R*C",        ext_tau, ext_sbp, ext_st, expected_trend=None)
        _analyze(f"{name} EXTERNAL", "c_slow",         ext_cs,  ext_sbp, ext_st, expected_trend=None)

    print(f"\n  ⚠️  CAVEAT — raw C (Compliance) does not preserve sign across")
    print(f"     datasets (see per-dataset tables above); τ=R×C and R alone")
    print(f"     are the identifiable, cross-dataset-consistent quantities.")
    print(f"     Do not use raw C for cross-dataset interpretability claims.")

    df_j = pd.DataFrame(records)
    csv_path = os.path.join(EXP_OUT_DIR, "exp_j_comprehensive_stage_analysis.csv")
    df_j.to_csv(csv_path, index=False)
    print(f"\n  ✅ Saved → {csv_path}")
    print("DONE ✓")
    return df_j

In [ ]:
# Domain-Gap Diagnostic
# ══════════════════════════════════════════════════════════════════════════════

def run_exp_k_domain_gap(ext_dict, exp2_result=None, exp2x_result=None,
                         df_i=None, df_h=None):
    print("\n" + "─" * 70)
    print("  Domain-Gap Diagnostic  [UCI vs ALL EXTERNAL]")
    print("─" * 70)
    if not ext_dict:
        print("  [SKIP] no external datasets available"); return None

    # ── UCI reference (population Ridge+Iso vs subject-bias stage) ────────────
    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    primary_uci, cal_uci, mask_uci, _ = full_pipeline_test(model, calib_proposed)
    eidx_uci = np.where(mask_uci)[0]
    uci_pre_sbp  = float(np.abs(primary_uci[eidx_uci, 0] - Ybp_te[eidx_uci, 0]).mean())
    uci_post_sbp = float(np.abs(cal_uci[eidx_uci, 0]     - Ybp_te[eidx_uci, 0]).mean())

    records = []
    records.append(dict(dataset="UCI-TEST", n_subj=len(np.unique(subj_te[eidx_uci])),
                        pre_sbp_mae=round(uci_pre_sbp, 3),
                        post_sbp_mae=round(uci_post_sbp, 3),
                        subj_bias_gain=round(uci_pre_sbp - uci_post_sbp, 3)))

    for name, tag, d in _iter_ext_datasets(ext_dict):
        primary_e, cal_e, mask_e = _get_primary_cal_mask(d)
        eidx_e = np.where(mask_e)[0]
        if len(eidx_e) == 0:
            print(f"  [{name}] n_eval=0 — skip"); continue
        pre_sbp  = float(np.abs(primary_e[eidx_e, 0] - d["Ybp_eval"][eidx_e, 0]).mean())
        post_sbp = float(np.abs(cal_e[eidx_e, 0]     - d["Ybp_eval"][eidx_e, 0]).mean())
        records.append(dict(dataset=f"{name}-EXTERNAL",
                            n_subj=len(np.unique(d["subj_eval"][eidx_e])),
                            pre_sbp_mae=round(pre_sbp, 3),
                            post_sbp_mae=round(post_sbp, 3),
                            subj_bias_gain=round(pre_sbp - post_sbp, 3)))

    df_gap = pd.DataFrame(records)
    df_gap["ratio_vs_uci_post"] = (df_gap["post_sbp_mae"] / uci_post_sbp).round(2)

    print(f"\n  {'Dataset':<18} {'Pre':>9} {'Post':>9} {'SubjBias Δ':>11} {'x UCI':>7}")
    print(f"  {'─'*58}")
    for _, r in df_gap.iterrows():
        print(f"  {r['dataset']:<18} {r['pre_sbp_mae']:>9.2f} {r['post_sbp_mae']:>9.2f} "
              f"{r['subj_bias_gain']:>11.2f} {r['ratio_vs_uci_post']:>6.2f}x")

    # ── Is the gap PulseDB-specific or a general zero-shot ceiling? ───────────
    ext_rows = df_gap[df_gap["dataset"] != "UCI-TEST"]
    if len(ext_rows) >= 2:
        spread = ext_rows["post_sbp_mae"].std()
        mean_ext = ext_rows["post_sbp_mae"].mean()
        cov = spread / mean_ext if mean_ext else float("nan")
        verdict = ("clustered — general zero-shot ceiling, not PulseDB-specific"
                  if cov < 0.25 else
                  "spread out — dataset-specific effects dominate, not a uniform ceiling")
        print(f"\n  External Post MAE: mean={mean_ext:.2f}  std={spread:.2f}  CoV={cov:.2f}")
        print(f"  → {verdict}")

    if df_i is not None and not df_i.empty:
        print("\n  R/C/τ shift by dataset (from the physics parameter (R/C) consistency analysis, Cohen's d vs UCI):")
        for _, r in df_i.iterrows():
            print(f"    [{r['dataset']:<12}] {r['parameter']:<14} d={r['cohens_d']:+.3f}")

    if df_h is not None and not df_h.empty:
        n_rev = int(df_h["sign_reversal"].sum()) if "sign_reversal" in df_h else 0
        print(f"\n  c_slow-SBP sign reversal (cross-dataset check): {n_rev}/{len(df_h)} external datasets")

    df_gap.to_csv(os.path.join(EXP_OUT_DIR, "exp_k_domain_gap_diagnostic.csv"), index=False)
    print(f"\n  Saved → {EXP_OUT_DIR}/exp_k_domain_gap_diagnostic.csv")

 # Plotting moved to the Figure Generation section (decoupled);
 # store only the payload it needs.
    _FIG_DATA["exp_k"] = dict(df_gap=df_gap.copy(), uci_post_sbp=float(uci_post_sbp))

    print("DONE ✓")
    return df_gap

In [ ]:
# Robust Calibration Candidate Comparison
# ══════════════════════════════════════════════════════════════════════════════

import csv as _csv
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.isotonic import IsotonicRegression

_C_COL = 7  # index of raw Windkessel compliance C in the 10-feature vector
_KEEP_9 = [i for i in range(10) if i != _C_COL]


def _moment_match(feats_ext, mu_ext, sig_ext, mu_uci, sig_uci):
    """Per-column re-center/re-scale an external dataset's raw calibration
    features to match UCI-VAL's own feature distribution (mean/std) —
    motivated by the base_arr unit-mismatch bug (pulsedb-window-bug-fix
    memory) that was the dominant cause of PulseDB's original +68mmHg bias:
    that was this exact failure mode for ONE column; this generalizes it.
    """
    sig_ext_safe = np.where(sig_ext < 1e-8, 1.0, sig_ext)
    z = (feats_ext - mu_ext) / sig_ext_safe
    return z * sig_uci + mu_uci


def run_exp_l_robust_calibration(ext_dict):
    print("\n" + "─" * 70)
    print("  Robust Calibration Candidate Comparison")
    print("─" * 70)

    r_s   = _calib_get(calib_proposed, "ridge_sbp")
    r_d   = _calib_get(calib_proposed, "ridge_dbp")
    iso_s = _calib_get(calib_proposed, "iso_s")
    iso_d = _calib_get(calib_proposed, "iso_d")

 # UCI-VAL feature-column moments (the space the baseline Ridge/Iso were fit on)
    feats_uci = _build_ridge_feats_10(model, SPLITS.val.X, SPLITS.val.Q,
                                       SPLITS.val.subj, SPLITS.val.base)
    mu_uci  = feats_uci.mean(axis=0)
    sig_uci = feats_uci.std(axis=0)

 # Candidate A2: refit Ridge+Iso on UCI-VAL with raw C dropped (9 features)
    feats_uci_9 = feats_uci[:, _KEEP_9]
    r_s_9 = Ridge(alpha=getattr(CALIB, "ridge_alpha", 10.0)).fit(feats_uci_9, SPLITS.val.Ybp[:, 0])
    r_d_9 = Ridge(alpha=getattr(CALIB, "ridge_alpha", 10.0)).fit(feats_uci_9, SPLITS.val.Ybp[:, 1])
    iso_s_9 = IsotonicRegression(out_of_bounds="clip").fit(
        r_s_9.predict(feats_uci_9), SPLITS.val.Ybp[:, 0])
    iso_d_9 = IsotonicRegression(out_of_bounds="clip").fit(
        r_d_9.predict(feats_uci_9), SPLITS.val.Ybp[:, 1])
    mu_uci_9, sig_uci_9 = feats_uci_9.mean(axis=0), feats_uci_9.std(axis=0)

    rows = []
    for name, tag, d in _iter_ext_datasets(ext_dict):
        _build_subj_cache(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])
        feats_ext = _build_ridge_feats_10(model, d["X_eval"], d["Q_eval"],
                                           d["subj_eval"], d["base_eval"])
        Ybp  = d["Ybp_eval"]
        subj = d["subj_eval"]

 # --- Baseline: zero-shot Ridge+Iso, SubjBias n_ref=3 (current default) ---
        primary_base = apply_ridge_iso(feats_ext, r_s, r_d, iso_s, iso_d)
        cal_base, mask_base = run_subject_bias_correction(primary_base, Ybp, subj)
        if mask_base.sum() < 10:
            print(f"  [{name}] n_eval={mask_base.sum()} — insufficient reference beats per subject, skip")
            continue
        mae_base_post = float(np.mean(np.abs(cal_base[mask_base, 0] - Ybp[mask_base, 0])))

 # --- Candidate A: 10-feature moment-matching (naive, includes raw C) ---
        mu_ext, sig_ext = feats_ext.mean(axis=0), feats_ext.std(axis=0)
        feats_a = _moment_match(feats_ext, mu_ext, sig_ext, mu_uci, sig_uci)
        primary_a = apply_ridge_iso(feats_a, r_s, r_d, iso_s, iso_d)
        cal_a, mask_a = run_subject_bias_correction(primary_a, Ybp, subj)
        mae_a_post = float(np.mean(np.abs(cal_a[mask_a, 0] - Ybp[mask_a, 0]))) if mask_a.sum() else float("nan")

 # --- Candidate A2: 9-feature moment-matching, raw C dropped entirely ---
        feats_ext_9 = feats_ext[:, _KEEP_9]
        mu_ext_9, sig_ext_9 = feats_ext_9.mean(axis=0), feats_ext_9.std(axis=0)
        feats_a2 = _moment_match(feats_ext_9, mu_ext_9, sig_ext_9, mu_uci_9, sig_uci_9)
        primary_a2 = apply_ridge_iso(feats_a2, r_s_9, r_d_9, iso_s_9, iso_d_9)
        cal_a2, mask_a2 = run_subject_bias_correction(primary_a2, Ybp, subj)
        mae_a2_post = float(np.mean(np.abs(cal_a2[mask_a2, 0] - Ybp[mask_a2, 0]))) if mask_a2.sum() else float("nan")

 # --- Candidate B: baseline Ridge+Iso, larger SubjBias n_ref=5 ---
        cal_b, mask_b = run_subject_bias_correction(primary_base, Ybp, subj, ref_beats=5)
        mae_b_post = float(np.mean(np.abs(cal_b[mask_b, 0] - Ybp[mask_b, 0]))) if mask_b.sum() else float("nan")

        print(f"  [{name}] SBP Post MAE  Baseline: {mae_base_post:6.2f}   "
              f"MomentMatch(10f,+C): {mae_a_post:6.2f}   "
              f"MomentMatch(9f,-C): {mae_a2_post:6.2f}   "
              f"n_ref=5: {mae_b_post:6.2f}")

        rows.append({
            "Dataset": name, "n_eval": int(mask_base.sum()),
            "Baseline_Post": round(mae_base_post, 3),
            "MomentMatch10f_withC_Post": round(mae_a_post, 3),
            "MomentMatch9f_noC_Post": round(mae_a2_post, 3),
            "NRef5_Post": round(mae_b_post, 3),
        })

    df = pd.DataFrame(rows)
    if len(df):
        cols = ["Baseline_Post", "MomentMatch10f_withC_Post", "MomentMatch9f_noC_Post", "NRef5_Post"]
        means = {c: df[c].mean() for c in cols}
        stds  = {c: df[c].std()  for c in cols}
        print(f"\n  Mean SBP Post MAE across {len(df)} datasets:")
        for c in cols:
            print(f"    {c:<28s} {means[c]:.2f} ± {stds[c]:.2f}")
        winner = min(means.items(), key=lambda t: t[1])
        print(f"  → Best mean MAE: {winner[0]} ({winner[1]:.2f} mmHg)")
 # Direct test of the C-poisoning hypothesis
        delta_c = means["MomentMatch10f_withC_Post"] - means["MomentMatch9f_noC_Post"]
        verdict = ("including raw C in moment-matching HURTS" if delta_c > 0.05 else
                    "raw C inclusion makes negligible/no difference" if abs(delta_c) <= 0.05 else
                    "including raw C unexpectedly helps")
        print(f"  → With-C vs without-C moment-matching Δ = {delta_c:+.2f} mmHg → {verdict}")

    out_csv = os.path.join(EXP_OUT_DIR, "exp_l_robust_calibration.csv")
    df.to_csv(out_csv, index=False)
    print(f"\n  Saved → {out_csv}")
    print("DONE ✓")
    return df

# ==========================================
# FIX APPLIED HERE
# ==========================================
# Ensure the model is explicitly mapped to the intended device
# before calling the function that relies on its layers.

In [ ]:
# Output-Distribution Quantile Mapping
# ══════════════════════════════════════════════════════════════════════════════

_QM_FIT_FRAC = 0.3   # subject fraction for the in-domain variant (matches the linear refit)


def _quantile_map(vals, ref_src, ref_tgt, n_q=101):
    """Monotonic quantile transform: place each `vals` element at its quantile
    in the `ref_src` distribution, then read the value at that quantile in
    `ref_tgt`. Linear-interpolated between 101 anchor quantiles."""
    qs    = np.linspace(0.0, 1.0, n_q)
    src_q = np.maximum.accumulate(np.quantile(ref_src, qs))  # monotone guard
    tgt_q = np.quantile(ref_tgt, qs)
    return np.interp(vals, src_q, tgt_q).astype(np.float32)


def _qm_eval_mae(aligned_sbp_eval, gcal_eval, Ybp_eval, subj_eval):
    """Subject-bias-correct an already-(quantile-mapped) SBP array on the eval
    rows and return (SBP Post MAE, n_eval). `gcal_eval` is the eval-row
    [N,2] GCal prediction whose SBP column gets overwritten by aligned_sbp_eval."""
    prim = gcal_eval.copy()
    prim[:, 0] = aligned_sbp_eval
    cal, m = run_subject_bias_correction(prim, Ybp_eval, subj_eval)
    if m.sum() < 10:
        return float("nan"), int(m.sum())
    return float(np.mean(np.abs(cal[m, 0] - Ybp_eval[m, 0]))), int(m.sum())


def run_exp_n_quantile_map(ext_dict, fit_frac=_QM_FIT_FRAC, seed=42):
    print("\n" + "─" * 70)
    print("  Output-Distribution Quantile Mapping  [zero-shot & in-domain]")
    print("─" * 70)

    r_s   = _calib_get(calib_proposed, "ridge_sbp")
    r_d   = _calib_get(calib_proposed, "ridge_dbp")
    iso_s = _calib_get(calib_proposed, "iso_s")
    iso_d = _calib_get(calib_proposed, "iso_d")

 # UCI-VAL reference for zero-shot QM: GCal-predicted SBP (source) + TRUE SBP (target)
    feats_uci = _build_ridge_feats_10(model, SPLITS.val.X, SPLITS.val.Q,
                                       SPLITS.val.subj, SPLITS.val.base)
    uci_pred  = apply_ridge_iso(feats_uci, r_s, r_d, iso_s, iso_d)[:, 0]
    uci_true  = SPLITS.val.Ybp[:, 0]

    rng = np.random.RandomState(seed)
    rows = []
    for name, tag, d in _iter_ext_datasets(ext_dict):
        subj = d["subj_eval"]; Ybp = d["Ybp_eval"]
        _build_subj_cache(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])
        feats_ext = _build_ridge_feats_10(model, d["X_eval"], d["Q_eval"],
                                           d["subj_eval"], d["base_eval"])
        gcal = apply_ridge_iso(feats_ext, r_s, r_d, iso_s, iso_d)  # [N,2]

 # subject split (in-domain fit vs eval) -- same protocol as the linear refit
        uniq = np.unique(subj); rng.shuffle(uniq)
        n_fit = max(5, int(len(uniq) * fit_frac))
        fit_set, eval_set = set(uniq[:n_fit]), set(uniq[n_fit:])
        fit_mask  = np.array([s in fit_set  for s in subj])
        eval_mask = np.array([s in eval_set for s in subj])

        gcal_e = gcal[eval_mask]; Ybp_e = Ybp[eval_mask]; subj_e = subj[eval_mask]

 # Baseline (no QM)
        mae_base, n_eval = _qm_eval_mae(gcal_e[:, 0], gcal_e, Ybp_e, subj_e)
        if not np.isfinite(mae_base):
            print(f"  [{name}] n_eval={n_eval} — insufficient, skip"); continue

 # (a) Zero-shot QM: external GCal-pred SBP -> UCI true-SBP quantiles
        qm_zs = _quantile_map(gcal_e[:, 0], uci_pred, uci_true)
        mae_zs, _ = _qm_eval_mae(qm_zs, gcal_e, Ybp_e, subj_e)

 # (b) In-domain QM: learn pred->true map on 30% fit subjects, apply to eval
        qm_id = _quantile_map(gcal_e[:, 0], gcal[fit_mask, 0], Ybp[fit_mask, 0])
        mae_id, _ = _qm_eval_mae(qm_id, gcal_e, Ybp_e, subj_e)

        print(f"  [{name:<16}] SBP Post MAE  Baseline: {mae_base:6.2f}   "
              f"QM-ZeroShot: {mae_zs:6.2f}   QM-InDomain(30%): {mae_id:6.2f}")
        rows.append(dict(Dataset=name, n_eval=n_eval,
                         Baseline_Post=round(mae_base, 3),
                         QM_ZeroShot_Post=round(mae_zs, 3),
                         QM_InDomain_Post=round(mae_id, 3)))

    df = pd.DataFrame(rows)
    if len(df):
        print(f"\n  Mean SBP Post MAE across {len(df)} datasets:")
        for col in ["Baseline_Post", "QM_ZeroShot_Post", "QM_InDomain_Post"]:
            print(f"    {col:<22s} {df[col].mean():.2f} ± {df[col].std():.2f}")
        best = min([(c, df[c].mean()) for c in
                    ["Baseline_Post", "QM_ZeroShot_Post", "QM_InDomain_Post"]],
                   key=lambda t: t[1])
        print(f"  → Best mean MAE: {best[0]} ({best[1]:.2f} mmHg)")

    out_csv = os.path.join(EXP_OUT_DIR, "exp_n_quantile_map.csv")
    df.to_csv(out_csv, index=False)
    print(f"\n  Saved → {out_csv}")
    print("DONE ✓")
    return df

In [ ]:
# c_slow Calibration Utility Test
# Compare 1-beat vs 2-beat vs 3-beat subject calibration MAE, DiBP-Net vs NoDis.
# Answers R2: "What does the end-user gain from DiBP-Net that NoDis does not?"
# ══════════════════════════════════════════════════════════════════════════════

N_BOOT_CALUTIL  = 1000
CALUTIL_OLS_GAP  = CALIB.ols_gap
CALUTIL_EVAL_MIN = CALIB.eval_min


def _run_calibration_n_ref(primary, true, subj, n_ref, ols_gap=CALUTIL_OLS_GAP, eval_min=CALUTIL_EVAL_MIN):
    """Same subject-bias-correction logic as run_subject_bias_correction, but with a
    caller-supplied n_ref (vs the notebook default CALIB.ref_beats) so we can sweep it."""
    cal_pred  = primary.copy()
    eval_mask = np.zeros(len(primary), dtype=bool)
    for s in np.unique(subj):
        m = np.where(subj == s)[0]; n = len(m)
        if n < n_ref + ols_gap + eval_min:
            continue
        ref_idx  = m[:n_ref]
        eval_idx = m[n_ref + ols_gap:]
        if len(eval_idx) < eval_min:
            continue
        bias_sbp = float(np.median(true[ref_idx, 0] - primary[ref_idx, 0]))
        bias_dbp = float(np.median(true[ref_idx, 1] - primary[ref_idx, 1]))
        cal_pred[eval_idx, 0] = primary[eval_idx, 0] + bias_sbp
        cal_pred[eval_idx, 1] = primary[eval_idx, 1] + bias_dbp
        eval_mask[eval_idx]   = True
    if not eval_mask.any():
        return None, None
    sbp_mae = float(np.mean(np.abs(cal_pred[eval_mask, 0] - true[eval_mask, 0])))
    dbp_mae = float(np.mean(np.abs(cal_pred[eval_mask, 1] - true[eval_mask, 1])))
    return sbp_mae, dbp_mae


def _bootstrap_calutil_ci(pred, true, subj, n_ref, n_boot=N_BOOT_CALUTIL, seed=42):
    rng  = np.random.default_rng(seed)
    uids = np.unique(subj)
    boot_sbp, boot_dbp = [], []
    for _ in range(n_boot):
        chosen = rng.choice(uids, size=len(uids), replace=True)
        idx    = np.concatenate([np.where(subj == s)[0] for s in chosen])
        sbp, dbp = _run_calibration_n_ref(pred[idx], true[idx], subj[idx], n_ref)
        if sbp is not None:
            boot_sbp.append(sbp); boot_dbp.append(dbp)
    if not boot_sbp:
        return None
    return dict(sbp_lo=float(np.quantile(boot_sbp, 0.025)), sbp_hi=float(np.quantile(boot_sbp, 0.975)),
                dbp_lo=float(np.quantile(boot_dbp, 0.025)), dbp_hi=float(np.quantile(boot_dbp, 0.975)))


def _calutil_primary_for(mdl, calib, X, Q, subj, base):
    r_s   = _calib_get(calib, "ridge_sbp")
    r_d   = _calib_get(calib, "ridge_dbp")
    iso_s = _calib_get(calib, "iso_s")
    iso_d = _calib_get(calib, "iso_d")
    _build_subj_cache(mdl, X, Q, subj, base)
    feats = _build_ridge_feats_10(mdl, X, Q, subj, base)
    return apply_ridge_iso(feats, r_s, r_d, iso_s, iso_d)


def _calutil_run_for_model(model_name, pred, true, subj, n_ref_list=(1, 2, 3)):
    rows = []
    for n_ref in n_ref_list:
        sbp_mae, dbp_mae = _run_calibration_n_ref(pred, true, subj, n_ref)
        if sbp_mae is None:
            print(f"  {model_name} n_ref={n_ref}: insufficient data")
            continue
        ci = _bootstrap_calutil_ci(pred, true, subj, n_ref)
        rows.append({
            "Model": model_name, "n_ref": n_ref,
            "SBP MAE": round(sbp_mae, 2), "DBP MAE": round(dbp_mae, 2),
            "SBP 95% CI [lo]": round(ci["sbp_lo"], 2) if ci else None,
            "SBP 95% CI [hi]": round(ci["sbp_hi"], 2) if ci else None,
            "DBP 95% CI [lo]": round(ci["dbp_lo"], 2) if ci else None,
            "DBP 95% CI [hi]": round(ci["dbp_hi"], 2) if ci else None,
        })
        print(f"  {model_name} n_ref={n_ref}: SBP={sbp_mae:.2f}  DBP={dbp_mae:.2f}")
    return rows


def run_exp_new2_calib_utility(n_ref_list=(1, 2, 3)):
    print("\n" + "─" * 70)
    print("  c_slow Calibration Utility Test  [TEST set]")
    print("─" * 70)

    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()

    mcnhm_pred = _calutil_primary_for(model,      calib_v192, X_te, Q_te, subj_te, base_te)
    nodis_pred = _calutil_primary_for(mdl_nodis,  calib_nodis, X_te, Q_te, subj_te, base_te)

    all_rows = []
    all_rows.extend(_calutil_run_for_model("DiBP-Net", mcnhm_pred, Ybp_te, subj_te, n_ref_list))
    all_rows.extend(_calutil_run_for_model("NoDis",    nodis_pred, Ybp_te, subj_te, n_ref_list))

    df = pd.DataFrame(all_rows)
    out_csv = os.path.join(EXP_OUT_DIR, "exp_new2_calibration_utility.csv")
    df.to_csv(out_csv, index=False)

    print("\n  --- Summary ---")
    for _, r in df.iterrows():
        print(f"  {r['Model']:<10} n_ref={r['n_ref']}  SBP={r['SBP MAE']:.2f}  DBP={r['DBP MAE']:.2f}")
    print(f"\n  Saved → {out_csv}")
    print("DONE ✓")
    return df

In [ ]:
# Deployment Complexity Profiling
# FLOPs, params, CPU latency, memory — DiBP-Net vs GRU/ResNet/Transformer.
# Reports the deployment table.
# ══════════════════════════════════════════════════════════════════════════════

import csv as _csv
import time

_DEPLOY_N_WARMUP = 20
_DEPLOY_N_ITER   = 200
_DEPLOY_BATCH    = 1
_DEPLOY_N_BEATS  = ARCH.n_beats + 1
_DEPLOY_SEQ_LEN  = ARCH.seq_len


def _deploy_count_parameters(mdl):
    return sum(p.numel() for p in mdl.parameters() if p.requires_grad)


def _deploy_measure_latency_cpu(mdl, dummy_inputs, n_warmup=_DEPLOY_N_WARMUP, n_iter=_DEPLOY_N_ITER):
    cpu_model  = mdl.to("cpu").eval()
    cpu_inputs = [x.cpu() if isinstance(x, torch.Tensor) else x for x in dummy_inputs]
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = cpu_model(*cpu_inputs)
    times = []
    with torch.no_grad():
        for _ in range(n_iter):
            t0 = time.perf_counter()
            _ = cpu_model(*cpu_inputs)
            times.append((time.perf_counter() - t0) * 1000)
    mdl.to(DEVICE)
 # Median + IQR, not mean + std: OS-scheduler spikes are heavy-tailed, so a
 # single preempted iteration can dominate the mean (ResNet once logged
 # 2.92 +/- 5.81 ms). The median is what a wearable would actually see.
    times = np.asarray(times)
    med = float(np.median(times))
    iqr = float(np.percentile(times, 75) - np.percentile(times, 25))
    return med, iqr


def _deploy_measure_peak_memory_mb(mdl, dummy_inputs):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            _ = mdl(*dummy_inputs)
        return torch.cuda.max_memory_allocated() / 1024 / 1024
    param_mb = sum(p.numel() * p.element_size() for p in mdl.parameters()) / 1024 / 1024
    return param_mb  # conservative lower bound on MPS/CPU (no CUDA memory API)


def _deploy_try_flops(mdl, dummy_inputs, model_name):
    try:
        from ptflops import get_model_complexity_info

        class _Wrapper(nn.Module):
            def __init__(self, m, inputs):
                super().__init__()
                self.m = m
                self._other = tuple(t.cpu() if isinstance(t, torch.Tensor) else t for t in inputs[1:])

            def forward(self, x):
                return self.m(x, *self._other)

        wrapper = _Wrapper(mdl, dummy_inputs).eval().cpu()
        x_shape = tuple(dummy_inputs[0].shape[1:])
        macs, _ = get_model_complexity_info(wrapper, x_shape, as_strings=False,
                                            print_per_layer_stat=False, verbose=False)
        return macs / 1e9
    except Exception as e:
        print(f"  [ptflops unavailable for {model_name}: {e}] — FLOPs skipped")
        return None


def _deploy_profile_model(name, mdl, dummy_inputs):
    mdl.eval()
    print(f"\n  {name}")
    params = _deploy_count_parameters(mdl)
    print(f"    Parameters: {params:,}")
    lat_med, lat_iqr = _deploy_measure_latency_cpu(mdl, dummy_inputs)
    print(f"    CPU Latency: {lat_med:.2f} ms  (IQR {lat_iqr:.2f})")
    mem_mb = _deploy_measure_peak_memory_mb(mdl, dummy_inputs)
    print(f"    Peak Memory: {mem_mb:.1f} MB")
    gflops = _deploy_try_flops(mdl, dummy_inputs, name)
    if gflops:
        print(f"    GFLOPs: {gflops:.3f}")
    return {
        "Model": name, "Params (M)": f"{params / 1e6:.2f}",
        "GFLOPs": f"{gflops:.3f}" if gflops else "N/A",
        "CPU Lat (ms)": f"{lat_med:.2f}", "CPU Lat IQR": f"{lat_iqr:.2f}",
        "Peak Mem (MB)": f"{mem_mb:.1f}",
    }


def run_exp_new3_deployment():
    print("\n" + "─" * 70)
    print("  Deployment Complexity Profiling")
    print("─" * 70)
    print(f"  Device: {DEVICE}")

    subj_in = torch.zeros(_DEPLOY_BATCH, dtype=torch.long).to(DEVICE)
    base_in = torch.zeros(_DEPLOY_BATCH).to(DEVICE)
    q_c_in  = torch.randn(_DEPLOY_BATCH, _DEPLOY_SEQ_LEN).to(DEVICE)
    x_in    = torch.randn(_DEPLOY_BATCH, _DEPLOY_N_BEATS, _DEPLOY_SEQ_LEN).to(DEVICE)

    dummy_mcnhm = (x_in, subj_in, base_in, q_c_in, 0.0)
    dummy_bl    = (x_in, subj_in, base_in)

 # TransformerBaseline's positional-embedding table is fixed-size, sized for
 # ARCH.n_beats (10) beats, not the +1 real-window convention used by
 # DiBP-Net/GRU/ResNet (checkpoint pos_emb.weight is [81,128] = 10*8+1, not
 # 11*8+1=89) — verified against the actual trained checkpoint. Feeding it
 # 11 beats overflows the embedding table (IndexError). GRU/ResNet are
 # recurrent/convolutional and sequence-length agnostic, so they keep the
 # real 11-beat dummy for realism.
    x_in_transformer = torch.randn(_DEPLOY_BATCH, ARCH.n_beats, _DEPLOY_SEQ_LEN).to(DEVICE)
    dummy_transformer = (x_in_transformer, subj_in, base_in)

    results = [
        _deploy_profile_model("DiBP-Net (v19.2)",     model,           dummy_mcnhm),
        _deploy_profile_model("GRU Baseline",         mdl_gru,         dummy_bl),
        _deploy_profile_model("ResNet-1D Baseline",   mdl_resnet,      dummy_bl),
        _deploy_profile_model("Transformer Baseline", mdl_transformer, dummy_transformer),
        _deploy_profile_model("SupCon Baseline",      mdl_supcon,      dummy_bl),
    ]

    out_csv = os.path.join(EXP_OUT_DIR, "exp_new3_deployment_profile.csv")
    with open(out_csv, "w", newline="") as f:
        w = _csv.DictWriter(f, fieldnames=results[0].keys())
        w.writeheader()
        w.writerows(results)
    print(f"\n  Saved → {out_csv}")
    print("DONE ✓")
    return pd.DataFrame(results)

In [ ]:
# the risk-ranking analysis — Zero-shot HTN risk ranking (Table XI + CI now lives in the HTN-stage analysis)
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.metrics import roc_auc_score

def _subj_level_scores(fwd, Ybp, subj):
    """Subject-mean c_slow, R, raw uncalibrated SBP, and htn_logit + ground-truth stage."""
    subjs   = np.unique(subj)
    s_cs    = np.array([fwd["c_slow"][subj == s].mean()   for s in subjs])
    s_R     = np.array([fwd["R"][subj == s].mean()        for s in subjs])
    s_htn   = np.array([fwd["htn_logit"][subj == s].mean() if fwd["htn_logit"].ndim == 1
                         else fwd["htn_logit"][subj == s].mean(0)[-1] for s in subjs])
 # raw uncalibrated SBP from the ensemble head (bp_ens[:,0]), mmHg-scale
    s_sbp_raw = np.array([fwd["bp_ens"][subj == s, 0].mean() for s in subjs])
    s_sbp_true = np.array([Ybp[subj == s, 0].mean() for s in subjs])
    s_dbp_true = np.array([Ybp[subj == s, 1].mean() for s in subjs])
    s_st = assign_stages(s_sbp_true, s_dbp_true)
    return dict(c_slow=s_cs, R=s_R, htn_logit=s_htn, sbp_raw=s_sbp_raw, stage=s_st)


def _auc_with_ci(score, y, n_boot=1000, seed=42):
    """Subject-block bootstrap CI for AUC. Skips resamples with a single class."""
    if len(np.unique(y)) < 2:
        return float("nan"), float("nan"), float("nan")
    pt = roc_auc_score(y, score)
    rng = np.random.default_rng(seed)
    n = len(y)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yb = y[idx]
        if len(np.unique(yb)) < 2:
            continue
        boots.append(roc_auc_score(yb, score[idx]))
    if not boots:
        return pt, float("nan"), float("nan")
    return pt, float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def run_exp_p1_zeroshot_phenotyping(ext_dict, high_risk_stages=(3, 4), n_boot=1000):
    """
    Compare the zero-shot AUC of c_slow / R / htn_logit (jointly-trained head)
    against raw uncalibrated SBP in separating high-risk (HTN-1+HTN-2) from
    low-risk (Normal+Elevated) subjects across domains, without fitting anything on
    external data at all (no calibration, no threshold fitting).

    Hypothesis under test: ranking-level signals (c_slow, R) should transfer better than
    absolute-scale signals (raw SBP), because absolute scale is what domain shift
    destroys (Section VI-D), while relative structure may survive.
    """
    print("\n" + "─" * 70)
    print("  Zero-shot HTN risk ranking (c_slow / R vs raw BP)")
    print("─" * 70)

    def _binary_label(stage):
        return np.isin(stage, high_risk_stages).astype(int)

    records = []

    def _analyze(dataset_name, scores_dict):
        y = _binary_label(scores_dict["stage"])
        n_pos, n_neg = int(y.sum()), int((1 - y).sum())
        print(f"\n  [{dataset_name}]  N={len(y)}  (high-risk={n_pos}, low-risk={n_neg})")
        if n_pos < 2 or n_neg < 2:
            print("    [SKIP] insufficient subjects in one class")
            return
        for score_name, score_key, flip in [
            ("c_slow",       "c_slow",   False),
            ("R",            "R",        False),
            ("htn_logit",    "htn_logit",False),
            ("raw_SBP","sbp_raw",  False),
        ]:
            s = scores_dict[score_key]
            auc, lo, hi = _auc_with_ci(s, y, n_boot=n_boot)
            print(f"    {score_name:<14} AUC = {auc:.3f}  [{lo:.3f}, {hi:.3f}]")
            records.append(dict(Dataset=dataset_name, Score=score_name,
                                 AUC=round(auc, 3), CI_lo=round(lo, 3), CI_hi=round(hi, 3),
                                 N=len(y), N_high_risk=n_pos, N_low_risk=n_neg))

    # ── UCI reference (in-domain, sanity check — should be highest AUC) ──
    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    fwd_uci = _infer_forward(model, X_te, Q_te, subj_te, base_te)
    scores_uci = _subj_level_scores(fwd_uci, Ybp_te, subj_te)
    _analyze("UCI (test, in-domain)", scores_uci)

    # ── External datasets (true zero-shot) ────────────────────────────────
    for name, tag, d in _iter_ext_datasets(ext_dict):
        fwd_ext = _infer_forward(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])
        scores_ext = _subj_level_scores(fwd_ext, d["Ybp_eval"], d["subj_eval"])
        _analyze(f"{name} (zero-shot)", scores_ext)

    df_p1 = pd.DataFrame(records)
    csv_path = os.path.join(EXP_OUT_DIR, "exp_p1_zeroshot_phenotyping.csv")
    df_p1.to_csv(csv_path, index=False)

    # ── Headline summary: does c_slow/R degrade less than raw SBP zero-shot? ──
    print("\n  " + "─" * 66)
    print("  Headline check: AUC drop from UCI -> external mean, by score type")
    print("  " + "─" * 66)
    uci_rows = df_p1[df_p1["Dataset"].str.contains("in-domain")]
    ext_rows = df_p1[~df_p1["Dataset"].str.contains("in-domain")]
    for score_name in ["c_slow", "R", "htn_logit", "raw_SBP"]:
        uci_auc = uci_rows.loc[uci_rows["Score"] == score_name, "AUC"]
        ext_auc = ext_rows.loc[ext_rows["Score"] == score_name, "AUC"]
        if uci_auc.empty or ext_auc.empty:
            continue
        drop = float(uci_auc.iloc[0]) - float(ext_auc.mean())
        print(f"    {score_name:<14} UCI={float(uci_auc.iloc[0]):.3f}  "
              f"ext_mean={float(ext_auc.mean()):.3f}  drop={drop:+.3f}")

    print(f"\n  Saved -> {csv_path}")
    print("DONE")
    return df_p1

In [ ]:
# Non-Linear (MLP) In-Domain Calibration Head
# ══════════════════════════════════════════════════════════════════════════════

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

_MLP_FIT_FRAC = 0.3  # matches the linear refit exactly, for a fair capacity-only comparison


def _fit_mlp_calibrator(feats, sbp, dbp, seed=42):
    """Small MLP calibrator: standardize features, then a shallow MLP per
    target. Kept deliberately small (single hidden layer, strong L2) given
    the ~30%-of-external-subjects sample sizes here (typically low hundreds
    to low thousands of beats) -- a large MLP would just overfit and produce
    a misleadingly bad "non-linearity doesn't help" result.
    """
    scaler = StandardScaler().fit(feats)
    feats_s = scaler.transform(feats)
    mlp_s = MLPRegressor(hidden_layer_sizes=(32,), alpha=1.0, max_iter=2000,
                          early_stopping=True, n_iter_no_change=20,
                          random_state=seed).fit(feats_s, sbp)
    mlp_d = MLPRegressor(hidden_layer_sizes=(32,), alpha=1.0, max_iter=2000,
                          early_stopping=True, n_iter_no_change=20,
                          random_state=seed).fit(feats_s, dbp)
    return scaler, mlp_s, mlp_d


def _apply_mlp_calibrator(feats, scaler, mlp_s, mlp_d):
    feats_s = scaler.transform(feats)
    return np.stack([mlp_s.predict(feats_s), mlp_d.predict(feats_s)], axis=1).astype(np.float32)


def run_exp_p_mlp_calibration(ext_dict, calib_fit_frac=_MLP_FIT_FRAC, seed=42):
    print("\n" + "─" * 70)
    print("  Non-linear (MLP) in-domain calibration vs. linear refit")
    print("─" * 70)

    r_s0   = _calib_get(calib_proposed, "ridge_sbp")
    r_d0   = _calib_get(calib_proposed, "ridge_dbp")
    iso_s0 = _calib_get(calib_proposed, "iso_s")
    iso_d0 = _calib_get(calib_proposed, "iso_d")

    rng = np.random.RandomState(seed)  # same seed as the linear refit -> identical split
    rows = []
    for name, tag, d in _iter_ext_datasets(ext_dict):
        subj = d["subj_eval"]
        Ybp  = d["Ybp_eval"]
        uniq_subj = np.unique(subj)
        if len(uniq_subj) < 10:
            print(f"  [{name}] only {len(uniq_subj)} subjects — skip")
            continue

        rng.shuffle(uniq_subj)
        n_fit = max(5, int(len(uniq_subj) * calib_fit_frac))
        fit_subj, eval_subj = set(uniq_subj[:n_fit]), set(uniq_subj[n_fit:])
        fit_mask = np.array([s in fit_subj for s in subj])
        eval_mask_subj = np.array([s in eval_subj for s in subj])

        _build_subj_cache(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])
        feats_ext = _build_ridge_feats_10(model, d["X_eval"], d["Q_eval"],
                                           d["subj_eval"], d["base_eval"])

 # --- Zero-shot baseline (identical to the linear refit) ---
        primary_zs = apply_ridge_iso(feats_ext[eval_mask_subj], r_s0, r_d0, iso_s0, iso_d0)
        cal_zs, mask_zs = run_subject_bias_correction(
            primary_zs, Ybp[eval_mask_subj], subj[eval_mask_subj])
        if mask_zs.sum() < 10:
            print(f"  [{name}] insufficient eval beats — skip")
            continue
        mae_zs = float(np.mean(np.abs(cal_zs[mask_zs, 0] - Ybp[eval_mask_subj][mask_zs, 0])))

 # --- Linear in-domain refit (re-derived here for a same-run comparison;
 # should reproduce the linear refit's numbers up to the shared RNG state) ---
        from sklearn.linear_model import Ridge
        from sklearn.isotonic import IsotonicRegression
        r_s1 = Ridge(alpha=getattr(CALIB, "ridge_alpha", 10.0)).fit(
            feats_ext[fit_mask], Ybp[fit_mask, 0])
        r_d1 = Ridge(alpha=getattr(CALIB, "ridge_alpha", 10.0)).fit(
            feats_ext[fit_mask], Ybp[fit_mask, 1])
        iso_s1 = IsotonicRegression(out_of_bounds="clip").fit(r_s1.predict(feats_ext[fit_mask]), Ybp[fit_mask, 0])
        iso_d1 = IsotonicRegression(out_of_bounds="clip").fit(r_d1.predict(feats_ext[fit_mask]), Ybp[fit_mask, 1])
        primary_lin = apply_ridge_iso(feats_ext[eval_mask_subj], r_s1, r_d1, iso_s1, iso_d1)
        cal_lin, mask_lin = run_subject_bias_correction(
            primary_lin, Ybp[eval_mask_subj], subj[eval_mask_subj])
        mae_lin = (float(np.mean(np.abs(cal_lin[mask_lin, 0] - Ybp[eval_mask_subj][mask_lin, 0])))
                   if mask_lin.sum() else float("nan"))

 # --- Non-linear MLP in-domain refit (same fit/eval split) ---
        scaler, mlp_s, mlp_d = _fit_mlp_calibrator(
            feats_ext[fit_mask], Ybp[fit_mask, 0], Ybp[fit_mask, 1], seed=seed)
        primary_mlp = _apply_mlp_calibrator(feats_ext[eval_mask_subj], scaler, mlp_s, mlp_d)
        cal_mlp, mask_mlp = run_subject_bias_correction(
            primary_mlp, Ybp[eval_mask_subj], subj[eval_mask_subj])
        mae_mlp = (float(np.mean(np.abs(cal_mlp[mask_mlp, 0] - Ybp[eval_mask_subj][mask_mlp, 0])))
                   if mask_mlp.sum() else float("nan"))

        print(f"  [{name}] n_fit_subj={n_fit}/{len(uniq_subj)}  SBP Post MAE  "
              f"ZeroShot: {mae_zs:6.2f}   LinearRefit: {mae_lin:6.2f}   MLP: {mae_mlp:6.2f}")

        rows.append({
            "Dataset": name, "n_subj_total": len(uniq_subj), "n_subj_fit": n_fit,
            "n_eval": int(mask_zs.sum()),
            "ZeroShot_Post": round(mae_zs, 3),
            "LinearRefit_Post": round(mae_lin, 3),
            "MLP_Post": round(mae_mlp, 3),
            "MLP_minus_Linear": round(mae_mlp - mae_lin, 3),
        })

    df = pd.DataFrame(rows)
    if len(df):
        print(f"\n  Mean SBP Post MAE across {len(df)} datasets:")
        print(f"    Zero-shot:      {df['ZeroShot_Post'].mean():.2f} ± {df['ZeroShot_Post'].std():.2f}")
        print(f"    Linear refit:   {df['LinearRefit_Post'].mean():.2f} ± {df['LinearRefit_Post'].std():.2f}")
        print(f"    MLP refit:      {df['MLP_Post'].mean():.2f} ± {df['MLP_Post'].std():.2f}")
        delta = df["MLP_minus_Linear"].mean()
        verdict = ("MLP meaningfully beats linear refit — the pred→true mapping IS "
                   "non-linear and Ridge lacked capacity, not just labels" if delta < -0.5 else
                   "MLP roughly matches linear refit — capacity is not the bottleneck; "
                   "the 30%-label ceiling itself (not model class) explains the linear refit's small gain"
                   if abs(delta) <= 0.5 else
                   "MLP is WORSE than linear refit — likely overfitting the small "
                   "in-domain fit set despite regularization")
        print(f"  → Mean MLP−Linear Δ = {delta:+.2f} mmHg → {verdict}")

    out_csv = os.path.join(EXP_OUT_DIR, "exp_p_mlp_calibration.csv")
    df.to_csv(out_csv, index=False)
    print(f"\n  Saved → {out_csv}")
    print("DONE ✓")
    return df

In [ ]:
# Multi-seed robustness of Table III 
# ─────────────────────────────────────────────────────────────────────────────
# Retrains DiBP-Net from N independent seeds and reports mean ± std of the
# main-table metrics (Test SBP/DBP MAE at Raw/Pre/Post + HTN-AUC). Uses the SAME
# train → fit-calib → evaluate pipeline as the master table , so the per-seed
# numbers are directly comparable to Table III. Checkpoints go to
# v192_seed{seed}_best — the canonical v192_best is never overwritten.
#
# COST: each seed = one full DiBP-Net two-phase training. Run manually:
# df_ms, summ_ms = run_exp_multiseed() # default 5 seeds
# df_ms, summ_ms = run_exp_multiseed(seeds=[42, 0, 1]) # cheaper subset

MULTISEED_SEEDS = [42, 0, 1, 2, 3]     # 42 == the canonical v192 seed

def _train_seed_model(seed: int, force_retrain: bool = False):
    """Fresh DiBP-Net trained under `seed`; returns (mdl, calib). Isolated to
    checkpoint tag v192_seed{seed} so v192_best is untouched. Resumable: reloads
    an existing seed checkpoint unless force_retrain."""
    global SPLITS
    if SPLITS is None: _load_data()
    tag = f"v192_seed{seed}"
    seed_everything(seed)
    if not force_retrain and _ckpt_exists(f"{tag}_best"):
        mdl, _ = load_checkpoint(f"{tag}_best")
        mdl._set_stats_from_splits(SPLITS)
        print(f"  [seed {seed}] loaded existing {tag}_best")
    else:
        mdl = DiBPNet().to(DEVICE); mdl._set_stats_from_splits(SPLITS)
        mdl, _ = _two_phase_train(mdl, tag, _empty_hist(), print_prefix=f"[seed{seed}] ")
    mdl.eval()
    calib = _fit_calib_on_val(mdl)          # Ridge+Iso on VAL, same as _load_mcnhm
    return mdl, calib


def run_exp_multiseed(seeds=None, force_retrain=False, n_boot=1000):
    """Train DiBP-Net over `seeds`, evaluate each with the master pipeline, and
    aggregate Table-III metrics to mean ± std. Writes multiseed_table3.csv."""
    seeds = list(seeds if seeds is not None else MULTISEED_SEEDS)
    _COLS = [("AUC",      "AUC (HTN)"),
             ("SBP_Raw",  "Beat SBP (Raw)"),  ("DBP_Raw",  "Beat DBP (Raw)"),
             ("SBP_Pre",  "Beat SBP (Pre)"),  ("DBP_Pre",  "Beat DBP (Pre)"),
             ("SBP_Post", "Beat SBP (Post)"), ("DBP_Post", "Beat DBP (Post)")]
    recs = []
    for sd in seeds:
        print(f"\n{'#'*70}\n#  SEED {sd}\n{'#'*70}", flush=True)
        mdl, calib = _train_seed_model(sd, force_retrain=force_retrain)
        res  = run_full_evaluation(mdl, calib, is_baseline=False, n_boot=n_boot)
        mrow = pd.DataFrame(_build_master_rows("DiBP-Net", res))
        test = mrow[mrow["Split"] == "Test"].iloc[0]
        rec  = {"seed": sd, **{k: float(test[col]) for k, col in _COLS}}
        recs.append(rec)
        print(f"  → seed {sd}: SBP Post={rec['SBP_Post']:.2f}  "
              f"DBP Post={rec['DBP_Post']:.2f}  AUC={rec['AUC']:.4f}", flush=True)

    df  = pd.DataFrame(recs)
    out = os.path.join(OUT_DIR, "multiseed_table3.csv"); df.to_csv(out, index=False)

    metrics = [k for k, _ in _COLS]
    summ = df[metrics].agg(["mean", "std"]).T
    summ["mean_pm_std"] = [f"{m:.3f} ± {s:.3f}" if k == "AUC" else f"{m:.2f} ± {s:.2f}"
                           for k, (m, s) in zip(summ.index, summ[["mean", "std"]].values)]

    print(f"\n  ✅ per-seed CSV → {out}")
    print(f"\n  ── Table III, N={len(df)} seeds {seeds} (mean ± std) ──")
    print(summ[["mean", "std", "mean_pm_std"]].to_string())
    print("\n  LaTeX-ready (Test): "
          f"SBP Post {summ.loc['SBP_Post','mean_pm_std']}, "
          f"DBP Post {summ.loc['DBP_Post','mean_pm_std']}, "
          f"AUC {summ.loc['AUC','mean_pm_std']}")
    return df, summ

In [ ]:
# section F FIGURE GENERATION (decoupled plotting; reads _FIG_DATA)
# ==============================================================================
def plot_exp_k(fd=None):
    fd = fd if fd is not None else _FIG_DATA.get("exp_k")
    if not fd: 
        print("  [plot_exp_k] no data")
        return
    
    df_gap, uci_post_sbp = fd["df_gap"], fd["uci_post_sbp"]
    
    clean_labels = df_gap["dataset"].replace({
        "UCI-TEST": "UCI",
        "PulseDB-EXTERNAL": "PulseDB",
        "Multi-Sensor PPG-EXTERNAL": "MS-PPG",
        "BCG-EXTERNAL": "BCG"
    })

    with plt.rc_context(_IEEE_RC):
        fig, ax = plt.subplots(figsize=(3.5, 2.75))
        x = np.arange(len(df_gap))
        w = 0.35
        
        ax.bar(x - w/2, df_gap["pre_sbp_mae"],  w, color="#90A4AE", alpha=0.85, label="GCal")
        ax.bar(x + w/2, df_gap["post_sbp_mae"], w, color="#1565C0", alpha=0.9,  label="SCal")
        
 # Plot Reference Line
        ax.axhline(uci_post_sbp, ls="--", color="#C62828", alpha=0.7, label="UCI SCal (ref)")
        
        ax.set_xticks(x)
        ax.set_xticklabels(clean_labels, rotation=0, fontsize=8) 
        
        ax.set_ylabel("SBP MAE (mmHg)", fontsize=8)
        ax.set_title("Domain-Gap: Zero-Shot SBP MAE", fontsize=8, fontweight="bold")
        
        max_val = max(df_gap["pre_sbp_mae"].max(), df_gap["post_sbp_mae"].max())
        ax.set_ylim(0, max_val * 1.35)
        
        ax.legend(loc="upper left", fontsize=8, framealpha=0.95)
        ax.grid(axis="y", alpha=0.3)
        ax.tick_params(axis="y", labelsize=8)
        
        fig.tight_layout(pad=0.4)
        
        out_path = os.path.join(EXP_OUT_DIR, "exp_k_domain_gap_diagnostic.png")
        fig.savefig(out_path, dpi=400, bbox_inches="tight", pad_inches=0.05)
        plt.close(fig)
        
    print(f"  [plot_exp_k] saved {out_path}")


def plot_exp_i(fd=None):
    fd = fd if fd is not None else _FIG_DATA.get("exp_i")
    if not fd: print("  [plot_exp_i] no data"); return
    box_R, box_C = fd["box_R"], fd["box_C"]

    with plt.rc_context(_IEEE_RC):
        fig, (ax_r, ax_c) = plt.subplots(2, 1, figsize=(3.5, 5.25))

        for ax, box, col, name in [(ax_r, box_R, "#3498DB", "Resistance ($R$)"),
                                    (ax_c, box_C, "#E74C3C", "Compliance ($C$)")]:
            ax.boxplot(list(box.values()), labels=list(box.keys()), patch_artist=True,
                       boxprops=dict(facecolor=col, alpha=0.6))
            ax.set_title(name, fontsize=8, fontweight="bold")
            ax.set_ylabel("Normalized Value", fontsize=8)
            ax.tick_params(axis="x", rotation=30, labelsize=8)
            ax.tick_params(axis="y", labelsize=8)
            ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)

        fig.suptitle("Distributions of Latent Windkessel Parameters", fontsize=8, fontweight="bold")
        
        fig.tight_layout(pad=0.5, h_pad=2.0)
        
        fig.savefig(os.path.join(EXP_OUT_DIR, "exp_i_physics_boxplot_combined.png"), dpi=400,
                    bbox_inches="tight", pad_inches=0.05)
        plt.close(fig)
    print("  [plot_exp_i] saved exp_i_physics_boxplot_combined.png")


def plot_ablation_icc(fd=None):
    fd = fd if fd is not None else _FIG_DATA.get("ablation_icc")
    if not fd: print("  [plot_ablation_icc] no data"); return
    
    cs_prop, cs_ablA = fd["cs_prop"], fd["cs_ablA"]
    stages_te, records_icc = fd["stages_te"], fd["records_icc"]
    
    with plt.rc_context(_IEEE_RC):
        fig = plt.figure(figsize=(3.5, 4.5))
        
        import matplotlib.gridspec as gridspec
        gs = gridspec.GridSpec(2, 2, figure=fig)
        
        ax0 = fig.add_subplot(gs[0, 0])
        ax1 = fig.add_subplot(gs[0, 1], sharey=ax0)
        ax2 = fig.add_subplot(gs[1, :])
        
        _hist = [("Proposed", cs_prop, ax0),
                 ("Ablation A", cs_ablA, ax1)]
        
        for idx, (title, cs, ax) in enumerate(_hist):
            for sid in [1, 2, 3, 4]:
                m = stages_te == sid
                if m.sum():
                    ax.hist(cs[m], bins=30, alpha=0.6, label=STAGE_LABELS[sid],
                            color=STAGE_COLORS[sid], density=True)
            ax.set_title(title, fontsize=8, fontweight="bold")
            ax.set_xlabel(r"$c_{\text{slow}}$", fontsize=8) 
            
            if idx == 0:
                ax.set_ylabel("Probability Density", fontsize=8)
            else:
                ax.tick_params(labelleft=False)
                
            ax.legend(fontsize=6, loc="upper right", framealpha=0.8, 
                      handlelength=0.8, handletextpad=0.3)
            ax.tick_params(labelsize=8)
            ax.margins(y=0.05)

        for i, r in enumerate(records_icc):
            ax2.barh(i, r["icc"],
                     xerr=[[max(r["icc"] - r["lo"], 0.0)], [max(r["hi"] - r["icc"], 0.0)]],
                     color=["#E74C3C", "#3498DB", "#27AE60"][i], alpha=0.8, capsize=4,
                     error_kw={"elinewidth": 1.5})
            ax2.text(r["hi"] + 0.005, i, f"{r['icc']:.4f}", va="center", fontsize=8)
            
        ax2.set_yticks(range(3))
        clean_ylabels = [r["var"].replace("Ablation ", "Abl ") for r in records_icc]
        ax2.set_yticklabels(clean_ylabels, fontsize=8)
        
        ax2.axvline(0.75, ls=":", color="gray", alpha=0.6, label="Good")
        ax2.axvline(0.90, ls="--", color="gray", alpha=0.6, label="Excellent")
        ax2.set_xlabel("ICC (1,1)", fontsize=8)
        ax2.set_title("Intraclass Correlation Coefficient", fontsize=8, fontweight="bold")
        
        ax2.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.28), ncol=2, framealpha=0.9)
        ax2.grid(axis="x", alpha=0.3)
        ax2.tick_params(labelsize=8)
        
        for artist in fig.findobj(plt.Text):
            try:
                artist.set_fontsize(8)
                artist.set_fontname("Times New Roman")
            except Exception:
                pass
        
        fig.tight_layout(pad=0.4, h_pad=1.5, w_pad=0.5)
        
        out_path = os.path.join(EXP_OUT_DIR, "exp_g_ablation_icc_combined.png")
        fig.savefig(out_path, dpi=400, bbox_inches="tight", pad_inches=0.05)
        plt.close(fig)
        
    print(f"  [plot_ablation_icc] saved {out_path}")

def plot_exp_b_drift(fd=None):
    fd = fd if fd is not None else _FIG_DATA.get("exp_b_drift")
    if not fd: print("  [plot_exp_b_drift] no data"); return
    df_b = fd["df_b"]
    with plt.rc_context(_IEEE_RC):
        x = list(range(len(df_b)))
        fig, ax = plt.subplots(figsize=(3.5, 2.25))
        ax.plot(x, df_b["sbp_mae"], "o-", color="#E74C3C", lw=2, label="SBP SCal")
        ax.plot(x, df_b["dbp_mae"], "s--", color="#3498DB", lw=2, label="DBP SCal")
        ax.axhline(5.0, ls=":", color="gray", alpha=0.6)
        ax.set_xticks(x); ax.set_xticklabels(df_b["beat_range"], rotation=0, ha="center", fontsize=8)
        ax.set_ylabel("SCal MAE (mmHg)", fontsize=8)
        ax.set_title("SCal MAE Temporal Drift", fontsize=8, fontweight="bold")
        ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3); ax.tick_params(labelsize=8)
        fig.tight_layout(pad=0.4)
        fig.savefig(os.path.join(EXP_OUT_DIR, "exp_b_drift_mae.png"), dpi=400,
                    bbox_inches="tight", pad_inches=0.05)
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(3.5, 2.25))
        ax.bar(x, df_b["n_beats"], color="#7F8C8D", alpha=0.75)
        ax.set_xticks(x); ax.set_xticklabels(df_b["beat_range"], rotation=0, ha="center", fontsize=8)
        ax.set_ylabel("Number of Windows", fontsize=8)
        ax.set_title("Evaluation Window Count per Bin", fontsize=8, fontweight="bold")
        ax.tick_params(labelsize=8); fig.tight_layout(pad=0.4)
        fig.savefig(os.path.join(EXP_OUT_DIR, "exp_b_drift_nbeats.png"), dpi=400,
                    bbox_inches="tight", pad_inches=0.05)
        plt.close(fig)
    print("  [plot_exp_b_drift] saved exp_b_drift_mae.png, exp_b_drift_nbeats.png")


def plot_exp9_umap(fd=None):
    fd = fd if fd is not None else _FIG_DATA.get("exp9_umap")
    if not fd: print("  [plot_exp9_umap] no data"); return
    markers, colors, domain_names = fd["markers"], fd["colors"], fd["domain_names"]
    domain, outlier_mask, Z2 = fd["domain"], fd["outlier_mask"], fd["Z2"]
    n_outliers, stage_all = fd["n_outliers"], fd["stage_all"]

    with plt.rc_context(_IEEE_RC):
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(3.5, 5.25))

        # ── Panel 1: Domain ──────────────────────────────────────────────
        clean_domain_names = [d.replace("-EXTERNAL", "").replace("Multi-Sensor PPG", "MS-PPG") for d in domain_names]
        
        for d_idx, (m, c, lbl) in enumerate(zip(markers, colors, clean_domain_names)):
            mask = (domain == d_idx) & ~outlier_mask
            ax1.scatter(Z2[mask, 0], Z2[mask, 1], s=6, alpha=0.5, marker=m,
                        color=c, label=lbl, linewidths=0, rasterized=True)
        if n_outliers > 0:
            ax1.scatter(Z2[outlier_mask, 0], Z2[outlier_mask, 1], s=10, alpha=0.8, marker="x",
                        color="#555", label=f"Outliers (n={n_outliers})", linewidths=0.7, rasterized=True)
        
        ax1.set_title(r"$z_{\mathrm{hemo}}$ Latent Space by Dataset", fontweight="bold", fontsize=8)
        ax1.set_xticks([]); ax1.set_yticks([])
        ax1.set_xlabel("UMAP Dimension 1", fontsize=8)
        ax1.set_ylabel("UMAP Dimension 2", fontsize=8)
        
        ax1.legend(fontsize=8, markerscale=1.5, loc="upper center", bbox_to_anchor=(0.5, -0.1),
                   ncol=2, handletextpad=0.3, columnspacing=0.9, framealpha=0.92, edgecolor="#ccc")

        # ── Panel 2: HTN Stage ───────────────────────────────────────────
        stage_labels = [
            "Normal (<120 mmHg)", 
            "Elevated (120-129 mmHg)",
            "HTN-1 (130-139 mmHg)", 
            r"HTN-2 ($\geq$140 mmHg)"
        ]
        for s_idx, (c, lbl) in enumerate(zip(["#2196F3", "#4CAF50", "#FF9800", "#F44336"], stage_labels)):
            mask = (stage_all == s_idx) & ~outlier_mask
            ax2.scatter(Z2[mask, 0], Z2[mask, 1], s=6, alpha=0.45, marker="o",
                        color=c, label=lbl, linewidths=0, rasterized=True)
                        
        ax2.set_title(r"$z_{\mathrm{hemo}}$ Latent Space by Hypertension Stage", fontweight="bold", fontsize=8)
        ax2.set_xticks([]); ax2.set_yticks([])
        ax2.set_xlabel("UMAP Dimension 1", fontsize=8)
        ax2.set_ylabel("UMAP Dimension 2", fontsize=8)
        
        ax2.legend(fontsize=8, markerscale=1.5, loc="upper center", bbox_to_anchor=(0.5, -0.1),
                   ncol=2, handletextpad=0.3, columnspacing=0.9, framealpha=0.92, edgecolor="#ccc")

        fig.subplots_adjust(left=0.1, right=0.95, top=0.92, bottom=0.16, hspace=0.7)
        
        fig.savefig(os.path.join(EXP_OUT_DIR, "latent_umap_combined.png"), dpi=400,
                    bbox_inches="tight", pad_inches=0.05)
        plt.close(fig)
        
    print("  [plot_exp9_umap] saved latent_umap_combined.png")

def plot_exp_e_forest(fd=None):
    fd = fd if fd is not None else _FIG_DATA.get("exp_e_forest")
    if not fd: print("  [plot_exp_e_forest] no data"); return
    records_e, n_pdb = fd["records_e"], fd["n_pdb"]
    
    with plt.rc_context(_IEEE_RC):
        fig_h = min(6.5, max(2.2, len(records_e) * 0.35))
        fig_f, ax_f = plt.subplots(figsize=(3.5, fig_h))
        y_pos = list(range(len(records_e)))[::-1]
        
        def clean_lbl(label_str):
            if "zero-order" in label_str:
                if "SBP" in label_str: return "SBP (Unadjusted)"
                if "DBP" in label_str: return "DBP (Unadjusted)"
                return "Age (Unadjusted)"
            else:
                lbl_no_space = label_str.replace(" ", "")
                if "SBP|" in lbl_no_space: return "SBP (Adjusted for Age)"
                if "DBP|" in lbl_no_space: return "DBP (Adjusted for Age)"
                return "Age (Adjusted for BP)"
        
        clean_yticklabels = [clean_lbl(r["label"]) for r in records_e]
        
        max_hi = max([fisher_ci(row["partial_r"], n_pdb)[1] for row in records_e])
        text_x = max(0.5, max_hi + 0.08)
        
        for i, row in enumerate(records_e):
            r = row["partial_r"]
            lo_f, hi_f = fisher_ci(r, n_pdb)
            
            is_main = "Adjusted for BP" in clean_yticklabels[i]
            m_color = "#C62828" if is_main else "#1565C0"
            
            ax_f.errorbar(r, y_pos[i], xerr=[[r - lo_f], [hi_f - r]], fmt="s",
                          color=m_color, capsize=4, markersize=5, lw=1.5)
            
            ci_text = f"{r:.3f} [{lo_f:.2f}, {hi_f:.2f}]"
            ax_f.text(text_x, y_pos[i], ci_text, va="center", fontsize=8)
        
        ax_f.axvline(0, color="#333333", lw=1.0, ls="--", alpha=0.6)
        
        ax_f.set_xlim(-0.05, text_x + 0.35)
        ax_f.set_yticks(y_pos)
        ax_f.set_yticklabels(clean_yticklabels, fontsize=8)
        
        ax_f.set_xlabel(r"Correlation $r$ (95% CI)", fontsize=8)
        
        ax_f.spines["right"].set_visible(False)
        ax_f.spines["top"].set_visible(False)
        ax_f.spines["left"].set_visible(False)
        ax_f.tick_params(axis="x", labelsize=8) 
        ax_f.tick_params(axis="y", left=False) 
        
        ax_f.grid(axis="x", alpha=0.3, ls=":")
        
        fig_f.suptitle(r"Zero-order and Partial Correlations (PulseDB)", 
                       fontsize=8, fontweight="bold", fontname="Times New Roman", x=0.5, y=1.0)
        
        for artist in fig_f.findobj(plt.Text):
            try:
                artist.set_fontsize(8)
                artist.set_fontname("Times New Roman")
            except Exception:
                pass
        
        fig_f.tight_layout(pad=0.4, rect=[0, 0, 1, 0.94])
        
        fig_f.savefig(os.path.join(EXP_OUT_DIR, "exp_e_partial_corr_forest.png"), dpi=400,
                      bbox_inches="tight", pad_inches=0.05)
        plt.close(fig_f)
    print("  [plot_exp_e_forest] saved exp_e_partial_corr_forest.png")
    
# Registry: (payload key in _FIG_DATA, plot function). Add new EXPs here.
_FIG_PLOTTERS = [
    ("exp_k",        plot_exp_k),
    ("exp_i",        plot_exp_i),
    ("ablation_icc", plot_ablation_icc),
    ("exp_b_drift",  plot_exp_b_drift),
    ("exp9_umap",    plot_exp9_umap),
    ("exp_e_forest", plot_exp_e_forest),
]

def regenerate_all_figures():
    """Regenerate every decoupled figure whose EXP payload is present in _FIG_DATA."""
    done = 0
    for name, fn in _FIG_PLOTTERS:
        if name in _FIG_DATA:
            fn(); done += 1
        else:
            print(f"  [skip] {name}: no payload (run its EXP first)")
    print(f"Figure generation complete ({done}/{len(_FIG_PLOTTERS)} plotted).")

## Run Experiments

### Temporal Stability and Physics Statistics

In [ ]:
df_i = run_exp_i_physics_consistency(ext_dict)

### HTN-Stage Analysis

How well R, C, tau, and c_slow separate the hypertension stages.

In [ ]:
run_exp_j_comprehensive_stage_analysis(ext_dict);

### Domain-Gap Diagnostic

Why external MAE stays about three times UCI's: unifies pre/post-calibration MAE, the R/C/tau shift, and c_slow-SBP consistency per dataset.

In [ ]:
df_k = run_exp_k_domain_gap(ext_dict, exp2_result=r2_all, exp2x_result=locals().get("r2x"), df_i=df_i, df_h=df_h)

### Robust Calibration Candidates

Baseline vs per-feature moment-matching (with and without raw C) vs a larger subject-bias reference window.

In [ ]:
df_l = run_exp_l_robust_calibration(ext_dict)

### Output-Distribution Quantile Mapping

A non-linear output-side calibration candidate: zero-shot and in-domain quantile mapping of the predicted blood pressure.

In [ ]:
df_n = run_exp_n_quantile_map(ext_dict)

### c_slow Calibration Utility

Whether adding c_slow / physics features to the calibrator changes accuracy.

In [ ]:
df_new2 = run_exp_new2_calib_utility()

### Deployment Complexity

Parameters, GFLOPs, CPU latency, and peak memory: DiBP-Net vs the GRU, ResNet-1D, and Transformer baselines.

In [ ]:
df_new3 = run_exp_new3_deployment()

In [ ]:
model = model.to(DEVICE)
df_p1 = run_exp_p1_zeroshot_phenotyping(ext_dict)

In [ ]:
df_p = run_exp_p_mlp_calibration(ext_dict)

### Multi-Seed Robustness

Retrains DiBP-Net from several seeds and reports the mean and standard deviation of the main-table metrics. Heavy; invoke manually.

In [ ]:
df_ms, summ_ms = run_exp_multiseed()

## Figure Generation

Plotting code separated from the experiment logic: each experiment stores its plot payload, and the plotting functions regenerate the figures. Run the experiments first.

In [ ]:
regenerate_all_figures()

## Additional Analyses

Lightweight analyses that reuse in-memory results, with no retraining:

- Demographics of the >=11-beat drift cohort per external dataset.
- Within-session drift versus session length, with a power analysis.

In [ ]:
# The demographics analysis : Demographics of the ≥11-beat periodic-calibration cohort
# ------------------------------------------------------------------------------
# Cohort = subjects with ≥11 beats — the SAME criterion used by
# test_periodic_calibration (long enough for within-session drift to exist).
# Per external dataset we report: cohort size vs total, per-subject BP
# distribution, and age where available.
# • Age: PulseDB only (phase-1 external npz carry no demographics).
# • Sex: dropped — gender_eval is all-missing (code -1) in this build.
# • Hospital PPG yields a ZERO cohort (no subject reaches 11 beats): this is
# the quantitative reason it is already excluded from External Validation
# and Table tab:aami_bhs, while still appearing in tab:crossdomain_wk
# (whose correlation / Cohen's-d metrics need no per-subject calibration).
# ══════════════════════════════════════════════════════════════════════════════
def _drift_datasets_iter():
    out = []
    if 'pdb' in globals() and pdb is not None and len(pdb.get("subj_eval", [])) > 0:
        out.append((pdb.get("name", "PulseDB"), pdb))
    if 'ext_dict' in globals() and ext_dict is not None:
        for tag in ("sensors", "bcg", "hospital"):
            d = ext_dict.get(tag)
            if d and len(d.get("subj_eval", [])) > 0:
                out.append((d["name"], d))
    return out

def build_drift_cohort_demographics(min_beats=11):
    rows = []
    for name, d in _drift_datasets_iter():
        ybp  = np.asarray(d["Ybp_eval"], dtype=float)
        subj = np.asarray(d["subj_eval"])
        age  = np.asarray(d["age_eval"]) if d.get("age_eval") is not None else None
        uniq = np.unique(subj)
        cohort = [s for s in uniq if np.sum(subj == s) >= min_beats]
        sbp, dbp, ages = [], [], []
        for s in cohort:
            idx = np.where(subj == s)[0]
            sbp.append(np.median(ybp[idx, 0]))
            dbp.append(np.median(ybp[idx, 1]))
            if age is not None and np.isfinite(age[idx[0]]):
                ages.append(float(age[idx[0]]))
        def ms(a):
            a = np.asarray(a, float)
            return (a.mean(), a.std()) if a.size else (np.nan, np.nan)
        def fmt(a):                       # "—" when the cohort is empty
            m, s_ = ms(a)
            return f"{m:.1f}±{s_:.1f}" if np.isfinite(m) else "—"
        rows.append(dict(
            Dataset=name, N_total=len(uniq), N_cohort=len(cohort),
            Cohort_pct=round(100 * len(cohort) / max(len(uniq), 1), 1),
            SBP_mmHg=fmt(sbp), DBP_mmHg=fmt(dbp),
            Age_yr=(fmt(ages) if ages else "N/A")))
    return pd.DataFrame(rows)

df_drift_demo = build_drift_cohort_demographics(min_beats=11)
print("=" * 78)
print("  Demographics of the ≥11-beat drift cohort (per external dataset)")
print("=" * 78)
print(df_drift_demo.to_string(index=False))
print("\n  Notes:")
print("   • BP = per-subject median over that subject's beats; mean±std across cohort.")
print("   • Age: PulseDB only — phase-1 external npz carry no demographics.")
print("     Sex omitted: the source gender field is all-missing in this build.")
print("   • Hospital PPG: cohort = 0 (no subject reaches 11 beats) — the reason it")
print("     is excluded from External Validation / AAMI-BHS, but retained in the")
print("     cross-domain Windkessel table (no per-subject calibration needed there).")
import os as _os
_out = "outputs/phase_2_v192/reviewer_R2_drift_cohort_demographics.csv"
_os.makedirs(_os.path.dirname(_out), exist_ok=True)
df_drift_demo.to_csv(_out, index=False)
print(f"\n  ✅ saved → {_out}")


In [ ]:
# the selection-bias check : INCLUDED vs EXCLUDED comparison
# ------------------------------------------------------------------------------
# The ≥11-beat cohort carries the entire drift-recovery result, but coverage is
# very uneven: PulseDB 99.1%, Multi-Sensor 95.4%, BCG 24.2%, Hospital 0%.
# If the subjects who happen to have ≥11 consecutive beats differ systematically
# from those who do not (e.g. calmer, more stable BP), the 31–39% recovery is
# confounded by selection. BCG is the stress case: 134 of 553 subjects used.
#
# For every dataset we therefore compare the INCLUDED (≥11 beats) against the
# EXCLUDED (<11 beats) subjects on per-subject SBP/DBP (and age where present),
# with Welch's t-test + Hedges' g. This is a falsification test: a null result
# defends the paper, a significant one must be reported as a limitation.
# ══════════════════════════════════════════════════════════════════════════════
from scipy.stats import ttest_ind, mannwhitneyu

def _hedges_g(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return np.nan
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    if sp == 0:
        return np.nan
    d = (a.mean() - b.mean()) / sp
    J = 1 - 3 / (4 * (na + nb) - 9)          # small-sample correction
    return d * J

def compare_included_excluded(min_beats=11):
    rows = []
    for name, d in _drift_datasets_iter():
        ybp  = np.asarray(d["Ybp_eval"], dtype=float)
        subj = np.asarray(d["subj_eval"])
        age  = np.asarray(d["age_eval"]) if d.get("age_eval") is not None else None

        inc, exc = {"sbp": [], "dbp": [], "age": [], "nb": []}, {"sbp": [], "dbp": [], "age": [], "nb": []}
        for s in np.unique(subj):
            idx = np.where(subj == s)[0]
            tgt = inc if len(idx) >= min_beats else exc
            tgt["sbp"].append(np.median(ybp[idx, 0]))
            tgt["dbp"].append(np.median(ybp[idx, 1]))
            tgt["nb"].append(len(idx))
            if age is not None and np.isfinite(age[idx[0]]):
                tgt["age"].append(float(age[idx[0]]))

        n_i, n_e = len(inc["sbp"]), len(exc["sbp"])
        row = {"Dataset": name, "N_incl": n_i, "N_excl": n_e,
               "Incl_%": round(100 * n_i / max(n_i + n_e, 1), 1)}

        for metric in ("sbp", "dbp", "age"):
            a, b = inc[metric], exc[metric]
            lab = metric.upper() if metric != "age" else "Age"
            if len(a) < 2 or len(b) < 2:
                row[f"{lab}_incl"] = f"{np.mean(a):.1f}" if a else "—"
                row[f"{lab}_excl"] = f"{np.mean(b):.1f}" if b else "—"
                row[f"{lab}_p"] = "—"; row[f"{lab}_g"] = "—"
                continue
            t, p = ttest_ind(a, b, equal_var=False)          # Welch
            g = _hedges_g(a, b)
            row[f"{lab}_incl"] = f"{np.mean(a):.1f}±{np.std(a):.1f}"
            row[f"{lab}_excl"] = f"{np.mean(b):.1f}±{np.std(b):.1f}"
            row[f"{lab}_p"] = f"{p:.3g}"
            row[f"{lab}_g"] = f"{g:+.2f}"
        rows.append(row)
    return pd.DataFrame(rows)

df_sel = compare_included_excluded(11)
print("=" * 96)
print("  Selection-bias check: INCLUDED (≥11 beats) vs EXCLUDED (<11 beats)")
print("=" * 96)
cols = ["Dataset", "N_incl", "N_excl", "Incl_%",
        "SBP_incl", "SBP_excl", "SBP_p", "SBP_g",
        "DBP_incl", "DBP_excl", "DBP_p", "DBP_g",
        "Age_incl", "Age_excl", "Age_p", "Age_g"]
cols = [c for c in cols if c in df_sel.columns]
print(df_sel[cols].to_string(index=False))

print("\n  Interpretation (|g|: <0.2 negligible, 0.2–0.5 small, >0.5 moderate+):")
for _, r in df_sel.iterrows():
    flags = []
    for lab in ("SBP", "DBP", "Age"):
        gv, pv = r.get(f"{lab}_g", "—"), r.get(f"{lab}_p", "—")
        if gv in ("—", None) or pv in ("—", None):
            continue
        try:
            g, p = abs(float(gv)), float(pv)
        except ValueError:
            continue
        if p < 0.05 and g >= 0.2:
            flags.append(f"{lab} (p={p:.3g}, g={float(gv):+.2f})")
    if r["N_excl"] == 0:
        verdict = "no excluded subjects — nothing to compare"
    elif r["N_incl"] == 0:
        verdict = "⚠️  ALL subjects excluded — dataset cannot support the analysis"
    elif flags:
        verdict = "🚨 SELECTION BIAS: " + "; ".join(flags)
    else:
        verdict = "✅ no meaningful difference (selection bias not supported)"
    print(f"   • {r['Dataset']:<18} incl {r['Incl_%']:>5}%  →  {verdict}")

_out = "outputs/phase_2_v192/reviewer_R2b_selection_bias.csv"
df_sel.to_csv(_out, index=False)
print(f"\n  ✅ saved → {_out}")
print("  NOTE: BCG is the stress case (lowest coverage). A null result there is what")
print("        defends the pooled 31–39% recovery claim; a positive one must be")
print("        reported as a limitation, not omitted.")


In [ ]:
# the stratified check : Drift recovery stratified by BP stage
# ------------------------------------------------------------------------------
# the selection-bias check found selection bias (≥11-beat cohort skews toward lower BP: BCG
# SBP p=0.006 g=-0.22, Multi-Sensor DBP p=0.010 g=-0.38). This does NOT yet
# tell us whether the pooled 31–39% drift-recovery claim (static→rolling)
# still holds within the higher-BP strata that are under-represented — i.e.
# the clinically relevant HTN groups. That is what this cell checks.
#
# Method: reuse the exact static/rolling residual computation from
# test_periodic_calibration (canonical rb=CALIB.ref_beats, gap=CALIB.ols_gap,
# anti-leak), but tag every subject by their AHA stage (Table I cutoffs,
# based on their own median observed SBP — descriptive only, not used in
# fitting) and report recovery separately per stage.
#
# Three views:
# (a) Pooled-by-stage — beat-pooled MAE/drop across all external
# datasets, directly comparable to the headline 31–39% figure.
# CAUTION: confounded with dataset composition per stage — see (c).
# (b) Per-dataset × stage — subject counts, shows which cells are thin
# (esp. BCG, N_incl=134; BCG has zero HTN-2 subjects).
# (c) WITHIN-dataset × stage — controls for the fact that each stage is
# dominated by a different dataset (Normal≈PulseDB 79%, HTN-2≈
# Multi-Sensor 78%), so any "recovery increases with stage" pattern
# in (a) could just be "recovery differs by dataset" (Simpson's
# paradox). This is the view that actually answers the question.
# ══════════════════════════════════════════════════════════════════════════════
from scipy.stats import ttest_rel as _ttest_rel2, pearsonr as _pearsonr2
try:
    from statsmodels.stats.power import TTestPower as _TTestPower2
    _has_sm2 = True
except Exception:
    _has_sm2 = False

def _aha_stage(sbp):
    if sbp < 120:   return "Normal"
    if sbp < 130:   return "Elevated"
    if sbp < 140:   return "HTN-1"
    return "HTN-2"

_STAGE_ORDER = ["Normal", "Elevated", "HTN-1", "HTN-2"]

def stratified_drift_recovery(min_beats=11, update_interval=15):
    rb, gap = CALIB.ref_beats, CALIB.ols_gap   # 3, 1 — canonical, same as the selection-bias and drift-length checks
    beat_rows, subj_rows = [], []

    for name, d in _drift_datasets_iter():
        primary = _primary_for(name, d, r2_all) if 'r2_all' in globals() else None
        if primary is None:
            print(f"  [{name}] no 'primary' preds in r2_all — skipped")
            continue
        ybp  = np.asarray(d["Ybp_eval"], dtype=float)
        subj = np.asarray(d["subj_eval"])

        for s in np.unique(subj):
            idx = np.where(subj == s)[0]
            if len(idx) < min_beats:
                continue

            stage = _aha_stage(np.median(ybp[idx, 0]))   # subject's own SBP, descriptive only
            static_bias = np.median(ybp[idx[:rb]] - primary[idx[:rb]], axis=0)
            periodic_bias = static_bias.copy()   # Periodic: refreshed every `update_interval` beats

            st_resid, pe_resid, rl_resid = [], [], []
            for j in range(rb + gap, len(idx)):
                gi = idx[j]
                st_e = abs((primary[gi, 0] + static_bias[0]) - ybp[gi, 0])
                # Periodic re-cal: refresh the offset every `update_interval` beats
                # (identical protocol to test_periodic_calibration / cell 85).
                if j > rb + gap and j % update_interval == 0:
                    recent_ref    = idx[j - gap - rb : j - gap]
                    periodic_bias = np.median(ybp[recent_ref] - primary[recent_ref], axis=0)
                pe_e = abs((primary[gi, 0] + periodic_bias[0]) - ybp[gi, 0])
                roll_ref  = idx[j - gap - rb : j - gap]
                roll_bias = np.median(ybp[roll_ref] - primary[roll_ref], axis=0)
                rl_e = abs((primary[gi, 0] + roll_bias[0]) - ybp[gi, 0])
                st_resid.append(st_e); pe_resid.append(pe_e); rl_resid.append(rl_e)
                beat_rows.append(dict(Dataset=name, Stage=stage, static_err=st_e,
                                      periodic_err=pe_e, rolling_err=rl_e))

            if len(st_resid) >= 3:
                subj_rows.append(dict(Dataset=name, Stage=stage,
                                       static_mae=np.mean(st_resid),
                                       periodic_mae=np.mean(pe_resid),
                                       rolling_mae=np.mean(rl_resid)))

    return pd.DataFrame(beat_rows), pd.DataFrame(subj_rows)


def _paired_recovery_line(sub, label, indent=4):
    """One paired static→rolling test line, with nan-safe power. Returns nothing, just prints."""
    pad = " " * indent
    if len(sub) < 5:
        print(f"{pad}{label:<28} N={len(sub):>3}  [skip: N<5]")
        return
    diff = sub.static_mae.values - sub.rolling_mae.values
    t, p = _ttest_rel2(sub.static_mae.values, sub.rolling_mae.values)
    sd = diff.std(ddof=1)
    dz = diff.mean() / sd if sd > 0 else np.nan

    pw_str = ""
    if _has_sm2 and np.isfinite(dz) and abs(dz) > 1e-6:
        try:
            pw = _TTestPower2().power(effect_size=abs(dz), nobs=len(diff), alpha=0.05, alternative='two-sided')
            if np.isfinite(pw):
                pw_str = f", power={pw:.3f}"
            else:
 # statsmodels/scipy's noncentral-t CDF returns nan erratically
 # here (ncp=29 evaluates fine while ncp=10.8 does not), so this
 # is a numerical quirk, NOT a monotonic "large ncp" failure.
 # Verified against the normal approximation
 # power ≈ Φ(ncp − z_{1−α/2}): every clamped case has
 # ncp ≥ 10.8, i.e. true power ≥ 1 − 7e-19. The clamp is sound.
                pw_str = ", power>0.999 (statsmodels underflow at large ncp, clamped)"
        except Exception as e:
            pw_str = f" [power calc failed: {e}]"

    print(f"{pad}{label:<28} N={len(sub):>3}  ΔMAE={diff.mean():+.2f} mmHg  "
          f"t={t:.2f}, p={p:.3g}, dz={dz:.2f}{pw_str}")


# ── Run ─────────────────────────────────────────────────────────────────────
df_beats, df_subj = stratified_drift_recovery(11)

# (a) Pooled-by-stage
print("=" * 100)
print("  Pooled-by-stage drift recovery (beat-level, all external datasets)")
print("  ⚠️  CONFOUNDED with dataset composition per stage — see (c) for the controlled view.")
print("=" * 100)
rows_a = []
for stage in _STAGE_ORDER:
    sub = df_beats[df_beats.Stage == stage]
    n_subj = df_subj[df_subj.Stage == stage].shape[0]
    if len(sub) == 0:
        rows_a.append(dict(Stage=stage, N_subj=0, N_beats=0, Static_MAE=np.nan,
                            Rolling_MAE=np.nan, Drop_pct=np.nan, Flag="no data"))
        continue
    mae_s = sub.static_err.mean()
    mae_r = sub.rolling_err.mean()
    drop  = (mae_s - mae_r) / mae_s * 100 if mae_s > 0 else np.nan
    flag  = "⚠️ underpowered (N<30)" if n_subj < 30 else "ok"
    rows_a.append(dict(Stage=stage, N_subj=n_subj, N_beats=len(sub),
                        Static_MAE=round(mae_s, 2), Rolling_MAE=round(mae_r, 2),
                        Drop_pct=round(drop, 1), Flag=flag))
df_stage = pd.DataFrame(rows_a)
print(df_stage.to_string(index=False))

print("\n  Paired static→rolling recovery (subject-level, per stage, POOLED across datasets):")
for stage in _STAGE_ORDER:
    sub = df_subj[df_subj.Stage == stage]
    _paired_recovery_line(sub, stage)

# (b) Per-dataset × stage subject counts
print("\n" + "=" * 100)
print("  Per-dataset × stage breakdown (subject counts — shows thin cells)")
print("=" * 100)
pivot_n = df_subj.pivot_table(index="Dataset", columns="Stage", values="static_mae",
                               aggfunc="count", fill_value=0)
pivot_n = pivot_n.reindex(columns=_STAGE_ORDER, fill_value=0)
print(pivot_n.to_string())
print("\n  Cells with N<10 are too thin to interpret directly (esp. expected for BCG).")

# (c) WITHIN-dataset × stage — the controlled view
print("\n" + "=" * 100)
print("  Within-dataset drift recovery by stage (controls for Simpson's paradox)")
print("=" * 100)
rows_c = []
for (dataset, stage), grp in df_beats.groupby(["Dataset", "Stage"]):
    n_subj = df_subj[(df_subj.Dataset == dataset) & (df_subj.Stage == stage)].shape[0]
    if len(grp) == 0:
        continue
    mae_s = grp.static_err.mean()
    mae_p = grp.periodic_err.mean()
    mae_r = grp.rolling_err.mean()
    drop    = (mae_s - mae_r) / mae_s * 100 if mae_s > 0 else np.nan
    drop_p  = (mae_s - mae_p) / mae_s * 100 if mae_s > 0 else np.nan
    flag  = "⚠️ N<10" if n_subj < 10 else ("~ N<30 (marginal)" if n_subj < 30 else "ok")
    rows_c.append(dict(Dataset=dataset, Stage=stage, N_subj=n_subj, N_beats=len(grp),
                        Static_MAE=round(mae_s, 2), Periodic_MAE=round(mae_p, 2),
                        Rolling_MAE=round(mae_r, 2),
                        Periodic_Drop_pct=round(drop_p, 1),
                        Drop_pct=round(drop, 1), Flag=flag))

df_within = pd.DataFrame(rows_c)
df_within["Stage"] = pd.Categorical(df_within["Stage"], categories=_STAGE_ORDER, ordered=True)
df_within = df_within.sort_values(["Dataset", "Stage"]).reset_index(drop=True)
print(df_within.to_string(index=False))

print("\n  Paired static→rolling recovery, WITHIN each dataset (controls for dataset confound):")
for dataset in sorted(df_within["Dataset"].unique()):
    for stage in _STAGE_ORDER:
        sub = df_subj[(df_subj.Dataset == dataset) & (df_subj.Stage == stage)]
        _paired_recovery_line(sub, f"{dataset} / {stage}")

# ── Save ────────────────────────────────────────────────────────────────────
import os as _os
_out_dir = "outputs/phase_2_v192"
_os.makedirs(_out_dir, exist_ok=True)
df_stage.to_csv(f"{_out_dir}/reviewer_R2c_stratified_recovery_pooled.csv", index=False)
df_subj.to_csv(f"{_out_dir}/reviewer_R2c_stratified_recovery_subjects.csv", index=False)
pivot_n.to_csv(f"{_out_dir}/reviewer_R2c_stage_dataset_counts.csv")
df_within.to_csv(f"{_out_dir}/reviewer_R2c_within_dataset_stage.csv", index=False)
print(f"\n  ✅ saved → {_out_dir}/reviewer_R2c_stratified_recovery_pooled.csv")
print(f"  ✅ saved → {_out_dir}/reviewer_R2c_stratified_recovery_subjects.csv")
print(f"  ✅ saved → {_out_dir}/reviewer_R2c_stage_dataset_counts.csv")
print(f"  ✅ saved → {_out_dir}/reviewer_R2c_within_dataset_stage.csv")

print("\n  Interpretation guide:")
print("   • Compare (a) pooled-by-stage vs (c) within-dataset-by-stage.")
print("   • If the stage trend in (a) SURVIVES within each dataset in (c) →")
print("     genuine BP-severity effect, not a dataset artifact. Report as-is.")
print("   • If the trend in (a) DISAPPEARS or reverses within datasets in (c) →")
print("     (a) was confounded by dataset composition (Simpson's paradox).")
print("     Report recovery as dataset-dependent, NOT stage-dependent, and")
print("     retract any 'recovery increases with BP severity' claim.")

In [ ]:
# the drift-vs-length analysis : Within-session drift vs session length + power analysis
# ------------------------------------------------------------------------------
# Per subject (≥11 beats) we fit the canonical t=0 static bias (rb=CALIB.ref_beats
# median, anti-leak gap) and track how the |static-calibrated SBP error| grows
# over the remaining beats — the "drift slope" (mmHg/beat). Then:
# (a) correlate drift magnitude with session length (Pearson r);
# (b) power-analyse the static→rolling recovery effect (paired, per subject).
# Caveat: session length is beat-capped at loading (beat_cap≈20), so the
# session-length range is narrow (~11–21 beats) — reported alongside r.
# Figure follows the paper house style (IEEE single column, Times New Roman 8pt,
# dpi 400) and is written to FIG_DIR so \graphicspath picks it up.
# ══════════════════════════════════════════════════════════════════════════════
import os
from scipy.stats import linregress, pearsonr as _pearsonr, ttest_rel as _ttest_rel
try:
    from statsmodels.stats.power import TTestPower
    _has_sm = True
except Exception:
    _has_sm = False


def _primary_for(name, d, R):
    """Return the model's raw 'primary' prediction array aligned to d, else None."""
    if isinstance(R, dict) and not any(k in R for k in ("uci", "PulseDB", d.get("name", ""))):
        R = next(iter(R.values()))            # unwrap {model: {dataset: ...}}
    for key in (name, d.get("name"), d.get("tag"), "PulseDB"):
        if key and key in R and isinstance(R[key], dict) and "primary" in R[key]:
            return np.asarray(R[key]["primary"], dtype=float)
    return None


def analyze_drift_vs_session(min_beats=11):
    rb  = CALIB.ref_beats     # 3
    gap = CALIB.ols_gap       # 1
    sess_len, drift_slope, drift_delta = [], [], []
    st_mae_l, rl_mae_l, ds_used = [], [], []
    for name, d in _drift_datasets_iter():
        primary = _primary_for(name, d, r2_all) if 'r2_all' in globals() else None
        if primary is None:
            print(f"  [{name}] no 'primary' preds in r2_all — skipped")
            continue
        ybp  = np.asarray(d["Ybp_eval"], dtype=float)
        subj = np.asarray(d["subj_eval"])
        n_before = len(sess_len)
        for s in np.unique(subj):
            idx = np.where(subj == s)[0]
            if len(idx) < min_beats:
                continue
            static_bias = np.median(ybp[idx[:rb]] - primary[idx[:rb]], axis=0)
            beats, st_resid, roll_resid = [], [], []
            for j in range(rb + gap, len(idx)):
                gi = idx[j]
                st_resid.append(abs((primary[gi, 0] + static_bias[0]) - ybp[gi, 0]))
                roll_ref  = idx[j - gap - rb:j - gap]
                roll_bias = np.median(ybp[roll_ref] - primary[roll_ref], axis=0)
                roll_resid.append(abs((primary[gi, 0] + roll_bias[0]) - ybp[gi, 0]))
                beats.append(j)
            if len(beats) < 3:
                continue
            sess_len.append(len(idx))
            drift_slope.append(linregress(beats, st_resid).slope)
            half = len(st_resid) // 2
            drift_delta.append(np.mean(st_resid[half:]) - np.mean(st_resid[:half]))
            st_mae_l.append(np.mean(st_resid))
            rl_mae_l.append(np.mean(roll_resid))
        if len(sess_len) > n_before:                 # only count datasets that contributed
            ds_used.append(f"{name} (n={len(sess_len) - n_before})")
    return (np.array(sess_len), np.array(drift_slope), np.array(drift_delta),
            np.array(st_mae_l), np.array(rl_mae_l), ds_used)


sl, dsl, dd, st_mae, rl_mae, ds_used = analyze_drift_vs_session(11)
print("=" * 78)
print("  Within-session drift vs session length + power analysis")
print("=" * 78)
print(f"  Datasets contributing: {', '.join(ds_used) or '(none — need r2_all in memory)'}")
print(f"  N subjects (≥11 beats): {len(sl)}")
if len(sl) >= 5:
    r, p = _pearsonr(sl, dsl)
    print(f"\n  Static drift slope : {dsl.mean():+.3f} ± {dsl.std():.3f} mmHg/beat")
    print(f"  Session length     : {sl.mean():.1f} ± {sl.std():.1f} beats "
          f"(range {sl.min()}–{sl.max()}; beat-capped at load)")
    print(f"  Corr(session length, drift slope): r={r:+.3f}, p={p:.3g}")
    diff = st_mae - rl_mae               # >0 ⇒ rolling recovers drift
    t, pp = _ttest_rel(st_mae, rl_mae)
    sd = diff.std(ddof=1)
    dz = diff.mean() / sd if sd > 0 else np.nan
    print(f"\n  Static→Rolling recovery (paired, N={len(diff)}):")
    print(f"    ΔMAE (static−rolling) = {diff.mean():+.3f} mmHg | paired t={t:.2f}, p={pp:.3g}")
    print(f"    Effect size Cohen's dz = {dz:.3f}")
    if _has_sm and np.isfinite(dz) and abs(dz) > 1e-6:
        try:
            pw  = TTestPower().power(effect_size=abs(dz), nobs=len(diff),
                                     alpha=0.05, alternative='two-sided')
            n80 = TTestPower().solve_power(effect_size=abs(dz), power=0.80,
                                           alpha=0.05, alternative='two-sided')
            print(f"    Achieved power @N={len(diff)}, α=0.05: {pw:.3f}")
            print(f"    N needed for 80% power: {int(np.ceil(n80))}")
        except Exception as e:
            print(f"    [power calc skipped: {e}]")

# ── Figure: paper house style (IEEE single column, TNR 8pt, dpi 400) ──────
    _COL1 = globals().get("_COL1_IN", 3.5)
    _FS_  = globals().get("_FS", 8)
    
    fig, axes = plt.subplots(1, 2, figsize=(_COL1, 2.55))

    ax = axes[0]
    ax.scatter(sl, dsl, s=6, alpha=0.25, color="#1f4e79", edgecolor="none",
               rasterized=True)
    b  = linregress(sl, dsl)
    xs = np.array([sl.min(), sl.max()], dtype=float)
    ax.plot(xs, b.intercept + b.slope * xs, color="#c00000", lw=1.2,
            label=f"$r={r:+.2f}$")
    ax.axhline(0, color="grey", lw=0.6, ls=":")
    ax.set_xlabel("Session length (beats)", fontsize=_FS_)
    
    ax.set_ylabel("Static drift slope (mmHg/beat)", fontsize=_FS_)
    
    ax.set_title("(a) Drift vs Session Length", fontsize=_FS_, fontweight="bold")
    ax.legend(fontsize=_FS_ - 1, loc="upper right", framealpha=0.95, handlelength=1.2, handletextpad=0.4)
    ax.tick_params(labelsize=_FS_)

    ax = axes[1]
    ax.scatter(st_mae, rl_mae, s=6, alpha=0.25, color="#1f4e79", edgecolor="none",
               rasterized=True)
    lim = [0.0, float(max(st_mae.max(), rl_mae.max()) * 1.05)]
    ax.plot(lim, lim, color="grey", lw=0.8, ls="--", label="no recovery")
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("Static SBP MAE (mmHg)", fontsize=_FS_)
    ax.set_ylabel("Rolling SBP MAE (mmHg)", fontsize=_FS_)
    ax.set_title(f"(b) Drift Recovery ($d_z={dz:.2f}$)", fontsize=_FS_, fontweight="bold")
    ax.legend(fontsize=_FS_ - 1, loc="upper left", framealpha=0.95, handlelength=1.2, handletextpad=0.4)
    ax.tick_params(labelsize=_FS_)

    for artist in fig.findobj(plt.Text):          # force house font on every label
        try:
            artist.set_fontsize(_FS_)
            artist.set_fontname("Times New Roman")
        except Exception:
            pass

    fig.tight_layout(pad=0.3, w_pad=1.2)
    
    save_dir = globals().get("FIG_DIR", "outputs/phase_2_v192/figures")
    os.makedirs(save_dir, exist_ok=True)
    out_path = os.path.join(save_dir, "drift_vs_session.png")
    fig.savefig(out_path, dpi=400, bbox_inches="tight", pad_inches=0.05)
    plt.show()
    print(f"\n  ✅ saved figure → {out_path}  (IEEE 1-col, TNR {_FS_}pt, dpi 400)")
    
    _df = pd.DataFrame(dict(session_len=sl, drift_slope=dsl, drift_delta=dd,
                            static_mae=st_mae, rolling_mae=rl_mae))
    _csv = "outputs/phase_2_v192/reviewer_R3_drift_per_subject.csv"
    _df.to_csv(_csv, index=False)
    print(f"  ✅ saved table  → {_csv}")
else:
    print("  [SKIP] too few subjects (need r2_all with 'primary' preds in memory).")


### Pre-Registered Drift Falsification Test

The one confirmatory test: primary endpoint c_slow, co-primary R, with the cutoff locked at three beats before any numbers were seen. It can refute the drift explanation.

In [ ]:
# Pre-registered falsification test of the "within-session drift" claim
# ------------------------------------------------------------------------------
# CLAIM UNDER TEST (paper Section 5): on Multi-Sensor PPG every Windkessel quantity
# loses its correlation with SBP (|r| < 0.01, the HTN-stage analysis). The paper currently
# explains this as a *within-session drift artifact*: the session is long enough
# that the subject-mean of a physics parameter is averaged across a drifting
# baseline, destroying the between-subject signal. The rival explanation is a
# plain *representation failure* on this sensor — in which case the sentence in
# the paper is wrong and must be retracted.
#
# PREDICTION THAT SEPARATES THEM: if it is drift, the correlation must be
# RECOVERABLE from the earliest beats of each session (before drift accumulates).
# If it is representation failure, the early beats are just as uncorrelated.
#
# ── PRE-REGISTRATION (locked before the numbers were seen) ────────────────────
# Primary endpoint ....... c_slow (the quantity the paper claims transfers)
# Co-primary ............. R (physics-anchored reference, paper Section 5)
# Secondary (reported, not decisive) ... C, tau = R*C
# Cutoff ................. EXACTLY 3 beats == CALIB.ref_beats. One value.
# No window sweep will be run or reported (no p-hacking).
# Cohort ................. subjects with >= 3 beats, IDENTICAL in both conditions,
# so the ONLY thing that varies is how many beats are
# averaged — not how many subjects enter the CI.
# Estimator .............. identical to per-subject MEAN of the parameter,
# per-subject MEAN of SBP, Pearson r, Fisher-z 95% CI.
# Decision rule .......... per parameter, on Multi-Sensor:
# CONFIRMED <=> early-window 95% CI EXCLUDES 0 AND full-session CI INCLUDES 0
# REFUTED <=> early-window CI still INCLUDES 0, WHILE >=1 control dataset
# does exclude 0 under the same 3-beat handicap (which proves
# 3 beats is enough to see the effect where it exists)
# INCONCLUSIVE <=> no control survives the 3-beat handicap => window too short,
# the test has no power and decides nothing.
#
# ── CIRCULARITY CHECK (verified in code before running) ───────────────────────
# R, C, tau, c_slow are emitted by the model's Windkessel head
# tau = softplus(wk_raw[:,0]) + 0.5 ; k = sigmoid(wk_raw[:,1])*2 + 0.1
# R = (tau*k).clamp(...) ; C = _compute_C(tau, k)
# They never touch the BP-calibration path. The correlation below is between a
# raw network output and the GROUND-TRUTH SBP label — no predicted BP enters it —
# so the SCal subject-bias correction cannot leak into this result.
# Beats 1-3 are also the SCal reference window (init_ref = idx[0:CALIB.ref_beats]).
# That is stated for transparency, but it contaminates nothing here for the reason
# above. Beat ORDER = array order: the same assumption SCal itself already makes.
#
# ── DIRECTION OF BIAS ─────────────────────────────────────────────────────────
# 3 beats is a noisier estimate of a subject's parameter than ~N beats, so
# attenuation pushes the early-window r TOWARD zero. The test is therefore
# CONSERVATIVE against the drift hypothesis: a positive result is strong, and a
# null is only interpretable via the control datasets (hence the rule above).
#
# ── CAVEAT TO CARRY INTO THE PAPER REGARDLESS OF OUTCOME ──────────────────────
# "Early beats" is not a pure drift probe: sensor settling and placement motion
# also live at the start of a session. A positive result is "consistent with"
# the drift account, never "proof" of it.
# ══════════════════════════════════════════════════════════════════════════════
from scipy.stats import pearsonr

EARLY_BEATS = int(CALIB.ref_beats)          # == 3, locked. Do not tune.
PARAMS      = ["c_slow", "R", "C", "tau"]   # c_slow primary, R co-primary
PRIMARY     = ["c_slow", "R"]


def _subject_corr(fwd, Ybp, subj, n_beats=None, min_beats=EARLY_BEATS):
    """Per-subject mean of each Windkessel param vs per-subject mean SBP.

    n_beats=None -> average over ALL beats (as in the paper).
    n_beats=k    -> average over the FIRST k beats only.
    Cohort is fixed to subjects with >= min_beats in BOTH cases.
    """
    subjs = [s for s in np.unique(subj) if (subj == s).sum() >= min_beats]
    out = {}
    R_, C_ = fwd["R"], fwd["C"]
    src = {"R": R_, "C": C_, "tau": R_ * C_, "c_slow": fwd["c_slow"]}
    sbp = []
    for p in PARAMS:
        out[p] = []
    for s in subjs:
        idx = np.where(subj == s)[0]
        use = idx if n_beats is None else idx[:n_beats]
        for p in PARAMS:
            out[p].append(float(np.mean(src[p][use])))
        sbp.append(float(np.mean(Ybp[use, 0])))
    sbp = np.asarray(sbp)
    res = {}
    for p in PARAMS:
        v = np.asarray(out[p])
        n = len(v)
        if n < 4 or np.std(v) == 0:
            res[p] = dict(r=np.nan, lo=np.nan, hi=np.nan, p=np.nan, n=n, excl0=False)
            continue
        r, pv = pearsonr(v, sbp)
        lo, hi = fisher_ci(r, n)
        res[p] = dict(r=float(r), lo=float(lo), hi=float(hi), p=float(pv), n=n,
                      excl0=bool(lo > 0 or hi < 0))
    return res, len(subjs)


def run_exp_r4_drift_falsification(ext_dict):
    print("=" * 100)
    print(f"  Pre-registered falsification test  |  cutoff = {EARLY_BEATS} beats "
          f"(= CALIB.ref_beats, locked)")
    print(f"  Primary endpoint: c_slow    Co-primary: R    Secondary: C, tau")
    print("=" * 100)

    # ── Assemble UCI TEST (in-domain control) + all externals ────────────────
    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    domains = [("UCI TEST (control)",
                _infer_forward(model, X_te, Q_te, subj_te, base_te), Ybp_te, subj_te)]
    for name, tag, d in _iter_ext_datasets(ext_dict):
        fwd = _infer_forward(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])
        domains.append((name, fwd, np.asarray(d["Ybp_eval"], float), np.asarray(d["subj_eval"])))

 # ══ GATE 1 — power/adequacy diagnostics, PRINTED BEFORE ANY CORRELATION ══
    print("\n  ── GATE: cohort size and beat counts (read this BEFORE any r below) ──")
    print(f"  {'Dataset':<22} {'N_all':>6} {'N(>=3bt)':>9} {'beats/subj  min / med / max':>30}")
    print("  " + "-" * 74)
    gate = {}
    for name, fwd, Ybp, subj in domains:
        cnt = np.array([(subj == s).sum() for s in np.unique(subj)])
        keep = cnt[cnt >= EARLY_BEATS]
        gate[name] = dict(n_all=len(cnt), n_keep=len(keep),
                          med=float(np.median(keep)) if len(keep) else 0.0)
        med = f"{keep.min():>3d} / {np.median(keep):>5.1f} / {keep.max():>4d}" if len(keep) else "—"
        print(f"  {name:<22} {len(cnt):>6d} {len(keep):>9d} {med:>30}")
    print("\n  ⚠️  If a domain's MEDIAN beats/subject is close to "
          f"{EARLY_BEATS}, the two conditions average nearly the same beats and the")
    print("     contrast is toothless there — that is a property of the data, not a result.")

 # ══ GATE 2 — the two conditions, same cohort, only the window changes ════
    rows = []
    verdict_src = {}
    for name, fwd, Ybp, subj in domains:
        full, n_f = _subject_corr(fwd, Ybp, subj, n_beats=None)
        early, n_e = _subject_corr(fwd, Ybp, subj, n_beats=EARLY_BEATS)
        assert n_f == n_e, "cohort must be identical across conditions"
        verdict_src[name] = dict(full=full, early=early, n=n_f)
        for p in PARAMS:
            for cond, res in (("full-session", full), (f"first-{EARLY_BEATS}", early)):
                rows.append(dict(Dataset=name, Param=p, Condition=cond, N=res[p]["n"],
                                 r=res[p]["r"], ci_lo=res[p]["lo"], ci_hi=res[p]["hi"],
                                 p_value=res[p]["p"], excludes_zero=res[p]["excl0"]))

    print("\n  ── r (Pearson, subject-level) with 95% Fisher-z CI ──")
    for p in PARAMS:
        star = " ★ PRIMARY" if p == "c_slow" else (" ☆ co-primary" if p == "R" else "")
        print(f"\n  [{p}]{star}")
        print(f"  {'Dataset':<22} {'N':>4} | {'full-session':^26} | {f'first-{EARLY_BEATS} beats':^26}")
        print("  " + "-" * 86)
        for name, _, _, _ in domains:
            v = verdict_src[name]
            f_, e_ = v["full"][p], v["early"][p]
            def fmt(x):
                if not np.isfinite(x["r"]):
                    return f"{'n/a':^26}"
                mark = "*" if x["excl0"] else " "
                return f"{x['r']:+.3f} [{x['lo']:+.3f},{x['hi']:+.3f}]{mark}"
            print(f"  {name:<22} {v['n']:>4d} | {fmt(f_):^26} | {fmt(e_):^26}")
    print("\n  (* = 95% CI excludes zero)")

 # ══ GATE 3 — apply the LOCKED decision rule ═════════════════════════════
    ms = next((n for n, _, _, _ in domains if "sensor" in n.lower()), None)
    print("\n" + "=" * 100)
    print("  LOCKED DECISION RULE — applied mechanically, no post-hoc reinterpretation")
    print("=" * 100)
    if ms is None:
        print("  ⚠️  Multi-Sensor not loaded — cannot decide.")
        verdicts = {}
    else:
        verdicts = {}
        for p in PRIMARY:
            f_, e_ = verdict_src[ms]["full"][p], verdict_src[ms]["early"][p]
 # controls = any OTHER domain where the effect exists (full) AND survives 3 beats
            adequate = [n for n, _, _, _ in domains if n != ms
                        and verdict_src[n]["full"][p]["excl0"]
                        and verdict_src[n]["early"][p]["excl0"]]
            if e_["excl0"] and not f_["excl0"]:
                v = "✅ CONFIRMED"
                why = (f"early-window CI [{e_['lo']:+.3f},{e_['hi']:+.3f}] excludes 0 while "
                       f"full-session CI [{f_['lo']:+.3f},{f_['hi']:+.3f}] includes it "
                       f"→ consistent with within-session drift (NOT proof: see settling caveat)")
            elif not e_["excl0"] and adequate:
                v = "🚨 REFUTED"
                why = (f"early-window CI [{e_['lo']:+.3f},{e_['hi']:+.3f}] still includes 0, yet "
                       f"{', '.join(adequate)} DO show the effect under the same {EARLY_BEATS}-beat "
                       f"handicap → the window is adequate, so the null is real. "
                       f"The 'within-session drift artifact' explanation must be RETRACTED "
                       f"and replaced with an honest 'unexplained / representation failure'.")
            elif not e_["excl0"] and not adequate:
                v = "⚪ INCONCLUSIVE"
                why = (f"no control dataset survives the {EARLY_BEATS}-beat handicap either "
                       f"→ the window is too short to detect anything; the test has no power "
                       f"and decides nothing in either direction.")
            else:  # e_ excl0 AND f_ excl0
                v = "⚠️  PREMISE VIOLATED"
                why = (f"full-session CI [{f_['lo']:+.3f},{f_['hi']:+.3f}] already excludes 0 — "
                       f"the |r|<0.01 anomaly this test was built to explain does not reproduce "
                       f"for this parameter. Re-check against the HTN-stage analysis before interpreting.")
            verdicts[p] = v
            tag = "PRIMARY" if p == "c_slow" else "co-primary"
            print(f"\n  [{p}] ({tag}) → {v}")
            print(f"      {why}")

        print("\n  ── HEADLINE (primary endpoint c_slow governs) ──")
        print(f"      {verdicts.get('c_slow', 'n/a')}")
        if verdicts.get("c_slow") != verdicts.get("R"):
            print(f"      NOTE: R disagrees ({verdicts.get('R')}). Report BOTH. Do not quietly")
            print( "      promote whichever agrees with the paper — that is the move this")
            print( "      pre-registration exists to prevent.")

    df_r4 = pd.DataFrame(rows)
    _out = os.path.join(EXP_OUT_DIR, "reviewer_R4_drift_falsification.csv")
    df_r4.to_csv(_out, index=False)
    print(f"\n  ✅ saved → {_out}")
    return df_r4, verdicts


df_r4, r4_verdicts = run_exp_r4_drift_falsification(ext_dict)


### Variogram: Noise versus Drift

A post-hoc analysis explaining the BCG counter-example to the falsification test.

In [ ]:
# the variogram analysis — VARIOGRAM: is the within-session variance NOISE or DRIFT?
# ------------------------------------------------------------------------------
# STATUS: POST-HOC / EXPLORATORY. Run AFTER seeing the pre-registered test. Not pre-registered.
# It is reported as such and cannot promote the pre-registered test's verdict; it only explains it.
#
# WHY IT EXISTS: the pre-registered test confirmed the drift account on Multi-Sensor (c_slow r:
# -0.018 -> +0.182 with the first 3 beats), but BCG moved the OTHER way
# (+0.474 -> +0.049). A reviewer will ask whether the 3-beat window is simply too
# noisy to trust — and if so, why we believe the Multi-Sensor number.
#
# THE LOGICAL ASYMMETRY THIS TEST MAKES CONCRETE:
# Pure measurement noise can ONLY attenuate r, and a mean over FEWER beats is
# ALWAYS less reliable than a mean over more. So a noise-only world predicts
# r(first 3) <= r(full session) for EVERY dataset.
# BCG and UCI obey this. PulseDB (+0.24) and Multi-Sensor (+0.20) VIOLATE it.
# Noise cannot manufacture that violation. Something systematic and
# time-progressive must corrupt the later beats. That is what a variogram sees.
#
# THE VARIOGRAM (standard drift diagnostic, no correlation involved):
# V(l) = 1/2 * E[(x_i - x_{i+l})^2] over beat pairs l apart, within subject
# pure i.i.d. noise ......... V(l) = sigma_e^2 -> FLAT
# noise + random-walk drift . V(l) = sigma_e^2 + l * sigma_d^2 -> RISING
# Fit V(l) = a + b*l. a = beat-to-beat noise floor. b = drift rate.
# Normalised by the between-subject variance so datasets are comparable and the
# units of R / c_slow drop out.
#
# PREDICTION (stated before running, so it can fail):
# Multi-Sensor: slope b > 0 (CI excludes 0) -> drift is real
# BCG: slope b ~ 0 (CI includes 0) -> its collapse was pure attenuation
# If Multi-Sensor comes out FLAT, the drift story loses its mechanism and the pre-registered test's
# CONFIRMED becomes unexplained coincidence — say so, do not bury it.
# ══════════════════════════════════════════════════════════════════════════════
N_BOOT_R4B = 1000
MAX_LAG    = 9
VAR_PARAMS = ["c_slow", "R", "SBP"]     # SBP included: does the LABEL drift too?


def _variogram_per_subject(vals, subj, max_lag=MAX_LAG, min_beats=4):
    """V_s(l) for every subject (rows) x lag (cols). NaN where the subject has no
    pair at that lag. Beat order == array order (same assumption SCal makes)."""
    subjs = [s for s in np.unique(subj) if (subj == s).sum() >= min_beats]
    M = np.full((len(subjs), max_lag), np.nan)
    means = np.empty(len(subjs))
    for i, s in enumerate(subjs):
        x = vals[subj == s].astype(float)
        means[i] = x.mean()
        for l in range(1, max_lag + 1):
            if len(x) > l:
                d = x[:-l] - x[l:]
                M[i, l - 1] = 0.5 * np.mean(d ** 2)
    return M, means, len(subjs)


def _fit_slope(V, lags):
    ok = np.isfinite(V)
    if ok.sum() < 3:
        return np.nan, np.nan
    A = np.vstack([np.ones(ok.sum()), lags[ok]]).T
    coef, *_ = np.linalg.lstsq(A, V[ok], rcond=None)
    return float(coef[0]), float(coef[1])          # intercept a, slope b


def run_exp_r4b_variogram(ext_dict):
    print("=" * 104)
    print("  Variogram:  —  is within-session variance NOISE (flat) or DRIFT (rising)?")
    print("  ⚠️  POST-HOC / EXPLORATORY — not pre-registered. Explains the pre-registered test; cannot overturn it.")
    print("=" * 104)

    X_te, Q_te, Ybp_te, subj_te, base_te = get_test_arrays()
    domains = [("UCI TEST (control)",
                _infer_forward(model, X_te, Q_te, subj_te, base_te), Ybp_te, subj_te)]
    for name, tag, d in _iter_ext_datasets(ext_dict):
        fwd = _infer_forward(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])
        domains.append((name, fwd, np.asarray(d["Ybp_eval"], float), np.asarray(d["subj_eval"])))

    lags = np.arange(1, MAX_LAG + 1, dtype=float)
    rows, curves = [], {}
    rng = np.random.default_rng(42)

    for name, fwd, Ybp, subj in domains:
        src = {"R": fwd["R"], "c_slow": fwd["c_slow"], "SBP": Ybp[:, 0]}
        for p in VAR_PARAMS:
            M, means, n = _variogram_per_subject(np.asarray(src[p], float), subj)
            if n < 10:
                print(f"\n  [{name}] {p}: only {n} subjects with >=4 beats — skipped")
                continue
            var_between = float(np.var(means, ddof=1))
            if var_between <= 0:
                continue
            V = np.nanmean(M, axis=0) / var_between          # normalised variogram
            a, b = _fit_slope(V, lags)

 # subject-block bootstrap on the slope (matches the paper's CI standard)
            bs = np.empty(N_BOOT_R4B)
            for k in range(N_BOOT_R4B):
                idx = rng.integers(0, n, n)
                Vb = np.nanmean(M[idx], axis=0) / max(float(np.var(means[idx], ddof=1)), 1e-12)
                bs[k] = _fit_slope(Vb, lags)[1]
            lo, hi = np.nanpercentile(bs, [2.5, 97.5])
            rising = bool(lo > 0)

            curves[(name, p)] = V
            rows.append(dict(Dataset=name, Param=p, N=n, noise_floor_a=a,
                             drift_slope_b=b, b_ci_lo=lo, b_ci_hi=hi, rising=rising))

    df = pd.DataFrame(rows)

    print(f"\n  Normalised variogram  V(l)/var_between  (l = beat lag)  — first {MAX_LAG} lags")
    for p in VAR_PARAMS:
        print(f"\n  [{p}]")
        hdr = "  " + f"{'Dataset':<22}" + "".join(f"{int(l):>7}" for l in lags)
        print(hdr); print("  " + "-" * (len(hdr) - 2))
        for name, _, _, _ in domains:
            V = curves.get((name, p))
            if V is None:
                continue
            print(f"  {name:<22}" + "".join(
                (f"{v:>7.2f}" if np.isfinite(v) else f"{'—':>7}") for v in V))

    print("\n" + "=" * 104)
    print("  DRIFT SLOPE b  (V(l) = a + b*l, normalised; b>0 with CI excluding 0 == accumulating drift)")
    print("=" * 104)
    print(f"  {'Dataset':<22} {'Param':<8} {'N':>5} {'noise a':>9} {'slope b':>9} "
          f"{'95% CI (subject bootstrap)':>30}  verdict")
    print("  " + "-" * 100)
    for _, r in df.iterrows():
        v = "🔺 DRIFT (rising)" if r["rising"] else "▬  flat (noise only)"
        print(f"  {r['Dataset']:<22} {r['Param']:<8} {r['N']:>5d} {r['noise_floor_a']:>9.3f} "
              f"{r['drift_slope_b']:>9.4f} {f'[{r.b_ci_lo:+.4f}, {r.b_ci_hi:+.4f}]':>30}  {v}")

    # ── the two predictions this test was built to check ────────────────────
    print("\n  ── Checking the predictions stated in the header ──")
    def _get(ds, p):
        m = df[(df.Dataset.str.contains(ds, case=False)) & (df.Param == p)]
        return m.iloc[0] if len(m) else None
    for ds, expect, why in [
        ("sensor", True,  "drift must be REAL here, or the pre-registered test's CONFIRMED verdict has no mechanism"),
        ("BCG",    False, "must be FLAT, or its 0.474->0.049 collapse is not pure attenuation"),
    ]:
        for p in ("c_slow", "R"):
            r = _get(ds, p)
            if r is None:
                continue
            ok = (r["rising"] == expect)
            mark = "✅ as predicted" if ok else "🚨 PREDICTION FAILED"
            print(f"   • {r['Dataset']:<20} {p:<7} rising={str(r['rising']):<5} "
                  f"(expected {expect}) {mark}")
            if not ok:
                print(f"       → {why}")

    print("\n  ── Does the LABEL (ground-truth SBP) drift too? ──")
    for _, r in df[df.Param == "SBP"].iterrows():
        print(f"   • {r['Dataset']:<22} slope b = {r['drift_slope_b']:+.4f} "
              f"[{r.b_ci_lo:+.4f},{r.b_ci_hi:+.4f}]  {'RISING' if r['rising'] else 'flat'}")
    print("     If SBP itself drifts as much as the physics params, the effect is")
    print("     physiological (real BP moving), not a model/sensor artifact. If SBP is")
    print("     flat while R/c_slow rise, the drift is in the MODEL's read-out. Both are")
    print("     publishable, but they are DIFFERENT claims — say which one this is.")

    _out = os.path.join(EXP_OUT_DIR, "reviewer_R4b_variogram.csv")
    df.to_csv(_out, index=False)
    print(f"\n  ✅ saved → {_out}")
    return df


df_r4b = run_exp_r4b_variogram(ext_dict)


In [ ]:
# the pre-registered testc — What actually breaks the Multi-Sensor correlation?
# Paste into phase2.ipynb as one cell. Needs: model, ext_dict, get_test_arrays,
# _infer_forward, _iter_ext_datasets, fisher_ci, EXP_OUT_DIR, CALIB.
# ------------------------------------------------------------------------------
# STATUS: POST-HOC / EXPLORATORY (run after the pre-registered test + R4b). Reported as such.
# It cannot overturn the pre-registered test's pre-registered verdict; it only explains it — or
# fails to, in which case we say the mechanism is unknown.
#
# WHERE WE ARE:
# the pre-registered test : Multi-Sensor c_slow r = -0.018 (full session) -> +0.182 (first 3
# beats). Pre-registered CONFIRMED. BUT BCG went the other way
# (+0.474 -> +0.049), which we could not explain.
# the variogram analysis : variograms rise for c_slow / R / SBP on ALL externals. The BCG
# slope is NOT trustworthy: at lag l only subjects with >l beats
# contribute, and BCG's median is 8, so lags 5-9 are computed on a
# different (minority) subset — its curve rises then FALLS
# (5.34 -> 4.59), which a random walk cannot do. Section 3 below
# refits it on a FIXED cohort so every lag uses the same subjects.
#
# A STORY THAT SOUNDS RIGHT BUT IS ALREADY DEAD (section 2 kills it explicitly):
# "Ground-truth SBP is so unstable within a session that the subject-mean is
# too noisy to correlate with anything." Arithmetic refutes this: with
# ICC~0.25 and ~18 beats, the reliability of the subject-mean is ~0.86, so
# the attenuation ceiling on r is ~0.93, nowhere near the observed 0.
# Worse, the 3-beat mean is LESS reliable (~0.5) yet correlates BETTER.
# Noise/attenuation predicts the opposite ordering. It is not a noise story.
# Section 2 computes these numbers instead of assuming them, so the refutation
# is on the record rather than in a comment.
#
# THE SURVIVING HYPOTHESIS (section 4 is its decisive test):
# c_slow is a SLOW, stable, subject-level quantity (ICC 0.9991) — it encodes
# the subject's BASELINE haemodynamic state, i.e. the state at the START of
# the recording. The ground-truth BP, meanwhile, WANDERS during the session
# (rising variogram). The session-MEAN BP is therefore an average over a
# trajectory that has drifted away from that baseline. A stable predictor of
# the baseline cannot track a target that has walked away from it.
#
# PREDICTION (stated before running, so it can fail):
# In the 3x3 matrix r(param-window, SBP-window):
# - r varies strongly ACROSS COLUMNS (which SBP window is the target)
# - r varies weakly DOWN ROWS (which window the param came from)
# i.e. it is the LABEL that moves, not the model's read-out.
# If instead r varies down rows too, the model's read-out drifts as well and
# the "stable predictor" story is wrong — say so.
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np, pandas as pd, os
from scipy.stats import pearsonr

K_EARLY   = int(CALIB.ref_beats)      # 3, same window the pre-registered test locked
N_BOOT    = 1000
_RNG      = np.random.default_rng(42)


# ── domain assembly (identical inference path to the HTN-stage analysis) ─────────────────
def _r4c_domains(ext_dict):
    X, Q, Ybp, subj, base = get_test_arrays()
    out = [("UCI TEST (control)", _infer_forward(model, X, Q, subj, base), Ybp, subj)]
    for name, tag, d in _iter_ext_datasets(ext_dict):
        fwd = _infer_forward(model, d["X_eval"], d["Q_eval"], d["subj_eval"], d["base_eval"])
        out.append((name, fwd, np.asarray(d["Ybp_eval"], float), np.asarray(d["subj_eval"])))
    return out


# ══════════════════════════════════════════════════════════════════════════════
# 1+2. ICC, and the attenuation ceiling it implies (this section KILLS the
# "unstable target" story if the ceiling sits far above the observed r)
# ══════════════════════════════════════════════════════════════════════════════
def _icc_oneway(vals, subj, min_beats=2):
    """ICC(1,1), unbalanced one-way random effects. sigma_b^2 / (sigma_b^2 + MSW)."""
    subjs = [s for s in np.unique(subj) if (subj == s).sum() >= min_beats]
    if len(subjs) < 3:
        return np.nan
    groups = [vals[subj == s].astype(float) for s in subjs]
    n_i    = np.array([len(g) for g in groups], float)
    m_i    = np.array([g.mean() for g in groups])
    N, a   = n_i.sum(), len(groups)
    grand  = np.concatenate(groups).mean()
    MSB    = float((n_i * (m_i - grand) ** 2).sum() / (a - 1))
    ssw    = float(sum(((g - g.mean()) ** 2).sum() for g in groups))
    MSW    = ssw / (N - a)
    if MSW <= 0:
        return np.nan
    k0     = (N - (n_i ** 2).sum() / N) / (a - 1)
    s2b    = (MSB - MSW) / k0
    icc    = s2b / (s2b + MSW)
    return float(np.clip(icc, 0.0, 1.0))


def _sb(icc, k):                       # Spearman-Brown: reliability of a k-beat mean
    return np.nan if not np.isfinite(icc) else k * icc / (1 + (k - 1) * icc)


def section_12(domains):
    print("=" * 108)
    print("  1+2 · ICC(1,1) and the ATTENUATION CEILING it implies")
    print("       Ceiling = sqrt(reliability of the k-beat mean). If the ceiling is FAR ABOVE the")
    print("       r we actually observe, then target instability does NOT explain the collapse.")
    print("=" * 108)
    rows = []
    for name, fwd, Ybp, subj in domains:
        cnt = np.array([(subj == s).sum() for s in np.unique(subj)])
        if len(cnt) == 0 or cnt.max() < 2:
            print(f"  [{name}] no multi-beat subjects — skipped"); continue
        k_med = float(np.median(cnt[cnt >= K_EARLY])) if (cnt >= K_EARLY).any() else np.nan
        src = {"SBP": Ybp[:, 0], "DBP": Ybp[:, 1], "c_slow": fwd["c_slow"], "R": fwd["R"]}
        for p, v in src.items():
            icc = _icc_oneway(np.asarray(v, float), subj)
            rows.append(dict(Dataset=name, Var=p, ICC=icc, k_median=k_med,
                             rel_k3=_sb(icc, K_EARLY), rel_kmed=_sb(icc, k_med),
                             ceil_k3=np.sqrt(_sb(icc, K_EARLY)),
                             ceil_kmed=np.sqrt(_sb(icc, k_med))))
    df = pd.DataFrame(rows)

    print(f"\n  {'Dataset':<22} {'Var':<7} {'ICC(1,1)':>9} {'k_med':>6} "
          f"{'rel(3bt)':>9} {'rel(k_med)':>11} {'ceil r (3bt)':>13} {'ceil r (k_med)':>15}")
    print("  " + "-" * 104)
    for _, r in df.iterrows():
        f = lambda x: f"{x:>9.3f}" if np.isfinite(x) else f"{'—':>9}"
        print(f"  {r.Dataset:<22} {r.Var:<7} {f(r.ICC)} {r.k_median:>6.0f} "
              f"{f(r.rel_k3)} {r.rel_kmed:>11.3f} {r.ceil_k3:>13.3f} {r.ceil_kmed:>15.3f}")

    print("\n  ── VERDICT on the 'unstable target' story ──")
    ms = df[(df.Dataset.str.contains("sensor", case=False)) & (df.Var == "SBP")]
    if len(ms):
        r = ms.iloc[0]
        print(f"   Multi-Sensor SBP: ICC={r.ICC:.3f}. A {r.k_median:.0f}-beat mean therefore has")
        print(f"   reliability {r.rel_kmed:.3f}, so ANY predictor could still reach r = {r.ceil_kmed:.3f}.")
        print(f"   Observed full-session r for c_slow was -0.018.")
        if r.ceil_kmed > 0.4:
            print("   => 🔴 target instability does NOT explain the collapse. Story REJECTED.")
            print("      (And the 3-beat mean is LESS reliable — ceiling "
                  f"{r.ceil_k3:.3f} — yet correlates BETTER (+0.182).")
            print("       Attenuation predicts the OPPOSITE ordering. It is not a noise story.)")
        else:
            print("   => the ceiling really is low; target instability may contribute. Investigate.")
    print("\n  ── The one number worth keeping from this section ──")
    for _, r in df[df.Var == "SBP"].iterrows():
        print(f"   • ICC(SBP) {r.Dataset:<22} = {r.ICC:.3f}   "
              f"({'BP is ~a per-subject constant' if r.ICC > 0.8 else 'BP genuinely moves within session'})")
    print("   UCI is the field's standard benchmark. If its ICC is ~0.9+ while real continuous")
    print("   recordings sit far lower, then UCI rewards learning a subject->constant map.")
    print("   That is a claim about the BENCHMARK, and it is worth stating carefully — but on")
    print("   its own it does NOT explain Multi-Sensor (see verdict above). Do not conflate them.")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# 3. BALANCED variogram — every lag on the SAME subjects (fixes the R4b defect)
# ══════════════════════════════════════════════════════════════════════════════
def section_3(domains, max_lag=6):
    print("\n" + "=" * 108)
    print(f"  3 · BALANCED VARIOGRAM — cohort fixed to subjects with >= {max_lag + 1} beats,")
    print( "      so every lag is computed on the SAME subjects. Fixes the variogram defect where")
    print( "      BCG's curve rose then FELL (a composition artifact, not a physical process).")
    print("=" * 108)
    lags = np.arange(1, max_lag + 1, dtype=float)
    rows = []
    for name, fwd, Ybp, subj in domains:
        src = {"c_slow": fwd["c_slow"], "R": fwd["R"], "SBP": Ybp[:, 0]}
        keep = [s for s in np.unique(subj) if (subj == s).sum() >= max_lag + 1]
        if len(keep) < 30:
            print(f"  [{name}] only {len(keep)} subjects with >= {max_lag+1} beats — skipped")
            continue
        for p, v in src.items():
            v = np.asarray(v, float)
            M = np.empty((len(keep), max_lag))
            means = np.empty(len(keep))
            for i, s in enumerate(keep):
                x = v[subj == s][: max_lag + 1]        # same beats for every lag
                means[i] = v[subj == s].mean()
                for l in range(1, max_lag + 1):
                    d = x[:-l] - x[l:]
                    M[i, l - 1] = 0.5 * np.mean(d ** 2)
            vb = float(np.var(means, ddof=1))
            V  = M.mean(axis=0) / vb
            A  = np.vstack([np.ones(max_lag), lags]).T
            b  = float(np.linalg.lstsq(A, V, rcond=None)[0][1])
            bs = np.empty(N_BOOT)
            for k in range(N_BOOT):
                ix = _RNG.integers(0, len(keep), len(keep))
                Vb = M[ix].mean(axis=0) / max(float(np.var(means[ix], ddof=1)), 1e-12)
                bs[k] = float(np.linalg.lstsq(A, Vb, rcond=None)[0][1])
            lo, hi = np.percentile(bs, [2.5, 97.5])
            rows.append(dict(Dataset=name, Param=p, N=len(keep), slope_b=b,
                             ci_lo=lo, ci_hi=hi, rising=bool(lo > 0),
                             curve=" ".join(f"{x:.2f}" for x in V)))
    df = pd.DataFrame(rows)
    print(f"\n  {'Dataset':<22} {'Param':<8} {'N':>5} {'slope b':>9} {'95% CI':>22}  "
          f"{'V(1..' + str(max_lag) + ')':<32} verdict")
    print("  " + "-" * 104)
    for _, r in df.iterrows():
        v = "🔺 DRIFT" if r.rising else "▬  flat"
        print(f"  {r.Dataset:<22} {r.Param:<8} {r.N:>5d} {r.slope_b:>9.4f} "
              f"{f'[{r.ci_lo:+.4f},{r.ci_hi:+.4f}]':>22}  {r.curve:<32} {v}")
    print("\n  Monotone rise on a FIXED cohort = real accumulating drift.")
    print("  If BCG is still rising here, then the variogram's 'pure attenuation' rescue of BCG is")
    print("  DEAD and BCG's 0.474 -> 0.049 collapse remains genuinely unexplained. Report it.")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# 4. THE DECISIVE TEST — 3x3: does the MODEL drift, or does the LABEL drift?
# ══════════════════════════════════════════════════════════════════════════════
def section_4(domains):
    print("\n" + "=" * 108)
    print("  4 · r( param-window , SBP-window )  —  which side is moving?")
    print("      Prediction: r varies ACROSS COLUMNS (SBP window) but NOT DOWN ROWS (param window)")
    print("                  => the LABEL wanders; the model's read-out is stable.")
    print("      If r ALSO varies down rows, the model's read-out drifts too and the")
    print("      'c_slow = stable baseline state' story is WRONG.")
    print("=" * 108)
    wins = ["first3", "last3", "full"]

    def _agg(v, idx, w):
        if w == "first3":  use = idx[:K_EARLY]
        elif w == "last3": use = idx[-K_EARLY:]
        else:              use = idx
        return float(np.mean(v[use]))

    rows = []
    for name, fwd, Ybp, subj in domains:
        keep = [s for s in np.unique(subj) if (subj == s).sum() >= 2 * K_EARLY]  # need disjoint ends
        if len(keep) < 30:
            print(f"\n  [{name}] only {len(keep)} subjects with >= {2*K_EARLY} beats — skipped")
            continue
        idxs = {s: np.where(subj == s)[0] for s in keep}
        for p in ("c_slow", "R"):
            v = np.asarray(fwd[p], float)
            P = {w: np.array([_agg(v, idxs[s], w) for s in keep]) for w in wins}
            Y = {w: np.array([_agg(Ybp[:, 0], idxs[s], w) for s in keep]) for w in wins}
            print(f"\n  [{name}] {p}   (N = {len(keep)} subjects with >= {2*K_EARLY} beats)")
            print(f"    {'param \\ SBP':<14}" + "".join(f"{'SBP ' + w:>22}" for w in wins))
            print("    " + "-" * 80)
            for wp in wins:
                line = f"    {wp:<14}"
                for wy in wins:
                    r, _ = pearsonr(P[wp], Y[wy])
                    lo, hi = fisher_ci(r, len(keep))
                    mark = "*" if (lo > 0 or hi < 0) else " "
                    line += f"{f'{r:+.3f} [{lo:+.2f},{hi:+.2f}]{mark}':>22}"
                    rows.append(dict(Dataset=name, Param=p, param_win=wp, sbp_win=wy,
                                     N=len(keep), r=r, ci_lo=lo, ci_hi=hi))
                print(line)
 # how much does r move along each axis? (range of r, averaged over the other axis)
            M = np.array([[pearsonr(P[wp], Y[wy])[0] for wy in wins] for wp in wins])
            col_spread = float(np.mean(M.max(axis=1) - M.min(axis=1)))   # vary SBP window
            row_spread = float(np.mean(M.max(axis=0) - M.min(axis=0)))   # vary param window
            print(f"    → mean spread when changing the SBP window   (across columns): {col_spread:.3f}")
            print(f"    → mean spread when changing the param window (down rows)     : {row_spread:.3f}")
            if col_spread > 2 * row_spread:
                print("      ✅ LABEL-dominated: the ground-truth BP is what wanders.")
            elif row_spread > 2 * col_spread:
                print("      🔴 MODEL-dominated: the read-out itself drifts — 'stable c_slow' story is WRONG.")
            else:
                print("      ⚪ BOTH move comparably — no clean attribution. Say exactly this; do not pick.")
    df = pd.DataFrame(rows)
    print("\n  (* = 95% Fisher-z CI excludes zero)")
    print("  NOTE: cohort here needs >= 6 beats so first-3 and last-3 are DISJOINT. That makes it")
    print("        a different (smaller) cohort than the pre-registered test — compare patterns, not absolute r.")
    return df


# ══════════════════════════════════════════════════════════════════════════════
if __name__ != "__main__":
    pass

_D = _r4c_domains(ext_dict)
df_icc  = section_12(_D)
df_vgb  = section_3(_D)
df_grid = section_4(_D)

for _df, _n in ((df_icc, "R4c_icc"), (df_vgb, "R4c_variogram_balanced"), (df_grid, "R4c_window_grid")):
    _p = os.path.join(EXP_OUT_DIR, f"reviewer_{_n}.csv")
    _df.to_csv(_p, index=False)
    print(f"  ✅ saved → {_p}")
